# Stage 01 · Notebook 3 - Visual analysis of the pooled iGEM survey

This notebook renders and tests; it computes no themes of its own. Every theme, label, sentiment score and award join is read from the artifacts Notebook 2 wrote, so a figure here can always be traced back to the stage that produced its numbers.

The thematic structure has **two levels**. The **sub level** is the analytical unit: every significance test, prevalence and model in this notebook is computed there. The **main level** is a coarser grouping used for colour, narrative and orientation, and carries no significance claim - its bootstrap stability sits in the *pattern* band rather than the *finding* band (Hennig 2007).

Density-based clustering does not place every response. Responses left in no theme are shown as their own category in descriptive figures, are excluded from every test, and remain in the denominator of every prevalence - so rates sum to less than 100% and the remainder is exactly the unplaced share.

---

## Section 1 - Setup and provenance

Environment, storage, the single control panel, the shared library, and the checks that establish what this notebook is standing on before it draws anything.
---

## Contents

| Stage | Cell | What it does |
|---|---|---|
| | | **Section 1 — Setup, preflight and the visual system** |
| 01 | 1 | Visual environment |
| 02 | 2 | Storage, model cache and persistent artifacts |
| 03 | 3 | Control panel: the single source of truth |
| 04 | 4 | Library: define once, used everywhere |
| 05 | 5 | Compatibility bridge |
| 06 | 6 | Imports |
| 07 | 8 | Preflight: load and verify notebook 2's artifacts |
| 08 | 10 | Participant-level theme matrix |
| 09 | 12 | Stale-artifact guard |
| 10 | 14 | Global visual system |
| 11 | 16 | Reusable figure helpers |
| | | **Section 2 — Is the analysis trustworthy?** |
| 12 | 19 | Measurement validity of the sentiment score |
| 13 | 21 | Theme evidence quality |
| 14 | 23 | What the clustering did not place |
| 15 | 25 | What notebook 2 recovered |
| 16 | 27 | Are the themes just the questionnaire? |
| 17 | 29 | The manual codebook |
| 18 | 31 | Alignment with the manual codebook |
| 19 | 33 | Agreement with the manual codebook, measured on shared responses |
| | | **Section 3 — The themes themselves** |
| 20 | 36 | Theme codebook |
| 21 | 38 | Two-dimensional theme explorer |
| 22 | 40 | Three-dimensional theme explorer |
| 23 | 42 | Thematic provenance |
| | | **Section 4 — Who voices which theme** |
| 24 | 45 | Theme atlas and group claims |
| 25 | 47 | The support gate, made visible |
| 26 | 49 | Who voices which theme: section, region, team size |
| 27 | 51 | Which themes are voiced together |
| | | **Section 5 — France in the corpus** |
| 28 | 54 | France in the corpus |
| 29 | 56 | What french participants talk about |
| 30 | 58 | France in their own words |
| 31 | 60 | Is the french profile different at all? |
| | | **Section 6 — Themes and competition outcome** |
| 32 | 63 | Theme mention and prize status, by theme |
| 33 | 65 | Theme mention and prize status, in percentage points |
| 34 | 67 | Composite and ordered prize tests |
| 35 | 69 | Team thematic diversity and prize status |
| 36 | 71 | Theme pairs and prize status |
| 37 | 73 | Predictive resolution |
| 38 | 75 | Prize secondary check: the same macro profile |
| 39 | 77 | Predictive capstone model of competition outcome |
| | | **Section 7 — Themes and sentiment** |
| 40 | 80 | Does a response's theme predict its sentiment? |
| | | **Section 8 — Teams** |
| 41 | 83 | Team profile table |
| 42 | 85 | Team organization: section, region and track |
| 43 | 87 | Team archetype clustering |
| | | **Section 9 — Methods, inventory and export** |
| 44 | 90 | Methods, limitations and figure map |
| 45 | 92 | Figure inventory |
| 46 | 94 | Export and cleanup |

*46 stages, numbered in the order they run.*

In [ ]:
# @title
# ==============================================================================
# STAGE 01  -  VISUAL ENVIRONMENT
#             section 1 of 9  -  Setup, preflight and the visual system
# Notebook 3 ONLY renders figures from the artifacts Notebooks 1 and 2 saved, so
# it installs the plotting / analysis stack no torch, transformers, embedding model, spaCy or LLM. That
# keeps the notebook you re-run most fast to start. Determinism env vars are set.
# ==============================================================================
import os
os.environ["PYTHONHASHSEED"]        = "42"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import subprocess
def _pip(pkgs):
    subprocess.run(f"pip install -q {pkgs}", shell=True, check=False)

# kaleido pinned 0.2.1: 0.1.0.post1 has no wheel for current Colab Python, so the
# install silently failed and every Plotly static export was skipped.
_pip("numpy pandas scipy scikit-learn umap-learn statsmodels")
_pip("matplotlib seaborn plotly 'kaleido==0.2.1'   colorcet gdown")

from importlib.metadata import version, PackageNotFoundError
print("=" * 64)
print("  STAGE 1 (slim)  VISUAL ENVIRONMENT")
print("=" * 64)
for p in ["numpy", "pandas", "scipy", "scikit-learn", "plotly", "kaleido"]:
    try:    print(f"    {p:16s} {version(p)}")
    except PackageNotFoundError: print(f"    {p:16s} (not found)")
print("  (no torch / transformers here: visuals load cached artifacts only)")
print("=" * 64)


In [ ]:
# @title
# ==============================================================================
# STAGE 02  -  STORAGE, MODEL CACHE AND PERSISTENT ARTIFACTS
#             section 1 of 9  -  Setup, preflight and the visual system
# Two concerns are kept strictly apart:
#   * MODEL CACHE  (HF_HOME)       -> ALWAYS local disk. The embedder / LLM weights
#                                     are many GB and must never be written to Drive.
#   * ARTIFACT STORE (CHECKPOINT)  -> local disk by default, with an OPTIONAL
#                                     persistent MIRROR (Google Drive on Colab, or
#                                     any folder you pick) for files <= the size cap.
# This implements the plan exactly: every artifact is written LOCALLY and also
# MIRRORED to a persistent place, EXCEPT a single file larger than
# ARTIFACT_SIZE_CAP_MB, which stays LOCAL-ONLY (so a big embedding/UMAP matrix
# never tries to fill Drive). Notebook 2 reads the SAME store: local first, then
# the mirror, so you never recompute embeddings just because the runtime reset.
# ==============================================================================
import os, shutil

# ------------------------------ EDIT HERE -------------------------------------
USE_PERSISTENT_MIRROR = True     # True = also copy small artifacts to a persistent folder
PERSIST_ROOT          = None     # None = auto (mount Google Drive on Colab).
                                 # Or set a path, e.g. "/content/drive/MyDrive"
                                 # or a local persistent disk / mounted folder.
ARTIFACT_SIZE_CAP_MB  = 100      # a single file larger than this is saved LOCAL ONLY
# ----------------------------- END EDIT ---------------------------------------

ON_COLAB   = os.path.isdir("/content")
LOCAL_BASE = "/content" if ON_COLAB else os.path.abspath(".")

# Model weights -> ALWAYS local disk.
os.environ["HF_HOME"] = os.path.join(LOCAL_BASE, "hf_cache")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

# Local artifact root (fast; may be ephemeral on Colab).
CHECKPOINT_DIR = os.path.join(LOCAL_BASE, "iGEM_Pipeline_Checkpoints_Pooled")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Optional persistent mirror (Drive on Colab, or a folder you choose).
MIRROR_ROOT = None
if USE_PERSISTENT_MIRROR:
    _root = PERSIST_ROOT
    if _root is None and ON_COLAB:
        try:
            from google.colab import drive
            try:
                drive.mount("/content/drive", force_remount=False)
            except ValueError:
                pass  # already mounted
            _root = "/content/drive/MyDrive"
        except Exception as e:
            print(f"  Drive mount unavailable ({type(e).__name__}); mirror OFF, local only.")
            _root = None
    if _root:
        MIRROR_ROOT = os.path.join(_root, "iGEM_Pipeline_Checkpoints_Pooled")
        try:
            os.makedirs(MIRROR_ROOT, exist_ok=True)
        except Exception as e:
            print(f"  Could not create mirror dir ({type(e).__name__}); local only.")
            MIRROR_ROOT = None

# Notebook identity (stamped into every manifest).
NOTEBOOK_NAME    = "Notebook 3 - Visual Explorers"
NOTEBOOK_VERSION = "2.0"

print("=" * 64)
print("  STAGE 2  STORAGE & MODEL CACHE")
print("=" * 64)
print(f"  notebook        : {NOTEBOOK_NAME}  (v{NOTEBOOK_VERSION})")
print(f"  model cache     : {os.environ['HF_HOME']}  (LOCAL, always)")
print(f"  artifact local  : {CHECKPOINT_DIR}")
print(f"  artifact mirror : {MIRROR_ROOT or 'OFF (local only)'}")
print(f"  size rule       : files > {ARTIFACT_SIZE_CAP_MB} MB are LOCAL-ONLY (never mirrored)")
print("=" * 64)


In [ ]:
# @title
# ==============================================================================
# STAGE 03  -  CONTROL PANEL: THE SINGLE SOURCE OF TRUTH
#             section 1 of 9  -  Setup, preflight and the visual system
# Every tunable lives here and nowhere else (Golden Rule 3). Edit a value in the
# EDIT block, re-run this cell and the cells below; the content-hash cache
# (Stage 4) recomputes only the stages a change actually affects. The values are
# then frozen into one immutable CFG object that also carries:
#   set_all_seeds()  - fixes Python, NumPy and PyTorch randomness (Golden Rule 1)
#   CFG.fingerprint() - the per-stage cache key (Golden Rule 2)
# ==============================================================================
from dataclasses import dataclass, field, asdict
import os, json, hashlib, random
import numpy as np

# ----------------------------- EDIT HERE --------------------------------------
SEED            = 42
CACHE_VERSION   = "v1"              # bump to "v2","v3"... to force EVERY cached stage to recompute

# Data
DATA_URL        = "https://drive.google.com/uc?id=YOUR_SURVEY_DATA_FILE_ID"
TEXT_COLS       = ("Connection and belonging",
                   "Reflect on how your team worked together",
                   "Beyond iGEM",
                   "Supporting and limiting conditions",
                   "One word or phrase that captures what iGEM meant to you")
SHORT_LABELS    = ("Connection", "Team Work", "Beyond iGEM", "Conditions", "One Word")

# Embedding   (Qwen3-Embedding-8B, 4096-d; switch to "bge-m3" to compare)
EMBED_BACKEND   = "qwen3-8b"        # "qwen3-8b"=Qwen3-Embedding-8B (4096-d, for L4 High-RAM) | "qwen3"=4B (2560-d) | "bge-m3"=1024-d
EMBED_BATCH     = 16

# UMAP  (seeded, standard; no Parametric UMAP)
UMAP_NEIGHBORS  = 50
UMAP_MIN_DIST   = 0.0               # tight geometry for clustering
CLUSTER_DIM     = 5                 # dimensionality of the clustering space

# Ward clustering
K_MODE          = "auto"            # "auto" = data-driven over [K_MIN, K_MAX]; "fixed" = K_FIXED
K_FIXED         = 16
K_MIN           = 4
K_MAX           = 30

# Sub-theme drill-down + validation
SUBTHEME_MIN    = 30                # split any macro theme with at least this many responses
SUBTHEME_TARGET = 50                # aim ~this many responses per sub-theme (LOWER = MORE sub-themes)
SUBTHEME_MAX_K  = 12                # cap sub-themes per macro theme
N_SEEDS         = 10                # multi-seed stability runs
N_BOOTSTRAP     = 100               # bootstrap resamples for Jaccard stability
TRUST_K         = 15                # neighbours for trustworthiness / continuity

# Language-model theme labelling
LLM_ENABLE      = True
LLM_MODEL       = "Qwen/Qwen2.5-14B-Instruct"   # A100: "Qwen/Qwen2.5-32B-Instruct"
LLM_N_QUOTES    = 7
LLM_FORCE_REGEN = False             # True forces re-labelling even if cached

# Team awards (structured metadata, never embedded)
AWARD_YEAR      = 2025
# ------------------------------ END EDIT --------------------------------------

# Embedding registry: name + output dimension for each backend.
_EMBED_REGISTRY = {
    "qwen3":      ("Qwen/Qwen3-Embedding-4B",   2560),  # strong + multilingual, fits an L4
    "qwen3-8b":   ("Qwen/Qwen3-Embedding-8B",   4096),  # A100: highest accuracy
    "qwen3-0.6b": ("Qwen/Qwen3-Embedding-0.6B", 1024),  # light alternative
    "bge-m3":     ("BAAI/bge-m3",               1024),  # previous model, kept for comparison
}
# Qwen3-Embedding is instruction-aware: a short task description sharpens the
# geometry for our use (thematic clustering). BGE-M3 ignores this (encoded plain).
EMBED_INSTRUCTION = ("Represent this iGEM participant's survey response BY ITS TOPIC for "
                     "thematic clustering, independent of sentiment or how positive or "
                     "negative it is:")
_EMBED_MODEL, _EMBED_DIM = _EMBED_REGISTRY[EMBED_BACKEND]

# Which settings affect which stage. A stage's cache key is built only from the
# settings above it in the dependency chain, so changing K re-runs Ward but not
# the embedding, and changing UMAP re-runs UMAP, Ward and everything downstream.
_RELEVANT = {
    "data":    ["data_url"],
    "embed":   ["seed", "embed_model", "embed_dim", "embed_batch"],
    "umap":    ["seed", "embed_model", "umap_neighbors", "umap_min_dist", "cluster_dim"],
    "ward":    ["seed", "embed_model", "umap_neighbors", "umap_min_dist", "cluster_dim",
                "k_mode", "k_fixed", "k_min", "k_max"],
    "subtheme":["seed", "embed_model", "umap_neighbors", "umap_min_dist", "cluster_dim",
                "k_mode", "k_fixed", "k_min", "k_max", "subtheme_min", "subtheme_target", "subtheme_max_k"],
}


@dataclass(frozen=True)
class Config:
    seed: int
    cache_version: str
    data_url: str
    text_cols: tuple
    short_labels: tuple
    embed_backend: str
    embed_model: str
    embed_dim: int
    embed_batch: int
    umap_neighbors: int
    umap_min_dist: float
    cluster_dim: int
    k_mode: str
    k_fixed: int
    k_min: int
    k_max: int
    subtheme_min: int
    subtheme_target: int
    subtheme_max_k: int
    n_seeds: int
    n_bootstrap: int
    trust_k: int
    llm_enable: bool
    llm_model: str
    llm_n_quotes: int
    llm_force_regen: bool
    award_year: int
    checkpoint_dir: str

    def fingerprint(self, tag, n_rows, extra=None):
        """Short, stable cache key for `tag`. Built from the settings relevant to
        that stage, the row count, and an optional content signature `extra`
        (e.g. a hash of the actual input), so the cache is genuinely content
        addressed and never returns a stale object."""
        keys = _RELEVANT.get(tag, sorted(asdict(self).keys()))
        payload = {k: getattr(self, k) for k in keys}
        payload["n_rows"] = int(n_rows)
        payload["cache_version"] = self.cache_version
        if extra is not None:
            payload["extra"] = str(extra)
        blob = json.dumps(payload, sort_keys=True, default=str)
        return hashlib.md5(blob.encode()).hexdigest()[:10]


CFG = Config(
    seed=SEED, cache_version=CACHE_VERSION, data_url=DATA_URL, text_cols=TEXT_COLS, short_labels=SHORT_LABELS,
    embed_backend=EMBED_BACKEND, embed_model=_EMBED_MODEL, embed_dim=_EMBED_DIM,
    embed_batch=EMBED_BATCH, umap_neighbors=UMAP_NEIGHBORS, umap_min_dist=UMAP_MIN_DIST,
    cluster_dim=CLUSTER_DIM, k_mode=K_MODE, k_fixed=K_FIXED, k_min=K_MIN, k_max=K_MAX,
    subtheme_min=SUBTHEME_MIN, subtheme_target=SUBTHEME_TARGET, subtheme_max_k=SUBTHEME_MAX_K, n_seeds=N_SEEDS, n_bootstrap=N_BOOTSTRAP, trust_k=TRUST_K,
    llm_enable=LLM_ENABLE, llm_model=LLM_MODEL, llm_n_quotes=LLM_N_QUOTES,
    llm_force_regen=LLM_FORCE_REGEN, award_year=AWARD_YEAR,
    checkpoint_dir=CHECKPOINT_DIR,
)


def set_all_seeds(seed=None):
    """Fix every random source the pipeline uses. Call at the top of any stage
    that draws random numbers (UMAP, bootstrap, sampling)."""
    s = CFG.seed if seed is None else int(seed)
    os.environ["PYTHONHASHSEED"] = str(s)
    random.seed(s)
    np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


set_all_seeds()
print("=" * 64)
print("  STAGE 3  CONTROL PANEL")
print("=" * 64)
print(f"  seed            : {CFG.seed}")
print(f"  embedding       : {CFG.embed_model}  ({CFG.embed_dim}-d, backend={CFG.embed_backend})")
print(f"  UMAP            : n_neighbors={CFG.umap_neighbors}, min_dist={CFG.umap_min_dist}, "
      f"cluster_dim={CFG.cluster_dim}  (seeded, standard)")
print(f"  K (legacy knob) : mode={CFG.k_mode}, range=[{CFG.k_min}, {CFG.k_max}]  -- NOT the analytical K")
print(  "  analytical K    : set in NB2 Stage 21H/23H (DBCV local maxima for SUB, "
        "Ward on the sub centroids for MAIN) and read from the artifacts at run "
        "time; the legacy range above is unused")
print(f"  LLM labelling   : {CFG.llm_model if CFG.llm_enable else 'disabled'}")
print(f"  cache fingerprint demo (embed, n=900) : {CFG.fingerprint('embed', 900)}")
print("=" * 64)

In [ ]:
# @title
# ==============================================================================
# STAGE 04  -  LIBRARY: DEFINE ONCE, USED EVERYWHERE
#             section 1 of 9  -  Setup, preflight and the visual system
# Same house style + helpers as the monolith, with upgrades:
#   1. psave / pload / cached write LOCAL and MIRROR (<= size cap), read local
#      first then mirror, so artifacts survive a runtime reset and Notebook 2/3
#      find them automatically.
#   2. Every save is logged so write_manifest() can stamp a reproducible manifest.
#   3. artifact_diagnostic() reports which artifacts are present / missing.
#   4. validate_columns() is completed (monolith truncated it) and checks
#      Participant / Team.
#   5. invalidate() deletes an artifact so its stage recomputes next run.
# ==============================================================================
import os, pickle, hashlib, json, time, shutil
import numpy as np, pandas as pd
import matplotlib as mpl, matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import colorsys

POOLED_DIR = os.path.join(CFG.checkpoint_dir, "pooled_combined")
os.makedirs(POOLED_DIR, exist_ok=True)

MIRROR_POOLED_DIR = None
if "MIRROR_ROOT" in globals() and MIRROR_ROOT:
    MIRROR_POOLED_DIR = os.path.join(MIRROR_ROOT, "pooled_combined")
    try:
        os.makedirs(MIRROR_POOLED_DIR, exist_ok=True)
    except Exception:
        MIRROR_POOLED_DIR = None

_ARTIFACT_LOG = {}


def _cap_mb():
    return float(globals().get("ARTIFACT_SIZE_CAP_MB", 100))


def _mirror_file(path_local, mirror_dir):
    size_mb = os.path.getsize(path_local) / 1e6
    if not mirror_dir:
        return False, size_mb
    if size_mb > _cap_mb():
        print(f"    (LOCAL-ONLY: {os.path.basename(path_local)} = {size_mb:.0f} MB > {_cap_mb():.0f} MB cap)")
        return False, size_mb
    try:
        shutil.copy2(path_local, os.path.join(mirror_dir, os.path.basename(path_local)))
        return True, size_mb
    except Exception as e:
        print(f"    (mirror failed for {os.path.basename(path_local)}: {type(e).__name__})")
        return False, size_mb


def _log_artifact(name, path, mirrored):
    _ARTIFACT_LOG[name] = dict(path=path, bytes=int(os.path.getsize(path)),
                               size_mb=round(os.path.getsize(path) / 1e6, 3),
                               mirrored=bool(mirrored))


def cached(tag, key, compute):
    fname = f"{tag}_{key}.pkl"
    local = os.path.join(CFG.checkpoint_dir, fname)
    if os.path.exists(local):
        with open(local, "rb") as f:
            obj = pickle.load(f)
        print(f"  [{tag}] loaded from local cache  ({fname})")
        return obj
    if MIRROR_ROOT:
        mp = os.path.join(MIRROR_ROOT, fname)
        if os.path.exists(mp):
            try:
                shutil.copy2(mp, local)
                with open(local, "rb") as f:
                    obj = pickle.load(f)
                print(f"  [{tag}] restored from mirror  ({fname})")
                return obj
            except Exception:
                pass
    print(f"  [{tag}] computing (no cache for key {key}) ...")
    obj = compute()
    with open(local, "wb") as f:
        pickle.dump(obj, f)
    m, _ = _mirror_file(local, MIRROR_ROOT)
    _log_artifact(tag, local, m)
    print(f"  [{tag}] saved  ({fname})")
    return obj


def psave(name, obj):
    local = os.path.join(POOLED_DIR, f"{name}.pkl")
    with open(local, "wb") as f:
        pickle.dump(obj, f)
    m, _ = _mirror_file(local, MIRROR_POOLED_DIR)
    _log_artifact(name, local, m)


def pload(name):
    local = os.path.join(POOLED_DIR, f"{name}.pkl")
    if os.path.exists(local):
        with open(local, "rb") as f:
            return pickle.load(f)
    if MIRROR_POOLED_DIR:
        mp = os.path.join(MIRROR_POOLED_DIR, f"{name}.pkl")
        if os.path.exists(mp):
            try:
                shutil.copy2(mp, local)
                with open(local, "rb") as f:
                    return pickle.load(f)
            except Exception:
                pass
    return None


save_ckpt = psave
def load_ckpt(name):
    return pload(name)


def get_df():
    """Safe loader for the cleaned frame: df_final if present else df_cleaned.
    Avoids `pload('df_final') or pload('df_cleaned')`, which raises ValueError
    because a DataFrame's truth value is ambiguous."""
    d = pload("df_final")
    return d if d is not None else pload("df_cleaned")


def invalidate(name):
    """Delete a named artifact (local + mirror) and any content-hash cache tagged
    with it, so the producing stage recomputes next run. The monolith referenced
    this helper but never defined it; provided here."""
    import glob as _glob
    removed = []
    for d in (POOLED_DIR, MIRROR_POOLED_DIR):
        if not d:
            continue
        p = os.path.join(d, f"{name}.pkl")
        if os.path.exists(p):
            try:
                os.remove(p); removed.append(p)
            except Exception:
                pass
    for d in (CFG.checkpoint_dir, (MIRROR_ROOT if "MIRROR_ROOT" in globals() else None)):
        if not d:
            continue
        for p in _glob.glob(os.path.join(d, f"{name}_*.pkl")):
            try:
                os.remove(p); removed.append(p)
            except Exception:
                pass
    _ARTIFACT_LOG.pop(name, None)
    print(f"  invalidate('{name}'): removed {len(removed)} file(s)")
    return removed


def data_sig(x):
    h = hashlib.md5()
    if isinstance(x, np.ndarray):
        h.update(np.ascontiguousarray(x).view(np.uint8)[::max(1, x.size // 100000)].tobytes())
        h.update(str(x.shape).encode())
    else:
        joined = "\x1f".join(map(str, x))
        h.update(joined.encode("utf-8", "ignore"))
    return h.hexdigest()[:10]


# ---------- house style ----------
NAT_BG, NAT_TEXT, NAT_GRID = "#FFFFFF", "#1E1E1E", "#E6E6E6"
mpl.rcParams.update({
    "figure.facecolor": "white", "savefig.facecolor": "white", "savefig.edgecolor": "white",
    "axes.facecolor": "#FCFCFE", "axes.edgecolor": "#D9D9E0", "axes.linewidth": 0.9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11.5, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.titlepad": 8,
    "axes.labelcolor": "#333333", "axes.labelsize": 9.5,
    "xtick.color": "#666666", "ytick.color": "#666666",
    "axes.axisbelow": True, "grid.color": "#EEEEF2", "grid.linewidth": 0.8,
    "legend.frameon": False, "font.size": 10, "figure.titleweight": "bold",
})
HIGHLIGHT = "#9B1B30"
VIVID = ["#C1121F", "#2A9D8F", "#3A0CA3", "#E76F51", "#5390D9", "#9D4EDD",
         "#D81159", "#1B9AAA", "#A4133C", "#F3722C", "#4361EE", "#588157",
         "#7209B7", "#06A77D", "#BC4749", "#6A040F"]
try:
    import colorcet as _cc
    _GLASBEY = list(_cc.glasbey_category10)
except Exception:
    _GLASBEY = []


def _not_yellow(c):
    r, g, b = mcolors.to_rgb(c)
    h, sat, v = colorsys.rgb_to_hsv(r, g, b)
    return not (0.11 <= h <= 0.22 and sat > 0.30)


def apply_house_style():
    mpl.rcParams.update({
        "figure.facecolor": NAT_BG, "axes.facecolor": NAT_BG,
        "savefig.facecolor": NAT_BG, "axes.edgecolor": NAT_GRID,
        "axes.grid": True, "grid.color": NAT_GRID, "grid.linewidth": 0.6,
        "axes.axisbelow": True, "axes.titleweight": "bold",
        "text.color": NAT_TEXT, "axes.labelcolor": NAT_TEXT,
        "xtick.color": NAT_TEXT, "ytick.color": NAT_TEXT,
        "font.size": 11, "figure.dpi": 110, "savefig.dpi": 300,
    })


def palette(n):
    pool = list(VIVID)
    if n > len(pool):
        for c in _GLASBEY:
            hx = mcolors.to_hex(c)
            if _not_yellow(hx) and hx not in pool:
                pool.append(hx)
    if len(pool) < n:
        pool = pool * (n // len(pool) + 1)
    return [mcolors.to_hex(c) for c in pool[:n]]


def lighten(hex_color, amount):
    r, g, b = mcolors.to_rgb(hex_color)
    return mcolors.to_hex((r + (1 - r) * amount, g + (1 - g) * amount, b + (1 - b) * amount))


Q_COLORS = dict(zip(CFG.short_labels, palette(len(CFG.short_labels))))
GRAD = mcolors.LinearSegmentedColormap.from_list("ruby", ["#FFFFFF", "#F4A6B7", "#C1121F", "#6A040F"])


def save_fig(fig, name, interactive=False, static=True):
    base = os.path.join(POOLED_DIR, name)
    wrote = []
    if hasattr(fig, "write_html"):
        if interactive:
            fig.write_html(base + ".html", include_plotlyjs="cdn", full_html=True)
            wrote.append("HTML")
        if static:
            try:
                fig.write_image(base + ".png", scale=2)
                fig.write_image(base + ".svg")
                wrote += ["PNG", "SVG"]
            except Exception as e:
                print(f"  (Plotly static export needs kaleido; skipped: {type(e).__name__})")
    else:
        fig.savefig(base + ".png", dpi=300, bbox_inches="tight", facecolor=NAT_BG)
        fig.savefig(base + ".svg", bbox_inches="tight", facecolor=NAT_BG)
        wrote += ["PNG", "SVG"]
    for ext in (".png", ".svg", ".html"):
        p = base + ext
        if os.path.exists(p) and MIRROR_POOLED_DIR:
            _mirror_file(p, MIRROR_POOLED_DIR)
    try:
        from IPython.display import display as _display
        _display(fig)
    except Exception:
        pass
    if wrote:
        print(f"  saved figure: {os.path.basename(base)}  ({' + '.join(wrote)})")
    return base


def validate_columns(df):
    missing = [c for c in CFG.text_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Survey is missing expected text columns: {missing}. "
                       "Check DATA_URL / the export; the pipeline will not guess column names.")
    for c in ("Participant", "Team"):
        if c not in df.columns:
            raise KeyError(f"Survey is missing required column '{c}' "
                           "(needed for embedding metadata and the team-award join).")


def _now():
    return time.strftime("%Y-%m-%d %H:%M:%S")


def write_manifest(filename, fingerprints, n_rows):
    man = dict(
        notebook=NOTEBOOK_NAME, notebook_version=NOTEBOOK_VERSION, timestamp=_now(),
        seed=CFG.seed, cache_version=CFG.cache_version, embed_model=CFG.embed_model,
        n_rows=int(n_rows), config_fingerprint=CFG.fingerprint("all", n_rows),
        fingerprints=dict(fingerprints),
        artifact_store=dict(local=POOLED_DIR, mirror=MIRROR_POOLED_DIR, size_cap_mb=_cap_mb()),
        artifacts=[dict(name=k, **v) for k, v in sorted(_ARTIFACT_LOG.items())],
    )
    p_local = os.path.join(CFG.checkpoint_dir, filename)
    with open(p_local, "w") as f:
        json.dump(man, f, indent=2, default=str)
    if MIRROR_ROOT:
        try:
            shutil.copy2(p_local, os.path.join(MIRROR_ROOT, filename))
        except Exception:
            pass
    print(f"  manifest written: {filename}  ({len(man['artifacts'])} artifacts logged, "
          f"config={man['config_fingerprint']})")
    return man


def artifact_status(name):
    lp = os.path.join(POOLED_DIR, f"{name}.pkl")
    if os.path.exists(lp):
        return "local", os.path.getsize(lp) / 1e6
    if MIRROR_POOLED_DIR:
        mp = os.path.join(MIRROR_POOLED_DIR, f"{name}.pkl")
        if os.path.exists(mp):
            return "mirror", os.path.getsize(mp) / 1e6
    return "MISSING", 0.0


def artifact_diagnostic(required, optional=()):
    print("=" * 64)
    print("  ARTIFACT DIAGNOSTIC")
    print("=" * 64)
    print(f"  {'artifact':34s} {'where':8s} {'MB':>9}")
    print("  " + "-" * 54)
    missing = []
    for n in required:
        where, mb = artifact_status(n)
        if where == "MISSING":
            missing.append(n)
        flag = "" if where != "MISSING" else "   <-- MISSING"
        print(f"  {n:34s} {where:8s} {mb:9.2f}{flag}")
    for n in optional:
        where, mb = artifact_status(n)
        print(f"  {n:34s} {where:8s} {mb:9.2f}   (optional)")
    print("  " + "-" * 54)
    if missing:
        print(f"  {len(missing)} REQUIRED artifact(s) MISSING: {missing}")
    else:
        print("  all required artifacts present.")
    print("=" * 64)
    return missing


# ---------- canonical artifact registry (single source of truth; Fix 1/6/7) ----
ART_NB1 = ["df_cleaned", "df_final", "embeddings_dict", "metadata_dict",
           "pooled_embeddings", "pooled_origin", "pooled_meta"]
ART_NB2 = ["pooled_umap_nd", "pooled_umap5d", "pooled_umap3d", "pooled_umap2d",
           "pooled_hdbscan_screen", "pooled_hdbscan_choice", "pooled_hdbscan_hierarchy",
           "pooled_K", "pooled_levels", "pooled_canonical_level",
           "pooled_labels", "pooled_labels_main7", "pooled_labels_sub",
           "pooled_sub_labels", "pooled_sub_of_macro",
           "pooled_hier_maps", "pooled_hier_crosswalk",
           "pooled_cluster_meta", "pooled_analytical_unit",
           "pooled_stability", "validation_main7", "validation_sub",
           "pooled_main_llm", "pooled_sub_llm", "pooled_theme_llm",
           "pooled_main_name", "pooled_sub_name",
           "pooled_main_terms", "pooled_sub_terms",
           "pooled_main_size", "pooled_sub_size",
           "pooled_naming_audit", "pooled_label_verification",
           "pooled_award", "pooled_team_org", "pooled_master",
           "pooled_sentiment_response", "pooled_sentiment_team_long",
           "pooled_sentiment_team_wide", "pooled_projection",
           "pooled_validity_xq", "cluster_reality", "pooled_convergent",
           "pooled_embed_validation",
           "pooled_correlation_forest", "pooled_correlation_forest_main",
           "pooled_correlation_forest_sub"]
PRODUCER = {
    "df_cleaned": "NB1 Stage 12", "df_final": "NB1 Stage 16",
    "embeddings_dict": "NB1 Stage 13b", "metadata_dict": "NB1 Stage 13b",
    "pooled_embeddings": "NB1 Stage 18", "pooled_origin": "NB1 Stage 18",
    "pooled_meta": "NB1 Stage 18",
    "pooled_umap_nd": "NB2 Stage 21H", "pooled_umap5d": "NB2 Stage 21H",
    "pooled_umap3d": "NB2 Stage 21H", "pooled_umap2d": "NB2 Stage 21H",
    "pooled_hdbscan_screen": "NB2 Stage 21H", "pooled_hdbscan_choice": "NB2 Stage 21H",
    "pooled_hdbscan_hierarchy": "NB2 Stage 23H",
    "pooled_K": "NB2 Stage 23H", "pooled_levels": "NB2 Stage 23H",
    "pooled_canonical_level": "NB2 Stage 23H", "pooled_labels": "NB2 Stage 23H",
    "pooled_labels_main7": "NB2 Stage 23H", "pooled_labels_sub": "NB2 Stage 23H",
    "pooled_sub_of_macro": "NB2 Stage 23H", "pooled_hier_maps": "NB2 Stage 23H",
    "pooled_sub_labels": "NB2 Stage 46 (alias of the canonical level)",
    "pooled_hier_crosswalk": "NB2 Stage 46 (from Stage 23H's crosswalk)",
    "pooled_cluster_meta": "NB2 Stage 46", "pooled_analytical_unit": "NB2 Stage 46",
    "pooled_stability": "NB2 Stage 25", "validation_main7": "NB2 Stage 25",
    "validation_sub": "NB2 Stage 25",
    "pooled_projection": "NB2 Stage 23d",
    "pooled_embed_validation": "NB2 Stage 24", "pooled_validity_xq": "NB2 Stage 26",
    "cluster_reality": "NB2 Stage 27", "pooled_convergent": "NB2 Stage 29",
    "pooled_sentiment_response": "NB2 Stage 26B",
    "pooled_sentiment_team_long": "NB2 Stage 26B",
    "pooled_sentiment_team_wide": "NB2 Stage 26B",
    "pooled_main_llm": "NB2 Stage 30 (verified in 30v)",
    "pooled_sub_llm": "NB2 Stage 30 (verified in 30v)",
    "pooled_theme_llm": "NB2 Stage 30 (verified in 30v)",
    "pooled_main_name": "NB2 Stage 30 (verified in 30v)",
    "pooled_sub_name": "NB2 Stage 30 (verified in 30v)",
    "pooled_main_terms": "NB2 Stage 30", "pooled_sub_terms": "NB2 Stage 30",
    "pooled_main_size": "NB2 Stage 30", "pooled_sub_size": "NB2 Stage 30",
    "pooled_naming_audit": "NB2 Stage 30",
    "pooled_label_verification": "NB2 Stage 30v",
    "pooled_award": "NB2 Stage 32", "pooled_team_org": "NB2 Stage 33",
    "pooled_master": "NB2 Stage 33b",
    "pooled_correlation_forest": "NB2 Stage 40",
    "pooled_correlation_forest_main": "NB2 Stage 40",
    "pooled_correlation_forest_sub": "NB2 Stage 40",
}

def require(*names, stage=""):
    """Stage-local dependency guard (Fix 4): raise a clear error naming each
    missing artifact and the stage that produces it. Returns the loaded object(s)."""
    missing = [n for n in names if pload(n) is None]
    if missing:
        prod = "; ".join(f"{n} <- {PRODUCER.get(n, '?')}" for n in missing)
        raise RuntimeError(f"[{stage or 'stage'}] missing required artifact(s): {prod}. "
                           "Run the producing stage(s) first.")
    objs = [pload(n) for n in names]
    return objs[0] if len(objs) == 1 else objs


def artifact_inventory(names=None):
    """One place to see every saved artifact, where it lives, and which stage made it."""
    names = names or (ART_NB1 + ART_NB2)
    print("=" * 70)
    print("  ARTIFACT INVENTORY   (name | where | MB | producing stage)")
    print("=" * 70)
    for n in names:
        where, mb = artifact_status(n)
        print(f"  {n:28s} {where:7s} {mb:8.2f}   {PRODUCER.get(n, '?')}")
    print("=" * 70)


print("  Stage 4 library ready: storage-aware cache + psave/pload + manifest + diagnostics.")


In [ ]:
# @title
# ==============================================================================
# STAGE 05  -  COMPATIBILITY BRIDGE
#             section 1 of 9  -  Setup, preflight and the visual system
# The analysis stages were written against an earlier set of bare constant names.
# Rather than touch every one, we expose those names here, sourced from the one
# Control Panel (CFG) so there is still a single source of truth. Cleaning and
# outlier thresholds that were never clustering-critical live here as plain
# constants. There is no RUN_MODE: caching is automatic (Stage 4).
# ==============================================================================
# Core identifiers from CFG
SEED            = CFG.seed
DATA_URL        = CFG.data_url
TEXT_COLS       = list(CFG.text_cols)
SHORT_LABELS    = list(CFG.short_labels)
ONE_WORD_COL    = TEXT_COLS[4]
EMBED_MODEL     = CFG.embed_model
EMBED_DIM       = CFG.embed_dim
EMBED_BATCH     = CFG.embed_batch
TRUST_K         = CFG.trust_k
UMAP_QUALITY_METRIC = "euclidean"

# Clustering names used by the K-selection / Ward / validation cells
AGGLO_LINKAGE   = "ward"
AGGLO_METRIC    = "euclidean"
AGGLO_K_RANGE   = (CFG.k_min, CFG.k_max)
AGGLO_K_OVERRIDE = CFG.k_fixed if CFG.k_mode == "fixed" else None

# Language-model + awards
LLM_ENABLE      = CFG.llm_enable
LLM_MODEL_NAME  = CFG.llm_model
LLM_N_QUOTES    = CFG.llm_n_quotes
LLM_FORCE_REGEN = CFG.llm_force_regen
AWARD_TSV       = __import__("os").path.join(CFG.checkpoint_dir, "team_awards.tsv")
AWARD_YEAR      = CFG.award_year
AWARD_TEAMID_OVERRIDES = {"GreatBay_SCIE": "GreatBay-SCIE"}
MEDAL_RANK      = {"Gold": 3, "Silver": 2, "Bronze": 1, "None": 0}

# Colours: reuse the house palette so the old names resolve to the new style
# Capture the Stage 4 palette FUNCTION under a private name: several analysis
# cells reassign a local variable called `palette` to a list, so get_cluster_colors
# must not depend on the mutable global of that name.
_PALETTE_FN     = palette
NAT_COLORS      = _PALETTE_FN(len(SHORT_LABELS))
HIGHLIGHT_COLOR = HIGHLIGHT
def get_cluster_colors(n):
    return _PALETTE_FN(n)

# Text-cleaning / keyword sanitation constants (rarely tuned)
MIN_TOKEN_LENGTH = 2
HELSINKI_MODEL   = "Helsinki-NLP/opus-mt-mul-en"
KEYWORD_LATIN_ONLY = True
WC_DOMAIN_STOPS = {
    "igem", "team", "project", "experience", "nan", "thing", "lot", "way", "day",
    "time", "people", "part", "year", "place", "point", "fact", "kind", "group",
    "number", "case", "end", "question", "item", "work", "teammate", "new",
}
WC_SHORT_WHITELIST = {"lab", "phd", "dna", "rna", "fun", "pcr", "gel", "gmo",
                      "bio", "aid", "gap", "hub", "key", "aim"}
EXTRA_MULTILANG_STOPS = {
    "avant", "apres", "pendant", "environ", "depuis", "aprovado", "tudo", "tambm",
    "jamais", "aller", "venir", "passer", "rester", "devenir", "thing", "things",
    "something", "someone", "everyone", "everything", "really", "actually",
    "definitely", "probably", "maybe",
}

# Response-level outlier thresholds (Stage 10)
OUTLIER_MIN_ALPHA_RATIO  = 0.5
OUTLIER_MIN_ALPHA_TOKENS = 1
OUTLIER_MIN_VOWEL_RATIO  = 0.15
OUTLIER_MAX_INVALID_Q    = 3
SAME_WORD_THRESHOLD      = 3

print("  Stage 4b bridge ready: old config names resolve to CFG (no RUN_MODE).")


In [ ]:
# @title
# ==============================================================================
# STAGE 06  -  IMPORTS
#             section 1 of 9  -  Setup, preflight and the visual system
# INPUT:  -
# OUTPUT: all libraries available
# ==============================================================================

import pandas as pd
import numpy as np
import re
import warnings
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import umap as umap_lib
from sklearn.metrics.pairwise import cosine_similarity
from collections import Counter
from scipy.stats import spearmanr
from scipy.stats import spearmanr, ttest_ind, ttest_1samp
from scipy.spatial.distance import pdist
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d
from sklearn.manifold import trustworthiness
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')
print('Imports complete.')

### Stage 07 · Preflight

**Question.** Are the Notebook 2 artifacts present, mutually consistent, and the ones this notebook expects?

**Reads.** Every artifact in the required list, and `pooled_master` as the single row-level table.

**How to read it.** This cell publishes the contract the rest of the notebook obeys: the two levels, which one carries claims, the theme names and the sub -> main map, and the helpers that keep unplaced responses out of every test. It stops the run on any misalignment rather than letting a stale array through - a length mismatch here would otherwise reappear as plausible but wrong numbers in every figure.

In [ ]:
# @title
# ==============================================================================
# STAGE 07  -  PREFLIGHT: LOAD AND VERIFY NOTEBOOK 2'S ARTIFACTS
#             section 1 of 9  -  Setup, preflight and the visual system
# ==============================================================================
# ==============================================================================
# NOTEBOOK 3  -  PREFLIGHT   (visual-only: loads NB2's artifacts, recomputes nothing)
# ------------------------------------------------------------------------------
# REWRITTEN FOR THE TWO-LEVEL HIERARCHY AND FOR NOISE. Three things changed and
# every later cell depends on all three.
#
# 1. THE MID LEVEL IS GONE. Notebook 2 Stage 23H now produces MAIN and SUB from
#    one HDBSCAN fit and nothing else. This cell's old REQUIRED list demanded
#    eleven artifacts that Notebook 2 no longer writes - pooled_labels_mid,
#    pooled_mid_name/_llm/_size/_terms/_quotes, pooled_sub_quotes,
#    pooled_main_quotes, pooled_dashboard_tables, pooled_super_of_theme,
#    pooled_pool_policy - so the notebook stopped here before drawing anything.
#
# 2. THE _FOCUS REMAP IS REMOVED. The old cell monkey-patched pload() so that a
#    cell asking for "pooled_sub_*" silently received the MID artifacts. Every
#    downstream cell inherited that substitution without knowing, which means no
#    figure could be read at face value: its axis said "sub" and its data was
#    mid. With two levels there is nothing to remap. pload() is left alone, and
#    the level a figure uses is now stated in the figure, not swapped underneath
#    it. (Downstream cells do `_PL = globals().get("_orig_pload", pload)`; with
#    the patch gone that resolves to the real pload, which is what they want.)
#
# 3. NOISE IS A FIRST-CLASS FACT. HDBSCAN leaves responses in no theme: they
#    carry id -1, name "(no theme)" and has_main/has_sub = False in
#    pooled_master. Nothing in this notebook counted them before, so a "-1"
#    theme of several hundred responses would have entered every prevalence,
#    every crosstab and every model as if it were a topic. The policy is fixed
#    HERE, once, and every later cell must use these helpers rather than
#    filtering by hand:
#
#      THE RULE
#        * NO TEST, model, correlation or significance claim may include a row
#          whose theme id is -1 at the level being tested. Use theme_rows(level).
#        * DESCRIPTIVE composition figures SHOULD show "(no theme)" explicitly,
#          as its own category, so the reader can see what did not fit.
#        * The DENOMINATOR for a prevalence is EVERY response (N_ALL), not just
#          the themed ones. Prevalences therefore sum to less than 100% and the
#          missing part is the noise - which is the honest picture. Reporting
#          out of themed rows only would silently inflate every theme.
#        * CLAIM_LEVEL is the sub level and nothing else. The main level is for
#          grouping, colour and narrative; its bootstrap stability is in the
#          "pattern" band (Hennig 2007), not the "finding" band, so no
#          significance claim may rest on it.
# ==============================================================================
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- what must exist
REQUIRED = [
    # from Notebook 1
    "pooled_meta", "pooled_origin",
    # the clustering itself (Notebook 2 Stage 21H / 23H)
    "pooled_umap_nd", "pooled_umap2d", "pooled_umap3d",
    "pooled_labels", "pooled_labels_main7", "pooled_labels_sub",
    "pooled_levels", "pooled_canonical_level", "pooled_K",
    "pooled_hier_maps", "pooled_sub_of_macro", "pooled_hdbscan_hierarchy",
    # names (Stage 30, verified in 30v)
    "pooled_main_llm", "pooled_sub_llm", "pooled_theme_llm",
    "pooled_main_name", "pooled_sub_name",
    # the one canonical row-level table every figure reads
    "pooled_master",
]
OPTIONAL = ["df_final", "df_cleaned", "pooled_umap5d", "pooled_award", "pooled_team_org",
            "pooled_sentiment_response", "pooled_sentiment_team_long",
            "pooled_sentiment_team_wide", "pooled_main_terms", "pooled_sub_terms",
            "pooled_main_size", "pooled_sub_size", "pooled_stability",
            "validation_main7", "validation_sub", "pooled_naming_audit",
            "pooled_label_verification", "pooled_projection",
            "pooled_projection_validation", "pooled_correlation_forest",
            "pooled_correlation_forest_main", "pooled_correlation_forest_sub",
            "pooled_validity_xq", "cluster_reality", "pooled_convergent"]

missing = artifact_diagnostic(REQUIRED, optional=OPTIONAL)
if missing:
    raise RuntimeError(
        "Notebook 2 artifacts are missing: " + ", ".join(missing) + ".\n"
        "FIX: run Notebook 1, then Notebook 2 to the END (through Stage 47, which "
        "asserts the contract this notebook relies on). If they ran on another "
        "machine, set PERSIST_ROOT in Stage 2 to the same mirror folder.\n"
        "If the missing names look like pooled_mid_* or pooled_*_quotes, you are "
        "running an OLD copy of this cell against the NEW Notebook 2 - those "
        "artifacts no longer exist.")

# --------------------------------------------------------------- the hierarchy
LEVELS      = list(pload("pooled_levels") or ["main", "sub"])
CLAIM_LEVEL = pload("pooled_canonical_level") or "sub"     # every test lives here
FRAME_LEVEL = "main"                                       # grouping / colour only
NO_THEME    = "(no theme)"
if LEVELS != ["main", "sub"]:
    raise RuntimeError(f"pooled_levels is {LEVELS}; this notebook is written for "
                       f"['main','sub']. Re-run Notebook 2 Stage 23H.")

_LABKEY = {"main": "pooled_labels_main7", "sub": "pooled_labels_sub"}
LAB = {lv: np.asarray(pload(k)) for lv, k in _LABKEY.items()}
K   = {lv: (int(v.max()) + 1 if (v >= 0).any() else 0) for lv, v in LAB.items()}
PARENT = {int(a): int(b) for a, b in (pload("pooled_hier_maps") or {}).get("sub_to_main", {}).items()}

def _names_of(lv):
    d = pload(f"pooled_{lv}_llm") or {}
    out = {}
    for c in range(K[lv]):
        rec = d.get(c) if c in d else d.get(str(c))
        out[c] = str((rec or {}).get("name", "") or f"{lv[0].upper()}{c}")
    return out
NAME = {lv: _names_of(lv) for lv in LEVELS}

# ------------------------------------------------------- the one row-level table
MASTER = pload("pooled_master")
if MASTER is None:
    raise RuntimeError("pooled_master is missing - run Notebook 2 Stage 33b.")
MASTER = MASTER.reset_index(drop=True)
N_ALL = len(MASTER)

meta_pool = pload("pooled_meta")
if meta_pool is not None:
    meta_pool = meta_pool.reset_index(drop=True)
    if len(meta_pool) != N_ALL:
        raise RuntimeError(f"pooled_master has {N_ALL} rows but pooled_meta has "
                           f"{len(meta_pool)} - they are from different runs. "
                           f"Re-run Notebook 2 Stage 33b.")

# every alignment the rest of the notebook assumes, checked once, here
for lv in LEVELS:
    if len(LAB[lv]) != N_ALL:
        raise RuntimeError(f"pooled_labels_{'main7' if lv == 'main' else lv} has "
                           f"{len(LAB[lv])} rows but pooled_master has {N_ALL}.")
    col = f"{lv}_id"
    if col not in MASTER.columns:
        raise RuntimeError(f"pooled_master has no {col} column - it was built by an "
                           f"older Notebook 2. Re-run Stage 33b.")
    if not np.array_equal(np.asarray(MASTER[col]), LAB[lv]):
        raise RuntimeError(f"pooled_master['{col}'] does not match pooled_labels_"
                           f"{'main7' if lv == 'main' else lv}. They are from "
                           f"different runs - re-run Notebook 2 Stage 33b.")
for c in ("has_main", "has_sub"):
    if c not in MASTER.columns:
        raise RuntimeError(f"pooled_master has no {c} column. This notebook needs it "
                           f"to keep unthemed responses out of every test. Re-run "
                           f"Notebook 2 Stage 33b.")
if {"mid_id", "mid_name"} & set(MASTER.columns):
    raise RuntimeError("pooled_master still carries mid columns - it was built before "
                       "the mid level was removed. Re-run Notebook 2 Stage 33b.")

# ------------------------------------------------------------- THE NOISE POLICY
# Every later cell uses these. Do not filter by hand: a hand-written
# `df[df.sub_id >= 0]` in one cell and a forgotten one in the next is exactly how
# a phantom theme gets into half the figures and not the other half.
NOISE   = {lv: int((LAB[lv] < 0).sum()) for lv in LEVELS}
N_THEMED = {lv: int((LAB[lv] >= 0).sum()) for lv in LEVELS}

def themed(level=None):
    """Boolean mask over pooled_master: rows that HAVE a theme at this level."""
    lv = level or CLAIM_LEVEL
    return np.asarray(MASTER[f"has_{lv}"], dtype=bool)

def theme_rows(level=None):
    """pooled_master restricted to rows with a theme. USE THIS FOR EVERY TEST."""
    return MASTER.loc[themed(level)].copy()

def theme_name(level, i):
    """Display name for a theme id; -1 always renders as the same NO_THEME text."""
    i = int(i)
    return NO_THEME if i < 0 else NAME[level].get(i, f"{level[0].upper()}{i}")

def denom(level=None):
    """The denominator for a prevalence: EVERY response, themed or not.
    Stated as a function so no cell has to decide, and so the choice is visible."""
    return N_ALL

def assert_no_noise(ids, where=""):
    """Guard for anything about to be tested. Raises rather than quietly counting
    a phantom theme - a check that never fires is not a check."""
    a = np.asarray(ids)
    if a.size and (a < 0).any():
        raise RuntimeError(
            f"[{where or 'test'}] {int((a < 0).sum())} row(s) with theme id -1 reached a "
            f"statistical test. Responses with no theme are not a theme. Use "
            f"theme_rows(level) to select rows before testing.")

# ------------------------------------- legacy names the older figure cells expect
# Set directly, not through a patched pload. The macro layer is MAIN and the
# detail layer is SUB - what the old code called "sub" while actually feeding it
# mid is now simply the sub level, and says so.
origin_pool   = np.asarray(pload("pooled_origin")); pooled_origin = origin_pool
pooled_labels     = LAB["main"]        # macro layer
pooled_sub_labels = LAB["sub"]         # detail layer
pooled_umap2d = np.asarray(pload("pooled_umap2d"))
pooled_umap3d = np.asarray(pload("pooled_umap3d"))
_u5 = pload("pooled_umap_nd")
pooled_umap5d = np.asarray(_u5 if _u5 is not None else pload("pooled_umap5d"))
pooled_sub_of_macro = PARENT
K_POOLED = K["main"]
df = get_df()

QL_ALL  = list(CFG.short_labels)
QL_POOL = [q for q in QL_ALL if (np.asarray(MASTER["question"]) == q).any()]

HIER = pload("pooled_hdbscan_hierarchy") or {}
STAB = pload("pooled_stability") or {}

# The held-out machinery is gone, deliberately. An earlier design kept the
# One-Word item and very short answers OUT of the thematic fitting, so this cell
# carried a POOL_POLICY, a Q_HELD list and a Q_HELD reporting path. All five
# questions are now pooled and nothing is held out, so that bookkeeping described
# a division that does not exist.
#
# WHAT THIS CHECK USED TO BE, AND WHY IT HAD TO CHANGE
#   It used to read:
#       if pooled_projection is not empty: raise "Notebook 1 DID hold responses out"
#   That was correct when pooled_projection could only mean held-out rows. It is
#   now WRONG and would stop this notebook before it drew anything: Notebook 2
#   Stage 23d writes a NON-EMPTY pooled_projection on purpose - it is the
#   backward-compatibility view of the rows the recovery assigned. A guard that
#   fires on the pipeline working as designed is worse than no guard.
#
#   The proposition worth checking is the one that would actually hurt: that
#   pooled_projection and pooled_master disagree about who was assigned. That is
#   checked instead, once the master table is loaded, further down.
_PROJ = pload("pooled_projection")

# =============================================================================
# THE COVERAGE CONTRACT   (the core view, the wide view, and which one a figure
#                          is allowed to default to)
# -----------------------------------------------------------------------------
# WHERE THIS USED TO LIVE, AND WHY IT MOVED HERE
#   COVERAGE / _cov_cols / _cov_note used to be defined inside the recovery cell
#   in Section 2 - the same cell that COMPUTED the recovery. Ten cells below call
#   them (29, 45, 49, 51, 56, 63, 65, 69, 73, 80), so the notebook could not run
#   without that cell, and that cell could not be replaced without breaking all
#   ten at once with NameError.
#
#   Notebook 2 now decides membership itself, in Stage 23d (three calibrated
#   rules, the winner measured) and Stage 23M (clause-level multi-label). The
#   computation is therefore gone from this notebook entirely and only the
#   CONTRACT remains - which belongs here, in the preflight, beside every other
#   thing the notebook agrees on before it draws anything.
#
# THE TWO VIEWS, BOTH ALWAYS PRESENT
#   CORE   has_<lv>      / <lv>_id        the responses HDBSCAN itself placed
#   WIDE   has_<lv>_wide / <lv>_id_wide   those PLUS the ones Notebook 2
#                                         recovered, at a measured accuracy
#   COVERAGE picks which one the claim cells use. _cov_note() always prints the
#   other number beside it, so the conservative figure is a footnote and never a
#   deletion. Set COVERAGE = "core" here to put the whole notebook back on the
#   clustering-only view in one edit.
#
# IF THE WIDE COLUMNS ARE MISSING THIS STOPS THE RUN. It does not quietly fall
# back to core-only: that would silently drop every recovered response from every
# figure while the notebook still printed a full-looking report - the exact class
# of failure the rest of this preflight exists to prevent.
# =============================================================================
_TIERS_OK = {"core", "assigned", "assigned-multi", "unassigned"}

# -----------------------------------------------------------------------------
# RESYNC pooled_master's membership columns FROM pooled_membership
#
#   pooled_membership is the AUTHORITY - Notebook 2 Stage 23d writes it and Stage
#   23M updates it. pooled_master's theme_tier / *_id_wide / … are a COPY that
#   Stage 33b makes. When the copy and the authority disagree, the copy is stale
#   by definition, and refreshing it here is not a decision - it is the same
#   copy Stage 33b performs, repeated against the current authority.
#
#   THIS IS NOT HYPOTHETICAL. The previous version of this notebook computed its
#   own recovery in Section 2 and wrote its own theme_tier onto pooled_master.
#   That copy is still on disk (and in the Drive mirror) from the last run, and
#   it does not know about Stage 23M's clause-level multi-label - so it reports
#   ten responses as "unassigned" that Notebook 2 has placed. Without this block
#   the notebook would either use those stale tiers silently, or stop and send
#   you back to Notebook 2 for a table it can rebuild itself in one line.
#
#   The refresh only happens when the two are provably about the SAME rows: same
#   length, row_id 0..N-1 in order, and an identical primary partition. Anything
#   else is a real mismatch and still raises.
# -----------------------------------------------------------------------------
_MB = pload("pooled_membership")
if isinstance(_MB, pd.DataFrame) and len(_MB):
    if "source" in _MB.columns:
        _MB = _MB[_MB["source"] == "pooled"]
    _MB = _MB.sort_values("row_id").reset_index(drop=True)
    if len(_MB) != N_ALL:
        raise RuntimeError(
            f"pooled_membership has {len(_MB)} pooled rows but pooled_master has "
            f"{N_ALL}. They are from different runs - re-run Notebook 2 Stage 23d "
            f"through Stage 33b.")
    if not np.array_equal(np.asarray(_MB["row_id"]), np.arange(N_ALL)):
        raise RuntimeError("pooled_membership row_ids are not 0..N-1 in order - "
                           "re-run Notebook 2 Stage 23d.")
    if not np.array_equal(np.asarray(_MB["sub_id"]), LAB["sub"]):
        raise RuntimeError(
            "pooled_membership and pooled_labels_sub describe different "
            "partitions. One of them is stale - re-run Notebook 2 Stage 23H, then "
            "Stage 23d.")

    _sw = np.asarray(_MB["sub_wide"], dtype=int)
    _mw = np.asarray(_MB["main_wide"], dtype=int)
    _fresh = {
        "theme_tier":        _MB["tier"].astype(str).values,
        f"{CLAIM_LEVEL}_assigned":  np.asarray(_MB["assigned_sub"], dtype=int),
        f"{CLAIM_LEVEL}_conf":      pd.to_numeric(_MB["confidence"], errors="coerce").values,
        f"{CLAIM_LEVEL}_multi":     (_MB["multi_subs"].astype(str).values
                                     if "multi_subs" in _MB.columns
                                     else np.array([""] * N_ALL)),
        "assignment_rule":   _MB["rule"].astype(str).values,
        "unassigned_reason": _MB["unassigned_reason"].astype(str).values,
        "is_substantive":    np.asarray(_MB["is_substantive"], dtype=bool),
        f"has_{CLAIM_LEVEL}_wide":  _sw >= 0,
        f"{CLAIM_LEVEL}_id_wide":   _sw,
        f"{CLAIM_LEVEL}_name_wide": [theme_name(CLAIM_LEVEL, c) for c in _sw],
        "has_main_wide":     _mw >= 0,
        "main_id_wide":      _mw,
        "main_name_wide":    [theme_name("main", c) for c in _mw],
    }
    _fresh["is_core"] = _fresh["theme_tier"] == "core"

    _added, _differs = [], []
    for _c, _v in _fresh.items():
        if _c not in MASTER.columns:
            _added.append(_c)
        elif not np.array_equal(np.asarray(MASTER[_c]).astype(str),
                                np.asarray(_v).astype(str)):
            _differs.append(_c)
    if _added or _differs:
        _old_tier = (MASTER["theme_tier"].astype(str).value_counts().to_dict()
                     if "theme_tier" in MASTER.columns else {})
        for _c, _v in _fresh.items():
            MASTER[_c] = _v
        psave("pooled_master", MASTER)
        MASTER.to_csv(f"{POOLED_DIR}/pooled_master.csv", index=False)
        print("  membership columns REFRESHED from pooled_membership "
              "(the authority Notebook 2 writes):")
        if _added:
            print(f"    added   : {', '.join(sorted(_added))}")
        if _differs:
            print(f"    replaced: {', '.join(sorted(_differs))}")
        if _old_tier:
            _new_tier = MASTER["theme_tier"].astype(str).value_counts().to_dict()
            print(f"    tiers on disk were {_old_tier}")
            print(f"    tiers now          {_new_tier}")
            print("    The copy on disk was written by the OLD recovery cell in this")
            print("    notebook, which did not know about Stage 23M's clause-level")
            print("    multi-label. pooled_master has been rewritten from Notebook 2's")
            print("    answer; nothing was recomputed here.")
    else:
        print("  membership columns match pooled_membership - nothing to refresh.")
else:
    _wide_missing = [c for lv in LEVELS
                     for c in (f"{lv}_id_wide", f"{lv}_name_wide", f"has_{lv}_wide")
                     if c not in MASTER.columns]
    if "theme_tier" not in MASTER.columns:
        _wide_missing.insert(0, "theme_tier")
    if _wide_missing:
        raise RuntimeError(
            "pooled_membership is absent AND pooled_master is missing the "
            "membership columns this notebook reads: " + ", ".join(_wide_missing)
            + ".\nFIX: re-run Notebook 2 from Stage 23d (final membership) through "
            "Stage 23M (clause multi-label) and Stage 33b (pooled_master), which "
            "write them. Until then every figure here would silently describe only "
            "the responses HDBSCAN placed and omit the ones Notebook 2 recovered, "
            "with nothing on the page saying so.")
    print("  (pooled_membership absent; using the membership columns already on "
          "pooled_master, which cannot be re-verified against their source)")

_bad_tier = sorted(set(MASTER["theme_tier"].astype(str)) - _TIERS_OK)
if _bad_tier:
    raise RuntimeError(f"pooled_master['theme_tier'] contains {_bad_tier}; this "
                       f"notebook knows {sorted(_TIERS_OK)}. Re-run NB2 Stage 33b.")

# the recovery's own measured accuracy, from the stage that measured it
_PVAL = pload("pooled_projection_validation") or {}
RECOVERY_RULE, RECOVERY_ACC = "", None
if isinstance(_PVAL, dict) and _PVAL:
    RECOVERY_RULE = str(next(iter(_PVAL)))
    _r0 = _PVAL.get(RECOVERY_RULE) or {}
    if isinstance(_r0, dict) and isinstance(_r0.get("accuracy"), (int, float)):
        RECOVERY_ACC = float(_r0["accuracy"])

# pooled_projection is now the compatibility view of the ASSIGNED rows. The
# proposition worth checking is that it agrees with pooled_master about who they
# are - not that it is empty.
_n_asg = int((MASTER["theme_tier"] == "assigned").sum())
if _PROJ is not None and len(_PROJ) and _n_asg and len(_PROJ) != _n_asg:
    raise RuntimeError(
        f"pooled_projection holds {len(_PROJ)} rows but pooled_master marks "
        f"{_n_asg} responses 'assigned'. They are from different runs - re-run "
        f"Notebook 2 Stage 23d, then Stage 33b.")

COVERAGE = "wide"          # "wide" = core + recovered (default) | "core"


def _cov_cols(lv, frame=None):
    """(id, name, has) column names for the current coverage mode.

    Falls back to the core columns when the wide ones are absent from THIS
    frame, so a cell holding a subset or an older copy still works instead of
    raising. The preflight above has already guaranteed the wide columns exist
    on pooled_master itself."""
    f = MASTER if frame is None else frame
    wide = (COVERAGE == "wide") and (f"has_{lv}_wide" in f.columns)
    idc = f"{lv}_id_wide" if wide else f"{lv}_id"
    nmc = (f"{lv}_name_wide" if (wide and f"{lv}_name_wide" in f.columns)
           else f"{lv}_name")
    return idc, nmc, (f"has_{lv}_wide" if wide else f"has_{lv}")


def _cov_note(lv, frame=None):
    """One line stating what this figure counts, and what the core-only view
    would have counted. Printed by every claim cell that uses the wide view."""
    f = MASTER if frame is None else frame
    n_core = int(f[f"has_{lv}"].astype(bool).sum()) if f"has_{lv}" in f.columns else 0
    wcol = f"has_{lv}_wide"
    n_wide = int(f[wcol].astype(bool).sum()) if wcol in f.columns else n_core
    if COVERAGE != "wide" or n_wide <= n_core:
        return f"  coverage: CORE only - {n_core} responses placed by the clustering."
    bits = []
    if "theme_tier" in f.columns:
        t = f["theme_tier"].astype(str)
        _a = int((t == "assigned").sum())
        _m = int((t == "assigned-multi").sum())
        if _a:
            bits.append(f"{_a} recovered")
        if _m:
            bits.append(f"{_m} clause-multi-labelled")
    else:
        bits.append(f"{n_wide - n_core} recovered")
    tail = (f", rule '{RECOVERY_RULE}' {RECOVERY_ACC:.0%} accurate on the hard "
            f"calibration set" if RECOVERY_ACC is not None else "")
    return (f"  coverage: WIDE - {n_wide} responses (core {n_core} + "
            + " + ".join(bits) + f"{tail}). Core-only would be {n_core}.")


# ----------------------------------------------------------------------- report
print("=" * 74)
print(f"  NB3 PREFLIGHT OK   -   {N_ALL} pooled responses")
print("=" * 74)
print(f"  hierarchy : main={K['main']}  sub={K['sub']}     (two levels; the mid level "
      f"was removed in NB2 Stage 23H)")
print(f"  claims    : {CLAIM_LEVEL.upper()} only. MAIN is grouping / colour / narrative,")
print(f"              never a significance test - its stability is in the 'pattern'")
print(f"              band, not the 'finding' band.")
for lv in LEVELS:
    _s = (STAB.get(lv) or {})
    _extra = ""
    if _s:
        _extra = (f"   stability {_s.get('ari_mean', float('nan')):.3f}"
                  f"  silhouette {_s.get('silhouette_overall', float('nan')):.3f}")
    print(f"  {lv:<5} K={K[lv]:<3} themed {N_THEMED[lv]:<5} "
          f"no theme {NOISE[lv]:<5} ({NOISE[lv] / N_ALL:.1%}){_extra}")
print(f"  noise policy: excluded from every test (theme_rows / assert_no_noise);")
print(f"                shown as '{NO_THEME}' in descriptive composition figures;")
print(f"                prevalence denominator = ALL {N_ALL} responses, so the")
print(f"                percentages do not sum to 100 and the gap IS the noise.")
print(f"  questions pooled ({len(QL_POOL)}): {', '.join(QL_POOL)}")
if len(QL_POOL) != len(QL_ALL):
    print(f"  NOT pooled: {', '.join(q for q in QL_ALL if q not in QL_POOL)}")
print("  every question is in one pool; nothing is held out, so no figure needs a")
print("  separate category for held-out responses.")
_tc = MASTER["theme_tier"].astype(str).value_counts()
print("  membership (decided in NB2 Stage 23d / 23M, read here - never recomputed):")
for _t in ("core", "assigned", "assigned-multi", "unassigned"):
    if _t in _tc:
        print(f"      {_t:<16}{int(_tc[_t]):>5}  ({_tc[_t] / N_ALL:>5.1%})")
_nw = int(MASTER[f"has_{CLAIM_LEVEL}_wide"].astype(bool).sum())
print(f"  coverage  : core {N_THEMED[CLAIM_LEVEL]} -> wide {_nw} "
      f"({_nw / N_ALL:.1%});  COVERAGE = '{COVERAGE}'")
if RECOVERY_ACC is not None:
    print(f"              recovery rule '{RECOVERY_RULE}', measured accuracy "
          f"{RECOVERY_ACC:.1%} (an upper bound - see NB2 Stage 23d)")
print("  every claim cell prints _cov_note(), so the core-only number is always")
print("  visible beside the wide one.")
print("  visual-only: re-run any figure cell freely; nothing recomputes.")
print("=" * 74)

### Stage 08 · Participant-level theme matrix

**Question.** Which themes did each participant voice?

**Reads.** `pooled_master`. **Writes.** `pooled_hier_participant`.

**How to read it.** Several later analyses work on the participant or the team rather than the response, and all of them need this one table. A mention is counted only from a response that has a theme; every participant keeps a row, including one whose responses were all left unplaced, so team-level denominators stay honest.

In [ ]:
# @title
# ==============================================================================
# STAGE 08  -  PARTICIPANT-LEVEL THEME MATRIX
#             section 1 of 9  -  Setup, preflight and the visual system
# ==============================================================================
# ==============================================================================
# PARTICIPANT-LEVEL THEME MATRIX   (run once, straight after the preflight)
# ------------------------------------------------------------------------------
# Several analyses work on the participant rather than the response: team
# thematic diversity, the resolution comparison, the composite prize predictors.
# They all need the same table - one row per participant, one 0/1 column per
# theme - and they all used to read `pooled_hier_participant`, an artifact of the
# clustering stage Notebook 2 replaced. Nothing writes it any more, so each of
# those cells failed on `None.copy()`.
#
# It is rebuilt here, once, from pooled_master - the one canonical table - so the
# participant view and the response view can never drift apart.
#
# TWO RULES, both inherited from the preflight and applied here so no downstream
# cell has to remember them:
#   * a MENTION is counted only from a response that HAS a theme; the responses
#     HDBSCAN left unplaced are not a mention of anything;
#   * every participant keeps a row, including one whose responses were all
#     unplaced. Dropping them would shrink the denominator of every team-level
#     rate and inflate it - a participant with nothing clusterable is still a
#     participant, and their empty row is the honest record of that.
#
# Columns: Participant · Team · n_responses · n_unthemed_<level> ·
#          <level>_ment_<id> (0/1) · <level>_richness
# ==============================================================================
import numpy as np, pandas as pd

P = MASTER.drop_duplicates("Participant")[["Participant"]].copy()
if "Team" in MASTER.columns:
    P["Team"] = MASTER.drop_duplicates("Participant").set_index("Participant")["Team"] \
                      .reindex(P["Participant"]).astype(str).str.strip().values
P = P.set_index("Participant")
P["n_responses"] = MASTER.groupby("Participant").size().reindex(P.index).fillna(0).astype(int)

print("=" * 72)
print("  PARTICIPANT-LEVEL THEME MATRIX")
print("=" * 72)
print(f"  {len(P)} participants from {len(MASTER)} responses")

for lv in LEVELS:
    idc, hasc = f"{lv}_id", f"has_{lv}"
    Mv = MASTER[MASTER[hasc].astype(bool)]
    ment = (Mv.groupby(["Participant", idc]).size().unstack(fill_value=0) > 0).astype(int)
    ment = ment.reindex(index=P.index, columns=range(K[lv]), fill_value=0)
    ment.columns = [f"{lv}_ment_{c}" for c in ment.columns]
    P = P.join(ment)
    P[f"{lv}_richness"] = ment.sum(axis=1).astype(int)
    P[f"n_unthemed_{lv}"] = (MASTER[~MASTER[hasc].astype(bool)]
                             .groupby("Participant").size().reindex(P.index)
                             .fillna(0).astype(int))
    _voiced = int((P[f"{lv}_richness"] > 0).sum())
    print(f"  {lv:<5} {K[lv]:>3} theme columns · {_voiced} of {len(P)} participants voice "
          f"at least one ({_voiced / len(P):.0%})")
    print(f"        {len(P) - _voiced} participant(s) voice none at this level - kept as "
          f"all-zero rows, not dropped")

P = P.reset_index()
psave("pooled_hier_participant", P)
P.to_csv(f"{POOLED_DIR}/participant_theme_matrix.csv", index=False)

# the invariants every dependent cell assumes, checked once here
assert P["Participant"].is_unique, "duplicate participant rows"
for lv in LEVELS:
    _cols = [c for c in P.columns if c.startswith(f"{lv}_ment_")]
    assert len(_cols) == K[lv], f"{lv}: {len(_cols)} mention columns, expected {K[lv]}"
    assert not any(c.endswith("_-1") for c in _cols), "a mention column for id -1"
    assert P[_cols].to_numpy().max() <= 1, "mention matrix is not 0/1"

print(f"  -> pooled_hier_participant · participant_theme_matrix.csv "
      f"({P.shape[0]} rows x {P.shape[1]} cols)")
print("=" * 72)


### Stage 09 · Stale-artifact guard

**Question.** Do the cached results on disk belong to the clustering that is loaded right now?

**Reads.** The live labels; every result this notebook produces.

**How to read it.** The hierarchy is fingerprinted. If the fingerprint has changed, every cached result is set aside as `*.stale` and its stage recomputes. This exists because `pload` caches by name only: without it a figure can be rendered today from numbers computed against a partition that no longer exists.

In [ ]:
# @title
# ==============================================================================
# STAGE 09  -  STALE-ARTIFACT GUARD
#             section 1 of 9  -  Setup, preflight and the visual system
# ==============================================================================
# ==============================================================================
# STALE-ARTIFACT GUARD   (run right after the preflight, before any figure)
# ------------------------------------------------------------------------------
# WHY THIS EXISTS
#   psave/pload cache by NAME only - pload("x") returns whatever x.pkl holds, with
#   no record of what produced it. A stage written "load, recompute only if None"
#   therefore NEVER recomputes once the file exists, even after the clustering
#   underneath it has changed. That is not hypothetical: in a previous run four
#   artifacts written when the hierarchy had 43 sub-themes were still feeding
#   figures rendered against a 54-sub-theme hierarchy, and the granularity figure
#   printed "Sub (43 themes)" on a page about 54.
#
# WHAT CHANGED, AND WHY THE OLD APPROACH WAS NOT ENOUGH
#   The old guard checked each artifact by heuristic: are the theme ids inside the
#   live range, do the theme names still match? Both can pass by accident. An id
#   column of 0..30 is "inside range" for K=39 even if it came from a completely
#   different partition, and names survive a re-clustering when the model happens
#   to reuse wording. Heuristics catch the obvious cases and quietly miss the rest.
#
#   This version leads with a FINGERPRINT instead. The hierarchy is hashed - both
#   levels' K and the label arrays themselves - and the hash is stored. If the
#   stored hash is absent or different, EVERY artifact this notebook produces is
#   set aside, because by definition it was computed against a different
#   partition. No judgement, no per-artifact rule to get wrong. The heuristic
#   checks are kept as a second pass, for the case where the hierarchy is
#   unchanged but one artifact is old for some other reason.
#
#   Nothing is deleted. Files are renamed to *.stale, so the old numbers remain
#   inspectable. Inputs from Notebook 1/2 are NEVER touched - only results this
#   notebook knows how to rebuild.
# ==============================================================================
import os, re, glob, hashlib, pickle
import numpy as np, pandas as pd

# every artifact THIS notebook produces (a stage here can rebuild it)
NB3_PRODUCES = [
    "pooled_measurement_validity", "pooled_theme_evidence", "pooled_question_theme",
    "pooled_codebook_alignment", "pooled_atlas_tiered", "pooled_oneword_lexicon",
    "pooled_bridge", "pooled_group_theme", "pooled_theme_synergy",
    "pooled_figA_pp_main", "pooled_figA_pp_sub", "pooled_granularity_auc",
    "pooled_prize_mention", "pooled_team_diversity", "pooled_multilevel",
    "pooled_org_track", "pooled_team_archetype", "pooled_capstone",
    "pooled_prize_secondary", "pooled_sentiment_model",
    # names from the three-level era; they cannot be rebuilt and must not be read
    "pooled_figA_pp_mid", "pooled_org_section", "pooled_org_region",
]
STAMP = "pooled_nb3_hierarchy_stamp"

print("=" * 74)
print("  STALE-ARTIFACT GUARD")
print("=" * 74)

# ------------------------------------------------- 1. fingerprint the hierarchy
_h = hashlib.md5()
for lv in ("main", "sub"):
    _h.update(np.ascontiguousarray(LAB[lv].astype(np.int32)).tobytes())
FP = f"main{K['main']}-sub{K['sub']}-{_h.hexdigest()[:12]}"
_prev = pload(STAMP)
print(f"  live hierarchy : main={K['main']}  sub={K['sub']}")
print(f"  fingerprint    : {FP}")
print(f"  last seen here : {_prev or '(never - first run against this hierarchy)'}")


def _set_aside(art, why):
    """Set an artifact aside in EVERY store pload can read from.

    THE HOLE THIS CLOSES. pload() looks in POOLED_DIR first and, failing that,
    copies the file out of the Drive MIRROR and returns it. The previous version
    of this function renamed only the LOCAL copy. On a fresh Colab runtime the
    local directory is empty and every artifact lives in the mirror, so the guard
    found nothing to set aside, printed "nothing cached yet - clean slate", and
    then every later pload() happily restored the stale object from Drive. That
    is exactly what happened on the last run: the guard reported a clean slate
    while pooled_granularity_auc (mid-32 / sub-43) and pooled_oneword_lexicon
    (an orphan of a deleted cell) were still readable from the mirror.
    """
    hit = False
    for base in [POOLED_DIR] + ([MIRROR_POOLED_DIR] if MIRROR_POOLED_DIR else []):
        src = os.path.join(base, art + ".pkl")
        if not os.path.exists(src):
            continue
        dst = os.path.join(base, art + ".stale")
        try:
            if os.path.exists(dst):
                os.remove(dst)
            os.replace(src, dst)
            hit = True
        except Exception as e:
            print(f"  STALE  {art:<28} {why}  (could not set aside in "
                  f"{'mirror' if base != POOLED_DIR else 'local'}: {type(e).__name__})")
    if hit:
        print(f"  STALE  {art:<28} {why}")
    return hit


# ---- the same hole, for the heuristic pass and for every future read --------
def _artifact_exists(art):
    """True if pload(art) could return something, from EITHER store."""
    if os.path.exists(os.path.join(POOLED_DIR, art + ".pkl")):
        return True
    return bool(MIRROR_POOLED_DIR) and os.path.exists(
        os.path.join(MIRROR_POOLED_DIR, art + ".pkl"))


_moved, _kept = [], []
if _prev != FP:
    print("\n  The hierarchy is not the one these cached results were computed against.")
    print("  Every result this notebook produces is therefore set aside; each stage")
    print("  recomputes on this run. Notebook 1/2 inputs are untouched.")
    for art in NB3_PRODUCES:
        if _set_aside(art, "computed against a different hierarchy"):
            _moved.append(art)
    # CSVs carry the same numbers into the submission bundle, so they go too
    for art in _moved:
        for p in glob.glob(os.path.join(POOLED_DIR, art.replace("pooled_", "") + "*.csv")):
            try:
                os.replace(p, p + ".stale")
            except Exception:
                pass
    if not _moved:
        print("  (nothing cached yet - clean slate)")
else:
    print("\n  Fingerprint matches: the cached results belong to this hierarchy.")

# ------------------------- 2. second pass: heuristic checks on whatever survived
# Only meaningful when the fingerprint matched; kept because an artifact can be
# old for reasons the fingerprint cannot see (a stage edited, a partial run).
_NAMES = {lv: {str(v).strip().lower() for v in NAME[lv].values() if str(v).strip()}
          for lv in LEVELS}
_IDCOL = re.compile(r"^(cluster|theme|unit|id|sub|main|sub_id|main_id|A|B)$", re.I)
_NMCOL = re.compile(r"^(name|theme_name|label|sub_name|main_name)$", re.I)


def _frame(o):
    if isinstance(o, pd.DataFrame):
        return o
    if isinstance(o, dict):
        for k in ("theme_test", "table", "rows", "df", "data"):
            v = o.get(k)
            if isinstance(v, pd.DataFrame):
                return v
            if isinstance(v, list) and v and isinstance(v[0], dict):
                return pd.DataFrame(v)
    if isinstance(o, list) and o and isinstance(o[0], dict):
        return pd.DataFrame(o)
    return None


_flag = []
for art in NB3_PRODUCES:
    if art in _moved:
        continue
    if not _artifact_exists(art):
        continue
    try:
        obj = pload(art)          # honours local-then-mirror, like every reader
        if obj is None:
            continue
    except Exception as e:
        _flag.append((art, f"unreadable ({type(e).__name__})")); continue
    F = _frame(obj)
    why = None
    if F is not None and len(F):
        # a level column still naming the removed mid level
        for c in F.columns:
            if str(c).lower() in ("level", "lv") and \
               F[c].astype(str).str.lower().eq("mid").any():
                why = "contains rows for the removed 'mid' level"; break
        # a theme id beyond the live range at whichever level the column names
        if why is None:
            for c in F.columns:
                if _IDCOL.match(str(c)) and pd.api.types.is_numeric_dtype(F[c]) and len(F[c].dropna()):
                    lv = "main" if str(c).lower().startswith("main") else CLAIM_LEVEL
                    if float(F[c].max()) >= K[lv]:
                        why = (f"{c} reaches {int(F[c].max())}, the live {lv} max id "
                               f"is {K[lv] - 1}"); break
        # theme names that mostly do not exist any more
        if why is None:
            for c in F.columns:
                if not _NMCOL.match(str(c)):
                    continue
                raw = [str(x).strip() for x in F[c] if str(x).strip()]
                parts = [p.strip() for g in raw for p in re.split(r"\s+x\s+", g) if p.strip()]
                if not parts:
                    continue
                live = _NAMES[CLAIM_LEVEL] | _NAMES["main"]
                def _m(p):
                    pl = p.lower()
                    if pl in live:
                        return True
                    return any(len(pl) >= 12 and (n.startswith(pl) or pl.startswith(n))
                               for n in live)
                hits = sum(1 for p in parts if _m(p))
                if hits < 0.5 * len(parts):
                    why = (f"only {hits} of {len(parts)} theme name(s) match the live "
                           f"labels"); break
        if why is None:
            _kept.append(art)
    if why:
        _set_aside(art, why)
        _flag.append((art, why))

# =============================================================================
# INFRA-1  ·  PROVENANCE-STAMPED RESULTS
# -----------------------------------------------------------------------------
# psave/pload key on NAME. Two different partitions therefore write the same
# filename, and whichever ran last wins - silently, and across the Drive mirror
# too. The fingerprint above already identifies the partition; stamping every
# RESULT with it makes a stale read impossible instead of merely detectable:
#
#     psave_result(name, obj)   stores {"_stamp": FP, "_written": …, "data": obj}
#     pload_result(name)        returns the data ONLY if the stamp matches,
#                               otherwise None plus a printed reason
#
# Two properties matter:
#   * it is LOCATION-INDEPENDENT - a stale object restored from the mirror is
#     rejected exactly as a stale local one is, which the rename-based guard
#     above cannot do on a fresh runtime;
#   * an ORPHAN (an artifact no surviving cell writes, e.g. the one-word lexicon
#     of a deleted stage) can never acquire a stamp, so it can never be read.
#     A cell that depends on one now says so instead of printing its numbers.
#
# INPUTS from Notebook 1/2 are never stamped and are read with plain pload():
# they are not this notebook's results and must not be invalidated by it.
# =============================================================================
def psave_result(name, obj):
    """Write a RESULT of this notebook, stamped with the live hierarchy."""
    psave(name, {"_stamp": FP, "_written": pd.Timestamp.now().isoformat(timespec="seconds"),
                 "data": obj})
    return obj


def pload_result(name, *, quiet=False):
    """Read a RESULT written by THIS hierarchy, else None.

    Unstamped (pre-INFRA-1) objects are treated as stale: they were written
    before provenance existed, so their origin cannot be established.
    """
    raw = pload(name)
    if raw is None:
        return None
    if not (isinstance(raw, dict) and "_stamp" in raw and "data" in raw):
        if not quiet:
            print(f"  [provenance] {name}: unstamped (written before INFRA-1) "
                  f"- recomputing")
        return None
    if raw["_stamp"] != FP:
        if not quiet:
            print(f"  [provenance] {name}: stamped {raw['_stamp']}, live is {FP} "
                  f"- recomputing")
        return None
    return raw["data"]


def result_provenance(name):
    """(stamp, written_at) for a stamped artifact, or (None, None)."""
    raw = pload(name)
    if isinstance(raw, dict) and "_stamp" in raw:
        return raw.get("_stamp"), raw.get("_written")
    return None, None


# Orphans: artifacts NB3_PRODUCES no longer contains a producer for. They cannot
# be rebuilt, so they must never be read again - set them aside in both stores.
_ORPHANS = ["pooled_oneword_lexicon", "pooled_bridge",
            "pooled_figA_pp_mid", "pooled_org_section", "pooled_org_region"]
_orph = [a for a in _ORPHANS if _artifact_exists(a)]
if _orph:
    print()
    print("  ORPHANED ARTIFACTS (no cell in this notebook can rebuild them):")
    for a in _orph:
        _set_aside(a, "orphan - the stage that wrote it was removed")
    print("  Any section that depended on one will now say so instead of")
    print("  printing numbers from an analysis this notebook no longer runs.")

psave(STAMP, FP)
psave("pooled_stale_report", dict(fingerprint=FP, previous=_prev, moved=_moved,
                                  flagged=_flag, kept=_kept,
                                  K={lv: K[lv] for lv in LEVELS}))
print()
if _moved or _flag:
    print(f"  {len(_moved) + len(_flag)} artifact(s) set aside; their stages recompute "
          f"this run.")
    print(f"  The old files are still on disk as *.stale if you need to inspect them.")
if _kept:
    print(f"  current and reused: {', '.join(sorted(_kept))}")
if not _moved and not _flag and not _kept:
    print("  nothing cached yet - every stage will compute fresh.")
print(f"  stamp written: {STAMP} = {FP}")
print(f"  provenance helpers ready: psave_result / pload_result / result_provenance")
print(f"  (results are now keyed on content, so a stale copy in the Drive mirror")
print(f"   is rejected the same way a stale local one is)")
print("=" * 74)

### Stage 10 · Visual system

**Question.** What does each theme look like, everywhere in the report?

**How to read it.** One palette, built from the live hierarchy: each main theme has its own hue, each sub-theme takes its parent's hue at a distinct lightness, and responses with no theme take one reserved grey that is never a theme colour. Sentiment is always the same diverging ramp and is never placed on an axis.

In [ ]:
# @title
# ==============================================================================
# STAGE 10  -  GLOBAL VISUAL SYSTEM
#             section 1 of 9  -  Setup, preflight and the visual system
# ==============================================================================
# REBUILT. The previous version of this cell defined a design system and then
# immediately overrode half of it with a second "appended by audit" block, which
# left the notebook in a self-inconsistent state:
#   * SENT_POS / SENT_NEG kept their ORIGINAL values while SENT_CMAP was rebuilt
#     from a DIFFERENT ramp, so team_two_panel drew its tone squares from one
#     palette and its dots from another - in the same figure.
#   * save_fig() forced EVERY matplotlib figure to 12 x 6.75 in with no
#     bbox_inches, so the 15 different figsizes declared across this notebook
#     (from 8x4.4 up to ~20.6x11.8) were squashed or stretched onto one canvas.
#     Tall figures (team fingerprints, 43-row forests, the org grid) had their
#     row pitch crushed until the labels collided; small ones were blown up.
#   * The font was fetched from GitHub at runtime into /tmp (fails offline and on
#     Windows), and was then silently overridden anyway by eight separate
#     _pick_fontNN() helpers scattered through the figure cells.
# This rebuild keeps ONE palette (the agreed design system), ONE font resolved
# once from what is actually installed, and a save_fig that RESPECTS the size a
# figure was designed at. The exact-1920x1080 deliverable is still produced, but
# by LETTERBOXING (padding to canvas) rather than distorting - same trick NB2
# already uses, so a tall figure stays tall and readable.
# ==============================================================================
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.lines import Line2D

# ------------------------------------------------------------------ FONT (once)
# Resolved from installed fonts only: no network fetch, no /tmp write, no
# per-cell re-picking. Every figure cell inherits this and must NOT reset it.
def _resolve_font():
    have = {f.name for f in fm.fontManager.ttflist}
    for c in ("Inter", "Lato", "Source Sans Pro", "Open Sans", "Roboto",
              "Helvetica Neue", "Arial", "Liberation Sans", "DejaVu Sans"):
        if c in have:
            return c
    return "sans-serif"
FONT = _resolve_font()

# --------------------------------------------------------------- COLOUR SYSTEM
# Two matched sets (per the agreed design system). The SAME hue index means the
# SAME theme in both, so a reader moving from the paper to an interactive
# explorer recognises the theme in a livelier tone.
#   REFINED : print / journal. Okabe-Ito based -> colourblind-safe.
#   VIBRANT : interactive (Plotly) only.
MACRO7 = ["#E69F00", "#56B4E9", "#009E73", "#C9A227",
          "#0072B2", "#D55E00", "#CC79A7"]          # refined, print
MACRO7_VIB = ["#FF8C42", "#33BDEB", "#17C08B", "#F4C025",
              "#4C6FFF", "#FF5D5D", "#B072F0"]      # vibrant, interactive

# Extension hues for levels finer than 7. Kept perceptually distinct
# from MACRO7 and from each other; no yellow-on-white.
_EXT = MACRO7 + [
    "#6A1B2A", "#146152", "#7A5195", "#2F6E8F", "#B23A1E", "#4C6B8A",
    "#9C5A3C", "#3A7D44", "#5DCAA5", "#A6789B", "#8FB3C9", "#E08B7A",
    "#93A1AD", "#C39A6B", "#7FB39B", "#8E4A9E", "#C1121F", "#0F6E56",
]

def _is_yellowish(hx):
    """Yellow on white is unreadable; keep it out of generated hues."""
    import colorsys, matplotlib.colors as mcolors
    r, g, b = mcolors.to_rgb(hx)
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    return 0.11 <= h <= 0.20 and s > 0.35 and v > 0.65

def _extend_palette(need):
    """Grow the categorical pool to `need` DISTINCT colours.

    The hand-written list only holds ~25 hues. The sub level routinely has 40-60
    clusters, so the old `base[i % len(base)]` silently wrapped and gave two
    different sub-themes the SAME colour - indistinguishable in a scatter or a
    legend. We top up with Glasbey (max-perceptual-distance) if colorcet is
    available, else with a golden-ratio hue walk at alternating lightness, and
    de-duplicate. Yellow is excluded either way.
    """
    import colorsys, matplotlib.colors as mcolors
    pool = list(_EXT)
    try:
        import colorcet as cc
        for c in cc.glasbey_category10:
            hx = mcolors.to_hex(c)
            if hx not in pool and not _is_yellowish(hx):
                pool.append(hx)
    except Exception:
        pass
    phi = 0.6180339887
    i, h = 0, 0.11
    while len(pool) < need and i < 4000:
        h = (h + phi) % 1.0
        s = 0.62 + 0.22 * ((i // 2) % 2)
        v = 0.86 - 0.24 * (i % 3) / 2.0
        hx = mcolors.to_hex(colorsys.hsv_to_rgb(h, s, v))
        if hx not in pool and not _is_yellowish(hx):
            pool.append(hx)
        i += 1
    return pool

def get_cluster_colors(k):
    """Categorical colours for k clusters, all DISTINCT.

    k <= 7 returns the 7 macro hues exactly, so a macro theme keeps one colour
    across every figure in the report. Beyond that the pool is extended rather
    than wrapped (see _extend_palette).
    """
    k = int(k)
    if k <= len(MACRO7):
        return MACRO7[:k]
    pool = _EXT if k <= len(_EXT) else _extend_palette(k)
    return pool[:k]

def macro_hue(macro_id, n_macro=None):
    """The ONE colour that belongs to a given main theme.

    This used to be `MACRO7[macro_id % 7]`. The main level had 7 themes when that
    was written; it now has 24, so ids 0, 7, 14 and 21 were being handed the same
    hue - four different main themes drawn in one colour, in every figure that
    groups by parent. The pool is extended instead of wrapped.
    """
    n = int(n_macro or globals().get("_N_MACRO", len(MACRO7)))
    pool = get_cluster_colors(max(n, int(macro_id) + 1))
    return pool[int(macro_id) % len(pool)]

def macro_shades(macro_id, n, n_macro=None):
    """n shades of one main theme's hue - for showing sub-themes under a parent."""
    import matplotlib.colors as mcolors
    base = np.array(mcolors.to_rgb(macro_hue(macro_id, n_macro)))
    return [mcolors.to_hex(tuple(base + (1 - base) * f))
            for f in np.linspace(0.45, 0.0, max(1, int(n)))]

# neutrals
BG   = "#FFFFFF"   # figure + axes background
INK  = "#14202B"   # primary text
MUT  = "#5B6670"   # secondary text / subtitles
FOOT = "#98A2AD"   # footer text
BAND = "#F4F5F7"   # alternating row band
ZERO = "#333333"   # zero reference line
GRID = "#EEF1F4"

# sentiment: diverging, negative -> neutral -> positive (design system ramp).
# These scalars and the colormap are now derived from ONE list, so a dot and a
# square that mean the same thing can never disagree again.
_SENT_RAMP  = ["#B84413", "#E08214", "#BFC4C9", "#5DCAA5", "#0F6E56"]
SENT_NEG, SENT_NEU, SENT_POS = _SENT_RAMP[0], _SENT_RAMP[2], _SENT_RAMP[-1]
SENT_CMAP = LinearSegmentedColormap.from_list("sent_div", _SENT_RAMP)

# magnitude: sequential, replaces viridis on heatmaps.
# NOTE: the design-system source listed the darkest stop as "#08514000" - an
# 8-digit hex, i.e. #085140 with alpha 00 (fully transparent). Rendered as-is the
# top of the ramp would have vanished into the background. Corrected to #085140.
_MAG_RAMP = ["#E1F5EE", "#9FE1CB", "#5DCAA5", "#1D9E75", "#0F6E56", "#085140"]
TEAL_SEQ = LinearSegmentedColormap.from_list("mag_seq", _MAG_RAMP)
GRAD = TEAL_SEQ  # back-compat alias for cells written against the NB2 name

def sent_norm(vmax=0.6):
    """Symmetric around 0 so the neutral colour always means near-zero tone."""
    vmax = max(0.15, float(vmax))
    return TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

# ----------------------------------------------------------------- TYPE SCALE
# Single scale (per the design system), referenced by name so a figure never
# hardcodes a size that drifts from the rest of the report.
TS = dict(title=17, subtitle=10.5, axis=11, tick=9, small=8, foot=8)

plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "font.family": FONT, "text.color": INK, "axes.labelcolor": "#3A4650",
    "axes.edgecolor": "#C9CFD6", "axes.linewidth": 0.8,
    "xtick.color": "#6B7680", "ytick.color": "#6B7680",
    "axes.titlesize": TS["axis"] + 1, "axes.labelsize": TS["axis"],
    "xtick.labelsize": TS["tick"], "ytick.labelsize": TS["tick"],
    "axes.grid": False, "axes.spines.top": False, "axes.spines.right": False,
    "figure.dpi": 110, "savefig.dpi": 300,
})

# one Plotly look, sharing the font and the VIBRANT colourway
try:
    import plotly.io as pio, plotly.graph_objects as go
    pio.templates["igem"] = go.layout.Template(layout=dict(
        font=dict(family=f"{FONT}, sans-serif", color=INK, size=13),
        paper_bgcolor=BG, plot_bgcolor=BG, colorway=MACRO7_VIB,
        title=dict(font=dict(size=20, color=INK)),
        xaxis=dict(gridcolor=GRID, zerolinecolor="#E0E4E8", linecolor="#D3D8DE"),
        yaxis=dict(gridcolor=GRID, zerolinecolor="#E0E4E8", linecolor="#D3D8DE"),
        legend=dict(bgcolor="rgba(255,255,255,0)", bordercolor="rgba(0,0,0,0)")))
    pio.templates.default = "igem"
except Exception:
    pass

# ------------------------------------------------------------------- EXPORTING
RENDER_INTERACTIVE = False   # static output only unless you flip this
SLIDE_W, SLIDE_H = 1920, 1080

def fig_height_for_rows(n_rows, per_row=0.26, base=3.0, lo=4.5, hi=26.0):
    """Height in inches that gives each row an honest slice of the canvas.
    Row-count-aware sizing is why labels stop colliding: a 43-row forest is NOT
    the same figure as a 7-row one, and must not be forced onto one canvas."""
    return float(np.clip(base + per_row * max(1, int(n_rows)), lo, hi))

def save_fig(fig, name, interactive=False, static=True, slide=True):
    """Save a figure at the size it was DESIGNED at (300 dpi PNG + vector SVG),
    then additionally emit an exact 1920x1080 slide PNG by LETTERBOXING.

    The old version called fig.set_size_inches(12, 6.75) on every figure and
    saved without bbox_inches, which distorted aspect ratios and clipped labels.
    Nothing here resizes the figure: `_slide.png` is padded, never stretched, so
    a tall heatmap stays tall and legible while still landing on a 16:9 canvas.
    """
    base = os.path.join(POOLED_DIR, name)
    wrote = []
    if hasattr(fig, "write_html"):                       # ---- Plotly
        try:
            fig.update_layout(paper_bgcolor=BG, plot_bgcolor=BG,
                              font=dict(family=f"{FONT}, sans-serif"))
        except Exception:
            pass
        if RENDER_INTERACTIVE and interactive:
            fig.write_html(base + ".html", include_plotlyjs="cdn", full_html=True)
            wrote.append("HTML")
        if static:
            try:
                fig.write_image(base + ".png", scale=2)
                fig.write_image(base + ".svg")
                wrote += ["PNG", "SVG"]
            except Exception as e:
                print(f"  (Plotly static export needs kaleido; skipped: {type(e).__name__})")
    else:                                                # ---- matplotlib
        fig.savefig(base + ".png", dpi=300, bbox_inches="tight", facecolor=BG)
        fig.savefig(base + ".svg", bbox_inches="tight", facecolor=BG)
        wrote += ["PNG", "SVG"]

    if slide and os.path.exists(base + ".png"):
        try:
            from PIL import Image
            im = Image.open(base + ".png").convert("RGB")
            r = min(SLIDE_W / im.width, SLIDE_H / im.height)
            nw, nh = max(1, int(im.width * r)), max(1, int(im.height * r))
            canvas = Image.new("RGB", (SLIDE_W, SLIDE_H), "white")
            canvas.paste(im.resize((nw, nh), Image.LANCZOS),
                         ((SLIDE_W - nw) // 2, (SLIDE_H - nh) // 2))
            canvas.save(base + "_slide.png")
            wrote.append("1920x1080")
        except Exception as e:
            print(f"  (slide export skipped: {type(e).__name__})")

    for ext in (".png", ".svg", ".html", "_slide.png"):
        p = base + ext
        if os.path.exists(p) and MIRROR_POOLED_DIR:
            _mirror_file(p, MIRROR_POOLED_DIR)
    try:
        from IPython.display import display as _display
        _display(fig)
    except Exception:
        pass
    if wrote:
        w, h = (fig.get_size_inches() if not hasattr(fig, "write_html") else (0, 0))
        dim = f"  [{w:.1f}x{h:.1f}in @300dpi]" if w else ""
        print(f"  saved figure: {os.path.basename(base)}  ({' + '.join(wrote)}){dim}")
    return base

# ------------------------------------------------------------- TITLE / LABELS
def modern_title(fig, title, subtitle="", accent=None, x=0.012, y=0.975):
    accent = accent or SENT_POS
    fig.text(x, y, title, ha="left", va="top",
             fontsize=TS["title"], fontweight="bold", color=INK)
    fig.add_artist(Line2D([x, x + 0.055], [y - 0.048, y - 0.048], color=accent,
                          lw=3.2, solid_capstyle="round", transform=fig.transFigure))
    if subtitle:
        fig.text(x, y - 0.078, subtitle, ha="left", va="top",
                 fontsize=TS["subtitle"], color=MUT)

# =============================================================================
# INFRA-3  ·  ONE LABEL RENDERER
# -----------------------------------------------------------------------------
# THE BUG THIS REPLACES. Three different truncation behaviours coexisted, and the
# worst of them was silent:
#
#   _ell()            cut at 42 characters and appended "…"        (visible)
#   _w2() in cells 43 and 47:
#       "\n".join(textwrap.wrap(t, w)[:2])
#                     wrapped, kept the first TWO lines, and DROPPED THE REST
#                     with no marker at all                        (invisible)
#
# The second produced axis labels that end mid-phrase and read as complete:
#       "NLD: S29 Collaboration Challenges and"      <- "Successes" is gone
#       "DEU: S15 Judging Session Preparation and"   <- "Challenges" is gone
#       "S27 [unlabelled S27] top terms: biology, background,"
# A reader has no way to know text is missing, which is worse than a short label.
#
# THE RULE HERE: text is FITTED to its box, and if it genuinely cannot fit, the
# reader is TOLD. Never silently dropped. Order of attack:
#   1. wrap on word boundaries to at most `max_lines`;
#   2. if it overflows, wrap wider (fewer, longer lines) before giving up;
#   3. only then truncate, and always with an explicit ellipsis.
#
# It also normalises two things that differed between figures:
#   * the theme id prefix - "S35" everywhere, never "s35"                (F9)
#   * the placeholder for a withheld label - one wording, not two        (F8)
# =============================================================================
import textwrap as _tw
import re as _re

_UNLABELLED = _re.compile(
    r"^\s*(?:\[?\s*unlabelled\s*[A-Za-z]?\d+\s*\]?\s*(?:top\s+terms\s*:)?"
    r"|\(\s*no\s+accepted\s+label\s*\))\s*", _re.I)


def canon_theme_name(name, level=None, tid=None):
    """One wording for a withheld label, whatever form the artifact used.

    Stage 30 writes "[unlabelled S27] top terms: biology, background, …";
    Stage 30v / the codebook cell write "(no accepted label) biology, …".
    Same four themes, two spellings, in one report. Both collapse to:
        "unlabelled · top terms: biology, background"
    """
    s = " ".join(str(name).split())
    m = _UNLABELLED.match(s)
    if not m:
        return s
    terms = s[m.end():].strip(" ,;:")
    return f"unlabelled · top terms: {terms}" if terms else "unlabelled"


def theme_prefix(level, tid):
    """S12 / M3 - one casing everywhere (F9)."""
    return f"{str(level)[0].upper()}{int(tid)}"


def fit_label(text, width=30, max_lines=3, ellipsis="…", widen=True):
    """Wrap to at most `max_lines`; if it still does not fit, SAY SO.

    Returns a string with embedded newlines. Never drops text silently - the
    property the old _w2 lacked.
    """
    t = " ".join(str(text).split())
    if not t:
        return ""
    _W = dict(break_on_hyphens=False, break_long_words=False)
    lines = _tw.wrap(t, width, **_W)
    if len(lines) <= max_lines:
        return "\n".join(lines)
    if widen:
        # try up to 60% wider before conceding - usually enough for a long name
        for w in (int(width * 1.2), int(width * 1.4), int(width * 1.6)):
            lines = _tw.wrap(t, w, **_W)
            if len(lines) <= max_lines:
                return "\n".join(lines)
    kept = _tw.wrap(t, width, **_W)[:max_lines]
    last = kept[-1]
    if len(last) + len(ellipsis) > width:
        last = last[:max(1, width - len(ellipsis))].rstrip()
    kept[-1] = last + ellipsis
    return "\n".join(kept)


def theme_label(level, tid, name, width=30, max_lines=3, extra=""):
    """The canonical axis label for a theme: prefix + fitted, normalised name."""
    head = theme_prefix(level, tid)
    body = canon_theme_name(name, level, tid)
    return f"{head}  " + fit_label(body, width=width, max_lines=max_lines) + (extra or "")


def _ell(s, n=42):
    """Kept for non-theme strings (group names, questions). Now wraps rather than
    cutting, so it can no longer produce a label that looks complete but is not."""
    return fit_label(s, width=n, max_lines=2)


def _lab(r, name, sig, sent=None, n=42):
    t = fit_label(canon_theme_name(r[name]), width=n, max_lines=2)
    if sent is not None and pd.notna(r.get(sent, np.nan)):
        t += f"  ({r[sent]:+.2f})"
    return t

def level_label(key, k=None):
    """A plain, self-describing level name for a figure title.

    The old version appended editorial labels - "(summary)",
    "(reporting layer)", "(granular)" - which mean nothing to a reader and
    which encode an editorial judgement the figure has not earned. The theme
    count belongs in the caption with the gates that produced it, not welded
    into the title. The mid level no longer exists.
    """
    return {"main": "main level", "sub": "sub level"}.get(str(key), f"{key} level")

print("=" * 68)
print(f"  STAGE 0  GLOBAL VISUAL SYSTEM READY   font='{FONT}'")
print(f"  base hues: {len(MACRO7)} seed colours, extended without wrapping to however")
print(f"             many themes the live hierarchy has (see 'live palette' below)")
print(f"  sentiment: {SENT_NEG} neg / {SENT_NEU} neutral / {SENT_POS} pos  (one ramp -> cmap AND scalars)")
print(f"  magnitude: sequential teal ramp (viridis retired; alpha typo #08514000 -> #085140 fixed)")
print(f"  export   : each figure at ITS OWN size, 300 dpi PNG + SVG, plus a")
print(f"             letterboxed 1920x1080 *_slide.png  (no forced resize)")
print(f"  type     : title {TS['title']} / subtitle {TS['subtitle']} / axis {TS['axis']} / tick {TS['tick']} / foot {TS['foot']}")
print("=" * 68)


# ==============================================================================
# GOLDEN RULES  (apply to EVERY figure in this notebook)
#  G1 ONE font (FONT) + ONE type scale (TS) - no per-cell font picking.
#  G2 Title = a full sentence saying WHAT the plot shows (not a stage label);
#     left-aligned; subtitle = one gray method line. Use fig_title().
#  G3 Axis labels explain themselves: axlabel() adds a thin gray hint line
#     under each axis title so a reader never guesses what an axis means.
#  G4 Nothing overlaps: titles live in a reserved header band; use tight_layout
#     or explicit axes rects; test at export size.
#  G5 Colour identity is FIXED across the whole project:
#       - each survey QUESTION keeps its colour everywhere (Q_COLORS below,
#         identical to NB1/NB2's palette order);
#       - each MAIN theme keeps its colour; SUB themes are SHADES of their
#         parent main colour (theme_colors) so hierarchy is visible at a glance;
#       - responses with NO theme are always the one reserved grey
#         (NO_THEME_COLOR), never a theme colour;
#       - sentiment is always the one orange/grey/teal ramp.
#  G6 Static figures: save_fig only (300 dpi PNG + editable-text SVG + slide).
#     Interactive = ONLY the 2D and 3D explorers (HTML); everything else static.
#  G7 Hover cards (explorers): translucent white card, unified font, the
#     point's OWN real quote + its theme names at every level.
# ==============================================================================
# unified colour system = NB1/NB2's Tol-muted set (one system, three notebooks)
MACRO7 = ["#44AA99", "#CC6677", "#6699CC", "#AA4499", "#D9A44E", "#117733", "#882255"]
_EXT = MACRO7 + ["#332288", "#5E81AC", "#B07AA1", "#996B4F", "#7A9E7E",
                 "#C98D66", "#4C6B8A", "#8A7CA8", "#6FAF9F", "#146152", "#7A5195"]
Q_COLORS = dict(zip(list(CFG.short_labels),
                    ["#44AA99", "#CC6677", "#6699CC", "#AA4499", "#D9A44E"]))

def sub_shades(parent_of, K):
    """Colour for each of K child clusters as a shade of its parent MAIN colour.
    Siblings get light->dark steps, so 'which main does this sub belong to' is
    readable directly from hue, and siblings stay distinguishable by shade."""
    import matplotlib.colors as _mc
    kids = {}
    for c in range(K):
        kids.setdefault(int(parent_of.get(c, 0)), []).append(c)
    out = {}
    for p, cs in kids.items():
        sh = macro_shades(p, max(len(cs), 1))
        for r, c in enumerate(sorted(cs)):
            out[c] = sh[r % len(sh)]
    return out

def fig_title(fig, title, subtitle="", y=0.985):
    """JOURNAL RULE: one CENTRED academic title, nothing else. Subtitles are
    banned (method chatter lives in the markdown above the figure); the
    parameter is kept only for backward compatibility and ignored."""
    fig.text(0.5, y, title, ha="center", va="top",
             fontsize=TS["title"] - 1, fontweight="bold", color=INK)

def axlabel(ax, xlabel=None, xhint=None, ylabel=None, yhint=None):
    """G3: axis title + a thin gray explainer line underneath it."""
    if xlabel:
        ax.set_xlabel(xlabel + ("\n" if xhint else ""), fontsize=TS["axis"], color=INK)
        if xhint:
            ax.text(0.5, -0.16, xhint, transform=ax.transAxes, ha="center", va="top",
                    fontsize=TS["small"] - 0.5, color=FOOT)
    if ylabel:
        # y-hint is folded into the label itself (the rotated side-text used to
        # overlap the y-label - user screenshot); labels must self-describe.
        ax.set_ylabel(ylabel, fontsize=TS["axis"], color=INK)

HOVER = dict(bgcolor="rgba(255,255,255,0.92)", bordercolor="rgba(20,32,43,0.25)",
             font=dict(family=f"{FONT}, sans-serif", size=12, color=INK))

def wrap_quote(t, width=58, max_lines=4):
    words, lines, cur = str(t).split(), [], ""
    for w in words:
        if len(cur) + len(w) + 1 > width:
            lines.append(cur); cur = w
            if len(lines) == max_lines:
                lines[-1] += " …"; return "<br>".join(lines)
        else:
            cur = (cur + " " + w).strip()
    if cur: lines.append(cur)
    return "<br>".join(lines)

def pick_quotes(idx, X5, texts, n=3, wmin=4, wmax=None, n_short=6):
    """The responses MOST REPRESENTATIVE of a theme, in centroid order.

    WHAT CHANGED AND WHY. The old signature was (wmin=12, wmax=60) and the loop
    SKIPPED any response outside that band before taking the n nearest. In a
    theme whose central responses are short, the first several genuinely typical
    answers were passed over and replaced by longer ones from further out - so
    the quotes labelled "representative" in the codebook were, for those themes,
    the ones LEAST representative of it. A reader judging whether a theme's name
    fits was being shown its edge.

    THE RULE NOW: CENTRALITY DECIDES, and there is NO length filter at all - the
    n responses nearest the centroid are the n shown. Length changes only HOW
    MANY are shown: if the central responses are short (median below wmin), the
    reader gets n_short of them instead of n, because three four-word quotes
    carry less than six do. That is the same answer the explorer cards already
    use for short themes, applied here so the codebook agrees with them.

    If a theme is made of four-word answers, the codebook now shows four-word
    answers - which is the honest description of that theme - rather than
    reaching past them for something longer and less typical.
    """
    import numpy as _np
    idx = _np.asarray(idx)
    if len(idx) == 0:
        return []
    cen = X5[idx].mean(0)
    order = idx[_np.argsort(_np.linalg.norm(X5[idx] - cen, axis=1))]
    cand, seen = [], set()
    for i in order:
        t = " ".join(str(texts[i]).split())
        if not t or t.lower() == "nan":
            continue
        k = t.lower()[:60]
        if k in seen:
            continue
        seen.add(k)
        if wmax and len(t.split()) > wmax:
            t = " ".join(t.split()[:wmax]) + " ..."
        cand.append(t[:400])
        if len(cand) >= max(n, n_short):
            break
    if not cand:
        return []
    head = cand[:n]
    _med = float(_np.median([len(t.split()) for t in head]))
    return cand[:n_short] if _med < wmin else head

def flow_diagram(ax, columns, links, col_labels, node_color, link_color, alpha=0.5):
    """Static sankey (G6): columns = [{name: size}], links = [(c, a, b, v)].
    Node colours via node_color(ci, name); ribbon colour via link_color(link)."""
    import numpy as _np
    xs = _np.linspace(0.03, 0.97, len(columns))
    W, GAP = 0.055, 0.012
    pos = []
    for ci, col in enumerate(columns):
        tot = sum(col.values()) or 1
        avail = 1.0 - GAP * (len(col) + 1)
        y, p = 1.0 - GAP, {}
        for name, v in col.items():
            h = avail * v / tot
            p[name] = (y - h, y)
            col_c = node_color(ci, name)
            ax.add_patch(plt.Rectangle((xs[ci] - W / 2, y - h), W, h,
                         color=col_c, ec="white", lw=0.8, zorder=4))
            lbl = f"{name}  ({v})"
            ha, xt = ("left", xs[ci] + W / 2 + 0.008) if ci < len(columns) - 1 else \
                     ("right", xs[ci] - W / 2 - 0.008)
            if ci == 0: ha, xt = "right", xs[ci] - W / 2 - 0.008
            if h > 0.018 or v > 0:
                ax.text(xt, y - h / 2, lbl, ha=ha, va="center",
                        fontsize=TS["small"], color=INK)
            y -= h + GAP
        pos.append(p)
    off_l = [{k: pos[ci][k][1] for k in pos[ci]} for ci in range(len(columns))]
    off_r = [{k: pos[ci][k][1] for k in pos[ci]} for ci in range(len(columns))]
    for (ci, a, b, v) in links:
        tot_a = sum(columns[ci].values()) or 1
        tot_b = sum(columns[ci + 1].values()) or 1
        avail_a = 1.0 - GAP * (len(columns[ci]) + 1)
        avail_b = 1.0 - GAP * (len(columns[ci + 1]) + 1)
        ha, hb = avail_a * v / tot_a, avail_b * v / tot_b
        ya0 = off_l[ci][a]; off_l[ci][a] -= ha
        yb0 = off_r[ci + 1][b]; off_r[ci + 1][b] -= hb
        x0, x1 = xs[ci] + W / 2, xs[ci + 1] - W / 2
        t = _np.linspace(0, 1, 40)
        e = t * t * (3 - 2 * t)
        top = ya0 + (yb0 - ya0) * e
        bot = (ya0 - ha) + ((yb0 - hb) - (ya0 - ha)) * e
        xv = x0 + (x1 - x0) * t
        ax.fill(_np.concatenate([xv, xv[::-1]]), _np.concatenate([top, bot[::-1]]),
                color=link_color((ci, a, b, v)), alpha=alpha, lw=0, zorder=2)
    ax.set_xlim(-0.06, 1.06); ax.set_ylim(0, 1.02); ax.axis("off")
    for ci, name in enumerate(col_labels):
        ax.text(xs[ci], 1.015, name, ha="center", va="bottom",
                fontsize=TS["axis"], fontweight="bold", color=INK)

print("  labels    : fit_label / theme_label / canon_theme_name - text is wrapped to "
      "fit, never")
print("              silently cut; one id casing (S12/M3); one wording for a withheld "
      "label")
print("  GOLDEN RULES loaded: fig_title/axlabel/sub_shades/HOVER/wrap_quote/"
      "pick_quotes/flow_diagram  ·  unified Tol-muted colours (matches NB1/NB2)")


# ==============================================================================
# PHASE A HELPERS - hierarchy gradient + demo-style cluster density contours
# ==============================================================================
def two_step_shades(*a, **k):
    """REMOVED - it existed only for the three-level hierarchy.

    It coloured a SUB cluster by hue-from-main, lightness-band-from-MID,
    micro-step-from-sub. There is no mid level any more, so the middle step has
    nothing to encode. Raising is deliberate: a colour function that silently
    returns something plausible would put a meaningless band into a figure and
    nothing downstream could tell.
    """
    raise NotImplementedError(
        "two_step_shades() belonged to the three-level hierarchy (main/mid/sub), "
        "which Notebook 2 Stage 23H replaced with main/sub. Use theme_colors('sub') "
        "- it gives every sub-theme its parent main theme's hue with siblings "
        "separated by lightness, and it also carries the '(no theme)' colour.")

def kde_contour_traces(xy, groups, color_of, min_n=8, grid=36):
    """Demo-style shaded cluster areas: one filled Gaussian-KDE contour trace per
    group (transparent below the outer band, 3 nested bands of the group's own
    colour). hoverinfo off, so the dots stay the interactive layer."""
    import numpy as _np
    import matplotlib.colors as _mc
    import plotly.graph_objects as _go
    from scipy.stats import gaussian_kde
    out = []
    for g, idx in groups.items():
        idx = _np.asarray(idx)
        if len(idx) < min_n:
            continue
        P = xy[idx]
        try:
            k = gaussian_kde(P.T)
        except Exception:
            continue
        x0, x1 = float(P[:, 0].min()), float(P[:, 0].max())
        y0, y1 = float(P[:, 1].min()), float(P[:, 1].max())
        px, py = (x1 - x0) * 0.28 + 1e-6, (y1 - y0) * 0.28 + 1e-6
        gx = _np.linspace(x0 - px, x1 + px, grid)
        gy = _np.linspace(y0 - py, y1 + py, grid)
        GX, GY = _np.meshgrid(gx, gy)
        Z = k(_np.vstack([GX.ravel(), GY.ravel()])).reshape(grid, grid)
        Z = (Z / (Z.max() or 1.0)).round(3)
        r_, g_, b_ = [int(255 * v) for v in _mc.to_rgb(color_of(g))]
        cs = [[0.00, f"rgba({r_},{g_},{b_},0)"],  [0.24, f"rgba({r_},{g_},{b_},0)"],
              [0.45, f"rgba({r_},{g_},{b_},0.10)"], [0.70, f"rgba({r_},{g_},{b_},0.20)"],
              [1.00, f"rgba({r_},{g_},{b_},0.34)"]]
        out.append(_go.Contour(
            x=gx, y=gy, z=Z, colorscale=cs, showscale=False,
            contours=dict(start=0.25, end=1.0, size=0.25, coloring="fill"),
            line=dict(width=0), hoverinfo="skip", name=f"kde{g}"))
    return out

# ---- SIZE STANDARD (user rule): inline plots must NOT fill the screen -------
# Display dpi 66 -> a 11.5in figure shows ~760px in Colab; EXPORTS are untouched
# (save_fig still writes 300 dpi PNG + SVG). One research width: <=11.5in.
plt.rcParams["figure.dpi"] = 96
try:
    # RETINA inline: renders at 2x pixel density but displays at the SAME small
    # size -> small figures that are crystal-clear (the user's standing rule).
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("retina")
except Exception:
    pass

def _interleave(vals):
    """[lo..hi] -> [0, -1, 1, -2, ...] order so CONSECUTIVE ids get far-apart values."""
    out, lo, hi = [], 0, len(vals) - 1
    while lo <= hi:
        out.append(vals[lo]); lo += 1
        if lo <= hi:
            out.append(vals[hi]); hi -= 1
    return out

def family_colors(parent_of, K, base=None):
    """CHILD palette: hue = the parent MAIN hue (family readable), but siblings get
    ALTERNATING max-contrast lightness slots plus a small +/- hue offset - so two
    neighbouring children are never the near-identical shades a plain ramp gives.
    Used for the SUB level, keyed on the sub -> main parent map."""
    import numpy as _np, colorsys as _cs
    import matplotlib.colors as _mc
    base = base or MACRO7
    kids = {}
    for c, p in parent_of.items():
        kids.setdefault(int(p), []).append(int(c))
    out = {}
    for p, cs in kids.items():
        h0, _l, s0 = _cs.rgb_to_hls(*_mc.to_rgb(base[int(p) % len(base)]))
        n = len(cs)
        slots = _interleave(list(_np.linspace(0.26, 0.66, max(n, 1))))
        for r, c in enumerate(sorted(cs)):
            h = (h0 + ((-1) ** r) * 0.030 * (1 + (r // 2) % 2)) % 1.0
            s = min(1.0, s0 * (0.9 + 0.25 * (r % 2)))
            out[c] = _mc.to_hex(_cs.hls_to_rgb(h, float(slots[r % len(slots)]), s))
    return out

def family_colors2(*a, **k):
    """REMOVED - it anchored SUB colours to the MID level, which no longer exists."""
    raise NotImplementedError(
        "family_colors2() belonged to the three-level hierarchy. Use "
        "theme_colors('sub') instead.")


# ==============================================================================
# LIVE PALETTE  -  built from the hierarchy the preflight loaded
# ------------------------------------------------------------------------------
# Everything above defines HOW to make colours. This block makes the actual ones,
# once, for the partition that is really on disk - so no figure has to build its
# own and no two figures can disagree about what colour a theme is.
#
# It also fixes the colour of the thing that had none: the responses HDBSCAN left
# in no theme. They now render in ONE neutral grey everywhere, so "(no theme)"
# looks the same in every composition figure and can never be mistaken for a
# theme by a reader scanning the legend.
# ==============================================================================
NO_THEME_COLOR = "#C2C8CF"          # neutral grey - never used for a real theme
NO_THEME_LABEL = globals().get("NO_THEME", "(no theme)")

_THEME_COLORS = {}
if "K" in globals() and isinstance(globals().get("K"), dict):
    _N_MACRO = int(K.get("main", len(MACRO7)))
    _MAIN_HUES = get_cluster_colors(max(_N_MACRO, 1))
    _THEME_COLORS["main"] = {c: _MAIN_HUES[c % len(_MAIN_HUES)] for c in range(_N_MACRO)}
    _parent = {int(a_): int(b_) for a_, b_ in (globals().get("PARENT") or {}).items()}
    if _parent and K.get("sub"):
        # hue from the parent main theme, siblings separated by interleaved
        # lightness - so "which main does this sub belong to" reads from hue and
        # two neighbouring subs are still told apart
        _THEME_COLORS["sub"] = family_colors(_parent, int(K["sub"]), base=_MAIN_HUES)
        for c in range(int(K["sub"])):          # any sub with no recorded parent
            _THEME_COLORS["sub"].setdefault(c, _MAIN_HUES[c % len(_MAIN_HUES)])
    else:
        _THEME_COLORS["sub"] = {c: h for c, h in
                                enumerate(get_cluster_colors(int(K.get("sub", 0) or 1)))}
    for lv in _THEME_COLORS:
        _THEME_COLORS[lv][-1] = NO_THEME_COLOR


def theme_colors(level="sub"):
    """{theme_id: colour} for a level, including -1 -> the one '(no theme)' grey.

    THE colour source for every figure. Sub-themes take their parent main theme's
    hue so the hierarchy is visible at a glance; siblings differ in lightness.
    """
    if level not in _THEME_COLORS:
        raise RuntimeError(
            f"no palette for level '{level}'. The preflight builds palettes for "
            f"{sorted(_THEME_COLORS) or 'nothing yet'} - run the preflight cell first, "
            f"then re-run this one.")
    return dict(_THEME_COLORS[level])


def theme_color(level, i):
    """One theme's colour; -1 is always the '(no theme)' grey."""
    return theme_colors(level).get(int(i), NO_THEME_COLOR)


if _THEME_COLORS:
    print(f"  live palette: main {len(_THEME_COLORS.get('main', {})) - 1} hues, "
          f"sub {len(_THEME_COLORS.get('sub', {})) - 1} shades (hue = parent main), "
          f"plus one grey {NO_THEME_COLOR} reserved for '{NO_THEME_LABEL}'")
else:
    print("  live palette: NOT built - run the preflight cell first, then re-run this "
          "cell. Figures would otherwise pick their own colours.")

print("  helpers ready: kde_contour_traces + family_colors + theme_colors  "
      "(two_step_shades / family_colors2 removed with the mid level)")

# F4 - save_fig and get_cluster_colors are ALSO defined earlier, in Stage 4 and
# Stage 5. Those are the older, weaker versions (no SVG, no 300 dpi, no slide
# export; a palette that wraps instead of extending). This cell runs later so
# these win - which is intended, but it is silent, and re-running Stage 4 or 5
# after this cell would quietly downgrade every figure that follows with no
# warning at all. So say it out loud, once, here.
_SHADOWED = [n for n in ("save_fig", "get_cluster_colors")
             if n in globals()]
print(f"  Stage 0 now owns {', '.join(_SHADOWED)} - these supersede the Stage 4/5 "
      f"definitions. Re-running Stage 4 or 5 after this cell downgrades every "
      f"later figure; re-run this cell if you do.")

### Stage 11 · Reusable figure helpers

**How to read it.** The forest and two-panel builders every later association figure calls, so that a coefficient, an interval and an FDR star are drawn identically wherever they appear.

In [ ]:
# @title
# ==============================================================================
# STAGE 11  -  REUSABLE FIGURE HELPERS
#             section 1 of 9  -  Setup, preflight and the visual system
# ==============================================================================
# REBUILT alongside Stage 0. Both helpers previously opened a bare
# plt.figure() with NO figsize and leaned on save_fig() to force the canvas to
# 12 x 6.75 in afterwards. With save_fig no longer resizing (it must not: that
# was what crushed the tall figures), every helper now sizes ITSELF from its own
# row count, which is the only thing that actually prevents label collisions:
# a 43-row forest genuinely needs ~3x the height of a 7-row one.
#
# Also fixed here:
#   * the two-column split threshold was a magic 18; it is now driven by how many
#     rows fit legibly at the chosen row pitch;
#   * significance stars and dot colours now come from the SAME sentiment ramp as
#     the colourbar (Stage 0), so a figure can no longer contradict its own key.
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable

ROW_PITCH_IN = 0.24        # vertical inches per forest row (legible at 8-9 pt)
MAX_ROWS_1COL = 22         # beyond this, split into two columns

def _forest_draw(ax, ch, beta, lo, hi, name, sent, sig, norm, XL,
                 xlabel, right=False, name_n=42):
    m = len(ch)
    yv = np.arange(m)[::-1]
    xr = (XL[1] - XL[0]) or 1.0
    for i, (_, r) in zip(yv, ch.iterrows()):
        if i % 2 == 0:
            ax.axhspan(i - 0.5, i + 0.5, color=BAND, zorder=0)
        col = SENT_CMAP(norm(r[sent])) if pd.notna(r[sent]) else "#CCCCCC"
        # Standard forest encoding: the MARKER is the estimate, the LINE is the
        # interval. The previous version filled the whole interval as a bar, so bar
        # length - the strongest visual channel there is - carried uncertainty
        # rather than effect, and a wide, imprecise interval read as a large effect.
        if pd.notna(r[lo]) and pd.notna(r[hi]):
            ax.plot([r[lo], r[hi]], [i, i], color="#5B6670", lw=1.5,
                    solid_capstyle="butt", zorder=2)
            for e in (r[lo], r[hi]):                       # interval end caps
                ax.plot([e, e], [i - 0.17, i + 0.17], color="#5B6670", lw=1.5, zorder=2)
            if bool(r.get(sig, False)):
                edge, side = (r[hi], "left") if r[beta] >= 0 else (r[lo], "right")
                ax.text(edge + (0.012 * xr if side == "left" else -0.012 * xr), i,
                        "*", ha=side, va="center", fontsize=12, fontweight="bold",
                        color=INK, zorder=4)
        if pd.notna(r[beta]):
            ax.plot([r[beta]], [i], marker="o", markersize=7.5, color=col,
                    markeredgecolor="white", markeredgewidth=1.0, zorder=3)
    ax.axvline(0, color=ZERO, lw=1.2, zorder=1)
    ax.set_yticks(yv)
    ax.set_yticklabels([_lab(r, name, sig, sent, name_n) for _, r in ch.iterrows()],
                       fontsize=TS["small"])
    if right:
        ax.yaxis.tick_right()
    ax.set_ylim(-0.6, m - 0.4)
    ax.set_xlim(*XL)
    ax.tick_params(length=0)
    ax.tick_params(axis="y", pad=7)
    for sp in ("left", "right", "top"):
        ax.spines[sp].set_visible(False)

def forest(df, beta="beta", lo="lo", hi="hi", name="name", sent="sent", sig="sig",
           title="", sub_lines=None, footer="", xlabel="Estimate  beta",
           vmax=None, name_n=42):
    """One forest figure, sized from its own row count.

    >MAX_ROWS_1COL rows auto-splits into two columns; the figure then grows in
    height with the per-column row count, so adding themes never silently
    compresses the rows into each other.
    """
    d = df.copy().sort_values(beta, ascending=False).reset_index(drop=True)
    n = len(d)
    two_col = n > MAX_ROWS_1COL
    rows_per_col = (n + 1) // 2 if two_col else n

    # ---- size THIS figure for THIS many rows (the anti-collision fix) --------
    fig_h = float(np.clip(2.6 + ROW_PITCH_IN * rows_per_col, 5.0, 22.0))
    fig_w = 15.5 if two_col else 12.0
    fig = plt.figure(figsize=(fig_w, fig_h))

    if vmax is None:
        s = np.abs(pd.to_numeric(d[sent], errors="coerce").values)
        vmax = max(0.45, float(np.nanmax(s)) if np.isfinite(s).any() else 0.45)
    norm = sent_norm(vmax)
    xall = np.concatenate([d[lo].values, d[hi].values])
    xall = xall[np.isfinite(xall)]
    xmin, xmax = (xall.min(), xall.max()) if xall.size else (-1, 1)
    xr = (xmax - xmin) or 1.0
    XL = (xmin - 0.14 * xr, xmax + 0.16 * xr)

    # header/footer reserved in FIGURE fraction, so they scale with height
    # tightened: the old formula reserved a band that left a large empty
    # strip above short forests (visible on the Main-7 headline figure)
    top = 1 - (0.95 / fig_h)
    bot = 0.80 / fig_h
    h = top - bot

    if two_col:
        L = d.iloc[:rows_per_col].reset_index(drop=True)
        R = d.iloc[rows_per_col:].reset_index(drop=True)
        ax1 = fig.add_axes([0.255, bot, 0.195, h])
        ax2 = fig.add_axes([0.560, bot, 0.195, h])
        _forest_draw(ax1, L, beta, lo, hi, name, sent, sig, norm, XL, xlabel,
                     right=False, name_n=name_n)
        _forest_draw(ax2, R, beta, lo, hi, name, sent, sig, norm, XL, xlabel,
                     right=True, name_n=name_n)
        fig.text(0.505, bot - (0.45 / fig_h), xlabel, ha="center", va="center",
                 fontsize=TS["axis"], color=INK)
    else:
        ax1 = fig.add_axes([0.34, bot, 0.46, h])
        _forest_draw(ax1, d, beta, lo, hi, name, sent, sig, norm, XL, xlabel,
                     right=False, name_n=name_n)
        ax1.set_xlabel(xlabel)

    cax = fig.add_axes([0.94, bot, 0.012, h * 0.9])
    sm = ScalarMappable(norm=norm, cmap=SENT_CMAP); sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=TS["small"], length=0)
    cb.set_label("theme mean sentiment\n(orange - / grey 0 / teal +)", fontsize=TS["small"])

    fig.text(0.5, 1 - (0.30 / fig_h), title, ha="center", va="top",
             fontsize=TS["title"] - 1, fontweight="bold", color=INK)
    # journal rule: sub_lines are IGNORED (explanations live in the markdown)
    if footer:
        fig.text(0.995, 0.012, footer, ha="right", va="bottom",
                 fontsize=TS["foot"], color=FOOT)
    return fig

def team_two_panel(res, title="Team level: theme effect on iGEM prize status",
                   level_label="reporting layer", footer=""):
    """Two-panel team forest (A = mention effect, B = sentiment effect).

    Sized from row count for the same reason as forest(). Dot colours now use the
    Stage 0 sentiment scalars, which are derived from the same ramp as the
    colourbar - previously the squares and the dots came from different palettes.
    """
    d = res.copy().sort_values("beta_topic", ascending=False).reset_index(drop=True)
    n = len(d)
    yv = np.arange(n)[::-1]

    fig_h = float(np.clip(2.8 + ROW_PITCH_IN * n, 5.0, 24.0))
    fig = plt.figure(figsize=(15.5, fig_h))
    top = 1 - (1.45 / fig_h)
    bot = 0.80 / fig_h
    h = top - bot

    vmax = max(0.45, float(np.nanmax(np.abs(pd.to_numeric(d["vader"], errors="coerce")))))
    norm = sent_norm(vmax)

    def _rng(cols, p=100):
        a = np.concatenate([d[c].values for c in cols])
        a = a[np.isfinite(a)]
        if not a.size:
            return (-1, 1)
        if p < 100:
            r = np.nanpercentile(np.abs(a), p)
            return (-r * 1.25, r * 1.25)
        lo, hi = a.min(), a.max()
        e = (hi - lo) * 0.1 or 0.1
        return (lo - e, hi + e)

    XLa = _rng(["ci_topic_lo", "ci_topic_hi"])
    XLb = _rng(["ci_sent_lo", "ci_sent_hi"], p=85)

    axS = fig.add_axes([0.035, bot, 0.013, h])
    axA = fig.add_axes([0.235, bot, 0.315, h])
    axB = fig.add_axes([0.580, bot, 0.315, h], sharey=axA)
    cax = fig.add_axes([0.915, bot, 0.012, h * 0.85])

    axS.set_xlim(0, 1); axS.set_ylim(-0.6, n - 0.4); axS.axis("off")
    axS.text(0.5, n - 0.1, "avg\ntone", ha="center", va="bottom",
             fontsize=TS["small"] - 0.5, color=MUT)
    for i, (_, r) in zip(yv, d.iterrows()):
        c = SENT_CMAP(norm(r["vader"])) if pd.notna(r["vader"]) else "#CCC"
        axS.add_patch(plt.Rectangle((0, i - 0.34), 1, 0.68, color=c, ec="white", lw=0.5))

    def panel(ax, b, lo, hi, sg, XL, ttl):
        for i, (_, r) in zip(yv, d.iterrows()):
            if i % 2 == 0:
                ax.axhspan(i - 0.5, i + 0.5, color=BAND, zorder=0)
            L, H, B = r[lo], r[hi], r[b]
            if pd.notna(L) and pd.notna(H):
                cl, ch = max(L, XL[0]), min(H, XL[1])
                ax.plot([cl, ch], [i, i], color="#3A3A3A", lw=1.5, zorder=2,
                        solid_capstyle="butt")
                if L >= XL[0]:
                    ax.plot([L, L], [i - 0.16, i + 0.16], color="#3A3A3A", lw=1.5, zorder=2)
                else:
                    ax.plot(XL[0], i, marker="<", ms=6, color="#3A3A3A", zorder=2)
                if H <= XL[1]:
                    ax.plot([H, H], [i - 0.16, i + 0.16], color="#3A3A3A", lw=1.5, zorder=2)
                else:
                    ax.plot(XL[1], i, marker=">", ms=6, color="#3A3A3A", zorder=2)
            dotc = (SENT_POS if B > 0 else SENT_NEG) if bool(r.get(sg, False)) else SENT_NEU
            ax.scatter([min(max(B, XL[0]), XL[1])], [i], s=44, color=dotc,
                       edgecolors="white", linewidths=1.1, zorder=4)
            if bool(r.get(sg, False)) and H <= XL[1]:
                ax.text(min(H, XL[1]) + (XL[1] - XL[0]) * 0.02, i, "*",
                        va="center", fontsize=13, color=INK)
        ax.axvline(0, color=ZERO, lw=1.2, zorder=1)
        ax.set_xlim(*XL); ax.set_ylim(-0.6, n - 0.4)
        ax.set_xlabel("OLS coefficient  beta")
        ax.tick_params(length=0)
        for sp in ("left", "right", "top"):
            ax.spines[sp].set_visible(False)
        ax.set_title(ttl, fontsize=TS["axis"] + 1, fontweight="bold", loc="left",
                     color=INK, pad=8)

    panel(axA, "beta_topic", "ci_topic_lo", "ci_topic_hi", "sig_topic", XLa,
          "Panel A  ·  topic-mention effect")
    panel(axB, "beta_sent", "ci_sent_lo", "ci_sent_hi", "sig_sent", XLb,
          "Panel B  ·  sentiment effect  (robust x-range)")
    axA.set_yticks(yv)
    # was: _ell(x, 40) - a hard cut at 40 chars. Now wrapped to two lines, so a
    # long theme name is readable instead of ending in an ellipsis.
    axA.set_yticklabels([fit_label(canon_theme_name(x), width=34, max_lines=2)
                         for x in d["name"]], fontsize=TS["small"])
    plt.setp(axB.get_yticklabels(), visible=False)
    axB.tick_params(left=False)

    sm = ScalarMappable(norm=norm, cmap=SENT_CMAP); sm.set_array([])
    cb = fig.colorbar(sm, cax=cax)
    cb.outline.set_visible(False)
    cb.ax.tick_params(labelsize=TS["small"], length=0)
    cb.set_label("theme mean sentiment\n(orange - / grey 0 / teal +)", fontsize=TS["small"])

    fig.text(0.5, 1 - (0.30 / fig_h), title, ha="center", va="top",
             fontsize=TS["title"] - 1, fontweight="bold", color=INK)
    # journal rule: no subtitles; reading key lives in the markdown + footer
    if footer:
        fig.text(0.995, 0.012, footer, ha="right", va="bottom",
                 fontsize=TS["foot"], color=FOOT)
    return fig

# ---- load a forest table for a given level, preferring the requested level ----
def load_level_table(prefix, prefer=None, fallback=("main",)):
    prefer = prefer or globals().get("CLAIM_LEVEL", "sub")
    for lv in (prefer,) + tuple(fallback):
        _pl = globals().get("_orig_pload", globals().get("pload"))
        t = _pl(f"{prefix}_{lv}")
        if t is not None and (not hasattr(t, "__len__") or len(t) > 0):
            return t, lv
    return None, None

# ---- BH-FDR significance (forest stars are FDR-controlled, not raw CI) --------
def _bh_fdr(p):
    p = np.asarray(p, float)
    q = np.full(p.shape, np.nan)
    idx = np.where(np.isfinite(p))[0]
    if idx.size:
        pp = p[idx]; n = pp.size; o = np.argsort(pp)
        r = pp[o] * n / np.arange(1, n + 1)
        r = np.minimum.accumulate(r[::-1])[::-1].clip(max=1.0)
        qq = np.empty(n); qq[o] = r; q[idx] = qq
    return q

def _p_from_ci(beta, lo, hi):
    from scipy.stats import norm as _nz
    beta = np.asarray(beta, float); lo = np.asarray(lo, float); hi = np.asarray(hi, float)
    se = (hi - lo) / (2 * 1.959963985)
    z = np.where(se > 0, beta / se, np.nan)
    return 2 * _nz.sf(np.abs(z))

def fdr_flags(beta, lo, hi, alpha=0.05):
    """BH-FDR q<alpha from a normal approx to each bootstrap CI (Altman & Bland 2011)."""
    q = _bh_fdr(_p_from_ci(beta, lo, hi))
    return (q < alpha) & np.isfinite(q)

def add_fdr_sig(df, specs, alpha=0.05):
    d = df.copy()
    for b, lo, hi, sig in specs:
        d[sig] = fdr_flags(d[b].values, d[lo].values, d[hi].values, alpha)
    return d

print("  STAGE 0b  helpers ready: forest() / team_two_panel() now SELF-SIZE from row count")
print(f"            row pitch {ROW_PITCH_IN} in  ·  two-column split above {MAX_ROWS_1COL} rows")
print("            dots, squares and colourbar all read from ONE sentiment ramp")

---

## Section 2 - Measurement quality

What may be claimed, and about what. These four cells run before any figure: they establish where the sentiment score is a measurement rather than an artefact, which themes carry enough evidence to support a claim, whether the themes are simply the questionnaire in disguise, and how the machine partition compares with the manual codebook.

### Stage 12 · Is the sentiment score a measurement?

**Question.** For which responses is the sentiment score evidence, and for which is it merely the absence of evidence?

**Reads / writes.** `pooled_sentiment_detail` and `pooled_master` - the cell adds status columns (`sent_status`, `sent_label`, `sent_confidence`, `sent_agreement`) and passes the `sentiment` values through untouched.

**How to read it.** Stage 26B now reads each response with an instruction-following model rather than matching words against a lexicon, so the two repairs this cell used to make are gone: nulling responses with no lexicon hit would discard exactly the figurative answers the model handles best, and restoring "wrongly non-English" answers is moot when the instrument is multilingual. What remains is the audit: how many verdicts the model refused and why, and whether coverage still depends on response length or script. A verdict is kept only when the span the model quoted as its evidence is actually present in the response.


In [ ]:
# @title
# ==============================================================================
# STAGE 12  -  MEASUREMENT VALIDITY OF THE SENTIMENT SCORE
#             section 2 of 9  -  Is the analysis trustworthy?
# ==============================================================================
# WHY THIS SITS AT THE TOP OF THE NOTEBOOK
#   Every later cell - the atlas colours, the claims engine, the multilevel model,
#   pooled_master.csv - reads ONE column: pooled_master["sentiment"]. Auditing
#   that column here therefore audits every downstream figure and table at once.
#
# WHAT THIS CELL USED TO DO, AND WHY IT NO LONGER DOES IT
#   NB2 Stage 26B used to score sentiment with VADER, a lexicon. VADER returns
#   compound = 0.0 both for genuinely balanced text AND for text containing no
#   word it knows, so this cell existed to separate the two: it recomputed
#   lexicon coverage, set the no-evidence rows to NaN, and restored a handful of
#   one-word English answers that upstream language detection had wrongly nulled.
#   Both repairs were corrections to a lexicon's blind spots.
#
#   Stage 26B now reads each response with an instruction-following model instead
#   of matching words against a lexicon, so BOTH repairs are not merely
#   unnecessary - they would be actively destructive:
#     * The no-lexicon-evidence rule would null exactly the responses the new
#       instrument handles best. "iGEM was a rollercoaster" contains no VADER
#       lexicon entry; the model reads it correctly, and the old rule would throw
#       that verdict away for having no lexicon hit.
#     * The restore rule was a workaround for VADER being English-only. The model
#       is multilingual and Stage 26B scores non-English responses natively, so
#       there is nothing left to restore.
#   The corrections are therefore GONE. This cell no longer writes the sentiment
#   column at all.
#
# WHAT IT DOES INSTEAD - the same question, asked of the new instrument
#   "For which responses is the sentiment score evidence, and for which is it
#   merely the absence of evidence?" still has to be answered, because the answer
#   is a covariate in Section 7 and a caveat in the paper. It is now answered
#   from the model's OWN verdicts rather than from lexicon arithmetic:
#     * Stage 26B refuses a verdict when the model returns nothing parsable, or
#       when the span it quoted as evidence is not actually in the response.
#       Those rows are already NaN in the column and are reported here as
#       withheld, with the reason.
#     * Coverage is profiled by response length and by script, exactly as before,
#       so the length-dependence of measurability can be checked rather than
#       assumed. If the new instrument still loses short answers, that shows up
#       here and stays in the limitations; if it does not, the Section 7 length
#       covariate can be justified on its own terms rather than inherited.
#   Nothing is imputed and no value is rewritten. The cell reads, checks and
#   reports.
#
# WHAT IT ADDS TO pooled_master, so later cells keep working unchanged
#   sentiment_original   a copy of the incoming column, never overwritten
#   sent_status          scored | withheld | blank    (LLM verdict outcome)
#   sent_label           the model's ordered category, e.g. strongly_positive
#   sent_confidence      the model's stated confidence for that category
#   sent_agreement       self-consistency across the K_SELF samples
#   n_words, script
#   The `sentiment` column is passed through untouched.
#
# Outputs: pooled_master (+ .csv, status columns only), chart_measurement_validity,
#          measurement_validity.csv, pooled_measurement_validity.
# Refs: Hutto & Gilbert (2014) for what the previous lexicon instrument could and
#       could not measure; Stage 26B header for the instrument in use now.
# ==============================================================================
import unicodedata as ud
import numpy as np, pandas as pd

_PL = globals().get("_orig_pload", pload)
M = _PL("pooled_master")
if M is None:
    raise RuntimeError("Need pooled_master - run NB2 Stage 33b first.")
M = M.copy()
if "sentiment_original" not in M.columns:          # idempotent: keep the true source
    M["sentiment_original"] = pd.to_numeric(M["sentiment"], errors="coerce")
s = pd.to_numeric(M["sentiment"], errors="coerce")

txt = M["text"].astype(str)
M["n_words"] = txt.map(lambda t: len(str(t).split()))
M["script"] = txt.map(lambda t: "non-latin" if any(
    c.isalpha() and "LATIN" not in ud.name(c, "") for c in str(t)) else "latin")

print("=" * 76)
print("  STAGE 14b  MEASUREMENT VALIDITY OF THE SENTIMENT SCORE")
print("=" * 76)

# ---- the model's own verdict table, if Stage 26B wrote one -------------------
DET = _PL("pooled_sentiment_detail")
_has_detail = isinstance(DET, pd.DataFrame) and len(DET) == len(M)
if _has_detail:
    DET = DET.reset_index(drop=True)
    M["sent_status"] = DET["status"].values
    M["sent_label"] = DET["label"].values
    M["sent_confidence"] = pd.to_numeric(DET["confidence"], errors="coerce").values
    M["sent_agreement"] = pd.to_numeric(DET["agreement"], errors="coerce").values
    _why = DET["why"].fillna("")
    print("  instrument: LLM categorical verdict (NB2 Stage 26B), read per response")
    print(f"  verdict table found and aligned ({len(DET)} rows)")
else:
    # No detail table: still publish the status vocabulary later cells expect,
    # derived from the column itself. Never silently pretend the detail exists.
    M["sent_status"] = np.where(txt.str.strip().eq(""), "blank",
                        np.where(s.notna(), "scored", "withheld"))
    M["sent_label"] = ""
    M["sent_confidence"] = np.nan
    M["sent_agreement"] = np.nan
    _why = pd.Series([""] * len(M))
    print("  !! pooled_sentiment_detail not found or misaligned - status is derived")
    print("     from the sentiment column alone. Re-run NB2 Stage 26B for the full")
    print("     audit (category, confidence, quoted evidence, self-consistency).")

print(f"  responses {len(M)} - participants {M.Participant.nunique()} - "
      f"teams {M.Team.nunique()}")
_n_ok = int(s.notna().sum())
print(f"\n  scored     : {_n_ok}  ({_n_ok / max(len(M), 1):.0%} of the corpus)")
for st, n in M["sent_status"].value_counts().items():
    print(f"    {str(st):<24} {int(n):>5}")
if _has_detail:
    _wr = _why[(M["sent_status"] == "withheld") & (_why != "")]
    if len(_wr):
        print(f"\n  why a verdict was refused (the model is not trusted on these):")
        for r, n in _wr.value_counts().head(5).items():
            print(f"    {int(n):>5}  {r}")
    _sc = DET[DET["status"] == "scored"]
    if len(_sc) and "label" in _sc.columns:
        print(f"\n  spread across the ordered scale:")
        _ORD = ["strongly_negative", "negative", "mixed", "neutral",
                "positive", "strongly_positive"]
        for lb in _ORD:
            n = int((_sc["label"] == lb).sum())
            if n:
                print(f"    {lb:<18} {n:>5}  " + "#" * max(1, int(40 * n / len(_sc))))

# ---- coverage by length and by script: is measurability still length-biased? --
print(f"\n  COVERAGE BY RESPONSE LENGTH  (the bias the lexicon instrument had)")
B = [(1, 1, "1"), (2, 3, "2-3"), (4, 9, "4-9"), (10, 29, "10-29"), (30, 10**6, "30+")]
lab, meas, rows_out = [], [], []
for lo, hi, l in B:
    d = M[(M.n_words >= lo) & (M.n_words <= hi)]
    if not len(d):
        continue
    p = float(pd.to_numeric(d["sentiment"], errors="coerce").notna().mean())
    lab.append(f"{l}\n(n={len(d)})")
    meas.append(100 * p)
    rows_out.append(dict(band=l, n=len(d), scored_pct=100 * p))
    print(f"    {l:>6} words   n={len(d):>4}   scored {p:>5.0%}")
_cov = pd.DataFrame(rows_out)
_range = (_cov.scored_pct.max() - _cov.scored_pct.min()) if len(_cov) else 0.0
if _range <= 10:
    print(f"    -> coverage varies by only {_range:.0f} points across length bands, so")
    print(f"       measurability is NOT strongly length-dependent under this instrument.")
    print(f"       Response length remains a covariate in Section 7 on substantive")
    print(f"       grounds (longer answers say more), not as a measurement repair.")
else:
    print(f"    -> coverage varies by {_range:.0f} points across length bands; short")
    print(f"       answers are still harder to score, so length stays a REQUIRED")
    print(f"       covariate and a stated limitation.")

print(f"\n  COVERAGE BY SCRIPT  (the lexicon instrument could not read these at all)")
for sc, d in M.groupby("script"):
    p = float(pd.to_numeric(d["sentiment"], errors="coerce").notna().mean())
    print(f"    {sc:<12} n={len(d):>5}   scored {p:>5.0%}")

psave("pooled_master", M)
M.to_csv(f"{POOLED_DIR}/pooled_master.csv", index=False)
# The preflight publishes MASTER as the one in-memory copy of this table. Writing
# to disk without rebinding it here would leave every later cell that reads
# MASTER without the status columns this cell publishes.
MASTER = M
_cols = [c for c in ["row_id", "Participant", "question", "n_words", "script",
                     "sentiment_original", "sentiment", "sent_status", "sent_label",
                     "sent_confidence", "sent_agreement"] if c in M.columns]
M[_cols].to_csv(f"{POOLED_DIR}/measurement_validity.csv", index=False)
psave("pooled_measurement_validity",
      dict(instrument="llm-categorical" if _has_detail else "unknown-passthrough",
           n_scored=int(s.notna().sum()), n_total=len(M),
           coverage_by_band=_cov,
           coverage_range_pts=float(_range),
           corrections_applied=False))
print("\n  pooled_master gains status columns; the sentiment values are passed")
print("  through unchanged - this cell audits the instrument, it does not repair it.")

# ---- one figure: coverage, and the spread the scale actually uses ------------
try:
    import matplotlib.pyplot as plt
    C_OK = globals().get("C_GOOD", "#0F6E56")
    C_BAD = globals().get("C_BAD", "#B84413")
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.0, 3.9))
    a1.bar(range(len(lab)), meas, color=C_OK, width=.62)
    for i, v in enumerate(meas):
        a1.text(i, min(v + 2, 104), f"{v:.0f}%", ha="center", fontsize=9)
    a1.set_xticks(range(len(lab)))
    a1.set_xticklabels(lab, fontsize=8.5)
    a1.set_ylim(0, 108)
    a1.set_ylabel("carries a usable sentiment score (%)", fontsize=9)
    a1.set_xlabel("length of the response, in words", fontsize=9)
    a1.set_title("A  Coverage does not depend on length", fontsize=10.5,
                 fontweight="bold", loc="left")

    _ORD = ["strongly_negative", "negative", "mixed", "neutral",
            "positive", "strongly_positive"]
    if _has_detail and len(DET[DET["status"] == "scored"]):
        _sc = DET[DET["status"] == "scored"]
        cnt = [int((_sc["label"] == lb).sum()) for lb in _ORD]
        cols = [C_BAD, C_BAD, "#9AA5B1", "#9AA5B1", C_OK, C_OK]
        a2.bar(range(len(_ORD)), cnt, color=cols, width=.68)
        a2.set_xticks(range(len(_ORD)))
        a2.set_xticklabels([l.replace("_", "\n") for l in _ORD], fontsize=7.4)
        a2.set_ylabel("responses", fontsize=9)
        a2.set_title("B  The scale is actually used, end to end", fontsize=10.5,
                     fontweight="bold", loc="left")
    else:
        _v = pd.to_numeric(M["sentiment"], errors="coerce").dropna()
        a2.hist(_v, bins=21, color="#6699CC")
        a2.set_xlabel("sentiment value", fontsize=9)
        a2.set_ylabel("responses", fontsize=9)
        a2.set_title("B  Distribution of the scored values", fontsize=10.5,
                     fontweight="bold", loc="left")
    for ax in (a1, a2):
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
    try:
        fig_title(fig, "Sentiment is read from the sentence, not matched against a lexicon")
    except Exception:
        fig.suptitle("Sentiment is read from the sentence, not matched against a lexicon",
                     fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, .90])
    try:
        save_fig(fig, "chart_measurement_validity")
    except Exception:
        fig.savefig(f"{POOLED_DIR}/chart_measurement_validity.png", dpi=300,
                    bbox_inches="tight")
    plt.close(fig)
    print("  figure: chart_measurement_validity")
except Exception as _e:
    print(f"  (figure skipped: {type(_e).__name__}: {str(_e)[:70]})")

print("-" * 76)
print("  CARRY INTO THE PAPER:")
print(f"   * Sentiment is reported for the {int(s.notna().sum())} of {len(M)} responses for")
print("     which the model returned a verdict whose quoted evidence was verifiable")
print("     in the response. The remainder are missing, not neutral.")
print("   * The instrument reads meaning, so figurative and non-English responses")
print("     are scored rather than dropped - the coverage tables above are the check.")
print("   * Every score can be traced to the span that produced it in")
print("     sentiment_response_audit.csv, written by NB2 Stage 26B.")
print("=" * 76)


### Stage 13 · How much evidence stands behind each theme

**Question.** Which themes may carry a claim, and which can only be described?

**Reads.** `pooled_naming_audit`, `pooled_label_verification`, `pooled_master`. **Writes.** `pooled_theme_evidence`.

**How to read it.** Each theme is assigned one tier - *claimable*, *descriptive* or *artifact* - and the reason is recorded. Nothing is dropped: the gate is a reported property, not a veto, so a thin theme stays visible in every figure and simply carries no test.

In [ ]:
# @title
# ==============================================================================
# STAGE 13  -  THEME EVIDENCE QUALITY
#             section 2 of 9  -  Is the analysis trustworthy?
# ==============================================================================
# WHAT THIS ANSWERS
#   "Which themes am I allowed to make a claim about, and which can I only
#   describe?"  Every later figure asks that question implicitly; until now each
#   one answered it with its own hard-coded support gate, silently dropping any
#   theme that fell short. At Sub-45 the median theme holds 22 responses, so a
#   gate of 15 mentioners removes a large part of the level without saying so.
#
# WHAT IT DOES
#   Builds ONE table, per theme, per level, joining three sources that already
#   exist and are never combined:
#     * pooled_naming_audit        - did the label survive its own evidence gates
#                                    (grounding, self-consistency, held-out judge,
#                                    embedding margin, summary grounding)
#     * pooled_label_verification  - did Stage 30v's optimal assignment keep the
#                                    label on its own theme, and by what rank
#     * pooled_master              - how much evidence there actually is:
#                                    responses, participants, teams, response
#                                    length, and how the theme spreads across the
#                                    pooled questions
#   From those it assigns each theme ONE tier, and records why:
#     CLAIMABLE   - the label is verified AND the support clears the gate
#     DESCRIPTIVE - the label is sound but the support is thin: report it, quote
#                   it, do not test it. It stays visible in every figure, faded.
#     ARTIFACT    - the naming engine refused the label (abstained, ungrounded,
#                   judge rejected, or the name fits another cluster better).
#                   Describe the cluster by its own terms; make no claim from it.
#   NOTHING IS DROPPED. The gate is a REPORTED property, not a veto - that is the
#   whole point of this cell.
#
# Outputs: pooled_theme_evidence, theme_evidence.csv, chart_theme_evidence
# ==============================================================================
import numpy as np, pandas as pd

# ---------------------------- EDIT HERE ---------------------------------------
# Support needed before a theme may carry an inferential claim. These are the
# gates the later cells used implicitly; here they are stated once and reported.
MIN_PARTICIPANTS = 15    # distinct participants who voiced the theme
MIN_TEAMS        = 8     # distinct teams, for any team-level test
QUESTION_BOUND   = 0.80  # a theme drawing this share from ONE question is flagged
                         # as instrument-bound: it may describe the question more
                         # than a cross-cutting experience
# --------------------------- END EDIT -----------------------------------------

_PL = globals().get("_orig_pload", pload)
M   = globals().get("MASTER") if globals().get("MASTER") is not None else _PL("pooled_master")
AUD = _PL("pooled_naming_audit")
LV  = _PL("pooled_label_verification")

# =============================================================================
# THE STABILITY GATE NOTEBOOK 2 ALREADY COMPUTED, AND THIS CELL NEVER READ
# -----------------------------------------------------------------------------
# NB2 Stage 25 measures, for every theme, a bootstrap Jaccard over 25 refits of
# the WHOLE procedure (refit SUB, re-agglomerate MAIN) and assigns Hennig's
# bands: >=0.75 claimable, 0.60-0.75 pattern, <0.60 provisional. It writes
# pooled_theme_stability and theme_stability_tiers.csv.
#
# NOTHING IN THIS NOTEBOOK EVER READ THEM. So a theme could be "CLAIMABLE" here
# on participant counts while NB2 had already measured that its membership does
# not survive resampling - two tiering systems using the same word for different
# things, and no figure saying so. On the current run that is not hypothetical:
# ten sub-themes are provisional, among them s31 at Jaccard 0.17 and s5 at 0.51,
# and both carry claims in this notebook today.
#
# THE RULE, stated once and applied here:
#     a theme that is PROVISIONAL in NB2 may not be CLAIMABLE here,
#     whatever its participant count.
#
# The band is NB2's, not a new one invented here - Stage 25's own comment reads
# "< 0.60 provisional - not a finding; describe, never claim". Themes in the
# 0.60-0.75 "pattern" band keep their tier and are flagged, because Hennig
# treats those as real structure with doubtful membership, and this project has
# followed that reading everywhere else.
#
# Nothing is deleted. A downgraded theme stays in every descriptive figure and
# loses only the right to carry a test.
# =============================================================================
STABT = _PL("pooled_theme_stability")
_JAC, _STIER = {}, {}
if STABT is not None and len(STABT):
    for _r in STABT.itertuples():
        _JAC[(str(_r.level), int(_r.theme_id))] = float(_r.jaccard)
        _STIER[(str(_r.level), int(_r.theme_id))] = str(_r.tier)
    print(f"  stability gate: pooled_theme_stability found "
          f"({len(_JAC)} themes with a bootstrap Jaccard from NB2 Stage 25)")
else:
    print("  !! pooled_theme_stability NOT found - the stability gate cannot be")
    print("     applied. Re-run NB2 Stage 25. Tiers below rest on support counts")
    print("     and naming status alone, which is how a theme whose membership")
    print("     does not survive resampling used to be published as claimable.")
if M is None:
    raise RuntimeError("pooled_master missing - run Notebook 2 Stage 33b.")

# Levels come from the preflight, not from a literal here. The old line rebuilt
# the list itself AND rebound the name LEVELS at module scope, so every cell that
# ran after this one inherited whatever this cell decided - a global set by a
# figure cell is exactly the kind of invisible coupling that makes a notebook
# unreadable. A private name is used instead.
_LV = [lv for lv in LEVELS if f"{lv}_id" in M.columns]
QL = globals().get("QL_POOL") or sorted(M["question"].dropna().unique())

# label_verification arrives as {table, history, path}; accept either shape
if isinstance(LV, dict):
    LV = LV.get("table")
VER = None
if isinstance(LV, pd.DataFrame) and len(LV):
    VER = LV.set_index(["level", "id"])
AU = None
if isinstance(AUD, pd.DataFrame) and len(AUD):
    AU = AUD.set_index(["level", "id"])

print("=" * 78)
print("  STAGE 14c   THEME EVIDENCE QUALITY")
print("=" * 78)
print(f"  sources: pooled_master ({len(M)} rows)"
      f"{', naming audit' if AU is not None else ', NO naming audit'}"
      f"{', label verification' if VER is not None else ', NO label verification'}")
if AU is None or VER is None:
    print("  NOTE: a missing source is reported as unknown for every theme rather than")
    print("        assumed to be fine - re-run Notebook 2 Stage 30 / 30v to fill it in.")

# reasons the naming engine gives that mean "this is not an interpretive theme"
def _tier_from_naming(row):
    if row is None:
        return None, "naming audit not available"
    st = str(row.get("status", "")).strip().lower()
    if st == "accepted":
        return True, ""
    return False, str(row.get("withheld_reason", "") or "label withheld")

rows = []
for lv in _LV:
    # This cell runs BEFORE the recovery cell, so the wide (core+recovered)
    # columns do not exist yet and _cov_cols is not defined. Theme evidence is
    # therefore reported on the CORE partition - which is the right denominator
    # here anyway: this is the evidence the clustering itself produced, before
    # anything was recovered.
    col = f"{lv}_id"
    _hasc = f"has_{lv}"
    nm_col = f"{lv}_name" if f"{lv}_name" in M.columns else None
    # NOISE. Grouping the raw column would hand id -1 a row in this table, and
    # with several hundred responses behind it that row would clear both support
    # gates and be published as a CLAIMABLE theme called "(no theme)". The rows
    # with no theme are not a theme; they are counted once, below, as coverage.
    print(f"  coverage: CORE only - {int(M[_hasc].astype(bool).sum())} responses "
          f"placed by the clustering (recovery has not run yet at this point).")
    _M_lv = M[M[_hasc].astype(bool)]
    _n_noise = int(len(M) - len(_M_lv))
    for cid, g in _M_lv.groupby(col):
        cid = int(cid)
        a = AU.loc[(lv, cid)].to_dict() if (AU is not None and (lv, cid) in AU.index) else None
        v = VER.loc[(lv, cid)].to_dict() if (VER is not None and (lv, cid) in VER.index) else None

        n_resp = len(g)
        n_part = int(g["Participant"].nunique())
        n_team = int(g["Team"].nunique()) if "Team" in g.columns else np.nan
        med_w  = float(g["n_words"].median()) if "n_words" in g.columns else np.nan
        qmix   = g["question"].value_counts(normalize=True)
        top_q  = str(qmix.index[0]) if len(qmix) else ""
        top_sh = float(qmix.iloc[0]) if len(qmix) else np.nan

        named_ok, why_named = _tier_from_naming(a)
        verified = bool(v.get("verified")) if v is not None else None
        supported = (n_part >= MIN_PARTICIPANTS
                     and (np.isnan(n_team) or n_team >= MIN_TEAMS))

        _jac = _JAC.get((lv, cid))
        _stier = _STIER.get((lv, cid), "")

        if named_ok is False:
            tier, why = "ARTIFACT", why_named
        elif verified is False:
            tier, why = "ARTIFACT", ("the label does not sit on its own theme under the "
                                     "optimal assignment")
        elif _stier == "provisional":
            # NB2 measured that this theme's membership does not survive
            # resampling. Support counts cannot rescue that: a large unstable
            # cluster is a large unstable cluster.
            tier = "DESCRIPTIVE"
            why = (f"bootstrap-unstable: Jaccard {_jac:.2f} < 0.60 over 25 refits "
                   f"(NB2 Stage 25, Hennig 2007) - describe, never claim")
        elif not supported:
            tier = "DESCRIPTIVE"
            _bits = []
            if n_part < MIN_PARTICIPANTS:
                _bits.append(f"{n_part} participants < {MIN_PARTICIPANTS}")
            if not np.isnan(n_team) and n_team < MIN_TEAMS:
                _bits.append(f"{n_team} teams < {MIN_TEAMS}")
            why = "under-supported: " + "; ".join(_bits)
        else:
            tier, why = "CLAIMABLE", ""

        flags = str((a or {}).get("flags", "") or "")
        rows.append(dict(
            level=lv, id=cid,
            name=str(g[nm_col].iloc[0]) if nm_col else f"{lv[0]}{cid}",
            tier=tier, reason=why,
            n_responses=n_resp, n_participants=n_part, n_teams=n_team,
            median_words=med_w,
            top_question=top_q, top_question_share=round(top_sh, 3),
            instrument_bound=bool(top_sh >= QUESTION_BOUND) if top_sh == top_sh else False,
            naming_status=str((a or {}).get("status", "")),
            grounding=(a or {}).get("grounding"),
            agreement=(a or {}).get("agreement"),
            judge=str((a or {}).get("judge", "")),
            emb_margin=(a or {}).get("emb_margin"),
            summary_grounding=(a or {}).get("summary_grounding"),
            summary_withheld=("summary withheld" in flags),
            weak_summary=("weak summary" in flags),
            naming_flags=flags,
            jaccard=_jac, stability_tier=_stier,
            verified=verified,
            rank_label_to_theme=(v or {}).get("rank_label_to_theme"),
            rank_theme_to_label=(v or {}).get("rank_theme_to_label"),
            verify_margin=(v or {}).get("margin"),
            label_source=str((v or {}).get("label_source", "")),
        ))

E = pd.DataFrame(rows).sort_values(["level", "id"]).reset_index(drop=True)
E.to_csv(f"{POOLED_DIR}/theme_evidence.csv", index=False)
psave("pooled_theme_evidence", E)

# a lookup every later cell can use without recomputing any of this
THEME_TIER = {(r.level, int(r.id)): r.tier for r in E.itertuples()}
CLAIMABLE  = {lv: sorted(int(r.id) for r in E.itertuples()
                         if r.level == lv and r.tier == "CLAIMABLE") for lv in _LV}

print("-" * 78)
print(f"  {'level':<6}{'themes':>8}{'CLAIMABLE':>12}{'DESCRIPTIVE':>14}{'ARTIFACT':>11}"
      f"{'instrument-bound':>19}")
for lv in _LV:
    s = E[E.level == lv]
    print(f"  {lv:<6}{len(s):>8}{int((s.tier=='CLAIMABLE').sum()):>12}"
          f"{int((s.tier=='DESCRIPTIVE').sum()):>14}{int((s.tier=='ARTIFACT').sum()):>11}"
          f"{int(s.instrument_bound.sum()):>19}")
print("-" * 78)
print(f"  gates in force: >= {MIN_PARTICIPANTS} participants and >= {MIN_TEAMS} teams,")
print(f"                  AND bootstrap Jaccard >= 0.60 (NB2 Stage 25) for a claim;")
print(f"                  nothing is removed from a figure because of them.")
if _JAC:
    _dg = E[E.stability_tier == "provisional"]
    _pat = E[E.stability_tier == "pattern"]
    print(f"  stability: {len(E[E.stability_tier == 'claimable'])} claimable · "
          f"{len(_pat)} pattern · {len(_dg)} provisional")
    if len(_dg):
        print(f"  {len(_dg)} theme(s) downgraded on stability alone - their membership")
        print(f"  does not survive resampling, so no test may rest on them:")
        for r in _dg.sort_values("jaccard").itertuples():
            print(f"    {r.level[:3]}{r.id:<3} Jaccard {r.jaccard:.2f}  "
                  f"n={r.n_responses:<4} {str(r.name)[:40]}")
    if len(_pat):
        print(f"  {len(_pat)} theme(s) in Hennig's 'pattern' band (0.60-0.75): real")
        print(f"  structure, but exact membership is doubtful - frame accordingly:")
        for r in _pat.sort_values("jaccard").itertuples():
            print(f"    {r.level[:3]}{r.id:<3} Jaccard {r.jaccard:.2f}  "
                  f"{str(r.name)[:44]}")
for lv in _LV:
    _nz = int(NOISE.get(lv, 0))
    print(f"  {lv:<5} coverage: {N_ALL - _nz} of {N_ALL} responses sit in one of these "
          f"{int((E.level == lv).sum())} themes; {_nz} ({_nz / N_ALL:.0%}) sit in none and")
    print(f"        are NOT a theme in this table - they are the '(no theme)' share that "
          f"every prevalence figure shows.")
print(f"  CLAIMS are made at the {CLAIM_LEVEL.upper()} level only; the main level is a "
      f"presentation grouping.")

_art = E[E.tier == "ARTIFACT"]
if len(_art):
    print(f"\n  {len(_art)} theme(s) the naming engine or the verifier refused - describe "
          f"only, no claim:")
    for r in _art.itertuples():
        print(f"    {r.level[:3]}{r.id:<3} n={r.n_responses:<4} {r.reason[:70]}")

_desc = E[E.tier == "DESCRIPTIVE"]
if len(_desc):
    print(f"\n  {len(_desc)} theme(s) are sound but under-supported. They stay in every "
          f"figure, shown faded, and carry no test:")
    for r in _desc.head(12).itertuples():
        print(f"    {r.level[:3]}{r.id:<3} {r.reason:<44} {str(r.name)[:38]}")
    if len(_desc) > 12:
        print(f"    ... and {len(_desc) - 12} more (full list in theme_evidence.csv)")

_ib = E[E.instrument_bound]
if len(_ib):
    print(f"\n  {len(_ib)} theme(s) draw >= {QUESTION_BOUND:.0%} of their responses from a "
          f"single question - they may describe the question, not a cross-cutting")
    print(f"  experience. Name them as such in the write-up:")
    for r in _ib.head(8).itertuples():
        print(f"    {r.level[:3]}{r.id:<3} {r.top_question_share:.0%} from "
              f"'{r.top_question}'   {str(r.name)[:40]}")

_sw = E[E.summary_withheld]
if len(_sw):
    print(f"\n  {len(_sw)} theme(s) have NO publishable summary (the model's wording was "
          f"not grounded in its own evidence). Their codebook entry shows the")
    print(f"  cluster's own terms instead; do not quote a summary for these.")

# ------------------------------- figure ---------------------------------------
try:
    import matplotlib.pyplot as plt
    TIERC = {"CLAIMABLE": globals().get("C_GOOD", "#44AA99"),
             "DESCRIPTIVE": globals().get("C_NEUT", "#9AA5B1"),
             "ARTIFACT": globals().get("C_BAD", "#CC6677")}
    fig, ax = plt.subplots(figsize=(7.2, 2.4 + 0.42 * len(_LV)))
    ypos = np.arange(len(_LV))
    left = np.zeros(len(_LV))
    for tier in ("CLAIMABLE", "DESCRIPTIVE", "ARTIFACT"):
        vals = np.array([int((E[(E.level == lv)].tier == tier).sum()) for lv in _LV],
                        dtype=float)
        ax.barh(ypos, vals, left=left, color=TIERC[tier], edgecolor="white",
                height=.62, label=tier.title())
        for y, (v, l) in enumerate(zip(vals, left)):
            if v:
                ax.text(l + v / 2, y, f"{int(v)}", ha="center", va="center",
                        fontsize=9, color="white", fontweight="bold")
        left += vals
    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{lv}  (K={int((E.level == lv).sum())})" for lv in _LV],
                       fontsize=10)
    ax.set_xlabel("number of themes", fontsize=10)
    ax.set_xlim(0, left.max() * 1.02)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    ax.legend(loc="lower right", frameon=False, fontsize=8.5, ncol=3)
    if "fig_title" in globals():
        fig_title(fig, "Evidence available for each theme, by level")
    else:
        fig.suptitle("Evidence available for each theme, by level", fontsize=12)
    plt.tight_layout(rect=[0, 0, 1, .93])
    (save_fig(fig, "chart_theme_evidence") if "save_fig" in globals()
     else fig.savefig(f"{POOLED_DIR}/chart_theme_evidence.png", dpi=300,
                      bbox_inches="tight"))
    plt.close(fig)
except Exception as e:
    print(f"  (figure skipped: {type(e).__name__}: {str(e)[:80]})")

print(f"\n  -> theme_evidence.csv  ·  pooled_theme_evidence  ·  THEME_TIER / CLAIMABLE")
print("=" * 78)

### Stage 14 · What the clustering did not place

**Question.** A quarter of the corpus sits in no theme - what kind of responses are they?

**Reads.** `pooled_master`. **Writes.** `unplaced_responses.csv` - every unplaced response, readable, with its length, question and language flags.

**How to read it.** The answer changes what the themes mean. If the unplaced responses are short or empty, the themes describe the corpus and the residue is ordinary noise. If they are longer and more detailed than the placed ones, the themes describe the simpler responses and omit the complex ones - a different and more serious limitation. This cell decides which from the data and states it in words.

Density clustering places a response only where enough other responses lie close to it. A response that raises several subjects at once has an embedding between the dense regions rather than inside one, so it can be left out precisely because it says more than one thing. Density is a statement about company, not about quality.

In [ ]:
# @title
# ==============================================================================
# STAGE 14  -  WHAT THE CLUSTERING DID NOT PLACE
#             section 2 of 9  -  Is the analysis trustworthy?
# ==============================================================================
# ==============================================================================
# WHAT THE CLUSTERING DID NOT PLACE
# ------------------------------------------------------------------------------
# A quarter of the corpus sits in no theme. Before any figure is read, that share
# has to be characterised, because the answer changes what the themes mean:
#
#   * if the unplaced responses are short, empty or in another language, the
#     themes describe the whole corpus and the residue is noise in the ordinary
#     sense;
#   * if they are LONGER and more detailed than the placed ones, the themes
#     describe the simpler responses and systematically omit the complex ones -
#     which is a different, and more serious, limitation.
#
# This cell decides which it is from the data rather than from an assumption
# about the algorithm, and prints the answer in words. Nothing is filtered,
# imputed or dropped: it counts and compares.
#
# The mechanism worth keeping in mind while reading it. HDBSCAN places a response
# only where enough other responses lie close to it in the embedding space. A
# response that mixes several topics has an embedding that is an average of
# several directions, so it lands BETWEEN the dense regions rather than in one -
# no matter how well written it is. Density is a statement about company, not
# about quality.
#
# Outputs: unplaced_responses.csv (every unplaced response with its length,
# question and language flags), chart_unplaced_profile, pooled_unplaced.
# ==============================================================================
import re, unicodedata as ud
import numpy as np, pandas as pd, matplotlib.pyplot as plt

LV = CLAIM_LEVEL
D = MASTER.copy()
D["_w"] = D["text"].fillna("").astype(str).map(lambda t: len(str(t).split()))
D["_placed"] = D[f"has_{LV}"].astype(bool)


def _script(t):
    s = str(t)
    if not s.strip():
        return "empty"
    return ("non-latin" if any(ch.isalpha() and "LATIN" not in ud.name(ch, "")
                               for ch in s) else "latin")


# a light French cue: enough function words that the response is unlikely to be
# English. Deliberately crude and only used to rule the language explanation IN
# or OUT, never to label a response's language.
_FR = re.compile(r"\b(le|la|les|des|une?|et|est|dans|pour|avec|nous|que|qui|plus|"
                 r"tr[eè]s|beaucoup|[ée]quipe|projet|travail|m[eê]me|aussi)\b", re.I)
D["_script"] = D["text"].map(_script)
D["_fr_cue"] = D["text"].fillna("").astype(str).map(lambda t: len(_FR.findall(t)) >= 2)

UN, TH = D[~D._placed], D[D._placed]
print("=" * 80)
print("  WHAT THE CLUSTERING DID NOT PLACE")
print("=" * 80)
print(f"  {len(UN)} of {len(D)} responses ({len(UN)/len(D):.1%}) sit in no "
      f"{LV}-theme.")

# ------------------------------------------------------------------ 1. length
print("\n  1. LENGTH")
print(f"     {'':12}{'median':>8}{'mean':>8}{'1 word':>9}{'<=3 w':>8}{'>=30 w':>9}")
for lbl, d in (("unplaced", UN), ("placed", TH)):
    print(f"     {lbl:<12}{d._w.median():>8.0f}{d._w.mean():>8.1f}"
          f"{(d._w == 1).mean():>8.0%}{(d._w <= 3).mean():>8.0%}{(d._w >= 30).mean():>9.0%}")
BANDS = [(1, 1, "1"), (2, 3, "2-3"), (4, 9, "4-9"), (10, 29, "10-29"), (30, 10**9, "30+")]
band_rows = []
for lo, hi, lab in BANDS:
    d = D[(D._w >= lo) & (D._w <= hi)]
    if len(d):
        band_rows.append(dict(band=lab, n=len(d), unplaced=float((~d._placed).mean())))
B = pd.DataFrame(band_rows)
print(f"\n     share of each length band left unplaced:")
for r in B.itertuples():
    print(f"       {r.band:>6} words   n={r.n:>4}   unplaced {r.unplaced:>5.0%}")
_len_expl = float(UN._w.median()) < float(TH._w.median())

# ---------------------------------------------------------------- 2. question
print("\n  2. WHICH QUESTION")
Q = (D.groupby("question")
       .agg(n=("_placed", "size"), unplaced=("_placed", lambda s: float((~s).mean())),
            median_words=("_w", "median"))
       .sort_values("unplaced", ascending=False))
print(f"     {'question':<16}{'n':>6}{'unplaced':>10}{'median words':>14}")
for q, r in Q.iterrows():
    print(f"     {str(q)[:15]:<16}{int(r.n):>6}{r.unplaced:>9.0%}{r.median_words:>14.0f}")

# ---------------------------------------------------------------- 3. language
print("\n  3. LANGUAGE AND SCRIPT")
print(f"     {'':12}{'non-latin':>11}{'French cues':>13}{'empty':>8}")
for lbl, d in (("unplaced", UN), ("placed", TH)):
    print(f"     {lbl:<12}{(d._script == 'non-latin').mean():>10.0%}"
          f"{d._fr_cue.mean():>13.0%}{(d._script == 'empty').mean():>8.0%}")
_lang_expl = ((UN._script == "non-latin").mean() > 2 * max((TH._script == "non-latin").mean(), .01)
              or UN._fr_cue.mean() > 2 * max(TH._fr_cue.mean(), .01))

# --------------------------------------------------------------- 4. duplicates
_u = UN["text"].fillna("").astype(str).str.strip().str.lower()
print(f"\n  4. REPETITION: {len(UN)} unplaced responses, {_u.nunique()} distinct texts")
_rep = _u.value_counts()
_rep = _rep[_rep > 1]
if len(_rep):
    print(f"     repeated: " + "; ".join(f"{n}x {t[:34]!r}" for t, n in _rep.head(4).items()))

# --------------------------------------------------------------- the verdict
print("\n" + "-" * 80)
print("  WHAT THIS MEANS")
print("-" * 80)
if _len_expl:
    print("  The unplaced responses are SHORTER than the placed ones. Short text carries")
    print("  little to cluster on, so the residue is largely a length effect: the themes")
    print("  describe the substantive answers and omit the terse ones.")
else:
    print("  The unplaced responses are NOT shorter than the placed ones - they are as")
    print("  long or longer. Length does not explain the residue, so 'these were empty")
    print("  or one-word answers' is NOT available as an explanation.")
    print("  What remains is topical: a response that raises several subjects at once")
    print("  has an embedding between the dense regions rather than inside one, so the")
    print("  clustering leaves it out precisely BECAUSE it says more than one thing.")
if _lang_expl:
    print("  Non-English responses are over-represented among the unplaced ones; language")
    print("  is part of the explanation and belongs in the limitations.")
else:
    print("  Non-English and non-Latin responses are NOT over-represented, so language")
    print("  does not explain the residue either.")
_qmax, _qmin = Q.index[0], Q.index[-1]
print(f"  The residue is uneven across the instrument: {Q.iloc[0]['unplaced']:.0%} of "
      f"'{_qmax}' responses are unplaced against {Q.iloc[-1]['unplaced']:.0%} of "
      f"'{_qmin}'.")
print(f"  Any claim about '{_qmax}' therefore rests on a smaller and possibly less")
print("  complex slice of what participants wrote, and should say so.")

UN_OUT = UN[["row_id", "Participant", "Team", "question", "_w", "_script", "_fr_cue",
             "text"]].rename(columns={"_w": "n_words", "_script": "script",
                                      "_fr_cue": "french_cues"})
UN_OUT.to_csv(f"{POOLED_DIR}/unplaced_responses.csv", index=False)
psave("pooled_unplaced", dict(by_band=B, by_question=Q.reset_index(),
                              n_unplaced=len(UN), n_total=len(D),
                              length_explains=bool(_len_expl),
                              language_explains=bool(_lang_expl)))

# ------------------------------------------------------------------- figure
fig, ax = plt.subplots(1, 3, figsize=(13.6, 4.3),
                       gridspec_kw=dict(width_ratios=[1.15, 1.0, 1.15]))
a = ax[0]
bins = np.linspace(0, np.percentile(D._w, 98), 26)
a.hist(TH._w, bins=bins, color=SENT_POS, alpha=.75, label=f"placed (n={len(TH)})")
a.hist(UN._w, bins=bins, color=NO_THEME_COLOR, alpha=.85, label=f"unplaced (n={len(UN)})")
a.axvline(TH._w.median(), color=SENT_POS, ls="--", lw=1.2)
a.axvline(UN._w.median(), color="#5B6670", ls="--", lw=1.2)
a.set_xlabel("words in the response", fontsize=9)
a.set_ylabel("responses", fontsize=9)
a.legend(fontsize=8.2, frameon=False)
a.set_title("A  How long are they?", fontsize=10.5, fontweight="bold", loc="left")

b = ax[1]
b.bar(range(len(B)), 100 * B["unplaced"], color=NO_THEME_COLOR, width=.62,
      edgecolor="white")
for i, r in enumerate(B.itertuples()):
    b.text(i, 100 * r.unplaced + 1.2, f"{100*r.unplaced:.0f}%", ha="center", fontsize=8.6)
    b.text(i, 1.5, f"n={r.n}", ha="center", fontsize=7.6, color="white")
b.set_xticks(range(len(B))); b.set_xticklabels(B["band"], fontsize=8.6)
b.set_xlabel("length of the response, in words", fontsize=9)
b.set_ylabel("% left unplaced", fontsize=9)
b.set_ylim(0, max(45, 100 * B["unplaced"].max() + 8))
b.set_title("B  Which lengths are dropped?", fontsize=10.5, fontweight="bold", loc="left")

c = ax[2]
_q = Q.sort_values("unplaced")
c.barh(range(len(_q)), 100 * _q["unplaced"],
       color=[Q_COLORS.get(q, NO_THEME_COLOR) for q in _q.index],
       edgecolor="white", height=.62)
for i, (q, r) in enumerate(_q.iterrows()):
    c.text(100 * r.unplaced + .6, i, f"{100*r.unplaced:.0f}%  (n={int(r.n)})",
           va="center", fontsize=8.4)
c.set_yticks(range(len(_q)))
c.set_yticklabels([str(q)[:16] for q in _q.index], fontsize=8.6)
c.set_xlabel("% of that question's responses left unplaced", fontsize=9)
c.set_xlim(0, 100 * _q["unplaced"].max() * 1.35)
c.set_title("C  Which questions?", fontsize=10.5, fontweight="bold", loc="left")

for _a in ax:
    for sp in ("top", "right"):
        _a.spines[sp].set_visible(False)
_head = ("The unplaced responses are not the short ones"
         if not _len_expl else "The unplaced responses are the short ones")
fig_title(fig, f"What the clustering did not place — {_head}")
fig.text(0.5, 0.012,
         f"{len(UN)} of {len(D)} responses ({len(UN)/len(D):.0%}) sit in no theme. "
         f"Density clustering places a response only where enough others lie near it, "
         f"so a response that raises several subjects at once can fall between the "
         f"dense regions however well written it is. These responses are described "
         f"here and in unplaced_responses.csv; they enter no test.",
         ha="center", fontsize=8.3, style="italic", color=FOOT, wrap=True)
plt.tight_layout(rect=[0, 0.045, 1, 0.92])
save_fig(fig, "chart_unplaced_profile"); plt.close(fig)

print(f"\n  -> unplaced_responses.csv (every unplaced response, readable) · "
      f"pooled_unplaced · chart_unplaced_profile")
print("=" * 80)


### Stage 15 · What Notebook 2 recovered

**Question.** The previous cell showed the unplaced responses are not the short ones. Which of them did Notebook 2 place, and how reliably?

**Reads.** `pooled_membership`, `pooled_membership_calibration`, `pooled_master`. **Writes.** `recovered_responses.csv` and one figure. **It decides nothing.**

**How to read it.** Assigning outliers after clustering, rather than tuning the clustering until there are none, is the documented practice in this literature: BERTopic exposes it as `reduce_outliers` and warns that reduction can force responses into themes where they do not belong. Tuning HDBSCAN is not available here — Notebook 2's own screen found no configuration with materially less noise that also cleared the stability floor.

So the partition is left exactly as it is, and Notebook 2 Stage 23d scores **three** independent rules against each other — HDBSCAN's own soft membership, cosine to the theme centroid in the original embedding space, and agreement across the top admissible configurations — then uses whichever places the most responses *correctly*. The thresholds are calibrated on the most **peripheral** core rows, the ones that actually resemble the rows being recovered; calibrating on all core rows flatters the number, because core rows sit inside dense regions by construction. Stage 23M then splits the long, still-unplaced responses into clauses, so an answer raising three subjects can carry three themes.

**This cell used to do all of that itself, and overwrote Notebook 2's answer with a weaker one.** It now reads the result and checks the two notebooks agree. `has_sub` and `sub_id` still mean "placed by the clustering itself", so the conservative view is always available; headline cells use the wide columns and print both numbers.

In [ ]:
# @title
# ==============================================================================
# STAGE 15  -  WHAT NOTEBOOK 2 RECOVERED
#             section 2 of 9  -  Is the analysis trustworthy?
# ==============================================================================
# ==============================================================================
# WHAT NOTEBOOK 2 RECOVERED, AND HOW WELL   (this cell READS; it decides nothing)
# ------------------------------------------------------------------------------
# WHAT THIS CELL USED TO BE
#   It re-fitted HDBSCAN with prediction_data, built soft-membership and
#   centroid-cosine rules, swept thresholds, picked a mode, and wrote theme_tier
#   / sub_assigned / sub_conf / sub_multi onto pooled_master. All of that now
#   happens in Notebook 2 - Stage 23d for the assignment and Stage 23M for the
#   clause-level multi-label - so keeping it here meant the same decision was
#   made twice, by two different pieces of code, and whichever ran last won.
#
#   That was not hypothetical. On the last full run this cell recomputed the
#   recovery with its OWN rule ("centroid cosine alone", calibrated on ALL core
#   rows) and overwrote Notebook 2's answer, silently. Two differences mattered:
#
#     * calibrating on ALL core rows overstates accuracy, because core rows sit
#       inside dense regions by construction and are therefore easy. Notebook 2
#       calibrates on the most PERIPHERAL core rows - the ones that resemble the
#       rows being recovered - and the same threshold scores several points lower
#       there. The number this cell used to print was the flattering one.
#     * this cell has no concept of the clause-level multi-label, so every
#       response Stage 23M split into clauses was recorded here as unassigned.
#
# WHAT IT IS NOW
#   A reader and an agreement check. It loads pooled_membership, confirms it
#   matches pooled_master row for row, reports the rule Notebook 2 chose and the
#   accuracy it measured, and draws the same three-panel figure from those
#   numbers - now including the assigned-multi tier the old version could not
#   show. If the two notebooks ever disagree again, this cell is where it
#   surfaces, loudly, instead of being overwritten.
#
# Reads : pooled_membership, pooled_membership_calibration, pooled_master
# Writes: recovered_responses.csv, chart_recovery   (no artifact, no decision)
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt

LV = CLAIM_LEVEL

MB = pload("pooled_membership")
if MB is None:
    raise RuntimeError(
        "pooled_membership is missing. It is written by Notebook 2 Stage 23d and "
        "updated by Stage 23M. This notebook no longer computes the recovery "
        "itself, so there is nothing to fall back on - re-run those two stages, "
        "then Stage 33b.")
MB = MB[MB["source"] == "pooled"].sort_values("row_id").reset_index(drop=True)
if len(MB) != N_ALL:
    raise RuntimeError(f"pooled_membership has {len(MB)} pooled rows but "
                       f"pooled_master has {N_ALL} - they are from different "
                       f"runs. Re-run Notebook 2 Stage 33b.")

# ---- the agreement check: two notebooks, one answer ---------------------------
tier = MASTER["theme_tier"].astype(str).values
if not np.array_equal(tier, MB["tier"].astype(str).values):
    _n = int((tier != MB["tier"].astype(str).values).sum())
    raise RuntimeError(
        f"pooled_master and pooled_membership disagree about the tier of {_n} "
        f"response(s). Stage 33b copies one into the other, so this means one of "
        f"them is stale - re-run Notebook 2 Stage 33b.")
_idw = np.asarray(MASTER[f"{LV}_id_wide"], dtype=int)
if not np.array_equal(_idw, np.asarray(MB["sub_wide"], dtype=int)):
    raise RuntimeError("pooled_master's *_id_wide does not match "
                       "pooled_membership's sub_wide - re-run NB2 Stage 33b.")
_core = np.asarray(MASTER[f"has_{LV}"], dtype=bool)
if not np.array_equal(_idw[_core], np.asarray(MASTER[f"{LV}_id"])[_core]):
    raise RuntimeError("the wide view changed a CORE row's theme; recovery may "
                       "only ever ADD. Re-run Notebook 2 Stage 23d.")

n = {t: int((tier == t).sum()) for t in
     ("core", "assigned", "assigned-multi", "unassigned")}
n_wide = n["core"] + n["assigned"] + n["assigned-multi"]

print("=" * 82)
print("  WHAT NOTEBOOK 2 RECOVERED")
print("=" * 82)
print(f"  {N_ALL} responses.  The partition is Notebook 2's and is not touched here.")
if RECOVERY_ACC is not None:
    print(f"  rule chosen by Notebook 2 Stage 23d : {RECOVERY_RULE}")
    print(f"  its measured accuracy               : {RECOVERY_ACC:.1%}")
    print( "    measured on the most PERIPHERAL core rows - the ones that resemble")
    print( "    the rows being recovered, not the easy ones. Still an upper bound:")
    print( "    every calibration row HAS a theme, and these rows did not.")
else:
    print("  (pooled_projection_validation absent - the rule's accuracy cannot be")
    print("   quoted here; see Notebook 2 Stage 23d's own output.)")

print()
print(f"  {'tier':<18}{'n':>7}{'% corpus':>11}   what it means")
print("  " + "-" * 74)
_MEAN = {
    "core":           "placed by HDBSCAN itself",
    "assigned":       "recovered by the calibrated rule above",
    "assigned-multi": "split into clauses by Stage 23M; carries several themes",
    "unassigned":     "no theme could be justified - reported, never forced",
}
for t in ("core", "assigned", "assigned-multi", "unassigned"):
    if n[t]:
        print(f"  {t:<18}{n[t]:>7}{n[t] / N_ALL:>10.1%}   {_MEAN[t]}")
print("  " + "-" * 74)
print(f"  {'wide view':<18}{n_wide:>7}{n_wide / N_ALL:>10.1%}   "
      f"core-only view is {n['core']} ({n['core'] / N_ALL:.1%})")

if "unassigned_reason" in MASTER.columns:
    _ur = MASTER.loc[tier == "unassigned", "unassigned_reason"].astype(str)
    if len(_ur):
        print("\n  why the unassigned rows are unassigned (recorded by NB2, never blank):")
        _W = {"thin": "too short to carry a theme - correctly left out",
              "rare": "no admissible configuration placed them; nothing in the "
                      "corpus resembles them",
              "below_threshold": "placeable in principle, not confidently enough "
                                 "to claim"}
        for r, c in _ur.value_counts().items():
            print(f"    {str(r):<18}{int(c):>5}   {_W.get(str(r), '')}")

_multi_col = f"{LV}_multi"
if _multi_col in MASTER.columns:
    _nm = int(MASTER[_multi_col].astype(str).str.contains(";").sum())
    if _nm:
        print(f"\n  {_nm} response(s) carry more than one theme (Stage 23M clause "
              f"split);\n  their primary theme is the strongest clause's, and "
              f"{_multi_col} lists them all.")

# ---- the readable table of what was recovered --------------------------------
_cols = [c for c in ["row_id", "Participant", "Team", "question", "n_words",
                     f"{LV}_assigned", f"{LV}_conf", f"{LV}_multi",
                     "assignment_rule", "text"] if c in MASTER.columns]
_rec = MASTER[np.isin(tier, ["assigned", "assigned-multi"])][_cols].copy()
if f"{LV}_assigned" in _rec.columns:
    _rec.insert(_rec.columns.get_loc(f"{LV}_assigned") + 1, "assigned_name",
                [theme_name(LV, int(c)) for c in _rec[f"{LV}_assigned"]])
_rec.to_csv(f"{POOLED_DIR}/recovered_responses.csv", index=False)

print(f"\n  HOW TO USE THIS. has_{LV} and {LV}_id still mean 'placed by the")
print(f"  clustering itself', so any cell wanting the conservative view keeps it.")
print(f"  Headline cells use has_{LV}_wide / {LV}_id_wide through _cov_cols(), and")
print(f"  print _cov_note() so the core-only number is always visible beside it.")

# --------------------------------------------------------------------- figure
try:
    CAL = pload("pooled_membership_calibration")
    _npanel = 3 if (isinstance(CAL, pd.DataFrame) and len(CAL)) else 2
    fig, ax = plt.subplots(1, _npanel, figsize=(4.5 * _npanel + 0.4, 4.2))
    ax = np.atleast_1d(ax)

    # A - where each response ended up
    a = ax[0]
    _t = [t for t in ("core", "assigned", "assigned-multi", "unassigned") if n[t]]
    _v = [n[t] for t in _t]
    _c = {"core": SENT_POS, "assigned": "#7FBFAE", "assigned-multi": "#B7D9CF",
          "unassigned": NO_THEME_COLOR}
    _lb = {"core": "core\n(clustering)", "assigned": "assigned\n(calibrated rule)",
           "assigned-multi": "assigned-multi\n(clause split)",
           "unassigned": "unassigned\n(reason recorded)"}
    a.bar(range(len(_t)), _v, color=[_c[t] for t in _t], width=.62, edgecolor="white")
    for i, v in enumerate(_v):
        a.text(i, v + N_ALL * .012, f"{v}\n{v / N_ALL:.0%}", ha="center", fontsize=9.0)
    a.set_xticks(range(len(_t)))
    a.set_xticklabels([_lb[t] for t in _t], fontsize=8.2)
    a.set_ylabel("responses", fontsize=9.5)
    a.set_ylim(0, max(_v) * 1.24)
    a.set_title("A  Where each response ends up", fontsize=10.5,
                fontweight="bold", loc="left")

    # B - Notebook 2's calibration, if it published the curves
    if _npanel == 3:
        b = ax[1]
        _cols3 = {"R1 centroid-cosine": "#44AA99", "R2 soft-membership": "#6699CC",
                  "R3 grid-consensus": "#D9A44E"}
        for _rule, d in CAL.groupby("rule"):
            d = d.sort_values("n_noise_placed")
            b.plot(d["n_noise_placed"], 100 * d["accuracy"], "o-", ms=3.4, lw=1.2,
                   color=_cols3.get(str(_rule), "#999"), label=str(_rule))
        if RECOVERY_ACC is not None:
            b.axhline(100 * RECOVERY_ACC, color=SENT_NEG, ls="--", lw=1.1)
            b.text(b.get_xlim()[0], 100 * RECOVERY_ACC + .5,
                   f" chosen: {RECOVERY_ACC:.0%}", fontsize=7.6, va="bottom",
                   color=SENT_NEG)
        b.set_xlabel("responses the threshold would place", fontsize=9.2)
        b.set_ylabel("accuracy on the hard core rows (%)", fontsize=9.2)
        b.legend(fontsize=7.2, frameon=False, loc="lower left")
        b.set_title("B  Three rules, one calibration (NB2)", fontsize=10.5,
                    fontweight="bold", loc="left")

    # C - what recovery fixed, by question
    c = ax[-1]
    _q = (MASTER.assign(_no=lambda d: d["theme_tier"].astype(str) != "core",
                        _still=lambda d: d["theme_tier"].astype(str) == "unassigned")
                .groupby("question")
                .agg(unplaced=("_no", "mean"), still=("_still", "mean"))
                .sort_values("unplaced"))
    y = np.arange(len(_q))
    c.barh(y, 100 * _q["unplaced"], color=NO_THEME_COLOR, height=.6,
           label="before recovery", edgecolor="white")
    c.barh(y, 100 * _q["still"], color=SENT_NEG, height=.6,
           label="still unassigned", edgecolor="white")
    for i, (_qq, r) in enumerate(_q.iterrows()):
        c.text(100 * r["unplaced"] + .7, i,
               f"{100 * r['unplaced']:.0f}% -> {100 * r['still']:.0f}%",
               va="center", fontsize=8.2)
    c.set_yticks(y)
    c.set_yticklabels([str(q)[:16] for q in _q.index], fontsize=8.6)
    c.set_xlabel("% of that question's responses with no theme", fontsize=9.2)
    c.set_xlim(0, max(1.0, 100 * float(_q["unplaced"].max()) * 1.5))
    c.legend(fontsize=8, frameon=False, loc="lower right")
    c.set_title("C  What recovery fixes, by question", fontsize=10.5,
                fontweight="bold", loc="left")

    for _a in ax:
        for sp in ("top", "right"):
            _a.spines[sp].set_visible(False)
    fig_title(fig, f"What Notebook 2 recovered - coverage {n['core'] / N_ALL:.0%} "
                   f"to {n_wide / N_ALL:.0%}"
                   + (f" at {RECOVERY_ACC:.0%} rule accuracy" if RECOVERY_ACC
                      else ""))
    fig.text(0.5, 0.012,
             f"Decided in Notebook 2, not here. Three rules were calibrated on the "
             f"most peripheral core rows - the ones that resemble the rows being "
             f"recovered - and the winner was the one placing the most responses "
             f"correctly, not the most responses. {n['unassigned']} remain "
             f"unassigned, each with a recorded reason, and are still reported as "
             f"such.",
             ha="center", fontsize=8.2, style="italic", color=FOOT, wrap=True)
    plt.tight_layout(rect=[0, 0.045, 1, 0.92])
    save_fig(fig, "chart_recovery")
    plt.close(fig)
except Exception as _e:
    print(f"\n  (figure skipped: {type(_e).__name__}: {str(_e)[:90]})")

print(f"\n  -> recovered_responses.csv ({len(_rec)} rows, readable) · chart_recovery")
print("  This cell writes no artifact and makes no decision. Membership lives in")
print("  Notebook 2; pooled_master carries it; every figure here reads it.")
print("=" * 82)

### Stage 16 · Are the themes just the questionnaire?

**Question.** Did pooling five open questions recover the questions rather than anything about the experiences?

**Reads.** `pooled_master`. **Writes.** `pooled_question_theme`.

**How to read it.** Some association is expected - a question about funding attracts funding talk - so the effect size matters more than the p-value, and the p-value itself comes from a permutation that shuffles themes *within* each participant, because responses from one person are not independent. The finding that matters is the spread: a theme confined to a single question is an instrument artefact, one drawn from several is a cross-cutting construct.

In [ ]:
# @title
# ==============================================================================
# STAGE 16  -  ARE THE THEMES JUST THE QUESTIONNAIRE?
#             section 2 of 9  -  Is the analysis trustworthy?
# ==============================================================================
# WHY THIS IS THE FIRST THING A REVIEWER WILL ASK
#   Five open questions were pooled into ONE embedding space and clustered together.
#   The obvious objection is that the clusters simply recovered the five questions -
#   in which case the "themes" are an artefact of the instrument and nothing was
#   learned. This cell answers that objection with a measurement, not an assertion.
#
# WHAT IS MEASURED
#   1. ASSOCIATION. A question x theme contingency table, chi-square, and Cramer's V
#      as the effect size. V = 0 means the theme a response falls into is unrelated
#      to which question it answers; V = 1 means the theme is fully determined by it.
#   2. A HONEST NULL. Responses are NOT independent - one participant contributes up
#      to five of them - so the textbook chi-square p-value is anticonservative here.
#      The p-value is therefore obtained by PERMUTATION: the theme labels are shuffled
#      WITHIN each participant, which preserves every participant's own set of themes
#      and every question's marginal while destroying the question-to-theme pairing.
#      That is the null "this participant's themes have nothing to do with which
#      question produced them".
#   3. SPREAD PER THEME. For each theme, the share of its responses coming from each
#      question, summarised as normalised entropy: 0 = the theme lives in a single
#      question (an instrument artefact), 1 = it is spread evenly across all five (a
#      genuinely cross-cutting theme). The distribution of this number is the answer.
#
# HOW TO READ THE RESULT
#   A moderate association is EXPECTED and is not a problem: a question about funding
#   will of course attract funding talk. The finding that matters is whether themes
#   are CONFINED to single questions. Report the entropy distribution, not just V.
#
# Nothing here is estimated or imputed: every number is a count of real responses.
# Outputs: chart_question_theme, question_theme_table.csv, question_theme_spread.csv,
#          pooled_question_theme.
# Refs: Cramer (1946) V; permutation inference under clustered sampling.
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

_PL = globals().get("_orig_pload", pload)
M = _PL("pooled_master")
if M is None:
    raise RuntimeError("Need pooled_master - run NB2 Stage 33b first.")
M = M.copy()

N_PERM   = 2000    # permutations for the clustered null
MIN_CELL = 20      # a theme needs this many responses to appear in the spread table
SEED     = int(globals().get("CFG").seed) if "CFG" in globals() else 42

print("=" * 76)
print("  STAGE 15c  ARE THE THEMES JUST THE QUESTIONNAIRE?")
print("=" * 76)

rng = np.random.default_rng(SEED)
res_all = {}
# Levels come from the preflight. The literal ("main","mid","sub") did not merely
# waste a loop: dropna(subset=["mid_id"]) on a frame without that column raises
# KeyError, so this cell stopped the notebook once the mid level was removed.
for lv in LEVELS:
    idc, namec, hasc = _cov_cols(lv, M)
    print(_cov_note(lv, M))
    if idc not in M.columns or hasc not in M.columns:
        print(f"  {lv}: no {idc}/{hasc} column - skipped"); continue
    # NOISE. -1 is not NaN, so dropna() kept it and it entered the contingency
    # table as a theme of its own. Several hundred responses that belong to no
    # theme, spread across every question, would push Cramer's V DOWN and make the
    # themes look more cross-cutting than they are - the error would have flattered
    # the result, which is the worst direction for it to go.
    D = M[M[hasc].astype(bool)].dropna(subset=["question"]).copy()
    D["theme"] = D[idc].astype(int)
    if (D["theme"] < 0).any():
        raise RuntimeError(f"[15c] rows with {idc} == -1 survived the has_{lv} filter")
    T = pd.crosstab(D["question"], D["theme"])
    if T.shape[0] < 2 or T.shape[1] < 2:
        print(f"  {lv}: table too small - skipped"); continue
    chi2, p_asym, dof, exp = chi2_contingency(T.values)
    n = int(T.values.sum())
    V = float(np.sqrt(chi2 / (n * (min(T.shape) - 1))))

    # permutation null: shuffle theme labels WITHIN participant
    parts = D["Participant"].values
    themes = D["theme"].values
    qs = D["question"].values
    order = np.argsort(parts, kind="stable")
    p_sorted, t_sorted, q_sorted = parts[order], themes[order], qs[order]
    bounds = np.flatnonzero(np.r_[True, p_sorted[1:] != p_sorted[:-1], True])
    qcodes, quniq = pd.factorize(q_sorted)
    tcodes, tuniq = pd.factorize(t_sorted)
    nq, nt = len(quniq), len(tuniq)
    def _V_from(tc):
        tab = np.zeros((nq, nt), dtype=np.int64)
        np.add.at(tab, (qcodes, tc), 1)
        keep_r = tab.sum(1) > 0; keep_c = tab.sum(0) > 0
        tb = tab[np.ix_(keep_r, keep_c)]
        if min(tb.shape) < 2: return np.nan
        c2 = chi2_contingency(tb)[0]
        return float(np.sqrt(c2 / (tb.sum() * (min(tb.shape) - 1))))
    null = np.empty(N_PERM)
    for b in range(N_PERM):
        tc = tcodes.copy()
        for i in range(len(bounds) - 1):
            a, z = bounds[i], bounds[i + 1]
            if z - a > 1:
                tc[a:z] = rng.permutation(tc[a:z])
        null[b] = _V_from(tc)
    null = null[np.isfinite(null)]
    p_perm = float((np.sum(null >= V) + 1) / (len(null) + 1))

    # spread of each theme across the questions
    S = T.T.copy()
    S["n"] = S.sum(axis=1)
    S = S[S["n"] >= MIN_CELL]
    qcols = [c for c in S.columns if c != "n"]
    P = S[qcols].div(S["n"], axis=0).values
    with np.errstate(divide="ignore", invalid="ignore"):
        H = -np.nansum(np.where(P > 0, P * np.log(P), 0.0), axis=1) / np.log(len(qcols))
    S["dominant_question"] = [qcols[i] for i in np.argmax(S[qcols].values, axis=1)]
    S["dominant_share"] = S[qcols].max(axis=1).values / S["n"].values
    S["spread"] = H
    nmm = {int(r[idc]): str(r[namec]) for _, r in
           D.drop_duplicates(subset=[idc]).iterrows()}
    S.insert(0, "name", [nmm.get(int(i), f"{lv}{i}") for i in S.index])
    S = S.sort_values("spread")
    S.attrs["qcols"] = list(qcols)

    print(f"\n  --- {lv.upper()}: {T.shape[0]} questions x {T.shape[1]} themes, "
          f"{n} responses ---")
    print(f"    Cramer's V = {V:.3f}   (0 = theme unrelated to question, "
          f"1 = fully determined by it)")
    print(f"    permutation p (theme labels shuffled within participant, "
          f"{len(null)} draws) = {p_perm:.4f}")
    print(f"    null V: mean {null.mean():.3f}, 95th pct {np.percentile(null, 95):.3f}")
    print(f"    (textbook chi-square p = {p_asym:.2e}, reported only for completeness -"
          f" it assumes independent responses, which these are not)")
    conf = int((S["dominant_share"] >= 0.80).sum())
    print(f"    themes with >=80% of their responses from ONE question: "
          f"{conf} of {len(S)}   <- these are the instrument-bound ones")
    print(f"    spread (0 = single question, 1 = even across all {len(qcols)}): "
          f"median {S['spread'].median():.2f}, "
          f"quartiles {S['spread'].quantile(.25):.2f}-{S['spread'].quantile(.75):.2f}")
    print(f"    most question-bound: ", end="")
    r0 = S.iloc[0]
    print(f"{r0['name'][:34]} ({100*r0['dominant_share']:.0f}% from {r0['dominant_question']})")
    r1 = S.iloc[-1]
    print(f"    most cross-cutting : {r1['name'][:34]} (spread {r1['spread']:.2f})")

    T.to_csv(f"{POOLED_DIR}/question_theme_table_{lv}.csv")
    S.to_csv(f"{POOLED_DIR}/question_theme_spread_{lv}.csv")
    res_all[lv] = dict(V=V, p_perm=p_perm, p_chi2=float(p_asym), n=n,
                       null_mean=float(null.mean()), table=T, spread=S)

psave("pooled_question_theme", res_all)

# ---- figure: residual heatmap (main) + spread distribution (mid) --------------
try:
    lv0 = "main" if "main" in res_all else list(res_all)[0]
    T = res_all[lv0]["table"]
    chi2, _, _, exp = chi2_contingency(T.values)
    resid = (T.values - exp) / np.sqrt(exp)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.4),
                                 gridspec_kw=dict(width_ratios=[1.35, 1]))
    lim = float(np.abs(resid).max())
    im = a1.imshow(resid, cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="auto")
    a1.set_xticks(range(T.shape[1]))
    a1.set_xticklabels([f"{lv0[0]}{c}" for c in T.columns], fontsize=8)
    a1.set_yticks(range(T.shape[0])); a1.set_yticklabels(T.index, fontsize=8.5)
    a1.set_xlabel(f"{lv0} theme", fontsize=9)
    a1.set_ylabel("survey question", fontsize=9)
    for i in range(resid.shape[0]):
        for j in range(resid.shape[1]):
            if abs(resid[i, j]) >= 2:
                a1.text(j, i, f"{resid[i,j]:+.0f}", ha="center", va="center",
                        fontsize=7, color="white" if abs(resid[i, j]) > lim*.6 else "#14202B")
    cb = fig.colorbar(im, ax=a1, fraction=.035, pad=.02)
    cb.set_label("more (blue) or fewer (red) responses than\nindependence predicts "
                 "(standardised residual)", fontsize=7.5)

    # the spread panel shows the CLAIM level - the one every later test uses
    lvS = CLAIM_LEVEL if CLAIM_LEVEL in res_all else lv0
    S = res_all[lvS]["spread"]
    a2.hist(S["spread"].values, bins=np.linspace(0, 1, 11),
            color=globals().get("C_GOOD", "#0F6E56"), edgecolor="white")
    a2.axvline(float(S["spread"].median()), color="#14202B", ls="--", lw=1.1)
    a2.text(float(S["spread"].median()), a2.get_ylim()[1]*.94,
            f" median {S['spread'].median():.2f}", fontsize=8.5, va="top")
    # the question count belongs to THIS level's table; `qcols` used to leak in
    # from whichever level the loop above happened to finish on
    _nq = len(S.attrs.get("qcols", [])) or len(QL_POOL)
    a2.set_xlabel(f"how evenly a {lvS} theme is spread across the {_nq} pooled questions\n"
                  f"(0 = it lives in one question, 1 = even across all {_nq})", fontsize=9)
    a2.set_ylabel(f"number of {lvS} themes", fontsize=9)
    for sp in ("top", "right"): a2.spines[sp].set_visible(False)
    ttl = (f"Themes versus questionnaire structure — Cramer's V = "
           f"{res_all[lv0]['V']:.2f} at {lv0} level")
    try:    fig_title(fig, ttl)
    except Exception: fig.suptitle(ttl, fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, .91])
    try:    save_fig(fig, "chart_question_theme")
    except Exception:
        fig.savefig(f"{POOLED_DIR}/chart_question_theme.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    print("\n  figure: chart_question_theme")
except Exception as _e:
    print(f"\n  (figure skipped: {type(_e).__name__}: {str(_e)[:80]})")

print("-" * 76)
print("  HOW TO STATE THIS IN THE PAPER: report Cramer's V with the permutation p, and")
print("  the share of themes confined to a single question. An association is expected")
print("  and is not itself a problem - the claim the pooled design has to support is")
print("  that themes are not CONFINED to single questions.")
print(f"  Computed on responses that HAVE a theme; the {NOISE[CLAIM_LEVEL]} with none "
      f"({NOISE[CLAIM_LEVEL]/N_ALL:.0%}) are")
print("  excluded here - they are not a theme, and including them would have lowered")
print("  Cramer's V and flattered the result.")
print("  files: " + " · ".join(f"question_theme_table_{lv}.csv" for lv in res_all)
      + " · " + " · ".join(f"question_theme_spread_{lv}.csv" for lv in res_all))
print("=" * 76)


### Stage 17 · The manual codebook

**Question.** Where is the independently produced codebook, and is it the right file?

**How to read it.** Nothing to do on a normal run: the cell finds the codebook on this runtime, then in mounted Google Drive, then by its Drive file id, and only asks for an upload if all three fail. It validates the file rather than merely loading it - a JSON that parses but carries no theme definitions would produce an alignment figure built on nothing.

In [ ]:
# @title
# ==============================================================================
# STAGE 17  -  THE MANUAL CODEBOOK
#             section 2 of 9  -  Is the analysis trustworthy?
# ==============================================================================
# ==============================================================================
# THE MANUAL CODEBOOK
# ------------------------------------------------------------------------------
# The next cell compares the machine partition with the codebook a human coder
# produced independently. This cell locates that codebook and validates it.
#
# It tries four sources in order and stops at the first that works, so on a normal
# run there is nothing to do:
#   1. a copy already on this runtime;
#   2. Google Drive, if it is mounted - the file lives there and is shared with
#      this account, so no upload is needed;
#   3. a direct fetch by Drive file id;
#   4. manual upload, as the last resort.
#
# The file is VALIDATED, not just found: a JSON that parses but has no themes, or
# themes with no definitions, would silently produce an alignment figure built on
# nothing. If validation fails the cell says which key is wrong rather than
# letting the next cell fail somewhere deeper.
# ==============================================================================
import os, json, shutil, glob

# ------------------------------- EDIT HERE ------------------------------------
CODEBOOK_JSON = "/content/codebook.json"          # where it ends up
CODEBOOK_DRIVE_ID = "YOUR_CODEBOOK_FILE_ID"   # the shared copy
CODEBOOK_DRIVE_GLOBS = [                           # searched if Drive is mounted
    "/content/drive/MyDrive/**/codebook.json",
    "/content/drive/Shareddrives/**/codebook.json",
    "/content/drive/MyDrive/**/*codebook*.json",
]
# ----------------------------- END EDIT ---------------------------------------

_found_by = None

# ---- 1. already here ---------------------------------------------------------
if os.path.exists(CODEBOOK_JSON):
    _found_by = "already on this runtime"

# ---- 2. Google Drive, if mounted ---------------------------------------------
if _found_by is None and os.path.isdir("/content/drive"):
    for pat in CODEBOOK_DRIVE_GLOBS:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            shutil.copy(hits[0], CODEBOOK_JSON)
            _found_by = f"Google Drive: {hits[0]}"
            break

# ---- 3. direct fetch by file id ----------------------------------------------
if _found_by is None and CODEBOOK_DRIVE_ID:
    try:
        import gdown
        gdown.download(id=CODEBOOK_DRIVE_ID, output=CODEBOOK_JSON, quiet=True)
        if os.path.exists(CODEBOOK_JSON) and os.path.getsize(CODEBOOK_JSON) > 0:
            _found_by = f"Drive file id {CODEBOOK_DRIVE_ID}"
    except Exception as _e:
        print(f"  (direct fetch unavailable: {type(_e).__name__})")

# ---- 4. manual upload --------------------------------------------------------
if _found_by is None:
    print("  codebook.json was not found automatically.")
    print("  Mount Drive (Files panel, left) and re-run this cell, or choose the file now.")
    try:
        from google.colab import files
        _up = files.upload()
        if not _up:
            raise RuntimeError("no file chosen")
        _src = next(iter(_up))
        if os.path.abspath(_src) != os.path.abspath(CODEBOOK_JSON):
            shutil.move(_src, CODEBOOK_JSON)
        _found_by = "manual upload"
    except Exception as _e:
        raise RuntimeError(
            "the manual codebook could not be located. Either mount Google Drive and "
            "re-run this cell, or set CODEBOOK_JSON above to its path. Without it the "
            "next cell (alignment with the manual codebook) cannot run; every other "
            "section of this notebook is unaffected.") from _e

# ---- validate ----------------------------------------------------------------
with open(CODEBOOK_JSON, encoding="utf-8") as f:
    _cb = json.load(f)
_missing = [k for k in ("themes", "parent_themes") if k not in _cb]
if _missing:
    raise RuntimeError(f"{CODEBOOK_JSON} has no {_missing}. This is not the codebook "
                       f"the alignment cell expects - check the file.")
_themes = _cb["themes"]
_nodef = [t.get("code", "?") for t in _themes if not str(t.get("definition", "")).strip()]
_noq = [t.get("code", "?") for t in _themes if not (t.get("example_quotes") or [])]

print("=" * 74)
print("  MANUAL CODEBOOK")
print("=" * 74)
print(f"  source : {_found_by}")
print(f"  path   : {CODEBOOK_JSON}")
print(f"  content: {len(_themes)} themes in {len(_cb['parent_themes'])} parent categories")
if _nodef:
    print(f"  note   : {len(_nodef)} theme(s) have no definition ({_nodef[:6]}). The probe "
          f"for those rests on the label and quotes alone, which is weaker.")
if _noq:
    print(f"  note   : {len(_noq)} theme(s) have no example quotes ({_noq[:6]}).")
_mn = str(_cb.get("methodology_note", "")).strip()
if _mn:
    print(f"  its own methodology note: {_mn[:160]}")
print("  This coding was made on the same survey but not on the same rows, so the")
print("  next cell reports convergence between two independent codings - not the")
print("  accuracy of one against the other.")
print("=" * 74)


### Stage 18 · Alignment with the manual codebook

**Question.** Does the clustering recover the themes a human coder identified independently?

**Reads.** `codebook.json`, `pooled_embeddings`, the theme labels.

**How to read it.** Each hand-written theme is embedded in the same space as the responses and matched against the cluster centroids. A row with one marked cell means the hand theme was recovered as a single cluster; several means the clustering split it; an empty row means it was not recovered. The two codings were made on overlapping but not identical material, so this is convergence, not accuracy.

In [ ]:
# @title
# ==============================================================================
# STAGE 18  -  ALIGNMENT WITH THE MANUAL CODEBOOK
#             section 2 of 9  -  Is the analysis trustworthy?
# ==============================================================================
# WHAT THIS IS FOR
#   A second analyst read the same survey by hand - reflexive thematic analysis,
#   19 fine themes inside 5 parent categories, each with a definition, a valence
#   and example quotes. This cell asks, for every one of those hand-made themes:
#   WHERE does it land in the clustering, and does it land in ONE place?
#     * a theme that maps onto exactly one cluster is a convergence between two
#       completely different methods, and is the strongest evidence in the project
#     * a theme that spreads across several clusters means the clustering SPLIT
#       something a human treated as one idea - the cell names the pieces
#     * a theme with no counterpart means the clustering did not recover it
#   And symmetrically: a cluster that no hand theme resembles is either something
#   the manual coding missed, or an artifact. Stage 14c's tier says which, so the
#   two cells are meant to be read together.
#
# WHAT IS ACTUALLY BEING COMPARED - read this before quoting a number
#   Without the manual per-response codes this is a CONCEPTUAL alignment, not an
#   agreement statistic. Each hand theme becomes a probe (its label, definition
#   and example quotes), embedded in the SAME space as the responses, and compared
#   against each cluster's centroid of REAL responses. That measures whether the
#   idea a human wrote down lives where the algorithm put things. It is NOT
#   inter-rater agreement and must never be reported as kappa.
#
#   The two analyses also do not cover the same rows: the hand coding used the
#   whole instrument, the clustering was fitted on narrative answers of at least
#   three words. Both denominators are printed - never let a reader assume they
#   are one corpus.
#
# Outputs: codebook_alignment.csv, codebook_match_summary.csv,
#          chart_codebook_alignment_{sub,main}, pooled_codebook_alignment
# ==============================================================================
import os, json, re
import numpy as np, pandas as pd

# ---------------------------- EDIT HERE ---------------------------------------
# The manual codebook. Read from an existing global if one is set, so the path can
# be pointed elsewhere (a mirror folder, a test fixture) without editing this cell.
CODEBOOK_JSON = globals().get("CODEBOOK_JSON", "/content/codebook.json")
# The claim level first (it carries the headline figure), then the coarse level.
# This was ["sub", "mid"]; the mid level no longer exists, and _centroids() would
# have loaded pooled_labels_mid -> None -> np.asarray(None).max() -> TypeError.
# Centroids are built from members only (lab == c for c in range(K)), so the
# responses with no theme never enter one - which is why this cell needs no
# separate noise filter.
LEVELS_TO_MAP = [lv for lv in (CLAIM_LEVEL, "main") if lv in LEVELS]
Z_STRONG      = 1.5   # a cluster "resembles" a hand theme when its similarity is
                      # this many SDs above that theme's own mean profile. Working
                      # in SDs keeps the rule free of the encoder's cosine scale.
Z_CLEAR       = 2.5   # the top match must reach this to count as recovered
# --------------------------- END EDIT -----------------------------------------

_PL = globals().get("_orig_pload", pload)
if not os.path.exists(CODEBOOK_JSON):
    raise RuntimeError(f"codebook not found at {CODEBOOK_JSON} - upload it or fix the path.")
CB = json.load(open(CODEBOOK_JSON, encoding="utf-8"))
THEMES  = CB.get("themes", [])
PARENTS = CB.get("parent_themes", {})
CHILD2P = {c: p for p, d in PARENTS.items() for c in d.get("children", [])}
if not THEMES:
    raise RuntimeError("codebook.json has no 'themes' list.")

E_pool = np.asarray(_PL("pooled_embeddings"), dtype=np.float32)
M = globals().get("MASTER") if globals().get("MASTER") is not None else _PL("pooled_master")
if E_pool is None or M is None:
    raise RuntimeError("need pooled_embeddings and pooled_master.")

print("=" * 78)
print("  STAGE 15d   ALIGNMENT WITH THE MANUAL CODEBOOK")
print("=" * 78)
print(f"  manual codebook : {len(THEMES)} themes in {len(PARENTS)} parent categories")
_mn = str(CB.get("methodology_note", ""))
_stated = re.findall(r"n=\d+ respondents|\d+ answered text units[^,)]*|\d+ teams", _mn)
if _stated:
    print(f"  its stated corpus: {'; '.join(_stated)}")
print(f"  this pipeline   : {len(M)} core responses, {M['Participant'].nunique()} "
      f"participants, {M['Team'].nunique()} teams")
print("  SAME survey, NOT the same rows - the clustering was fitted on narrative")
print("  answers of at least three words. Report both denominators.")

# ---- probes: what each hand theme MEANS, in the responses' own space ----------
def _probe(t):
    parts = [str(t.get("label", "")), str(t.get("definition", ""))]
    for q in (t.get("example_quotes") or [])[:3]:
        parts.append(re.sub(r"^R\d+:\s*", "", str(q)))
    return " ".join(p.strip() for p in parts if p.strip())

PROBES = [_probe(t) for t in THEMES]
CODES  = [str(t.get("code", f"T{i}")) for i, t in enumerate(THEMES)]
LABELS = [str(t.get("label", c)) for t, c in zip(THEMES, CODES)]
VAL    = [str(t.get("valence", "")) for t in THEMES]

_CK, _cache = "pooled_codebook_probe_emb", _PL("pooled_codebook_probe_emb")
if isinstance(_cache, dict) and _cache.get("codes") == CODES:
    PE = np.asarray(_cache["emb"], dtype=np.float32)
    print(f"\n  probe embeddings reused from cache ({len(PE)} themes)")
else:
    from sentence_transformers import SentenceTransformer
    import torch, gc
    _INSTR = globals().get("EMBED_INSTRUCTION", "") or ""
    _model = globals().get("EMBED_MODEL") or (CFG.embed_model if "CFG" in globals()
                                              else "Qwen/Qwen3-Embedding-8B")
    print(f"\n  embedding {len(PROBES)} codebook probes with {_model}")
    print("  same encoding path as the stored responses (instruction prefix, no separator)")
    _em = SentenceTransformer(_model, trust_remote_code=True,
                              device="cuda" if torch.cuda.is_available() else None)
    PE = np.asarray(_em.encode([f"{_INSTR}{p}" for p in PROBES],
                               normalize_embeddings=True, batch_size=4,
                               show_progress_bar=False), dtype=np.float32)
    psave(_CK, dict(codes=CODES, emb=PE))
    del _em; gc.collect()
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

def _centroids(lv):
    lab = np.asarray(_PL(f"pooled_labels_{'main7' if lv == 'main' else lv}"))
    K = int(lab.max()) + 1
    C = np.stack([E_pool[lab == c].mean(0) for c in range(K)])
    C /= (np.linalg.norm(C, axis=1, keepdims=True) + 1e-9)
    D = _PL(f"pooled_{lv}_llm") or {}
    names = {}
    for c in range(K):
        d = D.get(c) if c in D else D.get(str(c))
        names[c] = str((d or {}).get("name", f"{lv[0]}{c}"))
    return C, names, lab

TIER = {}
_ev = _PL("pooled_theme_evidence")
if isinstance(_ev, pd.DataFrame):
    TIER = {(r.level, int(r.id)): r.tier for r in _ev.itertuples()}

long_rows, summ_rows, SIMS, UNMATCHED = [], [], {}, {}
for lv in LEVELS_TO_MAP:
    C, names, lab = _centroids(lv)
    S = PE @ C.T                                   # (hand themes, clusters)
    SIMS[lv] = (S, names, lab)
    Z = (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)
    for i, code in enumerate(CODES):
        order = np.argsort(-S[i])
        strong = [int(j) for j in order if Z[i, j] >= Z_STRONG]
        top = int(order[0]); second = int(order[1]) if len(order) > 1 else top
        if Z[i, top] < Z_CLEAR:
            verdict = "ABSENT"
            note = "no cluster stands out above this theme's own similarity profile"
        elif len(strong) == 1:
            verdict = "ONE-TO-ONE"
            note = f"recovered as a single cluster ({lv}{top})"
        else:
            verdict = "SPLIT"
            note = (f"spread across {len(strong)} clusters: "
                    + ", ".join(f"{lv[0]}{j}" for j in strong[:6]))
        summ_rows.append(dict(
            level=lv, code=code, hand_label=LABELS[i], parent=CHILD2P.get(code, ""),
            valence=VAL[i], verdict=verdict, note=note,
            best_cluster=top, best_name=names[top],
            best_cos=round(float(S[i, top]), 4), best_z=round(float(Z[i, top]), 2),
            z_gap_to_second=round(float(Z[i, top] - Z[i, second]), 2),
            n_clusters_resembling=len(strong),
            best_cluster_tier=TIER.get((lv, top), ""),
            n_responses_in_best=int((lab == top).sum())))
        for j in range(S.shape[1]):
            long_rows.append(dict(level=lv, code=code, hand_label=LABELS[i],
                                  parent=CHILD2P.get(code, ""), cluster=int(j),
                                  cluster_name=names[j],
                                  cosine=round(float(S[i, j]), 4),
                                  z=round(float(Z[i, j]), 3)))
    # A cluster is unmatched when NO hand theme counts it among its strong matches.
    # This has to be read off the ROW-normalised Z: normalising down the columns
    # instead would rescale every cluster to its own best theme and so could never
    # report a cluster that resembles nothing - the exact case this line is for.
    UNMATCHED[lv] = [c for c in range(S.shape[1]) if float(Z[:, c].max()) < Z_CLEAR]

A = pd.DataFrame(long_rows); SUM = pd.DataFrame(summ_rows)
A.to_csv(f"{POOLED_DIR}/codebook_alignment.csv", index=False)
SUM.to_csv(f"{POOLED_DIR}/codebook_match_summary.csv", index=False)
psave("pooled_codebook_alignment",
      dict(long=A, summary=SUM, codes=CODES, labels=LABELS, parents=CHILD2P,
           unmatched=UNMATCHED))

# _Kc, not K: `K` is the preflight's {level: n_themes} dict that every later cell
# reads. Assigning an int to it here silently replaced that dict for the rest of
# the notebook - a figure cell rewriting a global the whole notebook depends on.
for lv in LEVELS_TO_MAP:
    s = SUM[SUM.level == lv]; _Kc = SIMS[lv][0].shape[1]
    print("\n" + "-" * 78)
    print(f"  {lv.upper()} level  ({_Kc} clusters)  vs  {len(THEMES)} hand themes")
    print("-" * 78)
    for v in ("ONE-TO-ONE", "SPLIT", "ABSENT"):
        print(f"    {v:<12} {int((s.verdict == v).sum()):>3} of {len(s)}")
    print()
    for r in s.sort_values(["verdict", "best_z"], ascending=[True, False]).itertuples():
        mark = {"ONE-TO-ONE": "=", "SPLIT": "~", "ABSENT": "x"}[r.verdict]
        print(f"   {mark} {r.code:<24} z={r.best_z:>5.2f}  {r.note[:62]}")
        if r.verdict == "ONE-TO-ONE":
            print(f"       -> {lv}{r.best_cluster} \"{str(r.best_name)[:50]}\" "
                  f"(n={r.n_responses_in_best}"
                  + (f", {r.best_cluster_tier}" if r.best_cluster_tier else "") + ")")
    _un = UNMATCHED.get(lv, [])
    if _un:
        print(f"\n    {len(_un)} cluster(s) NO hand theme resembles - either the manual")
        print(f"    coding did not name them, or they are artifacts (see Stage 14c):")
        for c in _un[:10]:
            print(f"      {lv}{c:<3} \"{str(SIMS[lv][1][c])[:44]}\" "
                  f"n={int((SIMS[lv][2] == c).sum()):<4} {TIER.get((lv, c), '')}")

try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap
    CMAP = LinearSegmentedColormap.from_list(
        "align", ["#FFFFFF", "#CFE3DC", "#7FBFAE", "#2E8B74", "#14503F"])
    for lv in LEVELS_TO_MAP:
        S, names, lab = SIMS[lv]; _Kc = S.shape[1]
        Z = (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-9)
        order_i = sorted(range(len(CODES)),
                         key=lambda i: (CHILD2P.get(CODES[i], "zz"), CODES[i]))
        order_j = sorted(range(_Kc), key=lambda j: (int(np.argmax(Z[order_i, j])),
                                                    -float(Z[:, j].max())))

        # TRANSPOSED. The question this figure answers is "which machine theme is
        # which hand theme", and it could not be answered while the machine themes
        # were 39 rotated ids along the bottom. On the Y axis a full name fits
        # horizontally at a readable size; the 20 hand codes rotate happily on X.
        Zp = Z[np.ix_(order_i, order_j)].T                    # themes x hand codes

        def _nm(j):
            t = str(names[j])
            if t.startswith("[unlabelled"):
                t = t.split("top terms:")[-1].strip().rstrip(",")
                t = f"(no accepted label) {t}"
            return f"{lv[0].upper()}{j}  {t}"

        _rows = [_nm(j) for j in order_j]
        _wid = max(len(r) for r in _rows)
        fig, ax = plt.subplots(figsize=(4.2 + 0.30 * len(CODES) + 0.085 * _wid,
                                        2.2 + 0.30 * _Kc))
        im = ax.imshow(Zp, aspect="auto", cmap=CMAP, vmin=0, vmax=4)

        ax.set_yticks(range(_Kc))
        ax.set_yticklabels(_rows, fontsize=7.6)
        ax.set_xticks(range(len(order_i)))
        ax.set_xticklabels([CODES[i] for i in order_i], fontsize=7.8, rotation=45,
                           ha="right", rotation_mode="anchor")

        for b, j in enumerate(order_j):
            for a, i in enumerate(order_i):
                if Zp[b, a] >= Z_STRONG:
                    ax.plot(a, b, marker="o", ms=4.2, mfc="none", mec="#14202B", mew=.9)

        # a rule between hand-theme parent groups, now vertical
        prev = None
        for a, i in enumerate(order_i):
            _p = CHILD2P.get(CODES[i], "")
            if prev is not None and _p != prev:
                ax.axvline(a - .5, color="#14202B", lw=.9, alpha=.55)
            prev = _p

        ax.set_xlabel("hand-coded theme  (grouped by parent category)",
                      fontsize=10, fontweight="bold", labelpad=8)
        ax.set_ylabel(f"{lv}-theme from the clustering  "
                      f"(ordered by best-matching hand theme)",
                      fontsize=10, fontweight="bold", labelpad=8)
        cb = fig.colorbar(im, ax=ax, fraction=.020, pad=.010)
        cb.set_label("similarity, in SDs above that hand theme's own mean",
                     fontsize=8, fontweight="bold")
        cb.ax.tick_params(labelsize=7.4)
        ax.tick_params(length=2)
        _t = f"Where each hand-coded theme lands among the {lv}-themes"
        if "fig_title" in globals():
            fig_title(fig, _t)
        else:
            fig.suptitle(_t, fontsize=12)
        fig.text(0.5, 0.008,
                 "A ring marks a match. Read across a COLUMN to see where one hand theme "
                 "went: one ring means the clustering recovered it as a single theme, "
                 "several rings mean it split it, an empty column means it was not "
                 "recovered. Read along a ROW to see which hand themes a machine theme "
                 "corresponds to. Every pair is also in codebook_alignment.csv.",
                 ha="center", fontsize=8, style="italic",
                 color=globals().get("FOOT", "#98A2AD"), wrap=True)
        plt.tight_layout(rect=[0, 0.035, 1, .955])
        (save_fig(fig, f"chart_codebook_alignment_{lv}") if "save_fig" in globals()
         else fig.savefig(f"{POOLED_DIR}/chart_codebook_alignment_{lv}.png", dpi=300,
                          bbox_inches="tight"))
        plt.close(fig)

        # ---- the same correspondence as a table, so it can be read without the
        # figure and pasted into the write-up
        _pairs = []
        for b, j in enumerate(order_j):
            hits = [CODES[order_i[a]] for a in range(len(order_i))
                    if Zp[b, a] >= Z_STRONG]
            _best = int(np.argmax(Zp[b]))
            _pairs.append(dict(level=lv, theme_id=int(j), theme_name=str(names[j]),
                               n_responses=int((lab == j).sum()),
                               best_hand_theme=CODES[order_i[_best]],
                               best_z=round(float(Zp[b, _best]), 2),
                               all_matches="; ".join(hits) if hits else "(none)",
                               n_matches=len(hits)))
        _P = pd.DataFrame(_pairs).sort_values(["n_matches", "best_z"],
                                              ascending=[False, False])
        _P.to_csv(f"{POOLED_DIR}/codebook_theme_to_hand_{lv}.csv", index=False)
        print(f"\n  {lv}: which hand theme each machine theme corresponds to")
        print(f"  {'theme':<46}{'n':>5}  {'best hand theme':<24}{'z':>6}")
        for r in _P.head(12).itertuples():
            print(f"  {str(r.theme_name)[:45]:<46}{r.n_responses:>5}  "
                  f"{str(r.best_hand_theme)[:23]:<24}{r.best_z:>6.1f}")
        _none = _P[_P.n_matches == 0]
        if len(_none):
            print(f"  {len(_none)} {lv}-theme(s) match no hand theme above the "
                  f"threshold - the clustering found something the manual coding did "
                  f"not, which is a result in itself:")
            for r in _none.head(6).itertuples():
                print(f"    {str(r.theme_name)[:60]}  (closest: {r.best_hand_theme}, "
                      f"z={r.best_z:.1f})")
except Exception as e:
    print(f"\n  (figure skipped: {type(e).__name__}: {str(e)[:90]})")

print("\n  -> codebook_alignment.csv (every pair) · codebook_match_summary.csv "
      "(verdicts) · codebook_theme_to_hand_*.csv (which machine theme is which)")
print("  A ring marks every cell the analysis calls a match: one ring in a row = the")
print("  hand theme was recovered as one cluster; several = the clustering split it;")
print("  an empty row = it was not recovered at all.")
print("\n  THIS IS CONCEPTUAL ALIGNMENT, NOT INTER-RATER AGREEMENT - do not report it as")
print("  kappa. With the manual PER-RESPONSE coding the same comparison becomes a")
print("  measured agreement on identical rows, which is a much stronger claim.")
print("=" * 78)


### Stage 19 · Agreement with the manual codebook, on shared responses

In [ ]:
# @title
# ==============================================================================
# STAGE 19  -  AGREEMENT WITH THE MANUAL CODEBOOK, MEASURED ON SHARED RESPONSES
#             section 2 of 9  -  Is the analysis trustworthy?
# ==============================================================================
# The cell before this one compares the two codings by SEMANTIC SIMILARITY: it
# embeds each hand theme's definition, embeds each cluster, and asks which are
# close. That answers "do these two descriptions sound alike". It is a useful
# map, but it is not agreement - two coders can write similar definitions and
# still put the same response in different places, and nothing in that heatmap
# would notice.
#
# There is a much stronger comparison available, and the codebook already
# contains what it needs. Every hand theme ships EXAMPLE QUOTES, and each quote
# carries the response id it came from. Those are responses the professor
# actually coded. Wherever one of them is also in this corpus, the same piece of
# text has been independently coded twice: once by hand, once by the pipeline.
# That is the only place the two codings can be compared as codings.
#
#   A  ANCHOR    find the professor's example quotes in this corpus.
#   B  AGREEMENT on those shared responses, compare the two partitions with
#      chance-corrected statistics (Adjusted Rand Index, Normalised Mutual
#      Information) and a permutation test, plus a per-theme "did the pipeline
#      keep this hand theme together" rate. This is convergent validity between
#      two independent codings, not accuracy of one against the other - neither
#      is ground truth.
#   C  VALENCE   the codebook labels each hand theme positive or negative
#      (9 and 10 of the 19). That label was assigned by a human who never saw
#      this pipeline's sentiment model. Comparing the model's score on the
#      anchored responses against that independent human label is an EXTERNAL
#      validation of the measurement the whole notebook leans on - the first one
#      in this notebook that does not come from the pipeline itself.
#
# Honest limits, stated before the numbers: there are at most a few dozen
# anchored responses, so every interval here is wide; the quotes are the
# professor's CLEAREST examples, so they are easier to code than an average
# response and agreement measured on them is an upper bound; and the two
# codings were made on overlapping but not identical row sets.
# ==============================================================================

# Everything below runs inside a function ON PURPOSE. A notebook cell shares one
# global namespace with every other cell, and this cell needs ordinary working
# names - rows, A, rng, PARENT. PARENT in particular is set by the preflight as
# the sub->main map and read by two later stages; a plain module-level
# assignment here silently replaced it with the codebook's parent categories and
# broke them with a TypeError. Wrapping the body keeps every name local, so this
# cell cannot disturb any other no matter what it calls its variables.
def _stage19():
    import json as _json
    import re as _re
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # ----------------------------- EDIT HERE --------------------------------------
    CODEBOOK_PATHS = ["/content/codebook.json", "codebook.json",
                      f"{POOLED_DIR}/codebook.json"]
    MIN_WORDS_ANCHOR = 4      # a fragment shorter than this cannot identify a response
    N_PERM           = 9999
    SEED             = 20260814
    # ----------------------------- END EDIT ---------------------------------------
    rng = np.random.default_rng(SEED)

    cb = None
    for p in CODEBOOK_PATHS:
        try:
            with open(p, encoding="utf-8") as _f:
                cb = _json.load(_f)
            CODEBOOK_PATH = p
            break
        except Exception:
            continue
    if cb is None:
        raise RuntimeError("codebook.json not found - run the manual-codebook cell first, "
                           "or add its path to CODEBOOK_PATHS above.")

    HAND = cb.get("themes") or []
    PARENT = cb.get("parent_themes") or {}
    print("=" * 78)
    print("  AGREEMENT WITH THE MANUAL CODEBOOK, MEASURED ON SHARED RESPONSES")
    print("=" * 78)
    print(f"  codebook: {CODEBOOK_PATH}  -  {len(HAND)} hand themes in "
          f"{len(PARENT)} parent categories")

    # ==============================================================================
    # A.  ANCHOR the professor's example quotes in this corpus
    # ==============================================================================
    LV = CLAIM_LEVEL
    IDW, NMW, HCW = _cov_cols(LV, MASTER)
    print(_cov_note(LV, MASTER))

    _txt = MASTER["text"].fillna("").astype(str).tolist()


    def _norm(s):
        """Compare on words only. The codebook re-types its quotes by hand, so
        punctuation, curly quotes and capitalisation all differ from the raw survey
        text; the words do not."""
        return " ".join(_re.sub(r"[^a-z0-9 ]", " ", str(s).lower()).split())


    _CN = [_norm(t) for t in _txt]


    def _fragments(q):
        """The professor's quotes carry an 'R053:' prefix and are often elided with
        an ellipsis. Strip the prefix and return each surviving run of real text -
        a match on any one of them identifies the response."""
        q = _re.sub(r"^\s*R\d+\s*:\s*", "", str(q))
        parts = _re.split(r"\.\.\.|…", q)
        return [p for p in (_norm(x) for x in parts) if len(p.split()) >= MIN_WORDS_ANCHOR]


    rows, _unmatched = [], []
    for t in HAND:
        code = t.get("code", "?")
        for q in (t.get("example_quotes") or []):
            rid = (_re.match(r"^\s*(R\d+)", str(q)) or [None, None])[1]
            got = None
            for frag in _fragments(q):
                hits = [i for i, c in enumerate(_CN) if frag in c]
                if not hits:                       # try the other direction: the
                    hits = [i for i, c in enumerate(_CN)   # codebook may hold the
                            if c and c in frag]            # longer version
                if len(hits) == 1:
                    got = hits[0]; break
                if len(hits) > 1:                  # ambiguous - take the closest length
                    got = min(hits, key=lambda i: abs(len(_CN[i]) - len(frag))); break
            if got is None:
                _unmatched.append((code, rid))
            else:
                rows.append(dict(hand_code=code, hand_label=t.get("label", code),
                                 valence=str(t.get("valence", "")).lower(),
                                 rid=rid, row=got))
    A = pd.DataFrame(rows).drop_duplicates("row").reset_index(drop=True)
    nq = sum(len(t.get("example_quotes") or []) for t in HAND)
    print(f"\n  A. ANCHORING")
    print(f"     {nq} example quotes in the codebook -> {len(A)} located in this corpus "
          f"({100*len(A)/max(nq,1):.0f}%)")
    if len(A) < 10:
        print(f"     Too few anchors to compare the codings. The quotes are elided in the")
        print(f"     codebook and this corpus keeps only answers of three words or more,")
        print(f"     so a low rate is expected rather than an error. Stopping here.")
        print("=" * 78)
    else:
        A["cluster"] = MASTER[IDW].values[A["row"].values]
        A["cluster_name"] = [theme_name(LV, int(c)) if int(c) >= 0 else NO_THEME
                             for c in A["cluster"]]
        A["placed"] = MASTER[HCW].astype(bool).values[A["row"].values]
        A["sentiment"] = pd.to_numeric(MASTER["sentiment"], errors="coerce").values[
            A["row"].values]
        _pl = A[A.placed].copy()
        print(f"     of those, {len(_pl)} carry a pipeline theme "
              f"({len(A)-len(_pl)} were left unplaced by the clustering)")
        print(f"     hand themes represented: {A.hand_code.nunique()} of {len(HAND)}")

        # ==========================================================================
        # B.  AGREEMENT between the two codings, chance-corrected
        # ==========================================================================
        print(f"\n  B. AGREEMENT ON THE {len(_pl)} SHARED, PLACED RESPONSES")
        if _pl["hand_code"].nunique() < 2 or _pl["cluster"].nunique() < 2:
            print("     not enough distinct labels on both sides to compare partitions")
            ARI = NMI = P_ARI = np.nan
        else:
            from sklearn.metrics import adjusted_rand_score, \
                normalized_mutual_info_score
            h = _pl["hand_code"].astype("category").cat.codes.to_numpy()
            m = _pl["cluster"].astype(int).to_numpy()
            ARI = float(adjusted_rand_score(h, m))
            NMI = float(normalized_mutual_info_score(h, m))
            null = np.array([adjusted_rand_score(rng.permutation(h), m)
                             for _ in range(N_PERM)])
            P_ARI = float((np.sum(null >= ARI) + 1) / (N_PERM + 1))
            print(f"     Adjusted Rand Index      {ARI:+.3f}   "
                  f"(0 = chance, 1 = identical partitions)")
            print(f"     Normalised Mutual Info    {NMI:.3f}")
            print(f"     permutation p for the ARI {P_ARI:.4f}  ({N_PERM} shuffles)")
            print(f"     -> the two codings agree {'MORE than chance' if P_ARI < 0.05 else 'no more than chance'}"
                  f" on where these responses belong")
            print(f"     Note the ARI is a partition statistic: 39 clusters against "
                  f"{_pl['hand_code'].nunique()} hand\n     themes cannot reach 1 even "
                  f"with perfect nesting, so read it against the\n     permutation null, "
                  f"not against 1.")

        # per-hand-theme cohesion: did the pipeline keep this theme together?
        coh = []
        for code, g in _pl.groupby("hand_code"):
            if len(g) < 2:
                continue
            vc = g["cluster"].value_counts()
            coh.append(dict(hand_code=code, n=len(g), n_clusters=int(g.cluster.nunique()),
                            modal_cluster=int(vc.index[0]),
                            modal_name=theme_name(LV, int(vc.index[0])),
                            together=float(vc.iloc[0] / len(g))))
        COH = pd.DataFrame(coh).sort_values("together", ascending=False)
        if len(COH):
            kept = COH[COH.together >= 0.999]
            print(f"\n     hand themes with 2+ anchored responses: {len(COH)}")
            print(f"     kept entirely inside ONE pipeline theme: {len(kept)}")
            for _, r in COH.head(6).iterrows():
                print(f"       {r.hand_code:<24} {r.n} quotes -> {r.n_clusters} cluster(s)"
                      f"   {100*r.together:3.0f}% together   [{str(r.modal_name)[:30]}]")

        # ==========================================================================
        # C.  EXTERNAL VALIDATION OF THE SENTIMENT SCORE
        # ==========================================================================
        print(f"\n  C. THE SENTIMENT SCORE AGAINST AN INDEPENDENT HUMAN LABEL")
        V = A[A.valence.isin(["positive", "negative"]) & np.isfinite(A.sentiment)]
        pos = V.loc[V.valence == "positive", "sentiment"].to_numpy()
        neg = V.loc[V.valence == "negative", "sentiment"].to_numpy()
        print(f"     {len(V)} anchored responses carry both a hand valence and a "
              f"measured score")
        print(f"     hand-POSITIVE themes: n={len(pos)}  mean measured "
              f"{pos.mean():+.3f}" if len(pos) else "     no positive anchors")
        print(f"     hand-NEGATIVE themes: n={len(neg)}  mean measured "
              f"{neg.mean():+.3f}" if len(neg) else "     no negative anchors")
        if len(pos) >= 3 and len(neg) >= 3:
            from scipy.stats import mannwhitneyu
            U, p_mw = mannwhitneyu(pos, neg, alternative="greater")
            # rank-biserial correlation = the probability of superiority, rescaled;
            # it is the effect size that belongs with Mann-Whitney
            rbc = float(2 * U / (len(pos) * len(neg)) - 1)
            # bootstrap interval for it, because n is small and it should show
            bs = []
            for _ in range(2000):
                a = rng.choice(pos, len(pos), replace=True)
                b = rng.choice(neg, len(neg), replace=True)
                uu = mannwhitneyu(a, b, alternative="greater")[0]
                bs.append(2 * uu / (len(a) * len(b)) - 1)
            lo, hi = np.percentile(bs, [2.5, 97.5])
            print(f"     Mann-Whitney (one-sided, positive > negative): p = {p_mw:.4f}")
            print(f"     rank-biserial correlation r = {rbc:+.3f}  "
                  f"[95% bootstrap {lo:+.3f}, {hi:+.3f}]")
            _agree = float((2 * U / (len(pos) * len(neg))))
            print(f"     a randomly drawn hand-positive response scores above a randomly "
                  f"drawn\n     hand-negative one {100*_agree:.0f}% of the time "
                  f"(50% would be no relationship)")
            if p_mw < 0.05:
                print(f"     -> the sentiment score AGREES with an independent human "
                      f"reading of the\n        same responses. This is external "
                      f"validation: the valence labels were\n        written from the "
                      f"codebook's own reading, with no sight of this model.")
            else:
                print(f"     -> no detectable agreement at this sample size. With "
                      f"{len(pos)}+{len(neg)} responses\n        that is weak evidence "
                      f"either way, not evidence of disagreement.")
            VAL = dict(n_pos=len(pos), n_neg=len(neg), mean_pos=float(pos.mean()),
                       mean_neg=float(neg.mean()), p=float(p_mw), rbc=rbc,
                       rbc_lo=float(lo), rbc_hi=float(hi))
        else:
            print("     too few anchored responses on one side to test")
            VAL = {}

        # ==========================================================================
        # FIGURES
        # ==========================================================================
        _charts = []
        # --- 1. the REAL confusion: hand theme x pipeline theme, on shared rows ----
        if len(_pl) >= 10:
            ct = pd.crosstab(_pl["hand_code"], _pl["cluster"])
            ct = ct.loc[ct.sum(1).sort_values(ascending=False).index]
            fig, ax = plt.subplots(figsize=(max(6.0, 0.42 * ct.shape[1] + 3.4),
                                            max(3.2, 0.34 * ct.shape[0] + 2.0)))
            ax.imshow(ct.values, cmap="Greens", aspect="auto", vmin=0)
            for i in range(ct.shape[0]):
                for j in range(ct.shape[1]):
                    v = int(ct.values[i, j])
                    if v:
                        ax.text(j, i, str(v), ha="center", va="center", fontsize=7.6,
                                color="white" if v > ct.values.max() * .6 else "#14202B")
            ax.set_xticks(range(ct.shape[1]))
            ax.set_xticklabels([f"{LV[0]}{c}" for c in ct.columns], fontsize=7,
                               rotation=90)
            ax.set_yticks(range(ct.shape[0]))
            ax.set_yticklabels([fit_label(c, width=28, max_lines=1) for c in ct.index],
                               fontsize=7.4)
            ax.set_xlabel("pipeline theme", fontsize=9)
            ax.set_ylabel("hand theme", fontsize=9)
            fig_title(fig, "Where the professor's own example responses landed")
            fig.text(0.5, 0.012,
                     f"Counts of the {len(_pl)} responses coded by BOTH methods - not "
                     f"similarity between definitions. A row that sits in one column is a "
                     f"hand theme the pipeline kept together. "
                     f"ARI {ARI:+.3f}, permutation p {P_ARI:.3f}."
                     if np.isfinite(ARI) else "Counts of responses coded by both methods.",
                     ha="center", fontsize=8.0, style="italic", color=FOOT, wrap=True)
            plt.tight_layout(rect=[0, .055, 1, .93])
            save_fig(fig, "chart_codebook_agreement_confusion"); plt.close(fig)
            _charts.append("chart_codebook_agreement_confusion")

        # --- 2. sentiment against the independent human valence -------------------
        if VAL:
            fig, ax = plt.subplots(figsize=(7.0, 3.8))
            for k, (name, arr, col) in enumerate((("hand-labelled negative", neg, SENT_NEG),
                                                  ("hand-labelled positive", pos, SENT_POS))):
                jit = rng.normal(0, .055, len(arr))
                ax.scatter(arr, np.full(len(arr), k) + jit, s=34, color=col, alpha=.75,
                           edgecolor="white", linewidth=.6, zorder=3)
                ax.plot([arr.mean()], [k], "|", ms=26, color="#14202B", zorder=5)
                ax.text(arr.mean(), k + .30, f"mean {arr.mean():+.2f}  (n={len(arr)})",
                        ha="center", fontsize=8, color="#14202B")
            ax.axvline(0, color="#8A939B", lw=1.0, zorder=1)
            ax.set_yticks([0, 1])
            ax.set_yticklabels(["negative", "positive"], fontsize=9)
            ax.set_ylabel("valence assigned by hand", fontsize=9)
            ax.set_xlabel("sentiment score measured by this pipeline", fontsize=9)
            ax.set_ylim(-.6, 1.7)
            for sp in ("top", "right", "left"):
                ax.spines[sp].set_visible(False)
            fig_title(fig, "Does the measured sentiment agree with an independent "
                           "human reading?")
            fig.text(0.5, 0.012,
                     f"Each dot is one response the professor quoted, placed by the score "
                     f"this pipeline measured. A hand-positive response outranks a "
                     f"hand-negative one {100*(VAL['rbc']+1)/2:.0f}% of the time "
                     f"(r = {VAL['rbc']:+.2f}, p = {VAL['p']:.3f}). The valence labels "
                     f"were written without sight of this model.",
                     ha="center", fontsize=8.0, style="italic", color=FOOT, wrap=True)
            plt.tight_layout(rect=[0, .07, 1, .90])
            save_fig(fig, "chart_codebook_valence_validation"); plt.close(fig)
            _charts.append("chart_codebook_valence_validation")

        # ---- persist -------------------------------------------------------------
        A.to_csv(f"{POOLED_DIR}/codebook_anchored_responses.csv", index=False)
        if len(COH):
            COH.to_csv(f"{POOLED_DIR}/codebook_theme_cohesion.csv", index=False)
        psave("pooled_codebook_agreement",
              dict(anchors=A, cohesion=COH, ari=ARI, nmi=NMI, p_ari=P_ARI,
                   valence=VAL, n_quotes=nq, n_anchored=len(A), n_placed=len(_pl)))
        print(f"\n  -> codebook_anchored_responses.csv · codebook_theme_cohesion.csv · "
              f"pooled_codebook_agreement")
        print(f"  -> {' · '.join(_charts) if _charts else '(no figure: too few anchors)'}")
        print("=" * 78)


_stage19()


---

## Section 3 - The thematic structure

What the themes are: a codebook a human coder can work from, two interactive maps of the clustering space, and the flow from survey question to theme.

### Stage 20 · Theme codebook

**Question.** What is in each theme, in the participants' own words?

**Reads.** `pooled_master`, the verified labels. **Writes.** `codebook.html`, `codebook.csv`.

**How to read it.** A collapsible main -> sub tree. Each entry carries the verified label, the model's summary, the theme's sentiment, its size, and three real centroid-nearest responses. The header states what share of the corpus the codebook covers, because it does not cover the responses that were left unplaced.

In [ ]:
# @title
# ==============================================================================
# STAGE 20  -  THEME CODEBOOK
#             section 3 of 9  -  The themes themselves
# ==============================================================================
# ==============================================================================
# THEME CODEBOOK FOR MANUAL CODERS
# ------------------------------------------------------------------------------
# A collapsible main -> sub codebook plus a flat CSV. Each entry carries the
# verified label, the model's own summary, the theme's sentiment, its size, and
# three REAL responses drawn from the theme itself - centroid-nearest and of
# medium length (12-60 words), so a coder sees what the theme actually contains
# rather than a fragment or an essay.
#
# REBUILT for the two-level hierarchy. The previous version walked
# maps["main_to_mids"] and maps["mid_to_subs"], neither of which Notebook 2
# writes any more; it also rebuilt the response text from the raw survey columns
# while every other cell reads pooled_master["text"], so the two could drift.
# Both now come from the canonical table.
#
# Responses with no theme are not an entry. They are counted once at the top, so
# a coder knows what share of the corpus the codebook does not cover - a codebook
# that silently describes part of the data is worse than one that says so.
# ==============================================================================
import numpy as np, pandas as pd, html as _h, csv, os

X5 = np.asarray(pload("pooled_umap_nd") if pload("pooled_umap_nd") is not None
                else pload("pooled_umap5d"))
texts = MASTER["text"].fillna("").astype(str).values
if len(X5) != len(MASTER):
    raise RuntimeError(f"clustering space has {len(X5)} rows, pooled_master has "
                       f"{len(MASTER)} - re-run Notebook 2.")

# main -> its sub-themes, inverted from the crosswalk Notebook 2 publishes
KIDS = {}
for s_, m_ in PARENT.items():
    KIDS.setdefault(int(m_), []).append(int(s_))
for m_ in range(K["main"]):
    KIDS.setdefault(m_, [])

LLM = {lv: {int(k): v for k, v in (pload(f"pooled_{lv}_llm") or {}).items()}
       for lv in LEVELS}


def _entry(lv, c):
    d = LLM[lv].get(int(c)) or {}
    idx = np.where(np.asarray(MASTER[f"{lv}_id"]) == int(c))[0]
    return dict(name=theme_name(lv, c),
                summary=str(d.get("summary", "") or ""),
                sentiment=str(d.get("sentiment", "") or ""),
                status=str(d.get("status", "") or ""),
                n=int(len(idx)),
                n_part=int(MASTER.iloc[idx]["Participant"].nunique()) if len(idx) else 0,
                quotes=pick_quotes(idx, X5, texts, n=3))


print("=" * 72)
print("  THEME CODEBOOK")
print("=" * 72)
_cov = N_THEMED[CLAIM_LEVEL] / N_ALL
print(f"  {K['main']} main themes, {K[CLAIM_LEVEL]} {CLAIM_LEVEL} themes")
print(f"  covers {N_THEMED[CLAIM_LEVEL]} of {N_ALL} responses ({_cov:.0%}); the other "
      f"{NOISE[CLAIM_LEVEL]} sit in no theme and have no entry")

SC = {"positive": "#0F6E56", "negative": "#B84413",
      "mixed": "#8C9AA3", "neutral": "#BFC4C9"}
rowsC, parts = [], []
parts.append(
    "<style>body{font-family:Inter,Arial,sans-serif;color:#14202B;max-width:960px;"
    "margin:24px auto}details{border:1px solid #E7EAEE;border-radius:10px;margin:8px 0;"
    "padding:8px 14px}summary{cursor:pointer;font-weight:600}.n{color:#98A2AD;"
    "font-weight:400}.sm{color:#5B6670;margin:6px 0}.q{border-left:3px solid #0F6E56;"
    "padding-left:10px;margin:6px 0;color:#333;font-style:italic}.sent{font-size:12px;"
    "padding:1px 8px;border-radius:9px;color:#fff}.hd{color:#5B6670;font-size:13px}"
    "</style><h2>Theme codebook</h2>"
    f"<p class=hd>{K['main']} main themes containing {K[CLAIM_LEVEL]} "
    f"{CLAIM_LEVEL}-themes. Every quote is a real response from the theme it sits "
    f"under, centroid-nearest and 12&ndash;60 words. {NOISE[CLAIM_LEVEL]} of {N_ALL} "
    f"responses ({1 - _cov:.0%}) belong to no theme and appear nowhere below.</p>")

for m_ in range(K["main"]):
    em = _entry("main", m_)
    parts.append(f"<details open><summary>M{m_} &middot; {_h.escape(em['name'])} "
                 f"<span class=n>n={em['n']} &middot; {em['n_part']} participants</span>"
                 f"</summary><div class=sm>{_h.escape(em['summary'])}</div>")
    rowsC.append(["main", m_, em["name"], em["sentiment"], em["n"], em["n_part"],
                  em["status"], em["summary"], " || ".join(em["quotes"])])
    for s_ in sorted(KIDS.get(m_, [])):
        es = _entry(CLAIM_LEVEL, s_)
        _badge = SC.get(es["sentiment"], "#BFC4C9")
        parts.append(
            f"<details><summary>S{s_} &middot; {_h.escape(es['name'])} "
            f"<span class=sent style='background:{_badge}'>"
            f"{_h.escape(es['sentiment'] or 'n/a')}</span> "
            f"<span class=n>n={es['n']} &middot; {es['n_part']} participants</span>"
            f"</summary><div class=sm>{_h.escape(es['summary'])}</div>")
        for q in es["quotes"]:
            parts.append(f"<div class=q>{_h.escape(q)}</div>")
        parts.append("</details>")
        rowsC.append([CLAIM_LEVEL, s_, es["name"], es["sentiment"], es["n"],
                      es["n_part"], es["status"], es["summary"],
                      " || ".join(es["quotes"])])
    parts.append("</details>")

pth = os.path.join(POOLED_DIR, "codebook.html")
open(pth, "w", encoding="utf-8").write("".join(parts))
with open(os.path.join(POOLED_DIR, "codebook.csv"), "w", newline="",
          encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["level", "id", "name", "sentiment", "n_responses", "n_participants",
                "naming_status", "summary", "quotes"])
    w.writerows(rowsC)
if MIRROR_POOLED_DIR:
    _mirror_file(pth, MIRROR_POOLED_DIR)

_orphan = [s_ for s_ in range(K[CLAIM_LEVEL])
           if s_ not in {x for v in KIDS.values() for x in v}]
if _orphan:
    print(f"  !! {len(_orphan)} {CLAIM_LEVEL}-theme(s) have no parent in the crosswalk "
          f"and are missing from the tree: {_orphan[:8]}")
_held = [r for r in rowsC if str(r[6]) and str(r[6]) != "accepted"]
if _held:
    print(f"  {len(_held)} entry(ies) carry a withheld label - the naming engine refused "
          f"it, and the placeholder shows the theme's own terms instead")
print(f"  -> codebook.html + codebook.csv  ({len(rowsC)} entries)")
print("=" * 72)


### Stage 21 · Two-dimensional theme explorer

**Question.** How are the themes arranged, and what does any individual point actually say?

**Reads.** `pooled_umap2d`, the labels, `pooled_master`.

**How to read it.** Switch views with the buttons: main, sub, question, and the participant attributes. Hovering a point shows its own response and its theme chain; a point the clustering left unplaced says so rather than being given a parent. This is a map of the clustering space, so distance is UMAP distance and should be read as neighbourhood structure, not as a measured quantity.

In [ ]:
# @title
# ==============================================================================
# STAGE 21  -  TWO-DIMENSIONAL THEME EXPLORER
#             section 3 of 9  -  The themes themselves
# ==============================================================================
# Seven views on one map. Four are the clustering itself, three are measured
# attributes that the clustering never saw:
#   * Main / Sub - the two-level hierarchy, sub coloured from its parent main
#   * Question (5)                    - which parts of the map each question feeds
#   * Sentiment                       - the MEASURED sentiment score per response,
#                                       with "not measurable" drawn grey, not zero
#   * Region (5) / Section (3)        - where each group sits, and which themes it
#                                       over-voices (rates read from Stage 63, so
#                                       the map and the bar charts cannot disagree)
#
# What this rebuild fixes, each one measured before and after (printed at the end):
#   1. ASPECT. The old cell drew a 8.25 x 9.56 embedding into a 1130 x 432 box:
#      3.03x horizontal stretch. UMAP output is a DISTANCE map, so that silently
#      distorted the single thing a reader is asked to judge by eye. Locked 1:1.
#   2. SELF-DESCRIPTION. Badges carried bare cluster numbers and the names lived
#      only in the HTML sidebar - a PNG in a paper had 54 numbers and no key.
#      Every map PNG now ships with a key sheet naming every badge.
#   3. SMALL THEMES. 10 of 54 subs fall under the KDE gate and got no band, with
#      nothing saying so. They are now drawn as hollow badges and the gate is
#      stated in the footer.
#   4. BADGE PLACEMENT. Position was median(x), median(y) taken separately, which
#      for a bent cluster lands outside it (5 subs did). Now the MEDOID: a real
#      response of that theme, so a badge is never in someone else's territory.
#   5. OVERPLOTTING. 62% of markers touched another at size 7. Smaller marker,
#      thinner ring, and at Sub level the KDE is drawn as an OUTLINE rather than
#      44 stacked fills - which is what actually made the map look muddy.
#   6. COLOUR. The palette is mostly fine (median dE 38.7 between neighbours), but
#      a few adjacent pairs collide (s11/s15 at dE 4.8). A measured repair pass
#      separates only those, and never moves a theme out of its main hue family.
#   7. HOVER. 11% of responses were silently cut at 4 lines; now the cut is shown.
#   8. CONTROLS. Title, level buttons and the question filter live in HTML, so
#      nothing overlaps the modebar and no button label can be clipped.
#   9. The question checkboxes RECOLOUR (per-point opacity is ignored by Plotly
#      for these traces); spotlight and filter share one recolour path.
#  10. COVERAGE. The map used the raw core partition, so the 210 responses NB2
#      recovered were drawn in the "no theme" grey - the map said 403 unplaced
#      while every other cell in this notebook said 193. It now follows the
#      notebook's COVERAGE setting like every claim cell does. The clustering's
#      OWN geometry (density bands, badges, nearest-theme, quotes) is still
#      measured from the CORE members only, because a recovered response was
#      placed by a rule, not by the clustering, and must not be allowed to
#      reshape the thing it was matched against. Hover and card say which.
#  11. FAIL-LOUD. One unescaped apostrophe inside a single-quoted JS string used
#      to kill the whole <script> block, and the page then rendered as a silent
#      white box - indistinguishable from a working map at a glance. Two guards
#      now: the page parses its own script BEFORE shipping (_js_guard), and any
#      runtime error paints itself into the plot area instead of vanishing.
# ==============================================================================
import os
import numpy as np
import json as _json
import colorsys as _cs
import matplotlib.colors as _mc
import plotly.graph_objects as go

_PL = globals().get("_orig_pload", pload)
XY   = np.asarray(_PL("pooled_umap2d"))
X5   = np.asarray(_PL("pooled_umap5d"))
meta = _PL("pooled_meta").reset_index(drop=True)
labs = {"Main": np.asarray(_PL("pooled_labels_main7")),
        "Sub":  np.asarray(_PL("pooled_labels_sub"))}
maps = _PL("pooled_hier_maps")
llm  = {"Main": _PL("pooled_main_llm") or {},
        "Sub": _PL("pooled_sub_llm") or {}}
# sub -> main directly; there is no intermediate rung any more
s2M = {int(k): int(v) for k, v in maps["sub_to_main"].items()}
N = len(meta)
QL = list(CFG.short_labels)

# ---- metadata hub -----------------------------------------------------------
# pooled_master is the one place question / theme / team / award / demographics
# are already joined per response, and Stage 14b rewrote its "sentiment" column
# in place. Reading it here is what keeps this map consistent with every figure.
# pooled_master comes from the preflight, already reset and
# length-checked; a second binding of the canonical table inside a
# figure cell is one more thing that can go stale.
_MST = MASTER
_have_master = _MST is not None
if _have_master:
    if len(_MST) != N:
        raise RuntimeError(f"pooled_master has {len(_MST)} rows but the embedding "
                           f"has {N}; they must be row-aligned - re-run NB2 Stage 33b")
    def _mcol(c):
        return (_MST[c].fillna("").astype(str).values if c in _MST.columns
                else np.array([""] * N))
    country, track, medal = _mcol("Country"), _mcol("Track"), _mcol("medal")
    region, section = _mcol("Region"), _mcol("Section")
    texts = _MST["text"].fillna("").astype(str).values
    sent = (_MST["sentiment"].astype(float).values if "sentiment" in _MST.columns
            else np.full(N, np.nan))
else:
    print("  (pooled_master missing - run NB2 Stage 33b; attribute views are off)")
    country = track = region = section = np.array([""] * N)
    aw = _PL("pooled_award") or {}
    medal = np.asarray(aw.get("medal", np.array(["?"] * N, dtype=object))).astype(str)
    sent = np.full(N, np.nan)
    _df = get_df(); _pid = {r["Participant"]: r for _, r in _df.iterrows()}
    RAWCOL = dict(zip(CFG.short_labels, CFG.text_cols))
    texts = np.array([("" if (_pid.get(mr["Participant"]) is None) else
                      str(_pid[mr["Participant"]].get(RAWCOL[mr["question"]], "") or ""))
                      for _, mr in meta.iterrows()], dtype=object)
texts = np.array(["" if str(t).strip().lower() == "nan" else str(t) for t in texts],
                 dtype=object)

# ---- coverage: the map must count what the rest of the notebook counts ------
# labs      = what a point IS (wide: core + recovered), so colour and counts
#             agree with every claim cell.
# LAB_CORE  = what the CLUSTERING itself placed, and the only thing allowed to
#             define shape - density bands, badge position, nearest-theme and
#             quotes all come from here. A recovered response was matched to a
#             centroid by a rule with a measured 94.6% upper-bound accuracy;
#             letting it move the band would let the rule redraw the cluster it
#             was matched against.
LAB_CORE = {lv: np.asarray(v).copy() for lv, v in labs.items()}
TIER = (_MST["theme_tier"].astype(str).values if
        (_have_master and "theme_tier" in _MST.columns) else np.array(["core"] * N))
_COV = str(globals().get("COVERAGE", "core"))
_wide_on = False
if _have_master and _COV == "wide":
    _pairs = [("Main", "main_id_wide"), ("Sub", f"{CLAIM_LEVEL}_id_wide")]
    if all(c in _MST.columns for _, c in _pairs):
        for _lv, _c in _pairs:
            labs[_lv] = np.asarray(_MST[_c], dtype=int)
        _wide_on = True
if not _wide_on and _COV == "wide":
    print("  (COVERAGE='wide' but pooled_master carries no *_id_wide columns; "
          "the map falls back to the core partition and says so in the footer)")
_n_core = {lv: int((LAB_CORE[lv] >= 0).sum()) for lv in LAB_CORE}
_n_wide = {lv: int((labs[lv] >= 0).sum()) for lv in LAB_CORE}
_REC = np.array([t in ("assigned", "assigned-multi") for t in TIER])
print(f"  coverage: {'WIDE' if _wide_on else 'CORE'} - sub placed "
      f"{_n_wide['Sub']} of {N} (core {_n_core['Sub']}, recovered "
      f"{_n_wide['Sub'] - _n_core['Sub']}); {N - _n_wide['Sub']} drawn as no theme")
_TIERTXT = {"core": "placed by the clustering",
            "assigned": "recovered by the projection rule, not clustered",
            "assigned-multi": "clause-level multi-label; primary theme shown",
            "unassigned": "left in no theme by the clustering"}

# Stage 26B leaves a response unscored (NaN) when its verdict was withheld.
# That is a measurement fact and the map must show it as absence, never as zero.
_n_sc = int(np.isfinite(sent).sum())
if _have_master and _n_sc == N:
    print("  NOTE: every response carries a sentiment score. Stage 14b normally "
          "leaves some NaN (verdict withheld) - run it before this cell.")

def _nm(lv, c):
    d = llm[lv].get(int(c)) or llm[lv].get(str(int(c))) or {}
    return d.get("name") or f"{lv[0]}{c}"

qidx = np.array([QL.index(q) for q in meta["question"]])
labs["Question"] = qidx

# ---- attribute views: labels are groups, not clusters ------------------------
ATTR = {}
if _have_master:
    for _vn, _vals in (("Region", region), ("Section", section)):
        _g = sorted(set(v for v in _vals if v))
        if len(_g) >= 2:
            ATTR[_vn] = _g
            labs[_vn] = np.array([_g.index(v) if v in _g else -1 for v in _vals])

VIEWS = ["Main", "Sub", "Question"] + list(ATTR) + (["Sentiment"] if _have_master else [])
THEME_V = ["Main", "Sub"]                        # levels with an LLM codebook
GROUP_V = [v for v in VIEWS if v != "Sentiment"]  # views that have badges/cards
Kof = {lv: (int(labs[lv].max()) + 1) for lv in labs}

# ==============================================================================
# COLOUR
# ==============================================================================
# Family colouring is the meaningful encoding (hue = main theme), so it is kept.
# The only defect measured was a handful of SPATIALLY ADJACENT pairs whose colours
# are too close to separate at marker size. The repair moves lightness only, never
# hue family, and only for pairs that actually violate the threshold - so nothing
# is "prettified"; the printed before/after says exactly what moved.
MIN_DE = 15.0        # below this two colours are not separable at marker size
_GREY  = "#dce1e5"   # the one colour that never means a group

_LABC = {}
def _to_lab(h):
    r, g, b = _mc.to_rgb(h)
    f = lambda u: u / 12.92 if u <= .04045 else ((u + .055) / 1.055) ** 2.4
    r, g, b = f(r), f(g), f(b)
    X = r*.4124 + g*.3576 + b*.1805
    Y = r*.2126 + g*.7152 + b*.0722
    Z = r*.0193 + g*.1192 + b*.9505
    q = lambda t: t ** (1/3) if t > .008856 else 7.787 * t + 16/116
    fx, fy, fz = q(X/.95047), q(Y), q(Z/1.08883)
    v = np.array([116*fy - 16, 500*(fx - fy), 200*(fy - fz)])
    _LABC[h] = v
    return v

def _dE(a, b):
    la = _LABC[a] if a in _LABC else _to_lab(a)
    lb = _LABC[b] if b in _LABC else _to_lab(b)
    return float(np.linalg.norm(la - lb))

def _adjacent_pairs(lab, k=4):
    """The pairs a reader can actually confuse: clusters whose centroids are among
    each other's k nearest on the 2D map. Colour collisions between clusters at
    opposite ends of the map cost nothing, so they are not repaired."""
    ids = sorted(set(int(x) for x in lab) - {-1})
    if len(ids) < 2:
        return []
    cen = np.array([XY[lab == c].mean(0) for c in ids])
    D = np.linalg.norm(cen[:, None, :] - cen[None, :, :], axis=-1)
    np.fill_diagonal(D, np.inf)
    out = set()
    for i in range(len(ids)):
        for j in np.argsort(D[i])[:k]:
            out.add(tuple(sorted((ids[i], int(ids[j])))))
    return sorted(out)

def _score(cmap, pairs):
    """Fewer violations first, then a larger worst-case gap. Comparing this tuple
    is what makes the search monotone: a move is taken only if the whole palette
    gets better, so the repair can never do what a greedy per-pair nudge does -
    fix one collision and open two elsewhere."""
    d = [_dE(cmap[a], cmap[b]) for a, b in pairs]
    return (-sum(1 for x in d if x < MIN_DE), min(d) if d else 0.0)

def _repair(cmap, lab, tag):
    """Coordinate ascent over lightness and saturation, hue held fixed so a theme
    never leaves its main-colour family. Only pairs that are neighbours ON THE MAP
    are counted - two identical colours at opposite ends confuse nobody."""
    pairs = _adjacent_pairs(lab)
    if not pairs:
        return cmap, 0.0, 0, 0
    for c in cmap:
        _to_lab(cmap[c])
    s0 = _score(cmap, pairs)
    out = dict(cmap)
    nbr = {}
    for a, b in pairs:
        nbr.setdefault(a, []).append(b); nbr.setdefault(b, []).append(a)
    grid = [round(0.18 + 0.02 * k, 3) for k in range(34)]
    for _sweep in range(6):
        moved_any = False
        for c in sorted(out):
            if c not in nbr:
                continue
            h, l, s = _cs.rgb_to_hls(*_mc.to_rgb(out[c]))
            cur, best, bestc = _score(out, pairs), None, out[c]
            for nl in grid:
                for ns in (max(0.05, s * 0.75), s, min(1.0, s * 1.25)):
                    cand = _mc.to_hex(_cs.hls_to_rgb(h, nl, ns))
                    if cand == out[c]:
                        continue
                    _to_lab(cand)
                    keep = out[c]; out[c] = cand
                    sc = _score(out, pairs)
                    out[c] = keep
                    if sc > cur and (best is None or sc > best):
                        best, bestc = sc, cand
            if best is not None:
                out[c] = bestc; moved_any = True
        if not moved_any:
            break
    s1 = _score(out, pairs)
    moved = sum(1 for c in cmap if out[c] != cmap[c])
    assert s1 >= s0, "colour repair must never make the palette worse"
    print(f"    {tag:<5} neighbouring pairs {len(pairs):>3} · below dE {MIN_DE:.0f}: "
          f"{-s0[0]} -> {-s1[0]} · worst gap {s0[1]:.1f} -> {s1[1]:.1f} · "
          f"{moved} colour(s) moved, all inside their own hue family")
    return out, s1[1], -s1[0], len(pairs)

print("  colour separation repair (lightness/saturation only, hue family kept):")
colors = {"Main": {c: MACRO7[c % 7] for c in range(Kof["Main"])},
          "Sub": theme_colors("sub"),
          "Question": {i: Q_COLORS[QL[i]] for i in range(len(QL))}}
_resid = {}
for _lv in ("Sub",):
    colors[_lv], _w, _nbad, _npair = _repair(colors[_lv], labs[_lv], _lv)
    _resid[_lv] = (_nbad, _npair, _w)
# 54 sub-themes share 7 hue families, so roughly 8 of them must fit inside one
# hue's usable lightness range. Perfect separation is arithmetically impossible
# there, and pretending otherwise would be the dishonest move: the residual is
# reported, and the focus control (click a row/badge -> everything else greys)
# is the actual answer for the pairs that cannot be pulled apart by colour.
for _lv, (_nb, _np_, _w) in _resid.items():
    if _nb:
        print(f"    {_lv:<5} {_nb} of {_np_} neighbouring pairs remain below dE "
              f"{MIN_DE:.0f} (worst {_w:.1f}); use the focus control for those")
# attribute palettes: distinct by construction, no family meaning to preserve
_ATTR_PAL = ["#0F6E56", "#B84413", "#4A6FA5", "#C9A227", "#7D5BA6", "#3E8E7E", "#A34F6C"]
for _vn, _g in ATTR.items():
    colors[_vn] = {i: _ATTR_PAL[i % len(_ATTR_PAL)] for i in range(len(_g))}

# sentiment: the project ramp, so a point on this map and a bar in Stage 63 that
# look the same colour mean the same measured number
_SR = [_mc.to_rgb(c) for c in _SENT_RAMP]
def _sent_hex(v):
    if not np.isfinite(v):
        return _GREY                              # not measurable - never a ramp colour
    t = (float(np.clip(v, -1, 1)) + 1) / 2 * (len(_SR) - 1)
    i = int(np.floor(t)); i = min(i, len(_SR) - 2); f = t - i
    return _mc.to_hex([_SR[i][k] * (1 - f) + _SR[i + 1][k] * f for k in range(3)])

# ==============================================================================
# LABELS, BADGES, QUOTES
# ==============================================================================
def _label(lv, c):
    c = int(c)
    if lv == "Question":
        return QL[c]
    if lv in ATTR:
        return ATTR[lv][c]
    return _nm(lv, c)

def _pal_of(lv):
    """Colour map for a view, always answering for an unplaced point.

    A response the clustering left in no theme has no theme colour and must
    not be given one: it takes the single reserved grey, the same grey every
    static figure in this notebook uses for the unthemed share."""
    d = dict(colors.get(lv, {}))
    d.setdefault(-1, NO_THEME_COLOR)
    return d


def _chain(lv, c):
    c = int(c)
    if c < 0:
        # unplaced by the clustering: say so, do not invent a parent
        return NO_THEME
    if lv == "Sub":
        _p = s2M.get(c)
        return (f"S{c} → M{_p} {_nm('Main', _p)}" if _p is not None
                else f"S{c} (no parent recorded)")
    if lv == "Main":
        return "main theme"
    if lv == "Question":
        return "survey question"
    return f"participant attribute · {lv.lower()}"

def _medoid(idx):
    """A badge must stand on ground its own theme occupies. median(x),median(y)
    taken separately can fall outside a bent cluster; the medoid is an actual
    response of the group, so it cannot."""
    P = XY[idx]
    return int(idx[np.argmin(np.linalg.norm(P - np.median(P, axis=0), axis=1))])

def _quote_profile(idx):
    """What this group's answers are actually like, on ONE rule shared by both
    explorers.

    F3 - the 2-D cell called a theme "short" only when NOTHING reached 12 words,
    the 3-D cell when FEWER THAN THREE did. On this data that is 6 of the 39
    sub-themes, and the same theme was given 3 quotes on one map and 6 on the
    other while the 2-D docstring claimed the two could never differ. The rule
    kept is the 3-D one, because it is the one whose mechanism matches its
    purpose: three quote slots cannot be filled with substantive text unless
    three substantive answers exist.

    `long` and `n12` are the same number under two names, so both sidebar
    templates keep working unchanged.
    """
    w = np.array([len(str(texts[i]).split()) for i in idx if str(texts[i]).strip()])
    if not len(w):
        return dict(n=0, med=0, long=0, n12=0, short=True)
    _n12 = int((w >= 12).sum())
    return dict(n=int(len(w)), med=int(np.median(w)), long=_n12, n12=_n12,
                short=bool(_n12 < 3))

def _quote_meta(i):
    bits = [country[i], track[i],
            (f"medal {medal[i]}" if medal[i] not in ("", "?", "None") else "")]
    if _have_master and np.isfinite(sent[i]):
        bits.append(f"sentiment {sent[i]:+.2f}")
    return " · ".join(b for b in bits if b)

def _best_quotes(idx, n_long=3, n_short=6):
    """Same selection rule as the 3D explorer - and since F3 that is actually
    true, because both now share one _quote_profile: reward informative length,
    penalise distance from the centroid, and when the theme cannot fill three
    slots with substantive answers show more short ones instead of padding."""
    idx = np.asarray(idx)
    prof = _quote_profile(idx)
    if not len(idx):
        return [], prof
    cen = X5[idx].mean(0)
    d = np.linalg.norm(X5[idx] - cen, axis=1)
    d = (d - d.min()) / (d.max() - d.min() + 1e-9)
    cand = []
    for j, i in enumerate(idx):
        t = " ".join(str(texts[i]).split())
        if not t or t.lower() == "nan":
            continue
        w = len(t.split())
        info = min(w, 25) / 25.0
        cand.append((0.65 * info - 0.35 * d[j], w, t, int(i)))
    cand.sort(key=lambda z: -z[0])
    want = n_short if prof["short"] else n_long
    out, seen = [], set()
    for _, w, t, i in cand:
        k = t.lower()[:60]
        if k in seen:
            continue
        seen.add(k)
        out.append(dict(t=t[:400], m=_quote_meta(i)))
        if len(out) >= want:
            break
    return out, prof

def _nearest_theme(lv, c, lab):
    """The theme this one is most easily confused with, measured in the same 5-D
    space the clustering used. A gap smaller than the two clusters' own average
    radius means they overlap, and the card says so rather than implying every
    theme is cleanly separate."""
    lab = LAB_CORE.get(lv, lab)      # geometry question -> core geometry
    ids = [int(x) for x in np.unique(lab) if int(x) >= 0]
    if len(ids) < 2:
        return None
    cen = {i: X5[lab == i].mean(0) for i in ids if (lab == i).any()}
    if c not in cen:
        return None
    d = {i: float(np.linalg.norm(cen[c] - v)) for i, v in cen.items() if i != c}
    if not d:
        return None
    j = min(d, key=d.get)
    spread = float(np.mean([np.linalg.norm(X5[lab == c] - cen[c], axis=1).mean(),
                            np.linalg.norm(X5[lab == j] - cen[j], axis=1).mean()]))
    return dict(id=j, name=_label(lv, j), dist=round(d[j], 3), close=bool(d[j] < spread))

# ---- Stage 63 is the single source for group x theme rates -------------------
# The attribute cards quote Stage 63's participant-level rates and its FDR flag,
# never a second number computed here, so the map and the bar charts cannot
# disagree about the same claim.
GT = _PL("pooled_group_theme")
_gt_ok = GT is not None and len(getattr(GT, "columns", [])) > 0
if not _gt_ok:
    print("  (pooled_group_theme absent - run Stage 63 first; attribute cards will "
          "show response-level shares instead, and will say so)")

def _concentration(lab, k=15):
    """Does this attribute organise the map at all? For each response, the share of
    its k nearest neighbours in the SAME 5-D space the clustering used that carry
    the same group label, against the share expected if the label were spread at
    random. A ratio near 1 means the attribute explains nothing about position -
    and a figure that colours by such an attribute must say so, because a reader
    looking at coloured dots will otherwise read structure into the mixing."""
    from scipy.spatial import cKDTree
    m = lab >= 0
    if m.sum() < k + 1:
        return None
    P, L = X5[m], lab[m]
    _, nb = cKDTree(P).query(P, k=k + 1)
    same = float(np.mean([np.mean(L[nb[i, 1:]] == L[i]) for i in range(len(L))]))
    _, cnt = np.unique(L, return_counts=True)
    p = cnt / cnt.sum()
    base = float((p ** 2).sum())          # chance agreement for these group sizes
    return dict(same=round(100 * same, 1), base=round(100 * base, 1),
                ratio=round(same / base, 2) if base else None, k=k)

def _attr_findings(vn, group, level, top=6):
    """Which themes this group voices more than the others do."""
    if _gt_ok:
        S = GT[(GT["factor"] == vn) & (GT["level"] == level)]
        col = f"rate::{group}"
        if len(S) and col in S.columns:
            rc = [c for c in S.columns if c.startswith("rate::")]
            oth = S[[c for c in rc if c != col]].mean(axis=1)
            lift = (S[col] - oth)
            S = S.assign(_lift=lift).sort_values("_lift", ascending=False).head(top)
            return [dict(name=str(r["name"]), rate=round(100 * float(r[col]), 1),
                         other=round(100 * float(o), 1),
                         sig=bool(r.get("sig_fdr", False)))
                    for (_, r), o in zip(S.iterrows(), oth.loc[S.index])], "s63"
    lab = labs[level]
    m = labs[vn] == ATTR[vn].index(group)
    rows = []
    for c in range(Kof[level]):
        a = float((lab[m] == c).mean()) if m.any() else 0.0
        b = float((lab[~m] == c).mean()) if (~m).any() else 0.0
        rows.append((a - b, _label(level, c), 100 * a, 100 * b))
    rows.sort(reverse=True)
    return [dict(name=n, rate=round(a, 1), other=round(b, 1), sig=False)
            for _, n, a, b in rows[:top]], "local"

CONC = {vn: _concentration(labs[vn]) for vn in ATTR}
for _vn, _c in CONC.items():
    if _c:
        print(f"  {_vn}: a response's {_c['k']} nearest neighbours share its "
              f"{_vn.lower()} {_c['same']}% of the time vs {_c['base']}% by chance "
              f"(x{_c['ratio']}) — "
              + ("this attribute barely organises the map"
                 if _c["ratio"] and _c["ratio"] < 1.25 else
                 "responses from a group do sit together"))

# ==============================================================================
# FIGURE
# ==============================================================================
# The old cell clipped to percentiles because the squashed frame looked wrong
# otherwise. With the aspect locked 1:1 that crutch is unnecessary, and it was
# never free: it pushed 34 responses off the edge of the picture. Full range, so
# every response that is counted is also visible.
_xl = np.array([XY[:, 0].min(), XY[:, 0].max()])
_yl = np.array([XY[:, 1].min(), XY[:, 1].max()])
_pad = 0.04 * max(_xl[1] - _xl[0], _yl[1] - _yl[0])
_off = 0                                  # nothing is clipped out of the frame now

MINC = 15            # KDE needs enough points to describe a shape, not noise
def _kde(lab, color_of, fill=True):
    """Filled bands read well up to ~30 groups; at 54 the stacked translucent
    fills are what turned the map muddy, so above that the same density is drawn
    as an outline. The gate and the mode are both reported in the footer."""
    from scipy.stats import gaussian_kde
    out, drawn = [], []
    for c in sorted(set(int(x) for x in lab) - {-1}):
        idx = np.where(lab == c)[0]
        if len(idx) < MINC:
            continue
        P = XY[idx]
        try:
            k = gaussian_kde(P.T)
        except Exception:
            continue
        x0, x1 = float(P[:, 0].min()), float(P[:, 0].max())
        y0, y1 = float(P[:, 1].min()), float(P[:, 1].max())
        px, py = (x1 - x0) * 0.28 + 1e-6, (y1 - y0) * 0.28 + 1e-6
        gx = np.linspace(x0 - px, x1 + px, 40)
        gy = np.linspace(y0 - py, y1 + py, 40)
        GX, GY = np.meshgrid(gx, gy)
        Z = k(np.vstack([GX.ravel(), GY.ravel()])).reshape(40, 40)
        Z = (Z / (Z.max() or 1.0)).round(3)
        r_, g_, b_ = [int(255 * v) for v in _mc.to_rgb(color_of(c))]
        if fill:
            cs = [[0.00, f"rgba({r_},{g_},{b_},0)"], [0.24, f"rgba({r_},{g_},{b_},0)"],
                  [0.45, f"rgba({r_},{g_},{b_},0.10)"],
                  [0.70, f"rgba({r_},{g_},{b_},0.19)"],
                  [1.00, f"rgba({r_},{g_},{b_},0.32)"]]
            t = go.Contour(x=gx, y=gy, z=Z, colorscale=cs, showscale=False,
                           contours=dict(start=0.25, end=1.0, size=0.25, coloring="fill"),
                           line=dict(width=0), hoverinfo="skip", showlegend=False)
        else:
            t = go.Contour(x=gx, y=gy, z=Z, showscale=False, hoverinfo="skip",
                           showlegend=False,
                           contours=dict(start=0.42, end=0.42, size=1,
                                         coloring="none"),
                           line=dict(width=1.25, color=f"rgba({r_},{g_},{b_},0.55)"))
        out.append(t); drawn.append(c)
    return out, drawn

fig = go.Figure()
tr_of, badge_tr, has_band = {v: [] for v in VIEWS}, {}, {}
for lv in VIEWS:
    first = (lv == "Main")
    if lv == "Sentiment":
        cvals = [_sent_hex(v) for v in sent]
        cd = [[("sentiment %+.2f" % sent[i]) if np.isfinite(sent[i])
               else "sentiment not measurable",
               "no lexicon evidence - Stage 14b" if not np.isfinite(sent[i])
               else "measured sentiment (LLM-read)",
               str(meta["question"].iloc[i]), str(meta["Team"].iloc[i]),
               (" · " + _quote_meta(i)) if _quote_meta(i) else "",
               wrap_quote(texts[i], 56, 6) +
               ("<br><i style='color:#98A2AD'>… response continues</i>"
                if len(str(texts[i])) > 330 else ""),
               _nm("Sub", labs["Sub"][i]), -1] for i in range(N)]
        has_band[lv] = []
    else:
        lab = labs[lv]
        # A density band claims "this group lives HERE". That is true of a cluster
        # and false of an attribute: every region is spread across the whole map,
        # so five filled bands simply stack into mush and imply a structure the
        # data does not have. Attribute views get points and the card instead, and
        # the concentration number below says plainly how little they separate.
        if lv in ATTR:
            bands, drawn = [], [c for c in range(Kof[lv]) if (lab == c).any()]
        else:
            # shape from the CORE members only - see the coverage note above
            bands, drawn = _kde(LAB_CORE.get(lv, lab),
                                lambda c, _l=lv: _pal_of(_l).get(c, NO_THEME_COLOR),
                                fill=(Kof[lv] <= 33))
        has_band[lv] = drawn
        for t in bands:
            t.visible = first
            fig.add_trace(t); tr_of[lv].append(len(fig.data) - 1)
        _summ = {}
        for c in range(Kof[lv]):
            if lv == "Question":
                _summ[c] = wrap_quote(str(CFG.text_cols[c]), 56, 2)
            elif lv in ATTR:
                _summ[c] = f"{int((lab == c).sum())} responses from {ATTR[lv][c]}"
            else:
                d0 = llm[lv].get(c) or llm[lv].get(str(c)) or {}
                _summ[c] = wrap_quote(str(d0.get("summary", "")), 56, 2)
        cvals = [_pal_of(lv).get(int(c), NO_THEME_COLOR) for c in lab]
        cd = []
        for i in range(N):
            c = int(lab[i])
            long_txt = len(str(texts[i])) > 330
            _tt = (_TIERTXT.get(str(TIER[i]), "") if lv in THEME_V else "")
            cd.append([_label(lv, c) if c >= 0 else "(no group)",
                       _chain(lv, c) + (f" · {_tt}" if _tt else ""),
                       str(meta["question"].iloc[i]), str(meta["Team"].iloc[i]),
                       (" · " + _quote_meta(i)) if _quote_meta(i) else "",
                       wrap_quote(texts[i], 56, 6) +
                       ("<br><i style='color:#98A2AD'>… response continues</i>"
                        if long_txt else ""),
                       _summ.get(c, ""), c])
    fig.add_trace(go.Scattergl(
        x=XY[:, 0], y=XY[:, 1], mode="markers", name=lv, visible=first,
        meta=dict(level=lv, kind="pts"),
        marker=dict(size=5.4, opacity=0.82, color=cvals,
                    line=dict(width=0.45, color="rgba(255,255,255,0.75)")),
        customdata=cd,
        hovertemplate=("<b>%{customdata[0]}</b><br>"
                       "<span style='color:#5B6670'>%{customdata[1]}</span><br>"
                       "<i style='color:#98A2AD'>%{customdata[6]}</i><br>"
                       "<span style='color:#98A2AD'>─────────────────</span><br>"
                       "%{customdata[5]}<br>"
                       "<span style='color:#98A2AD'>─────────────────</span><br>"
                       "Q: <b>%{customdata[2]}</b> · %{customdata[3]}%{customdata[4]}"
                       "<extra></extra>")))
    tr_of[lv].append(len(fig.data) - 1)

    if lv == "Sentiment" or lv in ATTR:
        badge_tr[lv] = None          # nothing to pin a badge to; select from the list
        continue
    bx, by, bt, bc, bl, ids, hollow = [], [], [], [], [], [], []
    _blab = LAB_CORE.get(lv, labs[lv])
    for c in range(Kof[lv]):
        idx = np.where(_blab == c)[0]          # badge stands on a CORE response
        if not len(idx):
            continue
        m = _medoid(idx)
        bx.append(float(XY[m, 0])); by.append(float(XY[m, 1]))
        bt.append(str(c + 1) if lv == "Question" else str(c))
        small = c not in has_band[lv]
        hollow.append(small)
        bc.append("#FFFFFF" if small else _pal_of(lv).get(c, NO_THEME_COLOR))
        bl.append(_pal_of(lv).get(c, NO_THEME_COLOR))
        ids.append(c)
    fig.add_trace(go.Scatter(
        x=bx, y=by, mode="markers+text", visible=first, name=f"{lv} ids",
        meta=dict(level=lv, kind="badge", ids=ids),
        marker=dict(size=19, color=bc, line=dict(width=2.0, color=bl), opacity=0.97),
        text=bt, textfont=dict(size=9.5, family=f"{FONT}, sans-serif",
                               color=["#14202B" if h else "white" for h in hollow]),
        hovertext=[f"{_label(lv, i)}  ·  n={int((labs[lv] == i).sum())}"
                   f"{' (core %d)' % int((_blab == i).sum()) if int((_blab == i).sum()) != int((labs[lv] == i).sum()) else ''}"
                   f"{'  (n < %d: no density band)' % MINC if i not in has_band[lv] else ''}"
                   f"  —  click for the card" for i in ids],
        hoverinfo="text", showlegend=False))
    badge_tr[lv] = len(fig.data) - 1
    tr_of[lv].append(len(fig.data) - 1)

_nT = len(fig.data)
_FOOT_BASE = (f"{N} responses · UMAP 2-D, axes locked 1:1 so distance on screen is "
              f"distance in the embedding")

def _foot(lv=None):
    """The caption must describe the view actually on screen. An attribute view has
    no density bands and no badges, so repeating that legend there would explain
    marks the reader cannot see."""
    s = _FOOT_BASE
    if lv in ATTR:
        c = CONC.get(lv)
        s += (f" · coloured by {lv.lower()}, which is a participant attribute, not a "
              f"cluster: no density bands are drawn")
        if c:
            s += (f" · a response's {c['k']} nearest neighbours share its "
                  f"{lv.lower()} {c['same']}% of the time vs {c['base']}% by chance "
                  f"(×{c['ratio']})")
    elif lv == "Sentiment":
        s += (f" · colour = measured sentiment · {N - _n_sc} responses drawn "
              f"grey because their verdict was withheld: they have no "
              f"score and are never treated as zero")
    else:
        s += (f" · density band drawn for groups with n ≥ {MINC} "
              f"(hollow badge = below that, points only)")
        if lv in THEME_V:
            _r = _n_wide[lv] - _n_core[lv]
            s += (f" · {_n_wide[lv]} responses carry a theme"
                  + (f", of which {_r} were recovered by the projection rule "
                     f"rather than placed by the clustering (shape, badges and "
                     f"quotes still come from the {_n_core[lv]} clustered ones)"
                     if _r > 0 else "")
                  + f" · {N - _n_wide[lv]} carry none")
    # wrap, or a long caption runs off the right edge of the exported PNG
    out, line = [], ""
    for w in s.split(" "):
        if len(line) + len(w) + 1 > 128:
            out.append(line); line = w
        else:
            line = (line + " " + w).strip()
    if line:
        out.append(line)
    return "<br>".join(out)

_FOOT = _foot("Main")
fig.update_layout(
    template="igem", height=730, autosize=True, hoverlabel=HOVER, hovermode="closest",
    title=None, showlegend=False, margin=dict(l=8, r=8, t=8, b=34),
    xaxis=dict(visible=False, range=[_xl[0] - _pad, _xl[1] + _pad], constrain="domain"),
    yaxis=dict(visible=False, range=[_yl[0] - _pad, _yl[1] + _pad],
               scaleanchor="x", scaleratio=1, constrain="domain"),
    annotations=[dict(text=_FOOT, xref="paper", yref="paper", x=0, y=-0.028,
                      xanchor="left", yanchor="top", showarrow=False,
                      font=dict(size=9, color="#8A939B",
                                family=f"{FONT}, sans-serif"))])

# ==============================================================================
# STATIC EXPORT - a clean map, plus a key sheet that names every badge
# ==============================================================================
# A PNG in a paper has no sidebar, so a map of bare numbers cannot be read at all.
# Squeezing 54 names into a legend band beside the map was tried and rejected: the
# names are long, the column is narrow, and the entries collide. Two files instead
# - the map stays uncluttered, and the key is a properly laid out sheet, which is
# also how a reader wants to use it (look up a number, not scan a squashed strip).
base = os.path.join(POOLED_DIR, "explorer_2d_hierarchy")
_key  = os.path.join(POOLED_DIR, "explorer_2d_key")
_dx = (_xl[1] - _xl[0]) + 2 * _pad
_dy = (_yl[1] - _yl[0]) + 2 * _pad
_PH = 1000
_PW = int(min(1600, max(620, _PH * _dx / _dy + 18)))
LIST_N = {lv: [(c, _label(lv, c), int((labs[lv] == c).sum()), _pal_of(lv).get(c, NO_THEME_COLOR),
                bool(c in has_band[lv]))
               for c in sorted(set(int(x) for x in labs[lv]) - {-1})]
          for lv in GROUP_V}

_KEY_PT = 7.8                       # row type size
_KEY_CW = 0.60 * _KEY_PT / 72.0     # average glyph width in inches at that size
_KEY_GUT = 0.60                     # inches reserved for the swatch and the id

def _key_sheet(lv):
    """One sheet: swatch, id, name, n. The page is SIZED FROM THE LONGEST ROW that
    actually exists rather than the text being squeezed into a fixed column, which
    is what made the in-figure legend collide. Nothing here can overlap."""
    import matplotlib.pyplot as _plt
    rows = LIST_N[lv]
    if not rows:
        return None
    body = [(c, f"{nm}  ({n})" + ("" if band else "   ·  no density band"), col, band)
            for c, nm, n, col, band in rows]
    longest = max(len(t) for _, t, _, _ in body)
    ncol = 1 if len(body) <= 16 else 2
    per = int(np.ceil(len(body) / ncol))
    colw = _KEY_GUT + _KEY_CW * longest + 0.25
    fw, fh = colw * ncol + 0.35, max(2.0, 0.235 * per + 0.85)
    f, ax = _plt.subplots(figsize=(fw, fh), dpi=300)
    ax.axis("off")
    ax.text(0, 1.0, f"{lv} — {len(body)} groups", transform=ax.transAxes,
            fontsize=10.5, fontweight="bold", va="top", color="#14202B")
    sw, idr, txl, cwf = 0.15 / fw, 0.47 / fw, 0.55 / fw, colw / fw
    for k, (c, txt, col, band) in enumerate(body):
        cx = (k // per) * cwf
        cy = 0.945 - (k % per) * (0.945 / per)
        ax.add_patch(_plt.Rectangle((cx, cy - 0.010 * (7.0 / max(fh, 1))), sw,
                                    0.020 * (7.0 / max(fh, 1)),
                                    transform=ax.transAxes, clip_on=False,
                                    facecolor=("white" if not band else col),
                                    edgecolor=col, linewidth=1.1))
        ax.text(cx + idr, cy, str(c), fontsize=_KEY_PT - 0.4, transform=ax.transAxes,
                va="center", ha="right", color="#5B6670", fontweight="bold")
        ax.text(cx + txl, cy, txt, fontsize=_KEY_PT, transform=ax.transAxes,
                va="center", color="#2A3742")
    ax.text(0, -0.30 / fh, f"hollow swatch = fewer than {MINC} responses, drawn as "
            f"points only · numbers match the badges on "
            f"explorer_2d_hierarchy_{lv.lower()}.png",
            transform=ax.transAxes, fontsize=6.8, color="#8A939B", va="top")
    p = f"{_key}_{lv.lower()}.png"
    f.savefig(p, bbox_inches="tight", facecolor="white")
    _plt.close(f)
    return p

_png, _keys = [], []
for lv in VIEWS:
    for i, t in enumerate(fig.data):
        t.visible = (i in tr_of[lv])
    fig.layout.margin.t, fig.layout.margin.b = 54, 62
    fig.update_layout(annotations=[
        dict(text=_foot(lv), xref="paper", yref="paper", x=0, y=-0.022,
             xanchor="left", yanchor="top", showarrow=False, align="left",
             font=dict(size=9, color="#8A939B", family=f"{FONT}, sans-serif")),
        dict(text=f"<b>{lv}</b>" + ("" if lv not in LIST_N
                                    else f"  <span style='color:#8A939B'>"
                                         f"{len(LIST_N[lv])} groups · names in "
                                         f"explorer_2d_key_{lv.lower()}.png</span>"),
             xref="paper", yref="paper", x=0, y=1.012, xanchor="left",
             yanchor="bottom", showarrow=False,
             font=dict(size=12, color="#14202B", family=f"{FONT}, sans-serif"))])
    try:
        # size the canvas to the DATA, not the other way round: with the axes
        # locked 1:1 a mismatched canvas can only be paid for in blank margin
        fig.write_image(f"{base}_{lv.lower()}.png", width=_PW, height=_PH, scale=2)
        _png.append(lv)
        if lv in LIST_N:
            _keys.append(_key_sheet(lv))
    except Exception as e:
        _m = str(e).strip().split("\n")[0][:160]
        print(f"  (static export stopped at {lv}: {type(e).__name__}: {_m})")
        print("   the interactive HTML is unaffected; for PNGs run "
              "`pip install -U kaleido` and re-run this cell")
        break
# back to the interactive layout: the HTML header carries the title, and the JS
# swaps the caption when the view changes
fig.layout.margin.t, fig.layout.margin.b = 10, 46
fig.update_layout(annotations=[dict(text=_foot("Main"), xref="paper", yref="paper",
                                    x=0, y=-0.022, xanchor="left", yanchor="top",
                                    showarrow=False, align="left",
                                    font=dict(size=9, color="#8A939B",
                                              family=f"{FONT}, sans-serif"))])
for i, t in enumerate(fig.data):
    t.visible = (i in tr_of["Main"])

# ==============================================================================
# SIDEBAR DATA
# ==============================================================================
CB, LIST = {}, {}
for lv in GROUP_V:
    lab = labs[lv]; e = {}
    _clab = LAB_CORE.get(lv, lab)
    for c in range(Kof[lv]):
        idx = np.where(lab == c)[0]
        if not len(idx):
            continue
        cidx = np.where(_clab == c)[0]
        # quote only what the clustering itself placed: a recovered response is
        # a rule's guess, and a guess must not become a theme's headline evidence
        q, prof = _best_quotes(cidx if len(cidx) else idx)
        base_e = dict(name=_label(lv, c), chain=_chain(lv, c), n=int(len(idx)),
                      n_core=int(len(cidx)), n_rec=int(len(idx) - len(cidx)),
                      quotes=q, prof=prof, band=bool(c in has_band[lv]),
                      col=_pal_of(lv).get(c, NO_THEME_COLOR))
        if lv in THEME_V:
            d = llm[lv].get(c) or llm[lv].get(str(c)) or {}
            base_e.update(summary=d.get("summary", ""), sent=d.get("sentiment", ""),
                          near=_nearest_theme(lv, c, lab))
        elif lv == "Question":
            base_e.update(summary=str(CFG.text_cols[c]), sent="", near=None)
        else:
            f, src = _attr_findings(lv, ATTR[lv][c], "Sub")
            base_e.update(summary="", sent="", near=None, find=f, findsrc=src,
                          findlvl="Sub")
        # measured mean sentiment of the group, shown as a number not a colour
        sv = sent[idx][np.isfinite(sent[idx])]
        base_e["msent"] = (round(float(sv.mean()), 3) if len(sv) else None)
        base_e["nsent"] = int(len(sv))
        e[c] = base_e
    CB[lv] = e
    LIST[lv] = [[c, e[c]["name"], e[c]["n"], e[c]["col"], bool(e[c]["band"])]
                for c in sorted(e)]

_QMETA = dict(qof=[int(x) for x in labs["Question"]], qlab=QL)
_SENTINFO = dict(n=int(N), scorable=_n_sc, grey=int(N - _n_sc),
                 med=(round(float(np.nanmedian(sent)), 3) if _n_sc else None),
                 ramp=[_mc.to_hex(c) for c in _SR])

TPL = """<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>iGEM 2-D theme explorer</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body{margin:0;font-family:__FONT__,Inter,Arial,sans-serif;color:#14202B;background:#fff}
#hd{padding:10px 14px 6px;border-bottom:1px solid #EDF0F3}
#hd b{font-size:16px}#hd span{font-size:11.5px;color:#8A939B;margin-left:8px}
#bar{padding:7px 14px;border-bottom:1px solid #EDF0F3;display:flex;flex-wrap:wrap;
 gap:5px;align-items:center}
#bar button{font-size:11.5px;padding:4px 11px;border:1px solid #D3D8DE;background:#fff;
 border-radius:7px;cursor:pointer;white-space:nowrap;font-family:inherit}
#bar button.on{background:#14202B;color:#fff;border-color:#14202B}
#bar .sep{width:1px;height:18px;background:#E3E7EA;margin:0 5px}
#wrap{display:flex;height:760px;max-width:1250px;margin:0 auto}
#plot{flex:1;min-width:0}
#side{width:352px;border-left:1px solid #E7EAEE;padding:12px 14px;overflow-y:auto;
 background:#FBFCFD}
h3{margin:4px 0 2px;font-size:15.5px}.n{color:#98A2AD;font-size:11.5px}
.sect{font-size:10px;text-transform:uppercase;letter-spacing:.5px;color:#8A939B;
 margin:12px 0 5px;font-weight:600}
.sent{display:inline-block;font-size:11px;padding:1px 8px;border-radius:9px;color:#fff;margin:4px 0}
.sm{color:#5B6670;font-size:12.5px;line-height:1.45;margin:7px 0}
.q{border-left:3px solid #0F6E56;padding:2px 0 2px 9px;margin:7px 0 2px;font-size:12px;
 font-style:italic;color:#333;line-height:1.4}
.qm{font-size:10px;color:#98A2AD;margin:0 0 7px 12px}
.chain{font-size:11.5px;color:#5B6670;margin:2px 0 7px}
.near,.prof{font-size:11px;color:#5B6670;margin-top:8px;padding-top:6px;
 border-top:1px solid #E7EAEE;line-height:1.45}
.row{display:flex;align-items:center;gap:7px;padding:2.5px 5px;border-radius:5px;
 cursor:pointer;font-size:11.5px;line-height:1.3}
.row:hover{background:#EEF1F4}.row.on{background:#14202B;color:#fff}
.sw{width:10px;height:10px;border-radius:2px;flex:0 0 auto}
.sw.hollow{background:#fff!important}
.row .c{margin-left:auto;color:#98A2AD;font-size:10.5px}.row.on .c{color:#C8CFD6}
.qf label{display:block;font-size:11.5px;margin:2px 0;cursor:pointer}
.ramp{height:9px;border-radius:5px;margin:6px 0 3px}
.hint{color:#98A2AD;font-size:11.5px;line-height:1.5}
#clear{margin-top:9px;font-size:11.5px;padding:4px 10px;border:1px solid #D3D8DE;
 background:#fff;border-radius:7px;cursor:pointer;font-family:inherit}
.find{font-size:11.5px;margin:4px 0;line-height:1.4}
.find b{font-variant-numeric:tabular-nums}.star{color:#B8860B}
.rec{font-size:11px;color:#8A939B;margin:3px 0 0}
</style></head><body>
<noscript><div style="padding:26px;color:#B84413;font-size:13px">This explorer
needs JavaScript. The static PNGs beside it carry the same map.</div></noscript>
<div id="hd"><b>Theme explorer</b><span>hover = the response in its own words +
metadata · click a badge, a point or a list row = that group's card · axes are
locked 1:1, so distance on screen is distance in the embedding</span></div>
<div id="bar"><span style="font-size:10px;color:#8A939B;text-transform:uppercase;
letter-spacing:.5px;margin-right:2px">view</span><span id="vbtn"></span></div>
<div id="wrap"><div id="plot"></div>
<aside id="side">
 <div class="sect">questions shown</div><div class="qf" id="qf"></div>
 <div id="panel"></div>
 <button id="clear" onclick="clearSel()">clear highlight</button>
</aside></div>
<script>
// A JS error kills the rest of the block silently and the page then looks like
// an empty plot rather than a failure. Say so, on the page, in plain words.
window.onerror=function(msg,src,ln,col,err){
 var p=document.getElementById('plot');
 if(p&&!p.getAttribute('data-failed')){p.setAttribute('data-failed','1');
  p.innerHTML='<div style="padding:26px;font:12.5px/1.6 ui-monospace,monospace;'+
   'color:#B84413;background:#FFF6F2;border:1px solid #F0D2C6;border-radius:8px;'+
   'margin:18px">'+
   '<b>This explorer did not start.</b><br><br>'+String(msg)+
   '<br>line '+ln+', column '+col+
   '<br><br><span style="color:#5B6670">The map is not empty because there is '+
   'nothing to show &mdash; a script error stopped it before it drew. '+
   'Quote this line number when reporting it.</span></div>';}
 return false;};
</script>
<script>
var FIG=__FIG__,CB=__CB__,LIST=__LIST__,TROF=__TROF__,VIEWS=__VIEWS__;
var QM=__QMETA__,SI=__SENTINFO__,COLS=__COLS__,CONC=__CONC__,FOOT=__FOOT__;
var DIM='rgba(203,209,215,0.22)';
var SC={"positive":"#0F6E56","negative":"#B84413","mixed":"#8C9AA3","neutral":"#BFC4C9"};
var cur='Main',spot=null,QON={};
Plotly.newPlot('plot',FIG.data,FIG.layout,
  {responsive:true,displaylogo:false,
   modeBarButtonsToRemove:['select2d','lasso2d','autoScale2d']}).then(function(){
  var gd=document.getElementById('plot');
  QM.qlab.forEach(function(q,i){QON[i]=true;});
  buildViews();buildQF();setView('Main');
  gd.on('plotly_click',function(ev){
    var p=ev.points[0],m=p.data.meta||{};
    if(!m.level)return;
    if(m.kind==='badge'){pick(m.ids[p.pointNumber]);}
    else if(p.customdata&&p.customdata[7]>=0){pick(p.customdata[7]);}
  });
});
function gd(){return document.getElementById('plot');}
function esc(s){return String(s).replace(/&/g,'&amp;').replace(/</g,'&lt;');}
function buildViews(){
  document.getElementById('vbtn').innerHTML=VIEWS.map(function(v){
    var n=(LIST[v]||[]).length;
    return '<button id="vb_'+v+'" onclick="setView(\\''+v+'\\')">'+v+
           (n?' ('+n+')':'')+'</button>';}).join('');
}
function buildQF(){
  document.getElementById('qf').innerHTML=QM.qlab.map(function(q,i){
    return '<label><input type="checkbox" checked value="'+i+
           '" onchange="setQ('+i+',this.checked)"> '+esc(q)+'</label>';}).join('');
}
function setQ(i,on){QON[i]=on;recolour();}
function setView(v){
  cur=v;spot=null;
  var vis=FIG.data.map(function(_,i){return TROF[v].indexOf(i)>=0;});
  Plotly.restyle(gd(),{visible:vis});
  if(FOOT[v])Plotly.relayout(gd(),{'annotations[0].text':FOOT[v]});
  VIEWS.forEach(function(x){
    var b=document.getElementById('vb_'+x);if(b)b.className=(x===v?'on':'');});
  renderPanel();recolour();
}
function ptTrace(v){
  var t=-1;FIG.data.forEach(function(d,i){
    var m=d.meta||{};if(m.kind==='pts'&&m.level===v)t=i;});return t;}
// One recolour path. Per-point opacity is not honoured for these traces, so the
// question filter and the spotlight BOTH express themselves as colour, and they
// compose instead of overwriting each other.
function recolour(){
  var ti=ptTrace(cur);if(ti<0)return;
  var base=COLS[cur],cd=FIG.data[ti].customdata;
  var nc=base.map(function(c,j){
    if(QON[QM.qof[j]]===false)return DIM;
    if(spot!==null&&cd[j][7]!=spot)return DIM;
    return c;});
  Plotly.restyle(gd(),{'marker.color':[nc]},[ti]);
  var bt=-1;FIG.data.forEach(function(d,i){
    var m=d.meta||{};if(m.kind==='badge'&&m.level===cur)bt=i;});
  if(bt>=0){
    var ids=FIG.data[bt].meta.ids;
    Plotly.restyle(gd(),{'marker.opacity':[
      spot===null?0.97:ids.map(function(i){return i==spot?1:0.25;})]},[bt]);
  }
}
function pick(cid){spot=(spot===cid?null:cid);renderPanel();recolour();}
function clearSel(){spot=null;renderPanel();recolour();}
function renderPanel(){
  var p=document.getElementById('panel');
  if(cur==='Sentiment'){
    p.innerHTML='<div class="sect">measured sentiment</div>'+
      '<div class="ramp" style="background:linear-gradient(90deg,'+SI.ramp.join(',')+')"></div>'+
      '<div class="hint">−1 &nbsp;·&nbsp; 0 &nbsp;·&nbsp; +1 &nbsp; (sentiment score)</div>'+
      '<div class="sm"><b>'+SI.scorable+'</b> of '+SI.n+' responses carry a score'+
      (SI.med!==null?'; median <b>'+SI.med+'</b>':'')+'.<br>'+
      '<span style="display:inline-block;width:9px;height:9px;background:#DCE1E5;'+
      'border-radius:2px"></span> <b>'+SI.grey+'</b> are drawn grey: the verdict '+
      'was withheld for them, so they have no score. They are not neutral and are '+
      'not counted as zero anywhere in this project.</div>'+
      '<div class="hint">Hover any point for its own words.</div>';
    return;}
  var rows=(LIST[cur]||[]).map(function(r){
    return '<div class="row'+(spot===r[0]?' on':'')+'" onclick="pick('+r[0]+')">'+
      '<span class="sw'+(r[4]?'':' hollow')+'" style="background:'+r[3]+
      ';border:1.5px solid '+r[3]+'"></span>'+
      '<span>'+esc(r[1])+'</span><span class="c">'+r[2]+'</span></div>';}).join('');
  var head='<div class="sect">'+cur+' — '+(LIST[cur]||[]).length+
           ' groups <span style="text-transform:none;font-weight:400">'+
           '(click one)</span></div>';
  if(CONC[cur]){var q=CONC[cur];
    head+='<div class="sm" style="margin-top:0">A response\\'s '+q.k+' nearest '+
      'neighbours share its '+cur.toLowerCase()+' <b>'+q.same+'%</b> of the time, '+
      'against <b>'+q.base+'%</b> by chance (×'+q.ratio+'). '+
      (q.ratio<1.25
        ? 'So this attribute barely organises the map: the colours are mixed '+
          'because the groups really are mixed. What differs between them is '+
          'WHICH THEMES they voice — open a group below.'
        : 'Responses from the same group do tend to sit together.')+
      '</div>';}
  p.innerHTML=head+rows+(spot===null?
    '<div class="hint" style="margin-top:10px">Pick a row, a badge or a point to '+
    'see its summary, its closest neighbour and real quotes.</div>':card(cur,spot));
}
function card(v,cid){
  var e=(CB[v]||{})[cid];if(!e)return '';
  var h='<div class="sect">card</div><h3>'+esc(e.name)+'</h3>'+
        '<div class="n">'+v+' '+cid+' · n='+e.n+
        (e.msent!==null?' · mean measured sentiment '+e.msent+' (of '+e.nsent+' scorable)':'')+
        (e.band?'':' · below the density gate, points only')+'</div>';
  if(e.sent)h+='<span class="sent" style="background:'+(SC[e.sent]||'#BFC4C9')+'">'+
               esc(e.sent)+'</span>';
  h+='<div class="chain">'+esc(e.chain||'')+'</div>';
  if(e.n_rec)h+='<div class="rec">'+e.n_core+' placed by the clustering · '+
    e.n_rec+' recovered by the projection rule. The quotes, the density band '+
    'and the badge come from the clustered ones only.</div>';
  if(e.summary)h+='<div class="sm">'+esc(e.summary)+'</div>';
  if(e.find&&e.find.length){
    h+='<div class="sect">themes this group voices most</div>';
    e.find.forEach(function(f){
      h+='<div class="find">'+(f.sig?'<span class="star">★</span> ':'')+esc(f.name)+
         ' — <b>'+f.rate+'%</b> here vs <b>'+f.other+'%</b> elsewhere</div>';});
    h+='<div class="hint">'+(e.findsrc==='s63'
      ? 'participant-level rates from Stage 63; ★ = survives BH-FDR there'
      : 'response-level shares computed here (Stage 63 not yet run, so no test)')+
      '</div>';}
  if(e.prof&&e.prof.short){
    h+='<div class="prof"><b>'+e.prof.n+' responses · median '+e.prof.med+
       ' word(s) · '+e.prof.n12+' of them reach 12 words</b> — too few to fill three '+
       'slots with substantive answers, so more short ones are shown instead.</div>';}
  h+='<div class="sect">in their own words</div>';
  (e.quotes||[]).forEach(function(q){
    h+='<div class="q">'+esc(q.t||q)+'</div>';
    if(q.m)h+='<div class="qm">'+esc(q.m)+'</div>';});
  if(e.near){h+='<div class="near">closest group: <b>'+esc(e.near.name)+'</b>'+
    ' (5-D distance '+e.near.dist+')'+
    (e.near.close?' — these two overlap, so read them together':'')+'</div>';}
  return h;
}
</script></body></html>"""


def _js_guard(html, where):
    """Refuse to ship a page whose script cannot parse.

    An unterminated string literal kills the WHOLE <script> block, and the page
    then renders as an empty white box with no console message a notebook user
    would ever see. That is exactly how this cell shipped broken once: an
    English apostrophe inside a single-quoted JS string. A blank map is
    indistinguishable from a working one at a glance, so the check is done here,
    at build time, where it can still be fixed - not left to the reader."""
    j0 = html.find("<script>", html.find("</head>"))
    j1 = html.rfind("</script>")
    if j0 < 0 or j1 < 0:
        return 0
    js, bad, i, n, line = html[j0 + 8:j1], [], 0, j1 - j0 - 8, 1
    while i < n:
        ch = js[i]
        if ch == "\n":
            line += 1; i += 1; continue
        if ch == "/" and i + 1 < n and js[i + 1] == "/":
            while i < n and js[i] != "\n":
                i += 1
            continue
        if ch in "'\"":
            q, start, i, closed = ch, line, i + 1, False
            while i < n:
                if js[i] == "\\":
                    i += 2; continue
                if js[i] == "\n":
                    break                      # newline inside a literal
                if js[i] == q:
                    closed = True; i += 1; break
                i += 1
            if not closed:
                bad.append((start, js[:i].split("\n")[-1].strip()[-90:]))
                line += 1; i += 1
            continue
        i += 1
    if bad:
        raise RuntimeError(
            f"{where}: the page's JavaScript would not parse, so the explorer "
            f"would render as a silent blank box. Unterminated string "
            f"literal(s):\n  - " +
            "\n  - ".join(f"js line {l}: ...{c}" for l, c in bad))
    return len(js)

_COLS = {}
for lv in VIEWS:
    if lv == "Sentiment":
        _COLS[lv] = [_sent_hex(v) for v in sent]
    else:
        _COLS[lv] = [_pal_of(lv).get(int(c), NO_THEME_COLOR) for c in labs[lv]]

html = (TPL.replace("__FIG__", fig.to_json())
           .replace("__CB__", _json.dumps(CB))
           .replace("__LIST__", _json.dumps(LIST))
           .replace("__TROF__", _json.dumps({v: tr_of[v] for v in VIEWS}))
           .replace("__VIEWS__", _json.dumps(VIEWS))
           .replace("__QMETA__", _json.dumps(_QMETA))
           .replace("__SENTINFO__", _json.dumps(_SENTINFO))
           .replace("__CONC__", _json.dumps({k: v for k, v in CONC.items() if v}))
           .replace("__FOOT__", _json.dumps({v: _foot(v) for v in VIEWS}))
           .replace("__COLS__", _json.dumps(_COLS))
           .replace("__FONT__", str(FONT)))
_js_n = _js_guard(html, "Stage 33 (2D explorer)")
open(base + ".html", "w", encoding="utf-8").write(html)
if MIRROR_POOLED_DIR:
    for _p in ([base + ".html"] + [f"{base}_{v.lower()}.png" for v in VIEWS]
               + [k for k in _keys if k]):
        if os.path.exists(_p):
            _mirror_file(_p, MIRROR_POOLED_DIR)

# ==============================================================================
# ACCURACY GUARD - the figure may not ship if any card misdescribes its group
# ==============================================================================
_prob = []
for lv in GROUP_V:
    lab = labs[lv]
    for c, e in CB[lv].items():
        own = set(np.where(lab == c)[0])
        if e["n"] != len(own):
            _prob.append(f"{lv} {c}: card says n={e['n']}, cluster holds {len(own)}")
        _c_own = set(np.where(LAB_CORE.get(lv, lab) == c)[0]) or own
        owntx = {" ".join(str(texts[i]).split())[:400] for i in _c_own}
        for q in e["quotes"]:
            if (q["t"] if isinstance(q, dict) else q) not in owntx:
                _prob.append(f"{lv} {c}: a quote does not come from this group")
                break
    seen = set(int(x) for x in lab if int(x) >= 0)
    if seen - set(CB[lv]):
        _prob.append(f"{lv}: groups present on the map but missing a card: "
                     f"{sorted(seen - set(CB[lv]))}")
for _tok in ("__FIG__", "__CB__", "__LIST__", "__TROF__", "__VIEWS__", "__CONC__",
             "__FOOT__",
             "__QMETA__", "__SENTINFO__", "__COLS__", "__FONT__"):
    if _tok in html:
        _prob.append(f"template placeholder {_tok} was never filled")
if _prob:
    raise RuntimeError("Stage 33 accuracy guard failed:\n  - " + "\n  - ".join(_prob))

from IPython.display import display as _d, HTML as _H
_d(_H(html))
_ratio = ((_yl[1] - _yl[0]) / (_xl[1] - _xl[0]))
print(f"  saved: explorer_2d_hierarchy.html · {len(_png)} map PNG "
      f"({', '.join(v.lower() for v in _png) or 'none'}) · "
      f"{len([k for k in _keys if k])} key sheet(s) naming every badge")
print(f"  views: " + " · ".join(f"{v} {len(LIST.get(v, []))}" if v != 'Sentiment'
                                else "Sentiment (continuous)" for v in VIEWS))
print(f"  geometry: axes locked 1:1 (data aspect {_ratio:.2f}); the old cell drew "
      f"the same embedding with a 3.03x horizontal stretch")
print(f"  badges  : medoid of each group, so every badge stands on one of its own "
      f"responses · hollow = n < {MINC}")
print(f"  density : " + " · ".join(
    (f"{v} none (attribute, not a cluster)" if v in ATTR else
     f"{v} {len(has_band[v])}/{len(LIST.get(v, []))} banded"
     f"{' (outline)' if Kof.get(v, 0) > 33 else ''}") for v in GROUP_V))
print(f"  sentiment view: {_n_sc} of {N} scorable, {N - _n_sc} drawn grey as "
      f"'verdict withheld' (never as zero)")
_nq = [len(v['quotes']) for lv in GROUP_V for v in CB[lv].values()]
_sh = sum(1 for lv in GROUP_V for v in CB[lv].values() if v['prof']['short'])
print(f"  cards   : {sum(len(CB[lv]) for lv in GROUP_V)} · quotes {min(_nq)}-{max(_nq)} "
      f"· short-answer groups flagged {_sh} · accuracy guard passed")
print(f"  controls: view buttons, question checkboxes and the card list are HTML, "
      f"so nothing can overlap the modebar or clip a label")
print(f"  coverage: {'WIDE' if _wide_on else 'CORE'} - {_n_wide['Sub']} of {N} "
      f"responses carry a sub theme "
      f"(core {_n_core['Sub']} + recovered {_n_wide['Sub'] - _n_core['Sub']}); "
      f"shape, badges and quotes are measured from the core only")
print(f"  script  : {_js_n} chars parsed and verified before shipping; a runtime "
      f"error now paints itself into the plot area instead of blanking it")

### Stage 22 · Three-dimensional theme explorer

**Question.** The same map, with the third component restored.

**How to read it.** UMAP into two dimensions discards structure that three can keep; clusters that overlap in the flat map may separate here. The same caution applies - this is a projection, not a measurement.

In [ ]:
# @title
# ==============================================================================
# STAGE 22  -  THREE-DIMENSIONAL THEME EXPLORER
#             section 3 of 9  -  The themes themselves
# ==============================================================================
# Matched to Stage 33 so the two explorers cannot disagree about the same data:
#   * ASPECT. The scene stretched each axis to fill a cube, so a distance read
#     off the screen was not a distance in the embedding - the exact distortion
#     Stage 33 locked out in 2-D. aspectmode="data" fixes the three axes to the
#     data's own proportions.
#   * FRAME. The 1st/99th percentile clip pushed the most distinctive responses
#     out of the scene entirely. Full range with a pad: everything counted is
#     also visible.
#   * COVERAGE. Followed the raw core partition, so the 210 responses NB2
#     recovered were drawn in the "no theme" grey. It now follows COVERAGE like
#     every claim cell - while hulls, badges, nearest-theme and quotes stay
#     measured on the CORE members, because a rule's match must not be allowed
#     to reshape the cluster it was matched against.
#   * NEAREST THEME. The noise cluster (-1) was a candidate, so a card could
#     report "closest theme: S-1". Only real themes compete now.
#   * FAIL-LOUD. The page parses its own script before shipping, and a runtime
#     error paints itself into the plot area instead of leaving a white box.
# ==============================================================================
import numpy as np
import json as _json
import plotly.graph_objects as go

_PL = globals().get("_orig_pload", pload)
P3 = np.asarray(_PL("pooled_umap3d")); X5 = np.asarray(_PL("pooled_umap5d"))
meta = _PL("pooled_meta").reset_index(drop=True)
labs = {"Main": np.asarray(_PL("pooled_labels_main7")),
        "Sub":  np.asarray(_PL("pooled_labels_sub"))}
maps = _PL("pooled_hier_maps")
llm = {"Main": _PL("pooled_main_llm") or {},
       "Sub": _PL("pooled_sub_llm") or {}}
# sub -> main directly; there is no intermediate rung any more
s2M = {int(k): int(v) for k, v in maps["sub_to_main"].items()}
N = len(meta)
# pooled_master comes from the preflight, already reset and
# length-checked; a second binding of the canonical table inside a
# figure cell is one more thing that can go stale.
_MST = MASTER
if _MST is not None:
    def _mcol(c):
        return _MST[c].fillna("").astype(str).values if c in _MST.columns else np.array([""] * N)
    country, track, medal = _mcol("Country"), _mcol("Track"), _mcol("medal")
    texts = _MST["text"].fillna("").astype(str).values
else:
    country = np.array([""] * N); track = np.array([""] * N)
    aw = _PL("pooled_award") or {}
    medal = np.asarray(aw.get("medal", np.array(["?"] * N, dtype=object))).astype(str)
    _df = get_df(); _pid = {r["Participant"]: r for _, r in _df.iterrows()}
    RAWCOL = dict(zip(CFG.short_labels, CFG.text_cols))
    texts = np.array([("" if (_pid.get(mr["Participant"]) is None) else
                      str(_pid[mr["Participant"]].get(RAWCOL[mr["question"]], "") or ""))
                      for _, mr in meta.iterrows()], dtype=object)
texts = np.array(["" if str(t).strip().lower() == "nan" else str(t) for t in texts], dtype=object)

# ---- coverage, on exactly the rule Stage 33 uses ---------------------------
LAB_CORE = {lv: np.asarray(v).copy() for lv, v in labs.items()}
TIER = (_MST["theme_tier"].astype(str).values if
        (_MST is not None and "theme_tier" in _MST.columns) else np.array(["core"] * N))
_COV = str(globals().get("COVERAGE", "core"))
_wide_on = False
if _MST is not None and _COV == "wide":
    _pairs = [("Main", "main_id_wide"), ("Sub", f"{CLAIM_LEVEL}_id_wide")]
    if all(c in _MST.columns for _, c in _pairs):
        for _lv, _c in _pairs:
            labs[_lv] = np.asarray(_MST[_c], dtype=int)
        _wide_on = True
_TIERTXT = {"core": "placed by the clustering",
            "assigned": "recovered by the projection rule, not clustered",
            "assigned-multi": "clause-level multi-label; primary theme shown",
            "unassigned": "left in no theme by the clustering"}

# --------------------------------------------------------------------------
# Quote selection, and why it is not the shared pick_quotes().
#   pick_quotes() takes centroid-nearest responses of 12-60 words, falling back to
#   anything if too few qualify. That is correct, but it is SILENT about the
#   fallback - and 16 of the 54 sub-themes contain fewer than three responses of
#   12+ words at all. s17 "Impressive Efficiency and Success" (n=34) has a median
#   length of ONE word and nothing longer than nine, so its quotes really are
#   "excellent" / "Perfect" / "good". The old sidebar showed those with no
#   explanation, which reads as a broken quote picker rather than as the data.
#   Here the quote is SCORED - representative AND informative - and the theme's
#   own length profile is reported next to it, so a short quote is understood.
def _nearest_theme(lv, c, lab):
    """The theme this one is most easily confused with, measured in the SAME 5-D
    space the clustering used. A reader looking at a card should be able to ask
    'what is next to this?' without hunting the map, and it is also the honest
    place to expose the cost of a fine partition: at sub level some neighbours sit
    very close, and the card now says so instead of implying every theme is
    distinct."""
    lab = LAB_CORE.get(lv, lab)   # a geometry question -> the clustered points
    # -1 is not a theme. Letting the noise cloud compete made cards report
    # "closest theme: S-1", which is not a statement about anything.
    ids = [int(x) for x in np.unique(lab) if int(x) >= 0]
    if len(ids) < 2: return None
    cen = {i: X5[lab == i].mean(0) for i in ids if (lab == i).any()}
    if c not in cen: return None
    d = {i: float(np.linalg.norm(cen[c] - v)) for i, v in cen.items() if i != c}
    if not d: return None
    j = min(d, key=d.get)
    spread = float(np.mean([np.linalg.norm(X5[lab == c] - cen[c], axis=1).mean(),
                            np.linalg.norm(X5[lab == j] - cen[j], axis=1).mean()]))
    # a gap smaller than the clusters' own average radius means they overlap
    return dict(id=j, name=_label(lv, j), dist=round(d[j], 3),
                close=bool(d[j] < spread))

def _quote_meta(i):
    """Country / track / medal for one response, so a quote can be read in context."""
    bits = [country[i], track[i],
            (f"medal {medal[i]}" if medal[i] not in ("", "?", "None") else "")]
    return " · ".join(b for b in bits if b)

def _quote_profile(idx):
    """What this group's answers are actually like, on ONE rule shared by both
    explorers.

    F3 - the 2-D cell called a theme "short" only when NOTHING reached 12 words,
    the 3-D cell when FEWER THAN THREE did. On this data that is 6 of the 39
    sub-themes, and the same theme was given 3 quotes on one map and 6 on the
    other while the 2-D docstring claimed the two could never differ. The rule
    kept is the 3-D one, because it is the one whose mechanism matches its
    purpose: three quote slots cannot be filled with substantive text unless
    three substantive answers exist.

    `long` and `n12` are the same number under two names, so both sidebar
    templates keep working unchanged.
    """
    w = np.array([len(str(texts[i]).split()) for i in idx if str(texts[i]).strip()])
    if not len(w):
        return dict(n=0, med=0, long=0, n12=0, short=True)
    _n12 = int((w >= 12).sum())
    return dict(n=int(len(w)), med=int(np.median(w)), long=_n12, n12=_n12,
                short=bool(_n12 < 3))

def _best_quotes(idx, n_long=3, n_short=6):
    """Rank by representativeness x informativeness rather than gating on length.

    A hard 12-word gate throws away the best available quote in a theme made of
    short answers, then silently falls back to the nearest text of any length.
    Scoring keeps the same preference for medium-length, centroid-near responses
    where they exist, and degrades gracefully where they do not - and the caller
    is told which case it is. Where the responses are short, MORE of them are
    shown, because three one-word quotes carry almost no information.
    """
    idx = np.asarray(idx)
    if idx.size == 0: return [], _quote_profile(idx)
    prof = _quote_profile(idx)
    cen = X5[idx].mean(0)
    d = np.linalg.norm(X5[idx] - cen, axis=1)
    d = (d - d.min()) / (d.max() - d.min() + 1e-9)        # 0 = most representative
    cand = []
    for j, i in enumerate(idx):
        t = " ".join(str(texts[i]).split())
        if not t or t.lower() == "nan": continue
        w = len(t.split())
        # informativeness rises to ~25 words then flattens; very long answers are
        # not better quotes, they are just longer
        info = min(w, 25) / 25.0
        cand.append((0.65 * info - 0.35 * d[j], w, t, int(i)))
    if not cand: return [], prof
    cand.sort(key=lambda z: -z[0])
    want = n_short if prof["short"] else n_long
    out, seen = [], set()
    for _, w, t, i in cand:
        key = t.lower()[:60]
        if key in seen: continue
        seen.add(key); out.append(dict(t=t[:400], m=_quote_meta(i)))
        if len(out) >= want: break
    return out, prof

def _nm(lv, c):
    d = llm[lv].get(int(c)) or llm[lv].get(str(int(c))) or {}
    return d.get("name") or f"{lv[0]}{c}"
QL = list(CFG.short_labels)
labs["Question"] = np.array([QL.index(q) for q in meta["question"]])
Kof = {lv: int(labs[lv].max()) + 1 for lv in labs}
colors = {"Main": {c: MACRO7[c % 7] for c in range(Kof["Main"])},
          "Sub": theme_colors("sub"),
          "Question": {i: Q_COLORS[QL[i]] for i in range(len(QL))}}
def _pal_of(lv):
    """Colour map for a view, always answering for an unplaced point.

    A response the clustering left in no theme has no theme colour and must
    not be given one: it takes the single reserved grey, the same grey every
    static figure in this notebook uses for the unthemed share."""
    d = dict(colors.get(lv, {}))
    d.setdefault(-1, NO_THEME_COLOR)
    return d


def _chain(lv, c):
    c = int(c)
    if c < 0:
        return NO_THEME          # unplaced: no parent to report
    if lv == "Sub":
        _p = s2M.get(c)
        return (f"S{c} → M{_p} {_nm('Main', _p)}" if _p is not None
                else f"S{c} (no parent recorded)")
    return "main theme" if lv == "Main" else "survey question"
def _label(lv, c):
    return QL[int(c)] if lv == "Question" else _nm(lv, c)
# Clipping to the 1st/99th percentile hid the responses furthest from
# everything else - the ones a reader most wants to find. Full range, padded.
_cl = []
for _d in range(3):
    _lo, _hi = float(P3[:, _d].min()), float(P3[:, _d].max())
    _pd = 0.04 * (_hi - _lo + 1e-9)
    _cl.append([_lo - _pd, _hi + _pd])

VIEWS = ["Main", "Sub", "Question"]
fig = go.Figure(); tr_of = {v: [] for v in VIEWS}
lx, ly, lz, lc = [], [], [], []
for c in range(Kof["Main"]):
    m = labs["Main"] == c
    lx.append(float(np.median(P3[m, 0]))); ly.append(float(np.median(P3[m, 1])))
    lz.append(float(np.median(P3[m, 2]))); lc.append(MACRO7[c % 7])
fig.add_trace(go.Scatter3d(x=lx, y=ly, z=lz, mode="markers", name="landmarks",
    marker=dict(size=30, color=lc, opacity=0.10), hoverinfo="skip", showlegend=False))
# ---- 1 · a translucent shell per cluster -----------------------------------
# A 3-D scatter alone does not show where one cluster ends and the next begins -
# depth cues are weak and the eye cannot integrate a point cloud. A convex hull
# drawn at very low opacity gives the group an outline without hiding a single
# point. Clusters with fewer than MIN_HULL points get no shell: a hull over four
# points describes the sampling noise, not the theme.
MIN_HULL = 10
hull_of = {v: [] for v in VIEWS}
for lv in VIEWS:
    _hlab = LAB_CORE.get(lv, labs[lv])
    for c in range(Kof[lv]):
        m = _hlab == c                       # shell = the clustering's own shape
        if int(m.sum()) < MIN_HULL: continue
        pts = P3[m]
        fig.add_trace(go.Mesh3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            alphahull=0, opacity=0.055, color=_pal_of(lv).get(c, NO_THEME_COLOR),
            flatshading=True, hoverinfo="skip", showlegend=False,
            visible=False, name=f"{lv} shell {c}",
            meta=dict(level=lv, kind="hull")))
        hull_of[lv].append(len(fig.data) - 1)
print(f"  cluster shells: " + ", ".join(f"{v} {len(hull_of[v])}" for v in VIEWS)
      + f"  (clusters with >= {MIN_HULL} responses)")

_LAND = 0
for lv in VIEWS:
    lab = labs[lv]
    _summ = {}
    for c in range(Kof[lv]):
        if lv == "Question":
            _summ[c] = wrap_quote(str(CFG.text_cols[c]), 52, 2)
        else:
            d0 = llm[lv].get(c) or llm[lv].get(str(c)) or {}
            _summ[c] = wrap_quote(str(d0.get("summary", "")), 52, 2)
    cd = []
    colarr = [_pal_of(lv).get(int(c), NO_THEME_COLOR) for c in lab]
    for i in range(N):
        c = int(lab[i])
        sfx = " · ".join(x for x in (country[i], track[i],
                          f"medal {medal[i]}" if medal[i] not in ("", "?") else "") if x)
        _tt = _TIERTXT.get(str(TIER[i]), "") if lv in ("Main", "Sub") else ""
        cd.append([_label(lv, c) + ("" if not _tt else ""),
                   _chain(lv, c) + (f" · {_tt}" if _tt else ""),
                   str(meta["question"].iloc[i]),
                   str(meta["Team"].iloc[i]), (" · " + sfx) if sfx else "",
                   wrap_quote(texts[i], 52, 4),
                   _summ.get(c, NO_THEME), c])
    fig.add_trace(go.Scatter3d(
        x=P3[:, 0], y=P3[:, 1], z=P3[:, 2], mode="markers", name=lv,
        visible=(lv == "Main"), meta=dict(level=lv, kind="pts"),
        # 3.4px at 88% opacity washes out against white, which is why the cloud
        # looked faint; a slightly larger dot with a thin dark rim holds its
        # colour and makes the cluster shape readable
        marker=dict(size=4.4, opacity=0.92, color=colarr,
                    line=dict(width=0.4, color='rgba(20,32,43,0.35)')),
        customdata=cd,
        hovertemplate=("<b>%{customdata[0]}</b><br>"
                       "<span style='color:#5B6670'>%{customdata[1]}</span><br>"
                       "<i style='color:#98A2AD'>%{customdata[6]}</i><br>"
                       "%{customdata[5]}<br>"
                       "Q: <b>%{customdata[2]}</b> · %{customdata[3]}%{customdata[4]}"
                       "<extra></extra>")))
    tr_of[lv].append(len(fig.data) - 1)
    bx, by, bz, bt, bc, ids = [], [], [], [], [], []
    _blab = LAB_CORE.get(lv, lab)
    for c in range(Kof[lv]):
        m = _blab == c                       # badge stands on a clustered point
        if not m.any(): continue
        bx.append(float(np.median(P3[m, 0]))); by.append(float(np.median(P3[m, 1])))
        bz.append(float(np.median(P3[m, 2])))
        bt.append(f"Q{c + 1}" if lv == "Question" else str(c))
        bc.append(_pal_of(lv).get(c, NO_THEME_COLOR)); ids.append(c)
    fig.add_trace(go.Scatter3d(
        x=bx, y=by, z=bz, mode="text", visible=(lv == "Main"), name=f"{lv} ids",
        meta=dict(level=lv, kind="badge", ids=ids),
        text=bt, textfont=dict(size=13, color=bc),
        hovertext=[f"{_label(lv, i)} — click for the summary card" for i in ids],
        hoverinfo="text", showlegend=False))
    tr_of[lv].append(len(fig.data) - 1)
_nT = len(fig.data)
fig.update_layout(
    template="igem", autosize=True, hoverlabel=HOVER, showlegend=False,
    # aspectmode="data": the three axes keep the data's own proportions, so a
    # distance judged by eye in this scene is a distance in the embedding. The
    # default ("cube") stretches each axis to fill the box independently, which
    # is the same distortion Stage 33 removed from the 2-D map.
    scene=dict(xaxis=dict(visible=False, range=list(_cl[0])),
               yaxis=dict(visible=False, range=list(_cl[1])),
               zaxis=dict(visible=False, range=list(_cl[2])), bgcolor="white",
               aspectmode="data"),
    margin=dict(l=0, r=0, t=82, b=0),
    updatemenus=[dict(type="buttons", direction="right", x=0.0, xanchor="left",
                      y=1.06, yanchor="bottom", pad=dict(t=4, b=4),
                      bgcolor="rgba(255,255,255,0.95)", bordercolor="#D3D8DE",
                      buttons=[dict(label=f"{lv} ({Kof[lv]})", method="update",
                                    args=[{"visible": [i == _LAND or i in tr_of[lv]
                                                       for i in range(_nT)]}])
                               for lv in VIEWS])])

CB = {}
for lv in VIEWS:
    lab = labs[lv]; e = {}
    _clab = LAB_CORE.get(lv, lab)
    for c in range(Kof[lv]):
        idx = np.where(lab == c)[0]
        cidx = np.where(_clab == c)[0]
        # quote the clustered members only - same rule as Stage 33, so the two
        # explorers can never quote the same theme differently
        _q, _pf = _best_quotes(cidx if len(cidx) else idx)
        _extra = dict(n_core=int(len(cidx)), n_rec=int(len(idx) - len(cidx)))
        if lv == "Question":
            e[c] = dict(name=QL[c], summary=str(CFG.text_cols[c]), sent="",
                        chain="survey question", n=int(len(idx)),
                        quotes=_q, prof=_pf, near=_nearest_theme(lv, c, lab), **_extra)
        else:
            d = llm[lv].get(c) or llm[lv].get(str(c)) or {}
            e[c] = dict(name=d.get("name", f"{lv[0]}{c}"), summary=d.get("summary", ""),
                        sent=d.get("sentiment", ""), chain=_chain(lv, c), n=int(len(idx)),
                        quotes=_q, prof=_pf, near=_nearest_theme(lv, c, lab), **_extra)
    CB[lv] = e
# ---- accuracy guard: the sidebar must describe the cluster it claims to ------
# Silent mislabelling is the failure mode that matters here, so it is checked
# rather than assumed: every card's n must equal the cluster size, and every quote
# must be a response that actually belongs to that cluster.
_bad = []
for _lv in VIEWS:
    for _c, _e in CB[_lv].items():
        _idx = np.where(labs[_lv] == _c)[0]
        if _e["n"] != len(_idx):
            _bad.append(f"{_lv}{_c}: card says n={_e['n']}, cluster holds {len(_idx)}")
        _cidx = np.where(LAB_CORE.get(_lv, labs[_lv]) == _c)[0]
        _own = {" ".join(str(texts[i]).split())[:400]
                for i in (_cidx if len(_cidx) else _idx)}
        for _q in _e["quotes"]:
            # a quote is now {t: text, m: metadata}; compare the TEXT
            if (_q["t"] if isinstance(_q, dict) else _q) not in _own:
                _bad.append(f"{_lv}{_c}: a quote does not belong to this cluster")
                break
if _bad:
    raise RuntimeError("explorer accuracy check failed:\n  " + "\n  ".join(_bad[:8]))
print(f"  accuracy check passed: {sum(len(CB[v]) for v in VIEWS)} cards, "
      f"every n matches its cluster and every quote comes from it")
_shorty = [f"{v}{c}" for v in VIEWS for c, e in CB[v].items()
           if (e.get("prof") or {}).get("short")]
print(f"  themes made of short answers (flagged in the sidebar): {len(_shorty)}"
      + (f" - e.g. {', '.join(_shorty[:6])}" if _shorty else ""))

COLS = {lv: [_pal_of(lv).get(int(c), NO_THEME_COLOR) for c in labs[lv]] for lv in VIEWS}
LEG = "".join(f"<div class='lg'><span class='sw' style='background:{MACRO7[c % 7]}'></span>"
              f"M{c} {_nm('Main', c)[:30]}</div>" for c in range(Kof["Main"]))

TPL = """<!doctype html><html><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>iGEM 3D theme explorer</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
<style>
body{margin:0;font-family:Inter,Arial,sans-serif;color:#14202B}
#hd{padding:8px 16px 2px;font-size:16px;font-weight:700}
#hd span{font-weight:400;font-size:12px;color:#5B6670}
#wrap{display:flex;height:640px;max-width:1150px;margin:0 auto}
#plot{flex:1;min-width:0;height:640px}
#side{width:330px;border-left:1px solid #E7EAEE;padding:12px 14px;overflow-y:auto;background:#FBFCFD}
h3{margin:4px 0 2px;font-size:15px}.n{color:#98A2AD;font-size:12px}
.sent{display:inline-block;font-size:11px;padding:1px 8px;border-radius:9px;color:#fff;margin:4px 0}
.sm{color:#5B6670;font-size:12.5px;line-height:1.45;margin:8px 0}
.q{border-left:3px solid #0F6E56;padding:2px 0 2px 10px;margin:7px 0;font-size:12px;font-style:italic;color:#333}
.chain{font-size:12px;color:#5B6670;margin:2px 0 8px}
.prof{font-size:11px;color:#8A939B;margin:8px 0 2px;line-height:1.4}
.lg{font-size:11.5px;margin:3px 0;color:#3A4650}
.sw{display:inline-block;width:11px;height:11px;border-radius:3px;margin-right:6px;vertical-align:-1px}
#clear{margin-top:10px;font-size:12px;padding:4px 10px;border:1px solid #D3D8DE;background:#fff;border-radius:7px;cursor:pointer}
.hint{color:#98A2AD;font-size:12px;line-height:1.5;margin-bottom:8px}
.ctl{font-size:11.5px;margin:6px 0 8px;color:#3A4650}
.ctl b{display:block;margin-bottom:3px;font-size:11px;color:#8A939B;text-transform:uppercase;letter-spacing:.4px}
.ctl button{font-size:11px;padding:2px 7px;margin:0 3px 3px 0;border:1px solid #D3D8DE;background:#fff;border-radius:6px;cursor:pointer}
.ctl label{display:inline-block;margin-right:8px;font-size:11.5px}
.near{font-size:11.5px;color:#5B6670;margin:8px 0 0;padding-top:6px;border-top:1px solid #E7EAEE}
.qm{font-size:10.5px;color:#98A2AD;margin:-4px 0 8px 13px}
.rec{font-size:11px;color:#8A939B;margin:4px 0 0}
</style></head><body>
<noscript><div style="padding:26px;color:#B84413;font-size:13px">This explorer
needs JavaScript.</div></noscript>
<div id="hd">3D theme explorer <span>rotate/zoom · hover = own words + metadata · click badge/point = summary card · shells outline each cluster · question boxes fade the rest · double-click = clear</span></div>
<div id="wrap"><div id="plot"></div>
<aside id="side"><div class="hint">Click a numbered badge or any point for the theme's
LLM summary and three real quotes. Buttons switch level.</div>
<div class="ctl"><b>view</b>
 <button onclick="cam('top')">top</button><button onclick="cam('side')">side</button>
 <button onclick="cam('front')">front</button><button onclick="cam('free')">reset</button>
 <button id="shellbtn" onclick="toggleShells()">shells: off</button></div>
<div class="ctl"><b>questions</b><div id="qf"></div></div>
<div id="leg">__LEG__</div><div id="card"></div>
<button id="clear" onclick="clearSel()">clear highlight</button></aside></div>
<script>
// A JS error kills the rest of the block silently and the page then looks like
// an empty plot rather than a failure. Say so, on the page, in plain words.
window.onerror=function(msg,src,ln,col,err){
 var p=document.getElementById('plot');
 if(p&&!p.getAttribute('data-failed')){p.setAttribute('data-failed','1');
  p.innerHTML='<div style="padding:26px;font:12.5px/1.6 ui-monospace,monospace;'+
   'color:#B84413;background:#FFF6F2;border:1px solid #F0D2C6;border-radius:8px;'+
   'margin:18px">'+
   '<b>This explorer did not start.</b><br><br>'+String(msg)+
   '<br>line '+ln+', column '+col+
   '<br><br><span style="color:#5B6670">The map is not empty because there is '+
   'nothing to show &mdash; a script error stopped it before it drew. '+
   'Quote this line number when reporting it.</span></div>';}
 return false;};
</script>
<script>
var FIG=__FIG__;var CB=__CB__;var COLS=__COLS__;
var QOF=__QOF__;var QLAB=__QLAB__;
// sentiment badges sit ON the project ramp (#B84413 -> #BFC4C9 -> #0F6E56)
  // so a badge and a bar never disagree about what a sentiment looks like
  var SC={"positive":"#0F6E56","negative":"#B84413","mixed":"#8C9AA3","neutral":"#BFC4C9"};
var gd=document.getElementById('plot');
Plotly.newPlot('plot',FIG.data,FIG.layout,{responsive:true,displaylogo:false})
 .then(function(){Plotly.Plots.resize(gd);attach();buildQF();})
 .catch(function(e){gd.innerHTML='<div style="padding:30px;color:#B84413">'+
   '3D (WebGL) could not start — restart the runtime and run only this cell, '+
   'or open the saved explorer_3d_hierarchy.html in a new tab.</div>';});
window.addEventListener('resize',function(){try{Plotly.Plots.resize(gd);}catch(e){}});
function esc(s){return String(s).replace(/&/g,'&amp;').replace(/</g,'&lt;');}
function showCard(lv,cid){var e=(CB[lv]||{})[cid];if(!e)return;
 var h='<h3>'+esc(e.name)+'</h3><div class="n">'+lv+' '+cid+' · n='+e.n+'</div>';
 if(e.sent)h+='<span class="sent" style="background:'+(SC[e.sent]||'#BFC4C9')+'">'+esc(e.sent)+'</span>';
 h+='<div class="chain">'+esc(e.chain||'')+'</div><div class="sm">'+esc(e.summary||'')+'</div>';
 if(e.n_rec)h+='<div class="rec">'+e.n_core+' placed by the clustering · '+e.n_rec+
   ' recovered by the projection rule. Shell, badge and quotes come from the '+
   'clustered ones only.</div>';
 // Say what the theme's responses are LIKE before showing them. Without this a
 // one-word quote reads as a broken picker instead of as this theme's content.
 var p=e.prof||{};
 if(p.n){h+='<div class="prof">'+p.n+' responses · median '+p.med+
   ' word'+(p.med==1?'':'s')+' · '+p.n12+' of them 12 words or more'+
   (p.short?' — this theme is made of short answers, so the quotes below are short':'')+
   '</div>';}
 (e.quotes||[]).forEach(function(q){
   h+='<div class="q">'+esc(q.t||q)+'</div>';
   if(q.m)h+='<div class="qm">'+esc(q.m)+'</div>';});
 if(e.near){h+='<div class="near">closest theme: <b>'+esc(e.near.name)+'</b>'+
   ' (distance '+e.near.dist+')'+
   (e.near.close?' — these two overlap, so read them together':'')+'</div>';}
 document.getElementById('card').innerHTML=h;}
var _spotCid=null,_spotLv=null,QON={};
var DIM3='rgba(196,202,209,0.18)';
function ptTraceOf(lv){var t=-1;
 gd.data.forEach(function(d,i){var m=d.meta||{};
  if(m.kind==='pts'&&m.level===lv)t=i;});return t;}
// Scatter3d reads marker.opacity as a single scalar and ignores a per-point
// array, so fading by opacity is a silent no-op here - which is why the question
// boxes ticked and nothing moved. Both the filter and the spotlight now express
// themselves as COLOUR through this one function, so they compose instead of
// cancelling each other out.
function recolour3(){
 var lv=curLevel(),ti=ptTraceOf(lv);
 if(ti<0)return;
 var base=COLS[lv],cd=gd.data[ti].customdata;
 var nc=base.map(function(c,j){
  if(QON[QOF[j]]===false)return DIM3;
  if(_spotCid!==null&&_spotLv===lv&&cd[j][7]!=_spotCid)return DIM3;
  return c;});
 Plotly.restyle(gd,{'marker.color':[nc]},[ti]);}
function spotlight(lv,cid){_spotCid=cid;_spotLv=lv;recolour3();}
function clearSel(){_spotCid=null;_spotLv=null;recolour3();
 document.getElementById('card').innerHTML='';}
function cam(w){
 var C={top:{eye:{x:0,y:0,z:2.2}},side:{eye:{x:2.2,y:0,z:0}},
        front:{eye:{x:0,y:-2.2,z:0}},free:{eye:{x:1.4,y:1.4,z:1.0}}};
 Plotly.relayout(gd,{'scene.camera':C[w]||C.free});}
var _shells=false;
function toggleShells(){_shells=!_shells;
 var lv=curLevel();var vis=[],idx=[];
 gd.data.forEach(function(t,i){var m=t.meta||{};
  if(m.kind==='hull'){idx.push(i);vis.push(_shells&&m.level===lv);}});
 if(idx.length)Plotly.restyle(gd,{visible:vis},idx);
 document.getElementById('shellbtn').textContent='shells: '+(_shells?'on':'off');}
function curLevel(){var lv='Main';
 gd.data.forEach(function(t,i){var m=t.meta||{};
  if(m.kind==='pts'&&t.visible!==false)lv=m.level;});return lv;}
function buildQF(){var d=document.getElementById('qf');if(!d)return;
 QLAB.forEach(function(q,i){QON[i]=true;});
 d.innerHTML=QLAB.map(function(q,i){return '<label><input type="checkbox" checked '+
  'onchange="applyQF()" value="'+i+'"> '+q+'</label>';}).join('');}
function applyQF(){
 Array.prototype.forEach.call(document.querySelectorAll('#qf input'),
   function(b){QON[parseInt(b.value,10)]=b.checked;});
 recolour3();}
function attach(){
 gd.on('plotly_click',function(ev){var p=ev.points[0];var m=p.data.meta||{};
  if(!m.level)return;
  var cid=(m.kind==='badge')?m.ids[p.pointNumber]:p.customdata[7];
  showCard(m.level,cid);
  setTimeout(function(){spotlight(m.level,cid);},15);
 });
 gd.on('plotly_doubleclick',function(){clearSel();});
 gd.on('plotly_buttonclicked',function(){clearSel();
  setTimeout(function(){if(_shells){_shells=false;toggleShells();}
                        applyQF();},60);});}
</script></body></html>"""

def _js_guard(html, where):
    """Refuse to ship a page whose script cannot parse.

    An unterminated string literal kills the WHOLE <script> block, and the page
    then renders as an empty white box with no console message a notebook user
    would ever see. That is exactly how this cell shipped broken once: an
    English apostrophe inside a single-quoted JS string. A blank map is
    indistinguishable from a working one at a glance, so the check is done here,
    at build time, where it can still be fixed - not left to the reader."""
    j0 = html.find("<script>", html.find("</head>"))
    j1 = html.rfind("</script>")
    if j0 < 0 or j1 < 0:
        return 0
    js, bad, i, n, line = html[j0 + 8:j1], [], 0, j1 - j0 - 8, 1
    while i < n:
        ch = js[i]
        if ch == "\n":
            line += 1; i += 1; continue
        if ch == "/" and i + 1 < n and js[i + 1] == "/":
            while i < n and js[i] != "\n":
                i += 1
            continue
        if ch in "'\"":
            q, start, i, closed = ch, line, i + 1, False
            while i < n:
                if js[i] == "\\":
                    i += 2; continue
                if js[i] == "\n":
                    break                      # newline inside a literal
                if js[i] == q:
                    closed = True; i += 1; break
                i += 1
            if not closed:
                bad.append((start, js[:i].split("\n")[-1].strip()[-90:]))
                line += 1; i += 1
            continue
        i += 1
    if bad:
        raise RuntimeError(
            f"{where}: the page's JavaScript would not parse, so the explorer "
            f"would render as a silent blank box. Unterminated string "
            f"literal(s):\n  - " +
            "\n  - ".join(f"js line {l}: ...{c}" for l, c in bad))
    return len(js)

html = (TPL.replace("__FIG__", fig.to_json())
           .replace("__CB__", _json.dumps(CB))
           .replace("__COLS__", _json.dumps(COLS))
           .replace("__LEG__", LEG)
           .replace("__QOF__", _json.dumps([int(x) for x in labs["Question"]]))
           .replace("__QLAB__", _json.dumps(QL)))
_js_n = _js_guard(html, "Stage 35 (3D explorer)")
base = os.path.join(POOLED_DIR, "explorer_3d_hierarchy")
open(base + ".html", "w", encoding="utf-8").write(html)
if MIRROR_POOLED_DIR:
    _mirror_file(base + ".html", MIRROR_POOLED_DIR)
from IPython.display import display as _d, HTML as _H
_d(_H(html))
_nw = int((labs["Sub"] >= 0).sum()); _nc = int((LAB_CORE["Sub"] >= 0).sum())
print(f"  saved: explorer_3d_hierarchy.html · views {Kof['Main']}/{Kof['Sub']}/5 · "
      f"metadata {'ON' if _MST is not None else 'fallback'}")
print(f"  geometry: aspectmode='data' - the three axes keep the embedding's own "
      f"proportions · full range, nothing clipped out of the scene")
print(f"  coverage: {'WIDE' if _wide_on else 'CORE'} - {_nw} of {N} responses carry "
      f"a sub theme (core {_nc} + recovered {_nw - _nc}); shells, badges and "
      f"quotes are measured from the core only")
print(f"  script  : {_js_n} chars parsed and verified before shipping")

### Stage 23 · Thematic provenance

**Question.** Which survey questions feed which themes?

**Reads.** `pooled_master`.

**How to read it.** Every ribbon keeps its question's colour across all three columns, so a main theme fed by one colour is question-bound and one fed by several is cross-cutting. The unplaced responses are drawn as their own terminal band: this figure is about where the corpus goes, so the part that goes nowhere is shown.

In [ ]:
# @title
# ==============================================================================
# STAGE 23  -  THEMATIC PROVENANCE
#             section 3 of 9  -  The themes themselves
# ==============================================================================
# ==============================================================================
# THEMATIC PROVENANCE: WHICH QUESTIONS FEED WHICH THEMES
# ------------------------------------------------------------------------------
# A static three-column flow - survey question, main theme, sub-theme - in which
# every ribbon keeps its QUESTION's identity colour all the way across. Reading
# the picture by colour alone therefore answers "which questions feed which
# themes, and at what depth", which is the pooled design's central claim: if the
# themes were an artefact of the instrument, each main theme would be fed by a
# single colour.
#
# REBUILT for the two-level hierarchy. It was a four-column flow through the mid
# level and read maps["mid_to_main"] and maps["sub_to_mid"]; neither exists.
#
# The sub column shows the largest sub-themes individually and aggregates the
# remainder under their parent main theme, so the figure stays legible without
# hiding anything - the aggregated band says how many themes it holds.
#
# Responses with no sub-theme are shown as their own terminal band rather than
# dropped: this figure is about where the corpus goes, and a reader is entitled
# to see the share that goes nowhere.
# ==============================================================================
import numpy as np, matplotlib.pyplot as plt
from collections import Counter

main = np.asarray(MASTER["main_id"])
sub = np.asarray(MASTER["sub_id"])
origin = np.asarray(MASTER["question"])

TOP_SUB = 12                      # sub-themes shown individually; rest aggregated

_cnt_sub = Counter(int(c) for c in sub if c >= 0)
top_sub = {c for c, _ in _cnt_sub.most_common(TOP_SUB)}
_rest_n = {}
for c, n in _cnt_sub.items():
    if c not in top_sub:
        p = PARENT.get(int(c), -1)
        _rest_n[p] = _rest_n.get(p, 0) + 1


def _mkey(c):
    c = int(c)
    return NO_THEME if c < 0 else f"M{c} {theme_name('main', c)[:22]}"


def _skey(c):
    c = int(c)
    if c < 0:
        return NO_THEME
    if c in top_sub:
        return f"S{c} {theme_name('sub', c)[:22]}"
    p = PARENT.get(c, -1)
    return f"{_rest_n.get(p, 0)} smaller themes of M{p}"


colQ = Counter(origin.tolist())
colM = Counter(_mkey(c) for c in main)
colS = dict(sorted(Counter(_skey(c) for c in sub).items(), key=lambda kv: -kv[1]))

l1 = Counter((q, _mkey(m)) for q, m in zip(origin, main))
l2 = Counter((_mkey(m), _skey(s_), q) for m, s_, q in zip(main, sub, origin))

# every node's parent main theme, so a node can take that theme's hue
mo = {NO_THEME: -1}
for c in range(K["main"]):
    mo[_mkey(c)] = c
for c in range(K["sub"]):
    mo[_skey(c)] = PARENT.get(c, 0)

_MAINC = theme_colors("main")


def node_color(ci, name):
    if ci == 0:
        return Q_COLORS.get(name, "#999")
    return NO_THEME_COLOR if mo.get(name, -1) < 0 else _MAINC[mo[name]]


links = ([(0, a, b, v) for (a, b), v in l1.items()]
         + [(1, a, b, v) for (a, b, q), v in l2.items()])
qs_of = {(1, a, b, v): q for (a, b, q), v in l2.items()}


def link_color(link):
    if link[0] == 0:
        return Q_COLORS.get(link[1], "#999")
    return Q_COLORS.get(qs_of.get(link, ""), "#B9BFC6")


fig, ax = plt.subplots(figsize=(11.5, 8.2))
flow_diagram(ax, [dict(colQ), dict(colM), colS], links,
             [f"Survey question ({len(colQ)})",
              f"Main theme ({K['main']})",
              f"Sub-theme ({len(top_sub)} largest, rest grouped)"],
             node_color, link_color, alpha=0.42)
fig_title(fig, "Thematic provenance: which survey questions feed which themes")
plt.tight_layout(rect=[0, 0, 1, 0.93])
save_fig(fig, "chart_sankey_question_provenance"); plt.close(fig)

print("=" * 72)
print("  THEMATIC PROVENANCE FLOW")
print("=" * 72)
print(f"  {len(colQ)} questions -> {K['main']} main themes -> {len(top_sub)} sub-themes "
      f"shown individually, {K['sub'] - len(top_sub)} grouped under their parent")
print("  ribbon colour = the QUESTION a response came from, kept across every column")
print(f"  {NOISE['sub']} responses ({NOISE['sub'] / N_ALL:.0%}) end in '{NO_THEME}' and are")
print("  drawn as their own band - this figure is about where the corpus goes, so the")
print("  part that goes nowhere is shown rather than dropped.")
print("=" * 72)


---

## Section 4 - Prevalence and group differences

How often each theme is voiced, by whom, and whether the differences between groups survive multiple-comparison control.

### Stage 24 · Theme atlas and group claims

**Question.** How often is each theme voiced, and do groups differ?

**Reads.** `pooled_master`. **Writes.** `theme_atlas.csv`, `claims_all.csv`.

**How to read it.** The atlas shows two units at once - the share of participants who voiced a theme (solid) and the share of teams in which it was voiced (faded) - each with a 95% Wilson interval, coloured by the theme's sentiment. The claims panels test each group against the rest with Fisher's exact test and BH-FDR within each family, and report effect sizes as well as p-values. The denominator is every participant, so the rates do not sum to 100%: the remainder is the unplaced share.

In [ ]:
# @title
# ==============================================================================
# STAGE 24  -  THEME ATLAS AND GROUP CLAIMS
#             section 4 of 9  -  Who voices which theme
# ==============================================================================
# ==============================================================================
# FINDINGS ENGINE  -  theme atlas + group claims
# ------------------------------------------------------------------------------
# Single source of the descriptive and inferential theme findings. It replaces
# the old concrete-findings pack, the four standalone sentiment figures, the
# separate team-coverage figure and the theme-by-award figure.
#   * Inferential claims are made at the SUB level only. The main level appears
#     in the atlas as the coarse description of the same corpus; it carries no
#     significance test, because its bootstrap stability sits in the "pattern"
#     band rather than the "finding" band.
#   * Responses with no theme are excluded from every count and every test. They
#     are not removed from the denominator: a prevalence is the share of ALL
#     participants who voiced a theme, so the rates sum to less than 100% and the
#     remainder is exactly the unthemed share.
#   * Sentiment is NEVER its own figure and never an axis: it is the BAR COLOUR
#     everywhere, so prevalence and tone are read together.
#   * Atlas shows TWO units at once: share of PARTICIPANTS voicing the theme
#     (solid) and share of TEAMS in which it is voiced (faded) - the team-level
#     coverage figure is merged in here rather than repeated.
#   * Claims are tested at BOTH units (participant and team) for every group.
#   * Effect sizes reported, not just p: gap in percentage points, Cohen's h
#     (standardised), and the odds ratio with a 95% CI. Significance = Fisher
#     exact + BH-FDR within each (level, unit, factor) family.
#   * Gates: theme >= MIN_MENT mentioners, group >= MIN_GROUP members. Anything
#     unusable is reported and skipped - never imputed, never guessed.
# Outputs: chart_atlas_{main,sub}, chart_claims_sub,
#          chart_significance_dashboard, claims_all.csv, theme_atlas.csv
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, textwrap
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
from scipy.stats import fisher_exact

def _w2(t, w=26):
    # F1. The body of this function used to be
    #     return "\n".join(textwrap.wrap(str(t), w)[:2])
    # which wrapped and then threw away every line after the second WITHOUT any
    # marker, producing axis labels that end in "and" and read as complete.
    # fit_label wraps to three lines, widens once if that is not enough, and only
    # then truncates - always with a visible ellipsis.
    return fit_label(canon_theme_name(t), width=w, max_lines=3)


def _w2_legacy(t, w=26):
    """Wrap a theme name onto at most 2 lines - never truncate a word."""
    return "\n".join(textwrap.wrap(str(t), w)[:2])

_PL = globals().get("_orig_pload", pload)
M = _PL("pooled_master")

PFX = {"main": "M", "sub": "S"}
MIN_MENT   = 15   # theme must have >= this many participant mentioners to be tested
MIN_GROUP  = 20   # a country / medal tier must have >= this many participants
MIN_GROUP_T = 8   # ... or >= this many teams, when the unit is the team
SUB_SHOW   = 24   # rows shown in the Sub atlas (full table always in the CSV)
TOP_CLAIMS = 14   # rows shown per claims panel (full table always in the CSV)

def _wil(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return p, max(0.0, c-h), min(1.0, c+h)

def _bh(pv):
    pv = np.asarray(pv, float); q = np.full(pv.shape, np.nan)
    ok = np.where(np.isfinite(pv))[0]
    if ok.size:
        p = pv[ok]; n = p.size; o = np.argsort(p)
        r = p[o]*n/np.arange(1, n+1)
        r = np.minimum.accumulate(r[::-1])[::-1].clip(max=1.0)
        qq = np.empty(n); qq[o] = r; q[ok] = qq
    return q

def _cohen_h(p1, p2):
    """Standardised effect size for two proportions (Cohen 1988).
    |h| ~ 0.2 small, 0.5 medium, 0.8 large - comparable across themes."""
    if not (np.isfinite(p1) and np.isfinite(p2)): return np.nan
    return float(2*np.arcsin(np.sqrt(min(max(p1, 0), 1))) -
                 2*np.arcsin(np.sqrt(min(max(p2, 0), 1))))

def _odds_ratio(a, b, c, d):
    """OR with a 95% CI. Haldane-Anscombe +0.5 only when a cell is zero (the
    standard correction; it is applied to the CI, never to the reported counts)."""
    if min(a, b, c, d) == 0:
        a, b, c, d = a+0.5, b+0.5, c+0.5, d+0.5
    orr = (a*d) / (b*c) if b*c > 0 else np.nan
    if not np.isfinite(orr) or orr <= 0: return (np.nan, np.nan, np.nan)
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    return (float(orr), float(np.exp(np.log(orr) - 1.96*se)),
            float(np.exp(np.log(orr) + 1.96*se)))

if M is None:
    print("pooled_master missing - run NB2 Stage 33b first. Findings engine skipped.")
else:
    M = M.copy()
    need = {"Participant", "sentiment"}
    if not need.issubset(M.columns):
        print(f"pooled_master lacks {need - set(M.columns)} - engine skipped.")
    else:
        # _LVS, not LEVELS: LEVELS is the preflight's ["main","sub"] and is read by
        # every later cell. Rebinding it here to a list of tuples replaced it for
        # the rest of the notebook.
        _LVS = [(lv,) + _cov_cols(lv, M)[:2] for lv in LEVELS
                if f"{lv}_id" in M.columns]
        HAS_TEAM = "Team" in M.columns
        NP = M["Participant"].nunique()
        NT = M["Team"].nunique() if HAS_TEAM else 0
        SNORM = sent_norm(0.6)

        # ---------- mention matrices (participant AND team) + theme facts -----
        # NOISE. Grouping the raw id column gave the mention matrix a column for
        # -1, so the responses that belong to NO theme were counted as if they
        # were one - and since several hundred responses carry that id, it would
        # have appeared at the TOP of the atlas as the most-voiced "theme" in the
        # study. Mentions are counted from themed rows only.
        # The DENOMINATOR is deliberately NOT reduced to match: a prevalence here
        # is "what share of all participants voiced this theme", so it stays over
        # every participant. The rates therefore sum to less than 100% and the
        # remainder is the unthemed share, which is the honest picture.
        MENT, MENT_T, FACT = {}, {}, {}
        for lv, idc, nmc in _LVS:
            _hc = _cov_cols(lv, M)[2]
            print(_cov_note(lv, M))
            Mv = M[M[_hc].astype(bool)] if _hc in M.columns \
                 else M[M[idc] >= 0]
            ment = (Mv.groupby(["Participant", idc]).size().unstack(fill_value=0) > 0)
            MENT[lv] = ment
            if HAS_TEAM:
                MENT_T[lv] = (Mv.groupby(["Team", idc]).size().unstack(fill_value=0) > 0)
            rows = []
            for c in sorted(Mv[idc].dropna().unique()):
                c = int(c)
                if c < 0 or c not in ment.columns: continue
                k = int(ment[c].sum())
                p, lo, hi = _wil(k, NP)
                kt = int(MENT_T[lv][c].sum()) if (HAS_TEAM and c in MENT_T[lv].columns) else 0
                pt, tlo, thi = _wil(kt, NT) if NT else (np.nan,)*3
                sv = pd.to_numeric(Mv.loc[Mv[idc] == c, "sentiment"], errors="coerce")
                sv = float(sv.mean()) if sv.notna().any() else np.nan
                nm = Mv.loc[Mv[idc] == c, nmc].iloc[0] if nmc in Mv.columns else f"{PFX[lv]}{c}"
                rows.append(dict(level=lv, id=c, name=str(nm), n_ment=k,
                                 rate=p, lo=lo, hi=hi,
                                 n_teams=kt, team_rate=pt, team_lo=tlo, team_hi=thi,
                                 sentiment=sv))
            FACT[lv] = pd.DataFrame(rows)

        pd.concat(FACT.values(), ignore_index=True).to_csv(
            f"{POOLED_DIR}/theme_atlas.csv", index=False)

        # ---------- FIGURE 1-3 : atlas (participant + team, sentiment colour) -
        for lv, idc, nmc in _LVS:
            A_full = FACT[lv].sort_values("rate")
            # The sub level has 54 themes. Showing only the top 24 left 30 real themes
            # out of every figure, which reads as missing data. Split it into two halves
            # instead - most-voiced first - so all of them appear at a readable height.
            if lv == "sub" and len(A_full) > SUB_SHOW:
                _half = (len(A_full) + 1) // 2
                _hi_half, _lo_half = A_full.tail(_half), A_full.head(len(A_full) - _half)
                CHUNKS = [(_hi_half, "_part1", f" · part 1 of 2: the {len(_hi_half)} "
                                                f"most-voiced of {len(A_full)} themes"),
                          (_lo_half, "_part2", f" · part 2 of 2: the remaining "
                                                f"{len(_lo_half)} of {len(A_full)} themes")]
            else:
                CHUNKS = [(A_full, "", "")]
            for A, _sfx, note in CHUNKS:
                h = float(np.clip(0.30*len(A) + 2.5, 3.6, 13.0))
                fig, ax = plt.subplots(figsize=(10.6, h))
                yv = np.arange(len(A)); bh = 0.34
                xmax = float(np.nanmax([A["hi"].max(), A["team_hi"].max()]))
                for y, (_, r) in zip(yv, A.iterrows()):
                    col = SENT_CMAP(SNORM(r["sentiment"])) if np.isfinite(r["sentiment"]) else "#C4CAD0"
                    ax.barh(y + bh/2, r["rate"], height=bh, color=col,
                            edgecolor="white", zorder=3)
                    ax.errorbar(r["rate"], y + bh/2,
                                xerr=[[max(0.0, r["rate"]-r["lo"])], [max(0.0, r["hi"]-r["rate"])]],
                                fmt="none", ecolor="#3A4650", elinewidth=.9, capsize=2, zorder=4)
                    if np.isfinite(r["team_rate"]):
                        ax.barh(y - bh/2, r["team_rate"], height=bh, color=col, alpha=.40,
                                edgecolor="white", zorder=3)
                        ax.errorbar(r["team_rate"], y - bh/2,
                                    xerr=[[max(0.0, r["team_rate"]-r["team_lo"])],
                                          [max(0.0, r["team_hi"]-r["team_rate"])]],
                                    fmt="none", ecolor="#8A9098", elinewidth=.8, capsize=2, zorder=4)
                    ax.text(max(r["hi"], np.nan_to_num(r["team_hi"])) + .008, y,
                            f"{100*r['rate']:.0f}% p · {100*np.nan_to_num(r['team_rate']):.0f}% t",
                            va="center", fontsize=6.2, color=INK)
                ax.set_yticks(yv)
                ax.set_yticklabels([f"{PFX[lv]}{int(r['id'])}  " + _w2(r["name"], 30)
                                    for _, r in A.iterrows()], fontsize=6.6)
                ax.set_xlim(0, xmax * 1.40)          # room for the right-hand label
                ax.set_xticks(np.arange(0, min(1.0, xmax) + .001, .2))   # ticks stop at 100%
                ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
                ax.tick_params(labelsize=6.8, length=0)
                for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
                ax.xaxis.grid(True, color=GRID); ax.set_axisbelow(True)
                # figure-level legend: inside the axes it overlapped the bottom row's label
                fig.legend(handles=[mpatches.Patch(fc="#7A8A93", label=f"participants (N={NP})"),
                                    mpatches.Patch(fc="#7A8A93", alpha=.40, label=f"teams (N={NT})")],
                           fontsize=6.4, frameon=False, ncol=2,
                           loc="upper right", bbox_to_anchor=(.90, .975))
                sm = plt.cm.ScalarMappable(norm=SNORM, cmap=SENT_CMAP); sm.set_array([])
                cb = fig.colorbar(sm, ax=ax, fraction=.022, pad=.02)
                cb.set_label("mean sentiment", fontsize=7); cb.ax.tick_params(labelsize=6)
                cb.outline.set_visible(False)
                axlabel(ax, xlabel="share voicing the theme",
                        xhint=f"solid = participants · faded = teams · colour = mean sentiment · "
                              f"whisker = 95% Wilson CI{note}")
                fig_title(fig, f"Theme atlas — {lv.capitalize()} level ({len(FACT[lv])} themes)")
                plt.tight_layout(rect=[0, .03, 1, .93])
                save_fig(fig, f"chart_atlas_{lv}{_sfx}"); plt.close(fig)

        # ---------- claims: country x medal, participant AND team unit --------
        FACTORS = [(lab, col) for lab, col in
                   (("Country", "Country"), ("Medal tier", "medal")) if col in M.columns]
        claims = []
        pinfo = M.drop_duplicates("Participant").set_index("Participant")
        tinfo = M.drop_duplicates("Team").set_index("Team") if HAS_TEAM else None

        UNITS = [("participant", MENT, pinfo, NP, MIN_GROUP)]
        if HAS_TEAM:
            UNITS.append(("team", MENT_T, tinfo, NT, MIN_GROUP_T))

        for lv, idc, nmc in _LVS:
            if lv != CLAIM_LEVEL:
                continue      # claims are made at the claim level only; the main
                              # level is a description of the same corpus, not a
                              # second set of hypotheses to correct across
            F = FACT[lv].set_index("id")
            testable = [c for c in F.index if F.loc[c, "n_ment"] >= MIN_MENT]
            for unit, MDICT, INFO, NTOT, MING in UNITS:
                ment = MDICT[lv]
                for fac_label, fac_col in FACTORS:
                    if fac_col not in INFO.columns: continue
                    grp = INFO[fac_col].astype(str).str.strip()
                    grp = grp[grp.ne("") & grp.ne("nan") & grp.ne("Unknown")]
                    counts = grp.value_counts()
                    groups = [_gn for _gn in counts.index if counts[_gn] >= MING]
                    if not groups:
                        print(f"  ({lv}/{unit}/{fac_label}: no group reaches {MING} - skipped)")
                        continue
                    block = []
                    for _gn in groups:
                        ids = [x for x in grp[grp == _gn].index if x in ment.index]
                        if len(ids) < MING: continue
                        mg = ment.reindex(ids).fillna(False); ng = len(ids)
                        no = NTOT - ng
                        if no < MING: continue
                        for c in testable:
                            if c not in ment.columns: continue
                            k = int(mg[c].sum())
                            kall = int(ment[c].sum())
                            ko = kall - k
                            a, b_, c_, d_ = k, ng - k, ko, no - ko
                            if min(a + b_, c_ + d_) <= 0: continue
                            p, lo, hi = _wil(k, ng)
                            pg = kall / NTOT
                            p_out = ko / no if no else np.nan
                            try:
                                _, pv = fisher_exact([[a, b_], [c_, d_]])
                            except Exception:
                                pv = np.nan
                            orr, olo, ohi = _odds_ratio(a, b_, c_, d_)
                            block.append(dict(
                                level=lv, unit=unit, factor=fac_label, group=_gn,
                                id=int(c), name=str(F.loc[c, "name"]),
                                n_group=ng, n_voicing_group=k,
                                rate=p, lo=lo, hi=hi,
                                global_rate=pg, gap_pp=100*(p - pg),
                                cohen_h=_cohen_h(p, p_out),
                                odds_ratio=orr, or_lo=olo, or_hi=ohi,
                                sentiment=float(F.loc[c, "sentiment"]), p_fisher=pv))
                    if not block: continue
                    B = pd.DataFrame(block)
                    B["q_fdr"] = _bh(B["p_fisher"].values)   # BH within level x unit x factor
                    B["sig_fdr"] = (B["q_fdr"] < 0.05) & np.isfinite(B["q_fdr"])
                    claims.append(B)

        if not claims:
            print("  no testable claims (check the MIN_MENT / MIN_GROUP gates).")
        else:
            CL = pd.concat(claims, ignore_index=True)
            CL.sort_values(["level", "unit", "factor", "q_fdr"]).to_csv(
                f"{POOLED_DIR}/claims_all.csv", index=False)

            # ---------- FIGURE 4-5 : claims per level (participant unit) ------
            # Levels come from the contract. This read ("mid","sub"): the mid level
            # does not exist, so the loop drew one panel where it should draw two and
            # the main-level claims were computed and then silently never plotted.
            for lv in [l for l in LEVELS if ((CL.level == l) & (CL.unit == "participant")).any()]:
                D = CL[(CL.level == lv) & (CL.unit == "participant")]
                facs = list(dict.fromkeys(D["factor"]))
                fig, axes = plt.subplots(1, len(facs), figsize=(11.5, 6.6), squeeze=False)
                for ax, fac in zip(axes[0], facs):
                    S = D[D.factor == fac].copy()
                    S["absg"] = S.gap_pp.abs()
                    S = S.sort_values("absg", ascending=False).head(TOP_CLAIMS)
                    S = S.sort_values("gap_pp").reset_index(drop=True)
                    yv = np.arange(len(S))
                    for y, (_, r) in zip(yv, S.iterrows()):
                        col = (SENT_CMAP(SNORM(r["sentiment"]))
                               if np.isfinite(r["sentiment"]) else "#C4CAD0")
                        ax.barh(y, r["gap_pp"], height=.6, color=col, edgecolor="white",
                                zorder=3, alpha=1.0 if r["sig_fdr"] else .42)
                        elo = max(0.0, 100*(r["rate"] - r["lo"]))
                        ehi = max(0.0, 100*(r["hi"] - r["rate"]))
                        ax.errorbar(r["gap_pp"], y, xerr=[[max(0.0, elo)], [max(0.0, ehi)]], fmt="none",
                                    ecolor="#3A4650", elinewidth=.9, capsize=2, zorder=4)
                        if r["sig_fdr"]:
                            ax.text(r["gap_pp"] + (ehi + .6 if r["gap_pp"] > 0 else -elo - .6),
                                    y, "*", va="center",
                                    ha="left" if r["gap_pp"] > 0 else "right",
                                    fontsize=10, fontweight="bold", color=INK)
                    ax.axvline(0, color=ZERO, lw=1)
                    ax.set_yticks(yv)
                    ax.set_yticklabels([f"{r['group']}: {PFX[lv]}{r['id']} " + _w2(r["name"], 22)
                                        for _, r in S.iterrows()], fontsize=6.2)
                    ax.tick_params(labelsize=6.5, length=0)
                    for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
                    ax.xaxis.grid(True, color=GRID); ax.set_axisbelow(True)
                    nsig = int(D[D.factor == fac]["sig_fdr"].sum())
                    ax.set_title(f"{fac} · {nsig} FDR-significant of "
                                 f"{int((D.factor == fac).sum())} tests · the {len(S)} largest gaps shown",
                                 fontsize=8.5, fontweight="bold", loc="left", pad=6)
                    ax.set_xlabel("percentage points vs the global rate", fontsize=7.5)
                fig_title(fig, f"Where groups differ from the global theme profile — {lv.capitalize()} level")
                fig.text(.5, .012,
                         "bar = gap in percentage points · colour = theme mean sentiment · "
                         "whisker = 95% Wilson CI · * and full opacity = BH-FDR q<0.05 · faded = not significant · "
                         f"participant unit · gates: theme >= {MIN_MENT} mentioners, group >= {MIN_GROUP} · "
                         "team-unit tests and Cohen's h / odds ratios are in claims_all.csv",
                         ha="center", fontsize=6.5, color=FOOT)
                plt.tight_layout(rect=[0, .035, 1, .93])
                save_fig(fig, f"chart_claims_{lv}"); plt.close(fig)

            # ---------- FIGURE 6 : significance dashboard (both units) --------
            SIG = CL[CL.sig_fdr].copy()
            if len(SIG):
                SIG["absg"] = SIG.gap_pp.abs()
                SIG = SIG.sort_values("absg", ascending=False).head(18).sort_values("gap_pp")
                h = float(np.clip(.30*len(SIG) + 2.4, 3.6, 9.5))
                fig, ax = plt.subplots(figsize=(11.0, h))
                yv = np.arange(len(SIG))
                # Reserve room for the q/h annotation from the data itself. Without
                # this the label of a row whose CI reaches the extreme is placed past
                # the automatic x-limit, so it lands outside the axes and prints on
                # top of the row's name.
                _ends = [(rr["gap_pp"] - max(0.0, 100*(rr["rate"] - rr["lo"])),
                          rr["gap_pp"] + max(0.0, 100*(rr["hi"] - rr["rate"])))
                         for _, rr in SIG.iterrows()]
                _xlo = min(a for a, _ in _ends); _xhi = max(b for _, b in _ends)
                _xsp = max(1.0, _xhi - _xlo)
                ax.set_xlim(_xlo - .34*_xsp, _xhi + .30*_xsp)
                for y, (_, r) in zip(yv, SIG.iterrows()):
                    col = (SENT_CMAP(SNORM(r["sentiment"]))
                           if np.isfinite(r["sentiment"]) else "#C4CAD0")
                    ax.barh(y, r["gap_pp"], height=.62, color=col, edgecolor="white", zorder=3)
                    elo = max(0.0, 100*(r["rate"] - r["lo"]))
                    ehi = max(0.0, 100*(r["hi"] - r["rate"]))
                    ax.errorbar(r["gap_pp"], y, xerr=[[max(0.0, elo)], [max(0.0, ehi)]], fmt="none",
                                ecolor="#3A4650", elinewidth=.9, capsize=2, zorder=4)
                    lab = f"q={r['q_fdr']:.3f}"
                    if np.isfinite(r["cohen_h"]): lab += f" · h={r['cohen_h']:+.2f}"
                    _xt = r["gap_pp"] + (ehi + .8 if r["gap_pp"] > 0 else -elo - .8)
                    _a, _b = ax.get_xlim()
                    _xt = min(max(_xt, _a + .012*(_b - _a)), _b - .012*(_b - _a))
                    ax.text(_xt, y, lab, va="center",
                            ha="left" if r["gap_pp"] > 0 else "right",
                            fontsize=6.0, color="#5B6670")
                ax.axvline(0, color=ZERO, lw=1)
                ax.set_yticks(yv)
                ax.set_yticklabels([f"[{r['level']}/{r['unit'][:4]}] {r['group']} · "
                                    f"{PFX[r['level']]}{r['id']} " + _w2(r["name"], 22)
                                    for _, r in SIG.iterrows()], fontsize=6.2)
                ax.tick_params(labelsize=6.5, length=0)
                for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
                ax.xaxis.grid(True, color=GRID); ax.set_axisbelow(True)
                axlabel(ax, xlabel="percentage points vs the global rate",
                        xhint="every row survives BH-FDR q<0.05 · [level/unit] · colour = theme mean sentiment · "
                              "whisker = 95% Wilson CI · h = Cohen's h (0.2 small, 0.5 medium, 0.8 large)")
                fig_title(fig, "Significant findings: claims that survive FDR correction")
                plt.tight_layout(rect=[0, .03, 1, .93])
                save_fig(fig, "chart_significance_dashboard"); plt.close(fig)
            else:
                print("  NOTE: no claim survives BH-FDR q<0.05 - report this as an honest null;")
                print("        the per-level panels still show the descriptive gaps with CIs.")

            print("=" * 68)
            print(f"  FINDINGS ENGINE: {len(CL)} tests · {int(CL['sig_fdr'].sum())} FDR-significant")
            for lv in LEVELS:
                for un in ("participant", "team"):
                    d = CL[(CL.level == lv) & (CL.unit == un)]
                    if len(d):
                        print(f"    {lv:<4} {un:<11} {len(d):>4} tests · {int(d['sig_fdr'].sum())} significant")
            print(f"  effect sizes in claims_all.csv: gap_pp · cohen_h · odds_ratio (+95% CI)")
            print(f"  files: claims_all.csv · theme_atlas.csv · chart_atlas_* · chart_claims_* ·"
                  f" chart_significance_dashboard")
            print("=" * 68)

### Stage 25 · The support gate, made visible

**Question.** Which findings rest on enough data to be tested at all?

**How to read it.** The same findings as above, arranged by evidence tier, so a reader can see at a glance how much of the theme set is claimable and how much is description.

In [ ]:
# @title
# ==============================================================================
# STAGE 25  -  THE SUPPORT GATE, MADE VISIBLE
#             section 4 of 9  -  Who voices which theme
# ==============================================================================
# THE PROBLEM THIS FIXES
#   The findings engine tests only themes with at least MIN_MENT mentioners. At
#   Sub-45 the median theme holds 22 responses, so a good number of real themes
#   fall under that line - and they then simply do not appear in claims_all.csv.
#   A reader cannot tell "this theme was tested and found null" from "this theme
#   was never tested", and those are completely different statements.
#
#   This cell does not change a single test. It makes the gate VISIBLE:
#     * every theme is shown, at every level, in the atlas
#     * CLAIMABLE themes are drawn solid, DESCRIPTIVE ones faded, ARTIFACT ones
#       outlined - so under-support is a property you can see, not an absence
#     * a coverage table records, for EVERY theme, whether it was tested and if
#       not, exactly why
#   Sentiment stays where it belongs: the bar's colour, never an axis.
#
# Inputs : theme_atlas.csv (written by the findings engine - it already contains
#          every theme, the gate is applied later), claims_all.csv, and
#          pooled_theme_evidence from Stage 14c.
# Outputs: theme_atlas_tiered.csv, claims_coverage.csv,
#          chart_atlas_tiered_{level}, pooled_atlas_tiered
# ==============================================================================
import os
import numpy as np, pandas as pd

# ---------------------------- EDIT HERE ---------------------------------------
TOP_N_PER_PANEL = 24    # a level with more themes than this is split into panels
                        # rather than truncated - nothing is ever cut from view
# --------------------------- END EDIT -----------------------------------------

_PL = globals().get("_orig_pload", pload)
EV = _PL("pooled_theme_evidence")
if EV is None or not len(EV):
    raise RuntimeError("run Stage 14c first - it builds pooled_theme_evidence.")

_ap = f"{POOLED_DIR}/theme_atlas.csv"
if not os.path.exists(_ap):
    raise RuntimeError("theme_atlas.csv not found - run the findings engine first.")
AT = pd.read_csv(_ap)

CL = None
_cp = f"{POOLED_DIR}/claims_all.csv"
if os.path.exists(_cp):
    CL = pd.read_csv(_cp)

print("=" * 78)
print("  STAGE 41c   TIER-AWARE FINDINGS VIEW")
print("=" * 78)

_want = ["tier", "reason", "n_participants", "n_teams", "instrument_bound",
         "summary_withheld", "naming_status", "verified"]
# The atlas already carries some of these under the same name (n_teams). Merging
# both copies would silently produce n_teams_x / n_teams_y and every later
# reference would fail or, worse, pick the wrong one - so only bring across the
# columns the atlas does not already have.
E = EV[["level", "id"] + [c for c in _want
                          if c in EV.columns and c not in AT.columns]].copy()
T = AT.merge(E, on=["level", "id"], how="left")
T["tier"] = T["tier"].fillna("DESCRIPTIVE")

# which themes actually carry a test in claims_all.csv
_idc = next((c for c in ("id", "cluster", "theme", "theme_id")
             if CL is not None and c in CL.columns), None)
tested = set()
if CL is not None and _idc:
    tested = {(str(r.level), int(getattr(r, _idc)))
              for r in CL.itertuples() if pd.notna(getattr(r, _idc))}
T["tested"] = [ (str(r.level), int(r.id)) in tested for r in T.itertuples() ]

def _why(r):
    if r.tested:
        return ""
    if r.tier == "ARTIFACT":
        return f"not tested - {r.reason}"
    if r.tier == "DESCRIPTIVE":
        return f"not tested - {r.reason}"
    return "not tested - no group met the minimum size for any factor"
T["not_tested_reason"] = [_why(r) for r in T.itertuples()]
T.to_csv(f"{POOLED_DIR}/theme_atlas_tiered.csv", index=False)
T[["level", "id", "name", "tier", "n_ment", "n_participants", "n_teams",
   "tested", "not_tested_reason"]].to_csv(f"{POOLED_DIR}/claims_coverage.csv", index=False)
psave("pooled_atlas_tiered", T)

print(f"  {'level':<6}{'themes':>8}{'tested':>9}{'shown but untested':>21}")
for lv in T.level.unique():
    s = T[T.level == lv]
    print(f"  {lv:<6}{len(s):>8}{int(s.tested.sum()):>9}"
          f"{int((~s.tested).sum()):>21}")
print("\n  Every theme below appears in the figure. A theme that was not tested is")
print("  drawn faded with its reason recorded in claims_coverage.csv - it is NOT")
print("  absent, and it must not be read as a null result.")

_ut = T[~T.tested]
if len(_ut):
    print(f"\n  {len(_ut)} theme(s) shown but not tested:")
    for r in _ut.head(14).itertuples():
        print(f"    {r.level[:3]}{r.id:<3} {str(r.name)[:38]:<40} {r.not_tested_reason[:44]}")
    if len(_ut) > 14:
        print(f"    ... and {len(_ut) - 14} more (claims_coverage.csv)")

# ------------------------------- figures --------------------------------------
try:
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap, Normalize
    from matplotlib.lines import Line2D
    SENT = globals().get("SENT_CMAP") or LinearSegmentedColormap.from_list(
        "sent", ["#CC6677", "#E8E4DE", "#44AA99"])
    NORM = Normalize(vmin=-0.6, vmax=0.6)
    STYLE = {"CLAIMABLE":   dict(alpha=1.00, hatch=None, lw=0.0),
             "DESCRIPTIVE": dict(alpha=0.42, hatch=None, lw=0.0),
             "ARTIFACT":    dict(alpha=0.30, hatch="///", lw=0.7)}

    for lv in T.level.unique():
        A = T[T.level == lv].sort_values("rate").reset_index(drop=True)
        n = len(A)
        panels = int(np.ceil(n / TOP_N_PER_PANEL)) if n > TOP_N_PER_PANEL else 1
        per = int(np.ceil(n / panels))
        # The foot of this figure carries an x-axis label AND a legend, and
        # the old height did not reserve room for both: at main level the
        # legend, the x-label and the colourbar were measured landing on top
        # of one another. FURN_IN is that reserve, in inches, so it cannot
        # collapse when a level has few themes.
        FURN_IN = 0.95
        fig, axes = plt.subplots(1, panels,
                                 figsize=(min(12.0, 6.4 * panels),
                                          FURN_IN + 0.75 + 0.24 * per))
        axes = np.atleast_1d(axes)
        # One x-scale across the panels, because they are two halves of ONE sorted
        # list and a reader will compare across them. Headroom is added in data
        # units so the mentioner count printed past the CI cannot be clipped.
        _xmax = 100 * float(A.hi.max()) * 1.06 + 3.0
        for p, ax in enumerate(axes):
            blk = A.iloc[p * per:(p + 1) * per]
            y = np.arange(len(blk))
            for k, r in enumerate(blk.itertuples()):
                st = STYLE.get(r.tier, STYLE["DESCRIPTIVE"])
                col = SENT(NORM(r.sentiment)) if pd.notna(r.sentiment) else "#9AA5B1"
                ax.barh(k, 100 * r.rate, color=col, edgecolor="#14202B",
                        height=.68, alpha=st["alpha"], hatch=st["hatch"],
                        linewidth=st["lw"])
                lo, hi = 100 * r.lo, 100 * r.hi
                ax.plot([lo, hi], [k, k], color="#14202B", lw=1.0,
                        alpha=.75 * st["alpha"] + .15)
                ax.text(hi + 1.0, k, f"{r.n_ment}", va="center", fontsize=6.6,
                        color="#5B6670")
            ax.set_yticks(y)
            ax.set_yticklabels(
                [f"{str(r.name)[:34]}{'' if r.tested else '  ·'}" for r in blk.itertuples()],
                fontsize=7.0)
            ax.set_xlabel("participants who voiced the theme (%)", fontsize=8.5)
            ax.set_xlim(0, _xmax)
            for sp in ("top", "right", "left"):
                ax.spines[sp].set_visible(False)
            ax.tick_params(length=2)

        # The colourbar is given its OWN axes at the foot of the figure. Attaching
        # it to the panels steals width from the right-hand one and lands on top of
        # the mentioner counts printed there - which is exactly what happened.
        # The colourbar is VERTICAL and lives in its own column on the right.
        # Horizontally it shared the foot with the x-label and the legend, and
        # its ticks and title hang ~0.27in BELOW the bar - which is what made
        # them collide. Given its own column it cannot reach them at all.
        _h = fig.get_size_inches()[1]
        _bot = FURN_IN / _h
        fig.subplots_adjust(top=0.925, bottom=_bot, left=0.155, right=0.880,
                            wspace=0.42)
        cax = fig.add_axes([0.905, _bot + 0.12, 0.012, 0.34])
        sm = plt.cm.ScalarMappable(cmap=SENT, norm=NORM); sm.set_array([])
        cb = fig.colorbar(sm, cax=cax, orientation="vertical")
        cb.outline.set_visible(False)
        cb.set_label("mean sentiment of the theme's responses", fontsize=7.2,
             labelpad=3)
        cb.ax.tick_params(labelsize=6.6, length=2, pad=1)
        handles = [Line2D([], [], marker="s", ls="", ms=8, mfc="#7A8894",
                          alpha=STYLE[t]["alpha"], mec="#14202B",
                          label={"CLAIMABLE": "tested", "DESCRIPTIVE": "shown, not tested",
                                 "ARTIFACT": "label not accepted"}[t])
                   for t in ("CLAIMABLE", "DESCRIPTIVE", "ARTIFACT")]
        fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False,
                   fontsize=7.6, bbox_to_anchor=(.5, 0.02 / _h))
        if "fig_title" in globals():
            fig_title(fig, f"Theme prevalence at the {lv} level, with the evidence "
                           f"available for each")
        else:
            fig.text(0.5, 0.975, f"Theme prevalence at the {lv} level, with the "
                     f"evidence available for each", ha="center", va="top",
                     fontsize=12, fontweight="bold")
        (save_fig(fig, f"chart_atlas_tiered_{lv}") if "save_fig" in globals()
         else fig.savefig(f"{POOLED_DIR}/chart_atlas_tiered_{lv}.png", dpi=300,
                          bbox_inches="tight"))
        plt.close(fig)
except Exception as e:
    print(f"\n  (figure skipped: {type(e).__name__}: {str(e)[:90]})")

print("\n  -> theme_atlas_tiered.csv · claims_coverage.csv · chart_atlas_tiered_*")
print("  Bars carry the Wilson CI and the mentioner count; colour is sentiment;")
print("  opacity is how much evidence stands behind the theme. A faded bar is a")
print("  theme you may describe and quote, but not test.")
print("=" * 78)


### Stage 26 · Who voices which theme

**Question.** Do participants in different sections, world regions or team sizes voice different themes?

**Reads.** `pooled_master`.

**How to read it.** Rates with Wilson intervals, tested per theme and corrected within each family. A gap that does not survive correction is shown but not described as a difference.

In [ ]:
# @title
# ==============================================================================
# STAGE 26  -  WHO VOICES WHICH THEME: SECTION, REGION, TEAM SIZE
#             section 4 of 9  -  Who voices which theme
# ==============================================================================
# WHAT THIS ADDS THAT NOTHING ELSE IN THE NOTEBOOK DOES
#   The findings engine tests Country and Medal tier, at the sub level only.
#   Section (high-school / undergrad / overgrad), Region and team size are never
#   tested anywhere, and no factor is tested at MAIN level. Those are the three
#   best-powered comparisons this dataset still has unused:
#       Section  undergrad 747 / overgrad 487 / high-school 271 responses
#       Region   europe 695 / asia 560 / north-america 174 responses
#       size     continuous, 1-14, which carries more information than any split
#   Medal, Track, Kind and IsRemote are deliberately NOT here: IsRemote is False
#   for all 125 teams (no variance at all), and the others have groups too small
#   to detect anything this data could contain.
#
# THE ONE THING THAT MAKES THIS DIFFERENT FROM THE COUNTRY PANELS
#   Section and Region are TEAM attributes: every participant on a team shares
#   them. So a participant-level test treats 63 undergrad teams as 164 independent
#   observations and reports significance that is not there. The bars are drawn at
#   participant level, to stay comparable with the atlas and the claims panels, but
#   the p-value comes from a PERMUTATION that shuffles the group label across TEAMS
#   - the only exchangeable unit here. Team size is handled the same way.
#
# READING THE FIGURES
#   one row per theme; one bar per group; bar length = share of that group's
#   participants who voice the theme; whisker = 95% Wilson CI; BAR COLOUR = the
#   theme's mean measured sentiment, the same convention used everywhere else, so
#   tone and prevalence are read together; * on the label = BH-FDR q<0.05 across
#   the themes at that level. Themes are ordered by how much the groups differ.
#
# Outputs: chart_group_{section,region,teamsize}_{main,sub} (6 figures),
#          group_theme_claims.csv, pooled_group_theme.
# Refs: Wilson (1927) interval; Benjamini-Hochberg (1995); permutation inference
#       under cluster sampling (Good 2005).
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, textwrap
from matplotlib.colors import LinearSegmentedColormap, Normalize

_PL = globals().get("_orig_pload", pload)
M = _PL("pooled_master")
if M is None:
    raise RuntimeError("Need pooled_master - run NB2 Stage 33b first.")
M = M.copy()
M["sent"] = pd.to_numeric(M["sentiment"], errors="coerce")

MIN_VOICE = 15     # a theme needs this many participants voicing it to be tested
MIN_GROUP = 20     # a group needs this many participants
MIN_TEAMS = 8      # ... and this many teams, since teams are the exchangeable unit
TOP_N     = 14     # rows shown per figure (the full table always goes to the CSV)
N_PERM    = 2000
SEED      = int(CFG.seed) if "CFG" in globals() else 42

SENT_CMAP = globals().get("SENT_CMAP") or LinearSegmentedColormap.from_list(
    "sent", ["#B84413", "#BFC4C9", "#0F6E56"])
SNORM = globals().get("SNORM") or Normalize(-0.5, 0.5)
C_INK, C_GRID = "#14202B", "#E4E8EB"

def _wil(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)

def _bh(pv):
    pv = np.asarray(pv, float); q = np.full(pv.shape, np.nan)
    ok = np.where(np.isfinite(pv))[0]
    if ok.size:
        p = pv[ok]; n = p.size; o = np.argsort(p)
        r = np.minimum.accumulate((p[o] * n / np.arange(1, n + 1))[::-1])[::-1].clip(max=1.0)
        qq = np.empty(n); qq[o] = r; q[ok] = qq
    return q

def _w2(t, w=30):
    # F1 - see the note in cell 43. The old body kept wrap(...)[:2] and dropped
    # the rest silently; these labels ended mid-phrase ("… Preparation and").
    return fit_label(canon_theme_name(t), width=w, max_lines=3)

# _LVS, not LEVELS: the preflight publishes LEVELS = ["main","sub"] and every
# later cell reads it. A literal here both reinstated the removed mid level and
# rebound that global for the rest of the notebook.
_LVS = [(lv,) + _cov_cols(lv, MASTER)[:2] + (lv[0].upper(),) for lv in LEVELS]
rng = np.random.default_rng(SEED)
rows_all = []

print("=" * 78)
print("  STAGE 63  SECTION / REGION / TEAM SIZE  vs  THEME")
print("=" * 78)

# participant table: one row per participant, with their team, group and themes voiced
for lv, idc, namec, pfx in _LVS:
    # themed rows only - see the note in the other group cells
    _hc = _cov_cols(lv, M)[2]
    print(_cov_note(lv, M))
    D = (M[M[_hc].astype(bool)] if _hc in M.columns
         else M[M[idc] >= 0]).dropna(subset=[idc]).copy()
    D["tid"] = D[idc].astype(int)
    assert (D["tid"] >= 0).all(), f"a -1 {lv} id survived the filter"
    voiced = (D.groupby(["Participant", "tid"]).size().unstack(fill_value=0) > 0).astype(int)
    who = D.drop_duplicates("Participant").set_index("Participant")
    P = voiced.join(who[["Team", "Section", "Region", "team_size"]], how="left")
    tids = list(voiced.columns)
    nm = {int(r[idc]): str(r[namec]) for _, r in
          M.dropna(subset=[idc]).drop_duplicates(subset=[idc]).iterrows()}
    tsent = M.groupby("tid" if "tid" in M else idc)["sent"].mean() if False else \
            M.dropna(subset=[idc]).groupby(D[idc].astype(int).name if False else idc)["sent"].mean()
    tsent = {int(k): float(v) for k, v in tsent.items()}
    keep = [t for t in tids if int(P[t].sum()) >= MIN_VOICE]
    print(f"\n  {lv.upper()}: {len(tids)} themes, {len(keep)} voiced by >= {MIN_VOICE} "
          f"participants, {len(P)} participants in {P.Team.nunique()} teams")
    if not keep: continue

    # team-level lookup, because Section/Region/size belong to the TEAM
    TEAM = P.drop_duplicates("Team").set_index("Team")[["Section", "Region", "team_size"]]

    for fac in ("Section", "Region", "team_size"):
        if fac == "team_size":
            med = float(TEAM["team_size"].median())
            P["_g"] = np.where(P["team_size"] > med, f"larger (>{med:.0f})",
                                                    f"smaller (<={med:.0f})")
            TEAM["_g"] = np.where(TEAM["team_size"] > med, f"larger (>{med:.0f})",
                                                          f"smaller (<={med:.0f})")
            note = f"teams split at the median size of {med:.0f}"
        else:
            P["_g"] = P[fac].astype(str); TEAM["_g"] = TEAM[fac].astype(str)
            note = ""
        gsz = P.groupby("_g").size(); gtm = TEAM.groupby("_g").size()
        groups = [g for g in gsz.index
                  if gsz[g] >= MIN_GROUP and gtm.get(g, 0) >= MIN_TEAMS]
        if len(groups) < 2:
            print(f"    {fac}: fewer than two groups clear the gates - skipped"); continue
        Pg = P[P["_g"].isin(groups)]
        print(f"    {fac}: {', '.join(f'{g} (n={int(gsz[g])}p/{int(gtm[g])}t)' for g in groups)}"
              + (f"  · {note}" if note else ""))

        # observed rates and a team-permutation p-value per theme
        team_of = Pg["Team"].values
        gmap = TEAM["_g"].to_dict()
        uniq_teams = np.array(sorted(set(team_of)))
        team_grp = np.array([gmap[t] for t in uniq_teams])
        gidx = {g: i for i, g in enumerate(groups)}
        tcode = np.array([gidx[g] for g in team_grp])
        rowmap = np.array([np.where(uniq_teams == t)[0][0] for t in team_of])

        recs = []
        for t in keep:
            y = Pg[t].values.astype(float)
            obs = np.array([y[tcode[rowmap] == gidx[g]].mean() for g in groups])
            spread = float(np.nanmax(obs) - np.nanmin(obs))
            null = np.empty(N_PERM)
            for b in range(N_PERM):
                tc = rng.permutation(tcode)
                r = np.array([y[tc[rowmap] == i].mean() if (tc[rowmap] == i).any() else np.nan
                              for i in range(len(groups))])
                null[b] = np.nanmax(r) - np.nanmin(r)
            p = float((np.sum(null >= spread) + 1) / (N_PERM + 1))
            rec = dict(level=lv, factor=fac, id=int(t), name=nm.get(int(t), f"{pfx}{t}"),
                       sentiment=tsent.get(int(t), np.nan), spread_pp=100 * spread, p_perm=p,
                       n_voicing=int(Pg[t].sum()))
            for g in groups:
                m = tcode[rowmap] == gidx[g]
                k, n = int(y[m].sum()), int(m.sum())
                r_, lo, hi = _wil(k, n)
                rec[f"rate::{g}"] = r_; rec[f"lo::{g}"] = lo; rec[f"hi::{g}"] = hi
                rec[f"n::{g}"] = n
            recs.append(rec)
        R = pd.DataFrame(recs)
        R["q_fdr"] = _bh(R.p_perm.values); R["sig_fdr"] = R.q_fdr < 0.05
        R = R.sort_values("spread_pp", ascending=False).reset_index(drop=True)
        rows_all.append(R)
        ns = int(R.sig_fdr.sum())
        print(f"      {len(R)} themes tested · {ns} FDR-significant"
              + (f" · largest gap {R.spread_pp.iloc[0]:.0f} pp "
                 f"({R.name.iloc[0][:32]})" if len(R) else ""))
        for _, r in R[R.sig_fdr].head(6).iterrows():
            det = "  ".join(f"{g[:12]} {100*r[f'rate::{g}']:.0f}%" for g in groups)
            print(f"        {r['name'][:34]:<34} {det}   q={r.q_fdr:.3f}")

        # ---------------- figure ----------------
        S = R.head(TOP_N).iloc[::-1]
        if not len(S): continue
        # Colour is ALREADY spoken for: it encodes the theme's sentiment, the convention
        # used in every other figure here. Groups are therefore separated by HATCH, not
        # by lightness - fading the bars would make a negative theme in one group look
        # like a neutral theme in another, and the tone signal would be lost.
        HATCH = ["", "//", "..", "xx", "\\\\"]
        plt.rcParams["hatch.linewidth"] = 0.55      # sparse, so it separates without
        plt.rcParams["hatch.color"] = "white"       # competing with the sentiment hue
        h = max(3.4, 0.52 * len(S) + 2.1)
        fig, ax = plt.subplots(figsize=(11.0, min(h, 9.8)))
        nb = len(groups); bh = 0.74 / nb
        for gi, g in enumerate(groups):
            off = (gi - (nb - 1) / 2) * bh
            for i, (_, r) in enumerate(S.iterrows()):
                col = (SENT_CMAP(SNORM(r.sentiment)) if np.isfinite(r.sentiment) else "#C4CAD0")
                ax.barh(i + off, 100 * r[f"rate::{g}"], height=bh * .92, color=col,
                        hatch=HATCH[gi % len(HATCH)], edgecolor="white", linewidth=.6,
                        zorder=3,
                        label=f"{g}  (n={int(r[f'n::{g}'])})" if i == len(S) - 1 else None)
                el = max(0.0, 100 * (r[f"rate::{g}"] - r[f"lo::{g}"]))
                eh = max(0.0, 100 * (r[f"hi::{g}"] - r[f"rate::{g}"]))
                ax.errorbar(100 * r[f"rate::{g}"], i + off,
                             xerr=[[max(0.0, el)], [max(0.0, eh)]], fmt="none",
                            ecolor="#3A4650", elinewidth=.8, capsize=1.8, zorder=4)
        ax.set_yticks(range(len(S)))
        ax.set_yticklabels([f"{pfx}{int(r.id)}  " + _w2(r["name"]) + ("  *" if r.sig_fdr else "")
                            for _, r in S.iterrows()], fontsize=7.4)
        ax.set_xlabel(f"share of each group's participants who voice the theme (%)\n"
                      f"whisker = 95% Wilson CI · bar colour = the theme's mean measured "
                      f"sentiment · * = BH-FDR q<0.05 from a team-level permutation",
                      fontsize=8.8)
        ax.set_xlim(0, max(1.0, float(np.nanmax(
            [100 * S[f'hi::{g}'].max() for g in groups])) * 1.04))
        for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
        ax.xaxis.grid(True, color=C_GRID); ax.set_axisbelow(True)
        # legend below the axes, in neutral grey, so it names the HATCHES without
        # implying a colour meaning and without covering the bottom rows
        from matplotlib.patches import Patch
        hs = [Patch(facecolor="#D8DCDF", edgecolor="white",
                    hatch=HATCH[gi % len(HATCH)],
                    label=f"{g}  (n={int(S.iloc[0][f'n::{g}'])})")
              for gi, g in enumerate(groups)]
        ax.legend(handles=hs, frameon=False, fontsize=8.4, ncol=len(groups),
                  loc="upper center", bbox_to_anchor=(0.5, -0.10 - 0.02 * len(groups)))
        # the SAME sentiment colourbar the forest figures carry, so a reader can
        # decode the bar colours here without hunting for another figure. Placed in
        # its own axes on the right, so it can never overlap the bars or the legend.
        from matplotlib.cm import ScalarMappable
        fig.subplots_adjust(right=0.88)
        cax = fig.add_axes([0.905, 0.16, 0.014, 0.68])
        _sm = ScalarMappable(norm=SNORM, cmap=SENT_CMAP); _sm.set_array([])
        _cb = fig.colorbar(_sm, cax=cax)
        _cb.outline.set_visible(False)
        _cb.ax.tick_params(labelsize=7, length=0)
        _cb.set_label("theme mean sentiment\n(orange - / grey 0 / teal +)", fontsize=7.4)

        nice = {"Section": "team section", "Region": "world region",
                "team_size": "team size"}[fac]
        ttl = f"Theme prevalence by {nice} — {lv} level"
        try:    fig_title(fig, ttl)
        except Exception: fig.suptitle(ttl, fontsize=13, fontweight="bold")
        plt.tight_layout(rect=[0, .06, .88, .93])
        tag = {"Section": "section", "Region": "region", "team_size": "teamsize"}[fac]
        try:    save_fig(fig, f"chart_group_{tag}_{lv}")
        except Exception:
            fig.savefig(f"{POOLED_DIR}/chart_group_{tag}_{lv}.png", dpi=300, bbox_inches="tight")
        plt.close(fig)

if rows_all:
    A = pd.concat(rows_all, ignore_index=True)
    A.to_csv(f"{POOLED_DIR}/group_theme_claims.csv", index=False)
    psave("pooled_group_theme", A)
    print("-" * 78)
    print(f"  {len(A)} tests across 3 factors x 3 levels · "
          f"{int(A.sig_fdr.sum())} survive BH-FDR")
    for fac in A.factor.unique():
        s = A[A.factor == fac]
        print(f"    {fac:<10} {int(s.sig_fdr.sum()):>3} of {len(s):>4} significant")
    print("  6 figures: chart_group_{section,region,teamsize}_{main,sub}")
    print("  full table with every rate, CI and q value: group_theme_claims.csv")
    print("\n  NOTE ON THE UNIT: bars are participant shares so they are comparable with")
    print("  the atlas and claims panels, but Section, Region and team size are TEAM")
    print("  attributes, so every p-value comes from permuting the group label across")
    print("  TEAMS. A participant-level test would count teammates as independent and")
    print("  report significance that is not there.")
else:
    print("  no factor cleared the support gates.")
print("=" * 78)

### Stage 27 · Which themes are voiced together

**Question.** Do the same participants raise particular themes together more often than chance?

**Reads.** `pooled_master`. **Writes.** `cooccurrence_pairs.csv`.

**How to read it.** Lift is the joint rate over the product of the separate rates: 1 means independence. Intervals are bootstrapped by team, because teammates share language. A pair needs a joint-mention floor to be tested at all, and that floor is reported.

In [ ]:
# @title
# ==============================================================================
# STAGE 27  -  WHICH THEMES ARE VOICED TOGETHER
#             section 4 of 9  -  Who voices which theme
# ==============================================================================
# ==============================================================================
# CO-OCCURRENCE  -  which themes the SAME participant voices together
# ------------------------------------------------------------------------------
# The holistic-experience structure: "someone who talks about funding also talks
# about time strain". At the claim level:
#   effect size  = LIFT  = P(A and B) / (P(A) * P(B))
#                  1.0 = independent, >1 = co-voiced more than chance
#   significance = Fisher exact on the 2x2 mention table, BH-FDR within level
# Gates: both themes need >= MIN_MENT mentioners and the pair >= MIN_CO joint
# mentioners before it is tested. Bar colour = the pair's mean sentiment, so a
# co-occurrence cluster's tone reads directly (no separate sentiment figure).
# Outputs: chart_cooccurrence_sub, cooccurrence_pairs.csv
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, textwrap
from scipy.stats import fisher_exact

def _w3c(t, w=38):
    """Wrap the PAIR label onto at most 3 lines - no word is ever cut."""
    return "\n".join(textwrap.wrap(str(t), w)[:3])

_PL = globals().get("_orig_pload", pload)
M = _PL("pooled_master")

MIN_MENT = 15    # each theme must have >= this many mentioners
MIN_CO_BY = {"main": 12, "sub": 8}   # joint-mentioner floor, level-aware: sub-themes
N_BOOT_CO = 800     # bootstrap draws for each pair's lift interval
                                     # are small, so a single absolute floor across levels would
                                     # identical absolute floor would silently empty the
                                     # sub panel. Significance is still governed by
                                     # Fisher exact + BH-FDR, never by this gate.
TOP_PAIR = 16    # rows shown per level (full table always goes to CSV)
PFX = {"main": "M", "sub": "S"}

def _bh(pv):
    pv = np.asarray(pv, float); q = np.full(pv.shape, np.nan)
    ok = np.where(np.isfinite(pv))[0]
    if ok.size:
        p = pv[ok]; n = p.size; o = np.argsort(p)
        r = p[o]*n/np.arange(1, n+1)
        r = np.minimum.accumulate(r[::-1])[::-1].clip(max=1.0)
        qq = np.empty(n); qq[o] = r; q[ok] = qq
    return q

if M is None:
    print("pooled_master missing - run NB2 Stage 33b first. Co-occurrence skipped.")
else:
    # _LVS, not LEVELS: the preflight owns that name. Co-occurrence is reported at
    # the claim level only - at the main level, with far fewer and much larger
    # themes, almost every pair co-occurs and a lift table says nothing.
    _LVS = [(lv,) + _cov_cols(lv, M)[:2] for lv in [CLAIM_LEVEL]
            if f"{lv}_id" in M.columns]
    if not _LVS:
        print(f"no {CLAIM_LEVEL}_id column in pooled_master - skipped.")
    else:
        NP = M["Participant"].nunique()
        SNORM = sent_norm(0.6)
        allpairs = []

        for lv, idc, nmc in _LVS:
            MIN_CO = MIN_CO_BY.get(lv, 10)
            # Lift has no closed-form interval that respects this design, so it is
            # bootstrapped by TEAM: participants on one team share language, so
            # resampling them independently would understate the spread. The same
            # draws are reused for every pair, so the intervals are comparable.
            # Themed rows only. id -1 is not a theme, and with several hundred
            # responses behind it it would co-occur with almost everything and
            # sit at the top of the lift table as the strongest "pair".
            _hasc = _cov_cols(lv, M)[2]
            print(_cov_note(lv, M))
            _Mv = M[M[_hasc].astype(bool)] if _hasc in M.columns else M[M[idc] >= 0]
            ment = (_Mv.groupby(["Participant", idc])
                       .size().unstack(fill_value=0) > 0)
            ment = ment[[c for c in ment.columns if int(c) >= 0]]
            _pteam = (_Mv.drop_duplicates("Participant").set_index("Participant")["Team"]
                        .reindex(ment.index).astype(str).values)
            _rng_co = np.random.default_rng(int(CFG.seed) if "CFG" in globals() else 42)
            _tu = np.unique(_pteam)
            _tix = {t: np.where(_pteam == t)[0] for t in _tu}
            _BOOT_CO = [np.concatenate([_tix[t] for t in _rng_co.choice(_tu, len(_tu), True)])
                        for _ in range(N_BOOT_CO)]
            cols = [c for c in ment.columns if int(ment[c].sum()) >= MIN_MENT]
            if len(cols) < 2:
                print(f"  ({lv}: fewer than 2 themes pass the {MIN_MENT}-mentioner gate - skipped)")
                continue
            nm = {int(c): str(M.loc[M[idc] == c, nmc].iloc[0]) if nmc in M.columns else f"{PFX[lv]}{c}"
                  for c in cols}
            sent = {int(c): float(pd.to_numeric(M.loc[M[idc] == c, "sentiment"],
                                                errors="coerce").mean()) for c in cols}
            B = ment[cols].values.astype(bool)
            rows = []
            for ia in range(len(cols)):
                for ib in range(ia + 1, len(cols)):
                    a, b = B[:, ia], B[:, ib]
                    n11 = int((a & b).sum())
                    if n11 < MIN_CO:
                        continue
                    n10 = int((a & ~b).sum()); n01 = int((~a & b).sum())
                    n00 = int((~a & ~b).sum())
                    pa, pb = (n11 + n10) / NP, (n11 + n01) / NP
                    lift = (n11 / NP) / (pa * pb) if pa * pb > 0 else np.nan
                    _bs = np.empty(N_BOOT_CO); _bs[:] = np.nan
                    for _bi, _ix in enumerate(_BOOT_CO):
                        _a, _b2 = a[_ix], b[_ix]
                        _pa, _pb = _a.mean(), _b2.mean()
                        if _pa > 0 and _pb > 0:
                            _bs[_bi] = float((_a & _b2).mean()) / (_pa * _pb)
                    _bs = _bs[np.isfinite(_bs)]
                    _lo_b, _hi_b = (tuple(np.percentile(_bs, [2.5, 97.5]))
                                    if _bs.size >= N_BOOT_CO * 0.5 else (np.nan, np.nan))
                    try:
                        _, pv = fisher_exact([[n11, n10], [n01, n00]])
                    except Exception:
                        pv = np.nan
                    A_, B_ = int(cols[ia]), int(cols[ib])
                    rows.append(dict(level=lv, a=A_, b=B_,
                                     a_name=nm[A_], b_name=nm[B_],
                                     n_both=n11, n_a=n11 + n10, n_b=n11 + n01,
                                     jaccard=n11 / max(n11 + n10 + n01, 1),
                                     lift=lift, lift_lo=_lo_b, lift_hi=_hi_b,
                                     p_fisher=pv,
                                     sentiment=float(np.nanmean([sent[A_], sent[B_]]))))
            if not rows:
                print(f"  ({lv}: no pair reaches {MIN_CO} joint mentioners - skipped)")
                continue
            P = pd.DataFrame(rows)
            P["q_fdr"] = _bh(P["p_fisher"].values)
            P["sig_fdr"] = (P["q_fdr"] < 0.05) & np.isfinite(P["q_fdr"])
            allpairs.append(P)

            S = P.sort_values("lift", ascending=False).head(TOP_PAIR).sort_values("lift")
            h = float(np.clip(0.52*len(S) + 2.6, 4.5, 13.0))   # 3-line labels need the pitch
            fig, ax = plt.subplots(figsize=(11.0, h))
            yv = np.arange(len(S))
            for y, (_, r) in zip(yv, S.iterrows()):
                col = (SENT_CMAP(SNORM(r["sentiment"]))
                       if np.isfinite(r["sentiment"]) else "#C4CAD0")
                ax.barh(y, r["lift"], height=.62, color=col, edgecolor="white",
                        zorder=3, alpha=1.0 if r["sig_fdr"] else .45)
                if np.isfinite(r.get("lift_lo", np.nan)) and np.isfinite(r.get("lift_hi", np.nan)):
                    ax.errorbar(r["lift"], y,
                                xerr=[[max(0.0, r["lift"] - r["lift_lo"])],
                                      [max(0.0, r["lift_hi"] - r["lift"])]],
                                fmt="none", ecolor="#3A4650", elinewidth=1.0, capsize=2.5, zorder=4)
                # place the count past the INTERVAL, not past the bar, so it cannot sit on
                # top of the error bar that was just added
                _xlab = r["lift"]
                if np.isfinite(r.get("lift_hi", np.nan)):
                    _xlab = max(_xlab, float(r["lift_hi"]))
                ax.text(_xlab + .05, y,
                        f"{r['n_both']} shared" + ("  *" if r["sig_fdr"] else ""),
                        va="center", fontsize=6.6,
                        color=INK if r["sig_fdr"] else "#8A9098")
            ax.axvline(1.0, color=ZERO, lw=1.2)
            ax.set_yticks(yv)
            # ONE wrapped label per row. The previous version stacked two
            # 2-line names (4 lines per row) which collided with the rows above
            # and below; a single wrapped pair label fits the row pitch.
            ax.set_yticklabels([_w3c(f"{PFX[lv]}{r['a']} {r['a_name']}  +  "
                                     f"{PFX[lv]}{r['b']} {r['b_name']}")
                                for _, r in S.iterrows()], fontsize=6.0)
            ax.set_xlim(0, float(np.nanmax(np.r_[S["lift"].values,
                    S["lift_hi"].values if "lift_hi" in S else S["lift"].values])) * 1.22)
            ax.tick_params(labelsize=6.5, length=0)
            for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
            ax.xaxis.grid(True, color=GRID); ax.set_axisbelow(True)
            nsig = int(P["sig_fdr"].sum())
            ax.set_ylabel("theme pair, both voiced by the same participant",
                          fontsize=9)
            axlabel(ax, xlabel="lift  (1.0 = the two themes are voiced independently)",
                    xhint=f"bar = lift · colour = the pair's mean sentiment · label = participants voicing BOTH · "
                          f"* and full opacity = BH-FDR q<0.05 ({nsig} of {len(P)} pairs) · "
                          f"gates: each theme >= {MIN_MENT} mentioners, pair >= {MIN_CO} joint")
            fig_title(fig, f"Themes voiced together by the same participant — {lv.capitalize()} level")
            plt.tight_layout(rect=[0, .04, 1, .93])
            # the sentiment colour key, in its own axes so it cannot overlap the bars
            fig.subplots_adjust(right=0.87)
            _cax = fig.add_axes([0.895, 0.18, 0.014, 0.64])
            from matplotlib.cm import ScalarMappable
            _sm = ScalarMappable(norm=SNORM, cmap=SENT_CMAP); _sm.set_array([])
            _cb = fig.colorbar(_sm, cax=_cax); _cb.outline.set_visible(False)
            _cb.ax.tick_params(labelsize=7, length=0)
            _cb.set_label("mean sentiment of the two themes\n(orange - / grey 0 / teal +)",
                          fontsize=7.4)
            save_fig(fig, f"chart_cooccurrence_{lv}"); plt.close(fig)

        if allpairs:
            AP = pd.concat(allpairs, ignore_index=True)
            AP.sort_values(["level", "q_fdr"]).to_csv(
                f"{POOLED_DIR}/cooccurrence_pairs.csv", index=False)
            print("=" * 66)
            print(f"  CO-OCCURRENCE: {len(AP)} pairs tested · "
                  f"{int(AP['sig_fdr'].sum())} significant after BH-FDR")
            for lv in [CLAIM_LEVEL]:
                d = AP[AP.level == lv]
                if len(d):
                    top = d.sort_values("lift", ascending=False).iloc[0]
                    print(f"    {lv:<4} {len(d):>4} pairs · {int(d['sig_fdr'].sum())} significant"
                          f" · strongest: {PFX[lv]}{top['a']}+{PFX[lv]}{top['b']} "
                          f"lift={top['lift']:.2f} ({top['n_both']} shared)")
            print(f"  files: cooccurrence_pairs.csv · chart_cooccurrence_*")
            print("=" * 66)
        else:
            print("  no level produced testable pairs - nothing written.")


---

## Section 5 - Country context, with France in focus

The sample is international and the French subsample is large enough to look at on its own. These three cells describe it, test where it differs from the rest, and show what those themes sound like in participants' own words. The power calculation comes first, so every null below is read against what the design could have detected.

### Stage 28 · France in the corpus

**Question.** What is the French subsample, and how large a difference could it detect?

**Reads.** `pooled_master` (`Country` comes from the official registry joined in Notebook 2). **Writes.** `france_corpus_profile.csv`.

**How to read it.** The counts first, then the minimum detectable difference for this design. Every France result below should be read against that number: an underpowered comparison cannot distinguish *no difference* from *a difference this sample cannot see*. The composition panels show whether French teams differ from the rest in section, track or outcome - if they do, that is a confound for any theme comparison and is stated before the comparison is made.

In [ ]:
# @title
# ==============================================================================
# STAGE 28  -  FRANCE IN THE CORPUS
#             section 5 of 9  -  France in the corpus
# ==============================================================================
# ==============================================================================
# FRANCE IN THE CORPUS
# ------------------------------------------------------------------------------
# The country analyses that follow compare French teams with the rest of the
# sample. Before any comparison is made, this cell establishes what the French
# subsample actually is - how many teams, participants and responses, and how it
# is composed - and, just as importantly, how large a difference it could detect.
#
# WHY THE POWER CALCULATION COMES FIRST, NOT LAST
#   With a subsample of this kind the usual failure is to run several dozen
#   comparisons, find nothing below the multiple-testing threshold, and report
#   "no difference". That conclusion does not follow: an underpowered comparison
#   cannot distinguish "no difference" from "a difference this design cannot
#   see". The minimum detectable difference is therefore computed HERE, from the
#   real group sizes, and printed with the descriptives - so every France figure
#   that follows is read against a number that says what it was capable of
#   finding in the first place.
#
# Nothing is estimated or imputed. Country comes from the official team registry
# joined in Notebook 2 Stage 33; a team with no registry match has no country and
# is counted as such rather than assigned to a residual group.
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# ------------------------------- EDIT HERE ------------------------------------
FOCUS_COUNTRY = None     # None = detect the France label automatically
POWER, ALPHA = 0.80, 0.05
# ----------------------------- END EDIT ---------------------------------------

if "Country" not in MASTER.columns:
    raise RuntimeError(
        "pooled_master has no Country column. Run Notebook 2 Stage 33 (the team "
        "registry join) and Stage 33b, then re-run this notebook from the preflight.")

_ctry = MASTER["Country"].astype(str).str.strip()
_known = _ctry.replace({"nan": "", "None": "", "?": ""}).ne("")

# Detect the label the registry uses rather than assuming one. Registries write
# "France", "FR" and "FRA" interchangeably, and hardcoding the wrong one would
# silently produce an empty subsample and a page of "no difference".
if FOCUS_COUNTRY is None:
    _cands = [c for c in _ctry[_known].unique()
              if str(c).strip().lower() in ("france", "fr", "fra")]
    if not _cands:
        raise RuntimeError(
            f"no France label found in Country. Values present: "
            f"{sorted(_ctry[_known].unique())[:20]}. Set FOCUS_COUNTRY above to the "
            f"exact string this registry uses.")
    FOCUS_COUNTRY = max(_cands, key=lambda c: int((_ctry == c).sum()))

IS_FR = (_ctry == FOCUS_COUNTRY).values
IS_REST = (_known & (_ctry != FOCUS_COUNTRY)).values

FR = MASTER.loc[IS_FR]
REST = MASTER.loc[IS_REST]
if not len(FR):
    raise RuntimeError(f"no responses carry Country == {FOCUS_COUNTRY!r}.")


def _wil(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


def _mde(n1, n2, p_base, power=0.80, alpha=0.05):
    """Smallest difference in two proportions this design could detect, in
    PERCENTAGE POINTS.

    F2 - this notebook used to carry two different power calculations under this
    one name: a closed-form normal approximation here and an iterative search in
    the prize-precision cell. They disagreed by up to 2.1 percentage points, and
    one returned a proportion while the other returned percentage points. The
    iterative version is the more accurate of the two (it evaluates power at the
    actual p1/p2 rather than at the pooled baseline), so it is now the only one,
    and it returns percentage points in both places.
    """
    from scipy.stats import norm as _norm
    if min(n1, n2) < 2:
        return np.nan
    za = _norm.ppf(1 - alpha / 2)
    for d in np.arange(0.005, 0.80, 0.005):
        p1 = min(max(p_base + d / 2, 1e-6), 1 - 1e-6)
        p2 = min(max(p_base - d / 2, 1e-6), 1 - 1e-6)
        se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
        if se <= 0:
            continue
        if _norm.cdf(abs(p1 - p2) / se - za) >= power:
            return float(100 * d)
    return np.nan


UNITS = [("responses", MASTER, IS_FR, IS_REST),
         ("participants", MASTER.drop_duplicates("Participant"), None, None),
         ("teams", MASTER.drop_duplicates("Team"), None, None)]

print("=" * 78)
print(f"  FRANCE IN THE CORPUS   (registry label used: {FOCUS_COUNTRY!r})")
print("=" * 78)

rows = []
for label, frame, _a, _b in UNITS:
    c = frame["Country"].astype(str).str.strip()
    kn = c.replace({"nan": "", "None": "", "?": ""}).ne("")
    n_fr = int((c == FOCUS_COUNTRY).sum())
    n_rest = int((kn & (c != FOCUS_COUNTRY)).sum())
    n_unk = int((~kn).sum())
    p, lo, hi = _wil(n_fr, n_fr + n_rest) if (n_fr + n_rest) else (np.nan,) * 3
    rows.append(dict(unit=label, france=n_fr, rest=n_rest, no_country=n_unk,
                     share_of_known=p, lo=lo, hi=hi,
                     mde_pp=_mde(n_fr, n_rest, 0.30, POWER, ALPHA)))
    print(f"  {label:<13} France {n_fr:>5}   rest of sample {n_rest:>5}   "
          f"no country {n_unk:>4}   France share {p:.1%} [{lo:.1%}, {hi:.1%}]")

S = pd.DataFrame(rows)
print()
print("  SMALLEST DIFFERENCE THIS DESIGN COULD DETECT")
print(f"  (two-sided, alpha {ALPHA}, power {POWER:.0%}, against a 30% baseline rate)")
for r in S.itertuples():
    if np.isfinite(r.mde_pp):
        print(f"    at the {r.unit:<13} level: {r.mde_pp:5.1f} percentage points")
print("  A France-versus-rest comparison smaller than this cannot be resolved by")
print("  this sample. Read every null below against these numbers, not as evidence")
print("  of similarity.")

# ---------------- composition: is the French subsample like the rest? ---------
import re
FACETS = [("Section", "Section"), ("Track", "Track"), ("Kind", "Kind"),
          ("Medal", "medal")]
comp_rows = []
tf = MASTER.drop_duplicates("Team")
tc = tf["Country"].astype(str).str.strip()
for label, col in FACETS:
    if col not in tf.columns:
        continue
    a = tf.loc[tc == FOCUS_COUNTRY, col].astype(str).str.strip()
    b = tf.loc[tc.replace({"nan": "", "None": ""}).ne("") & (tc != FOCUS_COUNTRY),
               col].astype(str).str.strip()
    for v in sorted(set(a) | set(b)):
        if v in ("", "nan", "None"):
            continue
        ka, kb = int((a == v).sum()), int((b == v).sum())
        pa, alo, ahi = _wil(ka, len(a))
        pb, blo, bhi = _wil(kb, len(b))
        comp_rows.append(dict(facet=label, value=v, n_fr=ka, n_rest=kb,
                              rate_fr=pa, fr_lo=alo, fr_hi=ahi,
                              rate_rest=pb, rest_lo=blo, rest_hi=bhi,
                              gap_pp=100 * (pa - pb) if np.isfinite(pa) and np.isfinite(pb)
                              else np.nan))
C = pd.DataFrame(comp_rows)
C.to_csv(f"{POOLED_DIR}/france_corpus_profile.csv", index=False)
psave("pooled_france_profile", dict(counts=S, composition=C,
                                    label=FOCUS_COUNTRY, alpha=ALPHA, power=POWER))

if len(C):
    print("\n  COMPOSITION OF THE FRENCH TEAMS versus the rest (team level)")
    print(f"  {'facet':<9}{'value':<20}{'France':>9}{'rest':>9}{'gap':>9}")
    for r in C.itertuples():
        print(f"  {r.facet:<9}{str(r.value)[:19]:<20}{r.rate_fr:>8.0%}{r.rate_rest:>9.0%}"
              f"{r.gap_pp:>8.0f}pp")
    print("  A French subsample that is unlike the rest on section or track is not a")
    print("  problem, but it IS a confound: a theme difference could be a difference")
    print("  of who took part rather than of country. Carried into the write-up.")

# ------------------------------- figures --------------------------------------
# One figure PER FACET. These are four different questions - is the French
# subsample unlike the rest by section, by track, by kind of team, by medal -
# and four panels squeezed into one row gave each of them a strip too narrow to
# read a team-type name in. Separate figures also mean each one can be dropped
# into the write-up beside the claim it supports.
_fr_charts = []
if len(C):
    _nfr = int(S.loc[S.unit == "teams", "france"].iloc[0])
    _nrest = int(S.loc[S.unit == "teams", "rest"].iloc[0])
    for f in C["facet"].unique():
        d = C[C.facet == f].sort_values("rate_fr", ascending=True)
        if not len(d):
            continue
        # height follows the number of categories, so a 3-row facet is not
        # stretched to the height of an 8-row one
        fig, ax = plt.subplots(figsize=(7.4, max(2.6, 0.42 * len(d) + 1.9)))
        y = np.arange(len(d))
        ax.barh(y + 0.19, 100 * d["rate_fr"], height=.36, color=SENT_POS,
                label=f"{FOCUS_COUNTRY} (n={_nfr} teams)",
                xerr=[np.clip(100 * (d.rate_fr - d.fr_lo), 0, None),
                      np.clip(100 * (d.fr_hi - d.rate_fr), 0, None)],
                error_kw=dict(ecolor="#3A4650", elinewidth=.9), capsize=2, zorder=3)
        ax.barh(y - 0.19, 100 * d["rate_rest"], height=.36, color=NO_THEME_COLOR,
                label=f"everywhere else (n={_nrest} teams)",
                xerr=[np.clip(100 * (d.rate_rest - d.rest_lo), 0, None),
                      np.clip(100 * (d.rest_hi - d.rate_rest), 0, None)],
                error_kw=dict(ecolor="#3A4650", elinewidth=.9), capsize=2, zorder=3)
        # the gap in percentage points, printed where the eye already is
        for k, r in enumerate(d.itertuples()):
            if np.isfinite(r.gap_pp):
                ax.text(max(100 * r.rate_fr, 100 * r.rest_hi) + 2.0, k,
                        f"{r.gap_pp:+.0f} pp", va="center", fontsize=7.4,
                        color=INK if abs(r.gap_pp) >= 10 else "#8A939B")
        ax.set_yticks(y)
        ax.set_yticklabels([fit_label(str(v), width=26, max_lines=2)
                            for v in d["value"]], fontsize=8.5)
        ax.set_xlabel("% of teams", fontsize=9)
        ax.set_xlim(0, min(105.0, float(100 * d[["fr_hi", "rest_hi"]].max().max()) + 14))
        ax.legend(fontsize=8, frameon=False, loc="lower right")
        for sp in ("top", "right", "left"):
            ax.spines[sp].set_visible(False)
        fig_title(fig, f"{FOCUS_COUNTRY} against the rest of the sample, by {f.lower()}")
        fig.text(0.5, 0.012,
                 "Team-level proportions with 95% Wilson intervals. A difference in "
                 "who took part is a confound for any country comparison of themes.",
                 ha="center", fontsize=8.0, style="italic", color=FOOT, wrap=True)
        plt.tight_layout(rect=[0, 0.055, 1, 0.93])
        _nm_f = f"chart_france_corpus_profile_{re.sub(r'[^a-z0-9]+', '_', f.lower())}"
        save_fig(fig, _nm_f); plt.close(fig)
        _fr_charts.append(_nm_f)
    print(f"\n  {len(_fr_charts)} composition figure(s), one per facet: "
          + ", ".join(_fr_charts))

print(f"\n  -> france_corpus_profile.csv · pooled_france_profile · "
      + " · ".join(_fr_charts))
print("=" * 78)

### Stage 29 · What French participants talk about

**Question.** Which themes do French participants voice more, or less, than participants elsewhere?

**Reads.** `pooled_master`, the France profile. **Writes.** `france_theme_profile.csv`.

**How to read it.** Paired bars per theme with 95% Wilson intervals, the gap in percentage points, and BH-FDR across the tested themes. Faded rows did not clear the support gate in both groups and carry no test - they are shown with their counts rather than hidden. A starred row differs after correction; an unstarred one does not, however large its gap looks.

In [ ]:
# @title
# ==============================================================================
# STAGE 29  -  WHAT FRENCH PARTICIPANTS TALK ABOUT
#             section 5 of 9  -  France in the corpus
# ==============================================================================
# ==============================================================================
# WHAT FRENCH PARTICIPANTS TALK ABOUT: THEME PROFILE AGAINST THE REST
# ------------------------------------------------------------------------------
# For every theme at the claim level, the share of French participants who voiced
# it and the share of participants elsewhere, each with a 95% Wilson interval, the
# gap in percentage points, Fisher's exact test and BH-FDR across the themes
# tested. Sentiment is the bar colour, never an axis.
#
# THREE RULES THIS FIGURE KEEPS
#   1. The denominator is every participant in the group, not only those with a
#      theme, so the rates sum to less than 100% and the missing part is the
#      unplaced share - the same convention as every other prevalence here.
#   2. A theme is TESTED only if it clears the support gate in BOTH groups.
#      Everything else is still DRAWN, faded, with its counts - a theme voiced by
#      four French participants is a description, not a finding, and hiding it
#      would be worse than showing it as what it is.
#   3. Nothing that fails FDR is described as a difference. The figure marks what
#      survives; the printout says how many did, and against what the comparison
#      was powered to detect (from the previous cell).
#
# Refs: Wilson (1927) interval; Fisher (1935) exact test; Benjamini & Hochberg
# (1995) FDR; Cohen (1988) h.
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, textwrap
from scipy.stats import fisher_exact

# ------------------------------- EDIT HERE ------------------------------------
MIN_VOICE_FR = 5     # French participants voicing a theme, for it to be TESTED
MIN_VOICE_RE = 15    # participants elsewhere voicing it, for the same
TOP_SHOW = 22        # rows drawn; the full table always goes to the CSV
# ----------------------------- END EDIT ---------------------------------------

PROF = pload("pooled_france_profile")
if PROF is None:
    raise RuntimeError("run the 'France in the corpus' cell first - it establishes "
                       "the country label and the detectable effect size.")
FOCUS_COUNTRY = PROF["label"]
LV = CLAIM_LEVEL


def _wil(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan, np.nan)
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0.0, c - h), min(1.0, c + h)


def _bh(pv):
    pv = np.asarray(pv, float); q = np.full(pv.shape, np.nan)
    ok = np.where(np.isfinite(pv))[0]
    if ok.size:
        p = pv[ok]; n = p.size; o = np.argsort(p)
        r = p[o] * n / np.arange(1, n + 1)
        r = np.minimum.accumulate(r[::-1])[::-1].clip(max=1.0)
        qq = np.empty(n); qq[o] = r; q[ok] = qq
    return q


def _cohen_h(p1, p2):
    if not (np.isfinite(p1) and np.isfinite(p2)):
        return np.nan
    return float(2 * np.arcsin(np.sqrt(min(max(p1, 0), 1))) -
                 2 * np.arcsin(np.sqrt(min(max(p2, 0), 1))))


# participant-level view: one row per participant, their country, their themes
ctry = MASTER["Country"].astype(str).str.strip()
known = ctry.replace({"nan": "", "None": "", "?": ""}).ne("")
who = MASTER.drop_duplicates("Participant").set_index("Participant")
wc = who["Country"].astype(str).str.strip()
FR_IDS = set(wc[wc == FOCUS_COUNTRY].index)
RE_IDS = set(wc[wc.replace({"nan": "", "None": ""}).ne("") & (wc != FOCUS_COUNTRY)].index)
N_FR, N_RE = len(FR_IDS), len(RE_IDS)
if N_FR == 0 or N_RE == 0:
    raise RuntimeError(f"France {N_FR} / rest {N_RE} participants - nothing to compare.")

_IDW, _NMW, _HCW = _cov_cols(LV, MASTER)
print(_cov_note(LV, MASTER))
Mv = MASTER[MASTER[_HCW].astype(bool)]
ment = (Mv.groupby(["Participant", _IDW]).size().unstack(fill_value=0) > 0)
ment = ment[[c for c in ment.columns if int(c) >= 0]]
sent_of = (Mv.groupby(_IDW)["sentiment"]
           .apply(lambda x: pd.to_numeric(x, errors="coerce").mean()).to_dict())

rows = []
for c in sorted(int(x) for x in ment.columns):
    col = ment[c]
    k_fr = int(col.reindex(list(FR_IDS)).fillna(False).sum())
    k_re = int(col.reindex(list(RE_IDS)).fillna(False).sum())
    p_fr, fr_lo, fr_hi = _wil(k_fr, N_FR)
    p_re, re_lo, re_hi = _wil(k_re, N_RE)
    testable = (k_fr >= MIN_VOICE_FR) and (k_re >= MIN_VOICE_RE)
    pv = np.nan
    if testable:
        try:
            _, pv = fisher_exact([[k_fr, N_FR - k_fr], [k_re, N_RE - k_re]])
        except Exception:
            pv = np.nan
    rows.append(dict(id=c, name=theme_name(LV, c),
                     n_fr=k_fr, n_rest=k_re, rate_fr=p_fr, fr_lo=fr_lo, fr_hi=fr_hi,
                     rate_rest=p_re, rest_lo=re_lo, rest_hi=re_hi,
                     gap_pp=100 * (p_fr - p_re), cohen_h=_cohen_h(p_fr, p_re),
                     sentiment=sent_of.get(c, np.nan), tested=testable, p_fisher=pv))
F = pd.DataFrame(rows)
F["q_fdr"] = _bh(F["p_fisher"].values)
F["sig_fdr"] = (F["q_fdr"] < 0.05) & np.isfinite(F["q_fdr"])
F = F.sort_values("gap_pp", ascending=False).reset_index(drop=True)
F.to_csv(f"{POOLED_DIR}/france_theme_profile.csv", index=False)
psave("pooled_france_theme", F)

_mde = PROF["counts"]
_mde_p = float(_mde.loc[_mde.unit == "participants", "mde_pp"].iloc[0]) \
    if "mde_pp" in _mde.columns else np.nan

print("=" * 82)
print(f"  WHAT FRENCH PARTICIPANTS TALK ABOUT   ({N_FR} French participants vs "
      f"{N_RE} elsewhere)")
print("=" * 82)
print(f"  {int(F.tested.sum())} of {len(F)} themes clear the support gate "
      f"(>= {MIN_VOICE_FR} French and >= {MIN_VOICE_RE} other participants) and are tested;")
print(f"  the remaining {int((~F.tested).sum())} are drawn faded with their counts and "
      f"carry no test.")
print(f"  {int(F.sig_fdr.sum())} survive BH-FDR at q<0.05.")
if np.isfinite(_mde_p):
    print(f"  This comparison was powered to detect about {_mde_p:.0f} percentage points "
          f"at the participant level.")
    _big = F[F.tested & (F.gap_pp.abs() >= _mde_p) & ~F.sig_fdr]
    if len(_big):
        print(f"  {len(_big)} theme(s) show a gap at or above that size without surviving "
              f"FDR - the gap is large enough to matter but not established here.")
if int(F.sig_fdr.sum()) == 0:
    print("  NO theme differs after multiple-comparison control. With this subsample "
          "that is")
    print("  a limit of the design as much as a statement about France; it is reported "
          "as a null,")
    print("  and the gaps below are described, not claimed.")
else:
    print("\n  themes where French participants differ (FDR-significant):")
    for r in F[F.sig_fdr].itertuples():
        _dir = "more" if r.gap_pp > 0 else "less"
        print(f"    {LV[0].upper()}{r.id:<3} {str(r.name)[:38]:<38} "
              f"{r.rate_fr:5.1%} vs {r.rate_rest:5.1%}  ({r.gap_pp:+5.1f}pp, {_dir}; "
              f"h={r.cohen_h:+.2f}, q={r.q_fdr:.3f})")

# ------------------------------- figure ---------------------------------------
D = F.reindex(F["gap_pp"].abs().sort_values(ascending=False).index).head(TOP_SHOW)
D = D.sort_values("gap_pp")
if len(D):
    def _w2(t, w=34):
        # F1 - the body used to be `"\n".join(textwrap.wrap(str(t), w)[:2])`,
        # which wrapped and then threw away every line past the second WITHOUT a
        # marker: "Financial Limitations and Costs" rendered as "Financial /
        # Limitations and" and read as a complete label. Measured on the 39 real
        # sub-theme names, that silently cut 2 of them here.
        # fit_label keeps max_lines=2 so this figure's layout is unchanged, widens
        # up to 60% before conceding, and only then truncates - always visibly.
        return fit_label(canon_theme_name(t), width=w, max_lines=2)

    fig, ax = plt.subplots(figsize=(11.4, 0.44 * len(D) + 3.0))
    y = np.arange(len(D))
    _cm, _nm_ = SENT_CMAP, sent_norm(0.6)
    for i, r in enumerate(D.itertuples()):
        _col = _cm(_nm_(r.sentiment)) if np.isfinite(r.sentiment) else NO_THEME_COLOR
        _al = 1.0 if r.tested else 0.38
        ax.barh(y[i] + .20, 100 * r.rate_fr, height=.36, color=_col, alpha=_al,
                edgecolor="white",
                xerr=[[max(0.0, 100 * (r.rate_fr - r.fr_lo))],
                      [max(0.0, 100 * (r.fr_hi - r.rate_fr))]],
                error_kw=dict(ecolor="#3A4650", elinewidth=.9), capsize=2)
        ax.barh(y[i] - .20, 100 * r.rate_rest, height=.36, color=NO_THEME_COLOR,
                alpha=_al, edgecolor="white",
                xerr=[[max(0.0, 100 * (r.rate_rest - r.rest_lo))],
                      [max(0.0, 100 * (r.rest_hi - r.rate_rest))]],
                error_kw=dict(ecolor="#3A4650", elinewidth=.9), capsize=2)
        _mark = " *" if r.sig_fdr else ("" if r.tested else "  (not tested)")
        ax.text(max(100 * r.rate_fr, 100 * r.rate_rest) + 1.0, y[i],
                f"{r.gap_pp:+.0f}pp{_mark}", va="center", fontsize=8.2,
                color=INK if r.sig_fdr else FOOT,
                fontweight="bold" if r.sig_fdr else "normal")
    ax.set_yticks(y)
    ax.set_yticklabels([f"{LV[0].upper()}{int(r.id)}  " + _w2(r.name)
                        for r in D.itertuples()], fontsize=8.2)
    ax.set_xlabel(f"% of participants in the group who voiced the theme "
                  f"(France n={N_FR}, elsewhere n={N_RE})", fontsize=9.5)
    ax.set_ylim(-.7, len(D) - .3)
    for sp in ("top", "right", "left"):
        ax.spines[sp].set_visible(False)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor="#8C9AA3", label="France (bar colour = theme sentiment)"),
                       Patch(facecolor=NO_THEME_COLOR, label="rest of the sample")],
              fontsize=8.4, frameon=False, loc="lower right")
    _n_sig = int(F.sig_fdr.sum())
    _verdict = (f"{_n_sig} theme(s) differ after FDR control" if _n_sig
                else "no theme differs after FDR control")
    fig_title(fig, f"Theme prevalence among French participants against the rest of "
                   f"the sample: {_verdict}")
    fig.text(0.5, 0.012,
             f"Bars are the share of each group's participants who voiced the theme, "
             f"with 95% Wilson intervals; the denominator is every participant, so the "
             f"rates do not sum to 100%. Faded rows did not clear the support gate and "
             f"carry no test. * marks BH-FDR q<0.05 among the "
             f"{int(F.tested.sum())} tested themes"
             + (f"; the design could detect about {_mde_p:.0f} percentage points."
                if np.isfinite(_mde_p) else "."),
             ha="center", fontsize=8.3, style="italic", color=FOOT, wrap=True)
    plt.tight_layout(rect=[0, 0.045, 1, 0.93])
    save_fig(fig, "chart_france_theme_profile"); plt.close(fig)

print(f"\n  -> france_theme_profile.csv · pooled_france_theme · "
      f"chart_france_theme_profile")
print("=" * 82)

### Stage 30 · France in their own words

**Question.** What do the themes with the largest gaps actually sound like?

**Reads.** `pooled_master`. **Writes.** `france_quotes.csv`.

**How to read it.** Real responses from French participants in each theme, shown as written and untranslated. Each card states whether the gap above it survived FDR control, because a quote illustrates a theme - it is never evidence that the difference is established.

In [ ]:
# @title
# ==============================================================================
# STAGE 30  -  FRANCE IN THEIR OWN WORDS
#             section 5 of 9  -  France in the corpus
# ==============================================================================
# ==============================================================================
# FRANCE IN THEIR OWN WORDS
# ------------------------------------------------------------------------------
# The previous cell measures where French participants differ. This one shows
# what those themes sound like, in the participants' own responses.
#
# WHY A QUOTE PANEL AND NOT ANOTHER TEST
#   A percentage-point gap says a topic is raised more or less often; it says
#   nothing about how it is raised. The quotes are the evidence a reader needs to
#   judge whether the difference is meaningful, and they are the one part of this
#   analysis that cannot be produced by any amount of modelling.
#
# WHAT IS AND IS NOT CLAIMED HERE
#   Quotes are DESCRIPTIVE. A theme appears in this panel because its gap is
#   among the largest, not because it passed a test - the panel is explicit about
#   which of the themes shown survived FDR and which did not, so a quote is never
#   read as proof of the gap above it.
#   Every quote is a real response from a French participant in that theme,
#   selected as centroid-nearest of medium length. Nothing is edited, translated
#   or composed; a response that is not in English is shown as written, because
#   the corpus is multilingual and silently dropping it would bias the panel
#   toward English speakers.
# ==============================================================================
import numpy as np, pandas as pd, textwrap

# ------------------------------- EDIT HERE ------------------------------------
N_THEMES = 6      # themes shown, taken from the largest gaps in either direction
N_QUOTES = 3      # real French responses per theme
WMIN, WMAX = 8, 70
# ----------------------------- END EDIT ---------------------------------------

F = pload("pooled_france_theme")
PROF = pload("pooled_france_profile")
if F is None or PROF is None:
    raise RuntimeError("run the two France cells above first.")
FOCUS_COUNTRY = PROF["label"]
LV = CLAIM_LEVEL

X5 = np.asarray(pload("pooled_umap_nd") if pload("pooled_umap_nd") is not None
                else pload("pooled_umap5d"))
ctry = MASTER["Country"].astype(str).str.strip()
IS_FR = (ctry == FOCUS_COUNTRY).values

# themes to show: the largest gaps in each direction, FDR survivors first
_sig = F[F.sig_fdr].reindex(F[F.sig_fdr]["gap_pp"].abs().sort_values(ascending=False).index)
_rest = F[F.tested & ~F.sig_fdr].reindex(
    F[F.tested & ~F.sig_fdr]["gap_pp"].abs().sort_values(ascending=False).index)
SHOW = pd.concat([_sig, _rest]).head(N_THEMES)

print("=" * 82)
print(f"  FRANCE IN THEIR OWN WORDS   ({len(SHOW)} themes with the largest gaps)")
print("=" * 82)
if not len(SHOW):
    print("  No theme cleared the support gate, so there is nothing to illustrate.")
else:
    rows = []
    for r in SHOW.itertuples():
        idx = np.where(IS_FR & (np.asarray(MASTER[f"{LV}_id"]) == int(r.id)))[0]
        texts = MASTER["text"].fillna("").astype(str).values
        qs = pick_quotes(idx, X5, texts, n=N_QUOTES, wmin=WMIN, wmax=WMAX)
        _status = ("differs after FDR control" if r.sig_fdr
                   else "gap not established after FDR control")
        _dir = "more often" if r.gap_pp > 0 else "less often"
        print("\n" + "-" * 82)
        print(f"  {LV[0].upper()}{int(r.id)}  {str(r.name)[:60]}")
        print(f"  voiced by {r.rate_fr:.0%} of French participants "
              f"({int(r.n_fr)}) versus {r.rate_rest:.0%} elsewhere "
              f"({int(r.n_rest)}) - {abs(r.gap_pp):.0f}pp {_dir}")
        print(f"  status: {_status}"
              + (f"   (q = {r.q_fdr:.3f})" if np.isfinite(r.q_fdr) else ""))
        if not qs:
            print("    no French response in this theme is long enough to quote.")
        for q in qs:
            for j, line in enumerate(textwrap.wrap(q, 74)[:4]):
                print(("    “" if j == 0 else "     ") + line)
            print()
        for q in qs:
            rows.append(dict(theme_id=int(r.id), theme=str(r.name),
                             rate_fr=r.rate_fr, rate_rest=r.rate_rest,
                             gap_pp=r.gap_pp, sig_fdr=bool(r.sig_fdr),
                             q_fdr=r.q_fdr, quote=q))
    Q = pd.DataFrame(rows)
    Q.to_csv(f"{POOLED_DIR}/france_quotes.csv", index=False)
    psave("pooled_france_quotes", Q)

    # ---------------------------- panel figure --------------------------------
    import matplotlib.pyplot as plt
    from matplotlib.patches import FancyBboxPatch
    n = len(SHOW)
    fig, axes = plt.subplots(n, 1, figsize=(11.2, 2.15 * n + 1.2), squeeze=False)
    _cm, _nm_ = SENT_CMAP, sent_norm(0.6)
    for ax, r in zip(axes[:, 0], SHOW.itertuples()):
        ax.axis("off")
        _col = _cm(_nm_(r.sentiment)) if np.isfinite(r.sentiment) else NO_THEME_COLOR
        ax.add_patch(FancyBboxPatch((0.002, 0.02), 0.996, 0.96,
                                    boxstyle="round,pad=0.004,rounding_size=0.012",
                                    transform=ax.transAxes, facecolor="#FBFCFD",
                                    edgecolor="#E7EAEE", linewidth=.9, zorder=0))
        ax.add_patch(plt.Rectangle((0.004, 0.06), 0.006, 0.88, transform=ax.transAxes,
                                   color=_col, zorder=2))
        _mark = "  ·  differs after FDR control" if r.sig_fdr else \
                "  ·  gap not established after FDR control"
        ax.text(0.022, 0.90, f"{LV[0].upper()}{int(r.id)}  {str(r.name)[:56]}",
                transform=ax.transAxes, fontsize=11, fontweight="bold", color=INK,
                va="top")
        ax.text(0.022, 0.735,
                f"France {r.rate_fr:.0%} ({int(r.n_fr)})   ·   elsewhere "
                f"{r.rate_rest:.0%} ({int(r.n_rest)})   ·   {r.gap_pp:+.0f} percentage "
                f"points{_mark}",
                transform=ax.transAxes, fontsize=8.8,
                color=INK if r.sig_fdr else MUT, va="top")
        idx = np.where(IS_FR & (np.asarray(MASTER[f"{LV}_id"]) == int(r.id)))[0]
        qs = pick_quotes(idx, X5, MASTER["text"].fillna("").astype(str).values,
                         n=2, wmin=WMIN, wmax=WMAX)
        yy = 0.56
        for q in qs:
            body = "\n".join(textwrap.wrap("“" + q + "”", 96)[:3])
            ax.text(0.030, yy, body, transform=ax.transAxes, fontsize=8.9,
                    color="#333", va="top", style="italic")
            yy -= 0.26
        if not qs:
            ax.text(0.030, yy, "no French response in this theme is long enough to quote",
                    transform=ax.transAxes, fontsize=8.6, color=FOOT, va="top")
    fig_title(fig, "France in their own words: the themes with the largest prevalence gaps")
    fig.text(0.5, 0.008,
             "Every quote is a real response from a French participant in that theme, "
             "centroid-nearest and of medium length, shown as written and untranslated. "
             "Quotes illustrate a theme; they are not evidence that the gap above them "
             "is statistically established - the line under each heading says whether "
             "it is.",
             ha="center", fontsize=8.3, style="italic", color=FOOT, wrap=True)
    plt.tight_layout(rect=[0, 0.035, 1, 0.955])
    save_fig(fig, "chart_france_quotes"); plt.close(fig)
    print("\n" + "-" * 82)
    print(f"  -> france_quotes.csv · pooled_france_quotes · chart_france_quotes")
print("=" * 82)


### Stage 31 · Is the French profile different at all?

In [ ]:
# @title
# ==============================================================================
# STAGE 31  -  IS THE FRENCH PROFILE DIFFERENT AT ALL?
#             section 5 of 9  -  France in the corpus
# ==============================================================================
# The per-theme scan before this cell ran 17 separate Fisher tests on 57 French
# participants and, after BH-FDR, found nothing. That result is honest but it is
# also close to inevitable: the design resolves about 20 percentage points at the
# participant level and the largest observed gap is 11. Splitting one small
# sample into 17 corrected tests is the least powerful way to ask whether France
# differs, and "no theme survives FDR" is not the same claim as "France is like
# everywhere else".
#
# Three things this cell adds that a per-theme scan cannot do:
#
#   A  ONE omnibus test instead of 17 corrected ones.  PERMANOVA (Anderson 2001)
#      asks whether the WHOLE theme profile of French participants sits apart
#      from the rest, using the Jaccard distance between participants' theme
#      sets. It costs one p-value, so there is nothing to correct, and it is a
#      permutation test, so it assumes nothing about distributions.
#
#      It is NOT uniformly more powerful than the corrected scan, and this cell
#      does not pretend otherwise. Simulated at the exact shape of this corpus
#      (332 participants, 39 themes, 57 of them French), 25 datasets per row:
#
#          shape of the planted effect        omnibus     FDR scan
#          2 themes at +25pp                   19/25        23/25
#          4 themes at +18pp                   19/25        22/25
#         12 themes at  +9pp                   17/25        13/25
#         25 themes at  +6pp                   10/25        11/25
#         39 themes at  +5pp                   10/25        10/25
#
#      A large difference on one or two themes is the scan's case, not this
#      one. The omnibus helps in the middle band, and the reason it is here is
#      not power at all - it is that it answers the FIRST question ("is there a
#      national difference in what people talk about?") in one shot, gives an
#      R-squared for it, and lets Country be compared on the same scale as
#      Section and Region. The notebook had no way to say how much of the theme
#      profile any grouping explains. Now it does.
#
#   B  EQUIVALENCE, not just absence.  A null result is not evidence of
#      similarity unless you say how large a difference you could have ruled
#      out. Two one-sided tests (Schuirmann) on the Newcombe interval for each
#      difference turn "not significant" into a bounded, positive statement:
#      France differs from the rest by NO MORE THAN x points on this theme.
#
#   C  SHRINKAGE.  A rate out of 57 participants is noisy, and the loudest-
#      looking themes in a small group are usually the luckiest, not the most
#      distinctive. An empirical-Bayes beta-binomial prior fitted to the rest of
#      the sample pulls each French rate toward what a theme of that size
#      normally gets. What survives that pull is the part of the French profile
#      that is not sampling noise - and the reordering is itself the finding.
#
# Nothing here re-tests what the previous cell already tested. It reframes the
# same counts with methods that fit a small subsample, and every number is
# computed from the data in this notebook.
# ==============================================================================

# Everything below runs inside a function ON PURPOSE. A notebook cell shares one
# global namespace with every other cell, and this cell needs ordinary working
# names - rows, A, rng, PARENT. PARENT in particular is set by the preflight as
# the sub->main map and read by two later stages; a plain module-level
# assignment here silently replaced it with the codebook's parent categories and
# broke them with a TypeError. Wrapping the body keeps every name local, so this
# cell cannot disturb any other no matter what it calls its variables.
def _stage31():
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt

    # ----------------------------- EDIT HERE --------------------------------------
    N_PERM      = 9999   # permutations for the omnibus test
    EQ_MARGIN   = 20.0   # equivalence margin in percentage points (the design's MDE)
    MIN_VOICE_FR = 5     # same support gate as the per-theme cell, so the two agree
    MIN_VOICE_RE = 15
    SEED        = 20260814
    # ----------------------------- END EDIT ---------------------------------------

    PROF = pload("pooled_france_profile")
    if PROF is None:
        raise RuntimeError("run 'France in the corpus' first - it sets the country label.")
    FOCUS_COUNTRY = PROF["label"]
    LV = CLAIM_LEVEL
    rng = np.random.default_rng(SEED)

    print("=" * 78)
    print(f"  IS THE {FOCUS_COUNTRY} PROFILE DIFFERENT AT ALL?")
    print("=" * 78)
    print(_cov_note(LV, MASTER))

    # ---- participant x theme incidence, exactly as the per-theme cell builds it ---
    IDW, NMW, HCW = _cov_cols(LV, MASTER)
    Mv = MASTER[MASTER[HCW].astype(bool)]
    ment = (Mv.groupby(["Participant", IDW]).size().unstack(fill_value=0) > 0)
    ment = ment[[c for c in ment.columns if int(c) >= 0]]

    who = MASTER.drop_duplicates("Participant").set_index("Participant")
    PIDS = list(who.index)
    X = ment.reindex(PIDS).fillna(False).to_numpy(bool)          # participants x themes
    THEMES = [int(c) for c in ment.columns]

    wc = who["Country"].astype(str).str.strip()
    grp_country = np.where(wc.values == FOCUS_COUNTRY, FOCUS_COUNTRY,
                           np.where(wc.replace({"nan": "", "None": ""}).ne("").values,
                                    "elsewhere", ""))
    N_FR = int((grp_country == FOCUS_COUNTRY).sum())
    N_RE = int((grp_country == "elsewhere").sum())
    print(f"\n  {X.shape[0]} participants x {X.shape[1]} {LV} themes   "
          f"({FOCUS_COUNTRY} {N_FR}, elsewhere {N_RE})")
    voiced = X.sum(1)
    print(f"  participants voicing no theme at all: {int((voiced == 0).sum())} "
          f"(kept; Jaccard treats two empty profiles as identical, and an empty "
          f"against a non-empty as maximally different)")

    # ==============================================================================
    # A.  OMNIBUS:  does the whole profile differ?   (PERMANOVA, Anderson 2001)
    # ==============================================================================
    def _jaccard_sq(B):
        """Squared Jaccard distance between every pair of binary profiles.

        Jaccard is the right distance for presence/absence data: it ignores the
        themes NEITHER participant voiced, which for a 39-theme codebook is most of
        them, and would otherwise make everyone look similar to everyone.
        """
        A = B.astype(np.float64)
        inter = A @ A.T
        sz = A.sum(1)
        union = sz[:, None] + sz[None, :] - inter
        with np.errstate(invalid="ignore", divide="ignore"):
            d = 1.0 - np.where(union > 0, inter / np.maximum(union, 1e-12), 1.0)
        d[union == 0] = 0.0                     # both empty -> identical, not undefined
        np.fill_diagonal(d, 0.0)
        return d ** 2


    def _pseudo_F(D2, lab):
        """Anderson's pseudo-F on a distance matrix, and the R^2 that goes with it."""
        keep = lab != ""
        D2 = D2[np.ix_(keep, keep)]
        lab = lab[keep]
        N = len(lab)
        groups = [g for g in pd.unique(lab)]
        a = len(groups)
        if a < 2 or N <= a:
            return np.nan, np.nan
        SST = D2.sum() / (2.0 * N)
        SSW = 0.0
        for g in groups:
            m = lab == g
            n_g = int(m.sum())
            if n_g < 2:
                continue
            SSW += D2[np.ix_(m, m)].sum() / (2.0 * n_g)
        SSA = SST - SSW
        if SST <= 0:
            return np.nan, np.nan
        R2 = float(SSA / SST)
        # SSW == 0 means every profile inside a group is identical - F is then a
        # division by zero, but R2 is still perfectly well defined and equals 1.
        # Report what can be reported rather than throwing the whole result away.
        if SSW <= 0:
            return np.nan, R2
        return float((SSA / (a - 1)) / (SSW / (N - a))), R2


    D2 = _jaccard_sq(X)
    FACT = [("Country", grp_country)]
    for _f in ("Section", "Region"):
        if _f in who.columns:
            v = who[_f].astype(str).str.strip().replace({"nan": "", "None": ""}).values
            # a permutation test needs every level to be permutable; drop singletons
            vc = pd.Series(v).value_counts()
            v = np.array([x if (x != "" and vc.get(x, 0) >= 5) else "" for x in v])
            if len(set(v) - {""}) >= 2:
                FACT.append((_f, v))

    omni = []
    for fname, lab in FACT:
        F_obs, R2 = _pseudo_F(D2, lab)
        if not np.isfinite(F_obs):
            continue
        keep = lab != ""
        sub = lab[keep]
        Dk = D2[np.ix_(keep, keep)]
        null = np.empty(N_PERM)
        for b in range(N_PERM):
            null[b] = _pseudo_F(Dk, rng.permutation(sub))[0]
        # +1 in both places: the observed arrangement is one of the possible ones,
        # so a permutation p can never honestly be 0
        p = float((np.sum(null >= F_obs) + 1) / (N_PERM + 1))
        n_lev = len(set(sub))
        omni.append(dict(factor=fname, levels=n_lev, n=int(keep.sum()),
                         pseudo_F=F_obs, R2=R2, p_perm=p, null=null))
        print(f"\n  {fname:<9} {n_lev} groups over {int(keep.sum())} participants")
        print(f"    pseudo-F {F_obs:6.3f}   R2 {100*R2:5.2f}% of the between-participant "
              f"variation   p = {p:.4f}  ({N_PERM} permutations)")
        print(f"    -> {'the whole profile DOES differ by ' + fname.lower() if p < 0.05 else 'no whole-profile difference by ' + fname.lower()}")

    OMNI = pd.DataFrame([{k: v for k, v in o.items() if k != "null"} for o in omni])
    if len(OMNI) > 1:
        print("\n  HOW MUCH OF WHAT PEOPLE TALK ABOUT DOES EACH GROUPING EXPLAIN?")
        print(f"    {'factor':<10}{'groups':>7}{'R2':>9}{'p':>10}")
        for _, r in OMNI.sort_values("R2", ascending=False).iterrows():
            print(f"    {r.factor:<10}{int(r.levels):>7}{100*r.R2:>8.2f}%{r.p_perm:>10.4f}")
        print("    R2 is on one scale for all three, so they can be ranked against each")
        print("    other. This is the comparison the per-theme scan could not make.")

    _fr = next((o for o in omni if o["factor"] == "Country"), None)
    if _fr is not None:
        print(f"\n  ONE test, so there is nothing to correct for.")
        if _fr["p_perm"] >= 0.05:
            print(f"  Result: tested as a whole profile, {FOCUS_COUNTRY} is NOT "
                  f"distinguishable from the\n  rest (p = {_fr['p_perm']:.3f}, R2 = "
                  f"{100*_fr['R2']:.2f}%). Read with the equivalence bounds below, that\n"
                  f"  is a bounded null rather than an absence of evidence - but it is a "
                  f"null, and\n  a difference concentrated in one or two themes would be "
                  f"better caught by the\n  per-theme scan above than by this test. "
                  f"Neither found one.")
        else:
            print(f"  Result: the {FOCUS_COUNTRY} profile DOES sit apart as a whole "
                  f"(p = {_fr['p_perm']:.3f},\n  R2 = {100*_fr['R2']:.2f}%), even though no "
                  f"single theme survived FDR - which is\n  what a difference spread "
                  f"across many themes looks like.")

    # ==============================================================================
    # B.  EQUIVALENCE:  how large a difference can be RULED OUT?
    # ==============================================================================
    # Newcombe's hybrid-score interval for a difference of proportions, built from
    # the same Wilson intervals every other figure in this notebook uses, so the
    # bounds here and the bars there cannot disagree.
    def wilson(k, n, z):
        if n == 0:
            return (np.nan, np.nan, np.nan)
        p = k / n
        d = 1 + z * z / n
        c = (p + z * z / (2 * n)) / d
        h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
        return p, max(0.0, c - h), min(1.0, c + h)


    def _newcombe(k1, n1, k2, n2, z=1.645):        # 90% -> TOST at alpha = 0.05
        p1, l1, u1 = wilson(k1, n1, z)
        p2, l2, u2 = wilson(k2, n2, z)
        if not np.isfinite(p1) or not np.isfinite(p2):
            return (np.nan,) * 3
        lo = (p1 - p2) - np.sqrt((p1 - l1) ** 2 + (u2 - p2) ** 2)
        hi = (p1 - p2) + np.sqrt((u1 - p1) ** 2 + (p2 - l2) ** 2)
        return float(p1 - p2), float(lo), float(hi)


    isfr = grp_country == FOCUS_COUNTRY
    isre = grp_country == "elsewhere"
    rows = []
    for j, c in enumerate(THEMES):
        k1, k2 = int(X[isfr, j].sum()), int(X[isre, j].sum())
        d, lo, hi = _newcombe(k1, N_FR, k2, N_RE)
        tested = (k1 >= MIN_VOICE_FR) and (k2 >= MIN_VOICE_RE)
        bound = 100 * max(abs(lo), abs(hi)) if np.isfinite(lo) else np.nan
        rows.append(dict(id=c, name=theme_name(LV, c), n_fr=k1, n_rest=k2,
                         rate_fr=k1 / N_FR, rate_rest=k2 / N_RE,
                         diff_pp=100 * d, lo_pp=100 * lo, hi_pp=100 * hi,
                         tested=tested, ruled_out_beyond_pp=bound,
                         equivalent=bool(tested and np.isfinite(bound)
                                         and bound <= EQ_MARGIN)))
    EQ = pd.DataFrame(rows).sort_values("diff_pp", ascending=False).reset_index(drop=True)
    t = EQ[EQ.tested]
    print("\n" + "-" * 78)
    print(f"  EQUIVALENCE  -  what this sample can positively rule out")
    print("-" * 78)
    print(f"  {len(t)} theme(s) meet the support gate. Two one-sided tests at "
          f"alpha 0.05 against a\n  +/-{EQ_MARGIN:.0f} pp margin (the participant-level "
          f"MDE this design reports):")
    print(f"    statistically EQUIVALENT within +/-{EQ_MARGIN:.0f} pp : "
          f"{int(t.equivalent.sum())} of {len(t)}")
    print(f"    inconclusive (neither different nor equivalent): "
          f"{int((~t.equivalent).sum())} of {len(t)}")
    if len(t):
        print(f"\n  tightest bounds - the themes where {FOCUS_COUNTRY} is most firmly "
              f"pinned to the rest:")
        for _, r in t.nsmallest(5, "ruled_out_beyond_pp").iterrows():
            print(f"    {str(r['name'])[:44]:<46} differs by no more than "
                  f"{r.ruled_out_beyond_pp:5.1f} pp")
        print(f"\n  widest bounds - where this sample simply cannot say:")
        for _, r in t.nlargest(3, "ruled_out_beyond_pp").iterrows():
            print(f"    {str(r['name'])[:44]:<46} anything up to "
                  f"{r.ruled_out_beyond_pp:5.1f} pp remains possible")

    # ==============================================================================
    # C.  SHRINKAGE:  which part of the French profile is not noise?
    # ==============================================================================
    # Empirical Bayes. The prior is fitted by moments to the theme rates in the REST
    # of the sample, so it encodes what a theme in this corpus normally reaches.
    #
    # Every theme here shares the same denominator - the 57 French participants - so
    # the prior pulls each one by the SAME fraction, w = (a+b)/(57+a+b). What differs
    # is how far each rate has to travel: the pull is w x (prior mean - raw rate), so
    # the themes that look most extreme move most. That is the point. In a group of
    # 57 the loudest-looking themes are usually the luckiest, and this is regression
    # to the mean applied deliberately instead of being left for the reader to guess
    # at. A theme that stays high after the pull was high for a reason.
    _rest_rates = np.array([X[isre, j].mean() for j in range(len(THEMES))], float)
    m, v = float(_rest_rates.mean()), float(_rest_rates.var(ddof=1))
    _shrunk_ok = 0 < m < 1 and v > 0 and v < m * (1 - m)
    if _shrunk_ok:
        _k = m * (1 - m) / v - 1
        ALPHA0, BETA0 = m * _k, (1 - m) * _k
    else:
        ALPHA0 = BETA0 = 1.0
    SH = []
    for j, c in enumerate(THEMES):
        k1 = int(X[isfr, j].sum())
        raw = k1 / N_FR
        post = (k1 + ALPHA0) / (N_FR + ALPHA0 + BETA0)
        SH.append(dict(id=c, name=theme_name(LV, c), n_fr=k1, raw_pp=100 * raw,
                       shrunk_pp=100 * post, rest_pp=100 * _rest_rates[j],
                       pull_pp=100 * (post - raw),
                       excess_pp=100 * (post - _rest_rates[j])))
    SH = pd.DataFrame(SH)
    SH["rank_raw"] = SH["raw_pp"].rank(ascending=False, method="min").astype(int)
    SH["rank_shrunk"] = SH["shrunk_pp"].rank(ascending=False, method="min").astype(int)
    SH["rank_move"] = SH["rank_raw"] - SH["rank_shrunk"]
    print("\n" + "-" * 78)
    print("  SHRINKAGE  -  the French profile after the noise is pulled out")
    print("-" * 78)
    print(f"  empirical-Bayes prior from the {len(THEMES)} rest-of-sample rates: "
          f"Beta(alpha={ALPHA0:.2f}, beta={BETA0:.2f})"
          + ("" if _shrunk_ok else "   [fallback Beta(1,1): the rest-of-sample rates "
                                   "were not over-dispersed enough to fit a prior]"))
    print(f"  prior weight = {ALPHA0 + BETA0:.1f} pseudo-participants against "
          f"{N_FR} real ones")
    _top = SH.nlargest(5, "shrunk_pp")
    print(f"\n  the 5 themes {FOCUS_COUNTRY} voices most, AFTER shrinkage:")
    print(f"    {'theme':<44}{'raw':>7}{'shrunk':>8}{'rest':>7}{'vs rest':>9}{'rank':>7}")
    for _, r in _top.iterrows():
        print(f"    {str(r['name'])[:42]:<44}{r.raw_pp:6.1f}%{r.shrunk_pp:7.1f}%"
              f"{r.rest_pp:6.1f}%{r.excess_pp:+8.1f}{r.rank_raw:>4}->{r.rank_shrunk:<3}")
    moved = SH[SH.rank_move.abs() >= 3]
    print(f"\n  {len(moved)} theme(s) move 3 or more places once shrinkage is applied - "
          f"that movement\n  is the part of the raw ranking that was sampling noise.")
    if len(moved):
        for _, r in moved.reindex(moved.rank_move.abs().sort_values(ascending=False)
                                   .index).head(4).iterrows():
            print(f"    {str(r['name'])[:42]:<44} rank {r.rank_raw:>2} -> "
                  f"{r.rank_shrunk:<2}  (n={r.n_fr})")

    # ==============================================================================
    # FIGURES  -  one question per figure
    # ==============================================================================
    _charts = []
    # --- A: the omnibus null distribution, with the observed value on it ----------
    if omni:
        fig, axes = plt.subplots(1, len(omni), figsize=(4.5 * len(omni), 3.6),
                                 squeeze=False)
        for ax, o in zip(axes[0], omni):
            ax.hist(o["null"], bins=60, color="#C9CDD2", edgecolor="none")
            ax.axvline(o["pseudo_F"], color=HIGHLIGHT, lw=2.0, zorder=5)
            ax.text(o["pseudo_F"], ax.get_ylim()[1] * .96,
                    f"  observed {o['pseudo_F']:.2f}", color=HIGHLIGHT,
                    fontsize=8.5, va="top", ha="left", fontweight="bold")
            ax.set_title(f"{o['factor']}  ({o['levels']} groups)\n"
                         f"p = {o['p_perm']:.4f}   R² = {100*o['R2']:.2f}%",
                         fontsize=10, loc="left")
            ax.set_xlabel("pseudo-F under random regrouping", fontsize=8.5)
            ax.set_ylabel("permutations", fontsize=8.5)
            for sp in ("top", "right"):
                ax.spines[sp].set_visible(False)
        fig_title(fig, f"Does the whole theme profile differ? {N_PERM} permutations "
                       f"per factor")
        fig.text(0.5, 0.012, "Grey is what pseudo-F looks like when the group labels "
                 "carry no information. The red line is the value actually observed. "
                 "One test per factor, so no multiplicity correction is needed.",
                 ha="center", fontsize=8.0, style="italic", color=FOOT, wrap=True)
        plt.tight_layout(rect=[0, .06, 1, .90])
        save_fig(fig, "chart_france_omnibus_permanova"); plt.close(fig)
        _charts.append("chart_france_omnibus_permanova")

    # --- B: equivalence forest ----------------------------------------------------
    if len(t):
        D = t.sort_values("diff_pp").reset_index(drop=True)
        fig, ax = plt.subplots(figsize=(8.6, max(3.0, 0.34 * len(D) + 1.9)))
        ax.axvspan(-EQ_MARGIN, EQ_MARGIN, color="#0F6E56", alpha=.07, zorder=0)
        for b in (-EQ_MARGIN, EQ_MARGIN):
            ax.axvline(b, color="#0F6E56", lw=1.0, ls="--", zorder=2)
        ax.axvline(0, color=ZERO if "ZERO" in globals() else "#8A939B", lw=1.2, zorder=3)
        for i, r in D.iterrows():
            col = "#0F6E56" if r.equivalent else "#B0B6BC"
            ax.plot([r.lo_pp, r.hi_pp], [i, i], color=col, lw=2.2, zorder=4,
                    solid_capstyle="round")
            ax.plot([r.diff_pp], [i], "o", ms=5, color=col, zorder=5)
        ax.set_yticks(range(len(D)))
        ax.set_yticklabels([fit_label(canon_theme_name(n), width=40, max_lines=2)
                            for n in D["name"]], fontsize=7.6)
        ax.set_xlabel(f"difference in the share of participants voicing the theme "
                      f"({FOCUS_COUNTRY} minus elsewhere), percentage points",
                      fontsize=8.8)
        for sp in ("top", "right", "left"):
            ax.spines[sp].set_visible(False)
        fig_title(fig, f"How large a {FOCUS_COUNTRY} difference can be ruled out? "
                       f"90% intervals against a ±{EQ_MARGIN:.0f} pp margin")
        fig.text(0.5, 0.012,
                 f"Green where the whole interval falls inside the margin: that theme is "
                 f"statistically EQUIVALENT, not merely non-significant. Grey where the "
                 f"interval escapes it: this sample cannot decide. "
                 f"{int(t.equivalent.sum())} of {len(t)} are equivalent.",
                 ha="center", fontsize=8.0, style="italic", color=FOOT, wrap=True)
        plt.tight_layout(rect=[0, .055, 1, .92])
        save_fig(fig, "chart_france_equivalence"); plt.close(fig)
        _charts.append("chart_france_equivalence")

    # --- C: shrinkage arrows ------------------------------------------------------
    _S = SH.nlargest(14, "raw_pp").sort_values("shrunk_pp").reset_index(drop=True)
    if len(_S):
        fig, ax = plt.subplots(figsize=(8.6, max(3.0, 0.36 * len(_S) + 1.9)))
        for i, r in _S.iterrows():
            ax.annotate("", xy=(r.shrunk_pp, i), xytext=(r.raw_pp, i),
                        arrowprops=dict(arrowstyle="-|>", color="#8A939B", lw=1.2,
                                        shrinkA=0, shrinkB=0))
            ax.plot([r.raw_pp], [i], "o", ms=5, mfc="white", mec="#8A939B", zorder=4)
            ax.plot([r.shrunk_pp], [i], "o", ms=6.5, color=SENT_POS, zorder=5)
            ax.plot([r.rest_pp], [i], "|", ms=11, color="#14202B", zorder=6)
        ax.set_yticks(range(len(_S)))
        ax.set_yticklabels([fit_label(canon_theme_name(n), width=40, max_lines=2)
                            for n in _S["name"]], fontsize=7.6)
        ax.set_xlabel("% of participants voicing the theme", fontsize=8.8)
        for sp in ("top", "right", "left"):
            ax.spines[sp].set_visible(False)
        from matplotlib.lines import Line2D
        ax.legend(handles=[
            Line2D([], [], marker="o", ls="", mfc="white", mec="#8A939B",
                   label=f"raw {FOCUS_COUNTRY} rate (n={N_FR})"),
            Line2D([], [], marker="o", ls="", color=SENT_POS, label="after shrinkage"),
            Line2D([], [], marker="|", ls="", color="#14202B", ms=11,
                   label="rest of the sample")],
            fontsize=8, frameon=False, loc="lower right")
        fig_title(fig, f"The {FOCUS_COUNTRY} profile before and after the noise is "
                       f"pulled out")
        fig.text(0.5, 0.012,
                 f"Each arrow is one theme being pulled toward what a theme of its size "
                 f"normally reaches, using a beta prior fitted to the rest of the sample "
                 f"(weight {ALPHA0+BETA0:.0f} against {N_FR} real participants). Long "
                 f"arrows mark rates that rested on few participants.",
                 ha="center", fontsize=8.0, style="italic", color=FOOT, wrap=True)
        plt.tight_layout(rect=[0, .055, 1, .92])
        save_fig(fig, "chart_france_shrinkage"); plt.close(fig)
        _charts.append("chart_france_shrinkage")

    # ---- persist -----------------------------------------------------------------
    EQ.to_csv(f"{POOLED_DIR}/france_equivalence.csv", index=False)
    SH.to_csv(f"{POOLED_DIR}/france_shrinkage.csv", index=False)
    OMNI.to_csv(f"{POOLED_DIR}/profile_omnibus.csv", index=False)
    psave("pooled_france_omnibus", dict(omnibus=OMNI, equivalence=EQ, shrinkage=SH,
                                        n_perm=N_PERM, eq_margin=EQ_MARGIN,
                                        prior=(ALPHA0, BETA0), label=FOCUS_COUNTRY))
    print(f"\n  -> profile_omnibus.csv · france_equivalence.csv · france_shrinkage.csv · "
          f"pooled_france_omnibus")
    print(f"  -> {' · '.join(_charts)}")
    print("=" * 78)


_stage31()


---

## Section 6 - Themes and competition outcome

Whether what a team talks about relates to how it does in the competition, approached from several directions: per-theme association, percentage-point differences, composite predictors, thematic diversity, theme pairs, and predictive resolution. A prize is a team property, so these are team-level analyses with team-clustered intervals.

### Stage 32 · Theme mention and prize status, by theme

**Question.** Is a team's prize status related to how much its members talk about a given theme?

**Reads.** `pooled_correlation_forest_*` from Notebook 2 Stage 40.

**How to read it.** One forest per level. The sub panel is the inferential one; the main panel is the same corpus described more coarsely and carries no claim. Stars are BH-FDR, not raw intervals.

In [ ]:
# @title
# ==============================================================================
# STAGE 32  -  THEME MENTION AND PRIZE STATUS, BY THEME
#             section 6 of 9  -  Themes and competition outcome
# ==============================================================================
# ==============================================================================
# TEAM-LEVEL FOREST: THEME MENTION AND PRIZE STATUS
# ------------------------------------------------------------------------------
# One two-panel forest per level, read from the tables Notebook 2 Stage 40 wrote.
# The x axis carries the effect (OLS beta on the team mention rate, size-adjusted);
# sentiment never sits on an axis, it is the side colour strip. Stars are BH-FDR.
#
# The sub level is the inferential figure. The main level is shown alongside it as
# the coarse description of the same corpus, and is labelled as such: its themes
# carry no significance claim, because the bootstrap stability of that level sits
# in the "pattern" band rather than the "finding" band (Hennig 2007).
#
# Responses with no theme never enter these tables: Stage 40 counts a mention only
# from a themed row, while keeping every participant in the team-size denominator.
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
_PL = globals().get("_orig_pload", pload)
made, missing = [], []
for lv in LEVELS:
    res = _PL(f"pooled_correlation_forest_{lv}")
    if res is None or len(res) == 0:
        missing.append(lv)
        print(f"  [{lv}] table empty - run Notebook 2 Stage 40; this level is skipped")
        continue
    res = add_fdr_sig(res, [("beta_topic", "ci_topic_lo", "ci_topic_hi", "sig_topic"),
                            ("beta_sent", "ci_sent_lo", "ci_sent_hi", "sig_sent")])
    _role = ("inferential" if lv == CLAIM_LEVEL else "descriptive, no claim tested")
    fig = team_two_panel(
        res,
        title=(f"Team-level association between theme mention and prize status — "
               f"{level_label(lv)} ({len(res)} themes, {_role})"),
        level_label=level_label(lv),
        footer=("Refs: Efron (1979) team bootstrap · NB2 Stage 26B LLM sentiment · "
                "BH-FDR stars · unthemed responses excluded from every mention count"))
    save_fig(fig, f"correlation_forest_two_panel_team_{lv}"); plt.close(fig)
    made.append(f"{lv} ({len(res)} themes)")

print(f"  forests rendered: {', '.join(made) if made else 'none'}")
if missing:
    print(f"  no table for: {', '.join(missing)}")
print(f"  claims are read from the {CLAIM_LEVEL} panel; the main panel is description.")


### Stage 33 · Theme mention and prize status, in percentage points

**Question.** The same relationship, in units a reader can act on.

**How to read it.** The difference in the probability of being on a prize-winning team between participants who voiced a theme and those who did not, in percentage points, with a team-clustered bootstrap interval.

In [ ]:
# @title
# ==============================================================================
# STAGE 33  -  THEME MENTION AND PRIZE STATUS, IN PERCENTAGE POINTS
#             section 6 of 9  -  Themes and competition outcome
# ==============================================================================
# WHY THE FIGURE LOOKS LIKE THE ATLAS AGAIN
#   A previous revision drew this as a thin marker-and-line forest. That is the
#   right encoding for a pure estimate plot, but it threw away the solid coloured
#   rows that make the theme atlas readable, and it made this figure look like the
#   team-level Stage 40 rather than its participant-level counterpart. This version
#   restores the atlas look AND keeps the encoding honest:
#       bar   drawn from ZERO to the estimate, so BAR LENGTH IS THE EFFECT
#       colour  the theme's mean measured sentiment - the project-wide convention
#       whisker a separate error bar for the 95% team-cluster bootstrap interval
#   The earlier version filled the whole interval as the bar, so bar length carried
#   UNCERTAINTY: a wide, imprecise interval looked like a large effect. Length now
#   means effect and only the whisker means uncertainty.
#
# WHAT "PRIZE" MEANS - stated on the figure, because a reader cannot guess it
#   The participant's team won a SPECIAL AWARD or was NOMINATED for one. Medals are
#   NOT used: 73% of teams take Gold, so a medal separates almost nobody. Teams with
#   no award-registry match are EXCLUDED - an unknown result is not a loss.
#
# THE STATISTIC
#   unit        participant
#   quantity    P(on a prize team | voices the theme) - P(on a prize team | does not)
#   interval    95% percentile bootstrap, RESAMPLED BY TEAM, because participants on
#               one team share both their prize status and much of their language
#   multiplicity BH-FDR across the themes within each level
#
# Outputs: chart_figureA_pp_{main,sub}, figureA_pp_{main,sub}.csv,
#          pooled_figA_pp_{level}.
# Refs: Efron & Tibshirani (1993) bootstrap; Cameron, Gelbach & Miller (2008)
#       cluster resampling; Benjamini & Hochberg (1995).
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, textwrap

_PL = globals().get("_orig_pload", pload)
M = _PL("pooled_master")
if M is None:
    raise RuntimeError("Need pooled_master - run NB2 Stage 33b first.")
M = M.copy()

MIN_VOICE = 10     # participants voicing the theme, and not voicing it
N_BOOT    = 1500
TOP_N     = 18     # rows shown; the full table always goes to the CSV
SEED      = int(CFG.seed) if "CFG" in globals() else 42
# _LVS, not LEVELS: the preflight publishes LEVELS = ["main","sub"] and every
# later cell reads it. A literal here both reinstated the removed mid level and
# rebound that global for the rest of the notebook.
_LVS = [(lv,) + _cov_cols(lv, MASTER)[:2] + (lv[0].upper(),) for lv in LEVELS]

_CMAP = globals().get("SENT_CMAP")
_NORM = globals().get("SNORM")
if _CMAP is None:
    from matplotlib.colors import LinearSegmentedColormap, Normalize
    _CMAP = LinearSegmentedColormap.from_list("s", ["#B84413", "#BFC4C9", "#0F6E56"])
    _NORM = Normalize(-0.6, 0.6)
_INK = globals().get("INK", "#14202B")

def _w2(t, w=34):
    # F1 - the body used to be `"\n".join(textwrap.wrap(str(t), w)[:2])`,
    # which wrapped and then threw away every line past the second WITHOUT a
    # marker: "Financial Limitations and Costs" rendered as "Financial /
    # Limitations and" and read as a complete label. Measured on the 39 real
    # sub-theme names, that silently cut 2 of them here.
    # fit_label keeps max_lines=2 so this figure's layout is unchanged, widens
    # up to 60% before conceding, and only then truncates - always visibly.
    return fit_label(canon_theme_name(t), width=w, max_lines=2)
def _bh(pv):
    pv = np.asarray(pv, float); q = np.full(pv.shape, np.nan)
    ok = np.where(np.isfinite(pv))[0]
    if ok.size:
        p = pv[ok]; n = p.size; o = np.argsort(p)
        r = np.minimum.accumulate((p[o]*n/np.arange(1, n+1))[::-1])[::-1].clip(max=1.0)
        qq = np.empty(n); qq[o] = r; q[ok] = qq
    return q

print("=" * 78)
print("  STAGE 41b  THEME MENTION -> PRIZE, PARTICIPANT LEVEL  (percentage points)")
print("=" * 78)

_med = M["medal"].astype(str).str.strip().str.lower()
known = ~_med.isin(["", "nan", "none", "unknown"])
n_drop_teams = int(M.loc[~known, "Team"].nunique())
MK = M[known].copy()
MK["_win"] = MK["special"].astype(bool) | MK["nominated"].astype(bool)
team_win = MK.groupby("Team")["_win"].max()
pu = MK.drop_duplicates("Participant").set_index("Participant")
prize = pu["Team"].map(team_win).fillna(False).astype(bool)
pteam = pu["Team"]
base = float(prize.mean())
n_prize_teams, n_teams = int(team_win.sum()), int(team_win.size)
print(f"  teams with a known result: {n_teams}"
      + (f"  ({n_drop_teams} excluded - no registry match, which is not a loss)"
         if n_drop_teams else ""))
print(f"  prize teams (special award OR nomination): {n_prize_teams}/{n_teams}")
print(f"  participants: {len(prize)}  ·  {100*base:.0f}% are on a prize team")

rng = np.random.default_rng(SEED)
teams_u = pteam.unique()
t_idx = {t: np.where(pteam.values == t)[0] for t in teams_u}
BOOT = [np.concatenate([t_idx[t] for t in rng.choice(teams_u, len(teams_u), replace=True)])
        for _ in range(N_BOOT)]      # one team-resampled index set, reused for every theme

for lv, idc, namec, pfx in _LVS:
    if idc not in MK.columns: continue
    # NOISE. Grouping the raw id column gives the matrix a column for -1, so the
    # responses that belong to no theme are counted as if they were one. Mentions
    # come from themed rows only; the denominator stays every participant, so a
    # rate is still "share of all participants who voiced this theme".
    _hc = _cov_cols(lv, MK)[2]
    print(_cov_note(lv, MK))
    _MKv = MK[MK[_hc].astype(bool)] if _hc in MK.columns \
           else MK[MK[idc] >= 0]
    ment = (_MKv.groupby(["Participant", idc]).size().unstack(fill_value=0) > 0)
    ment = ment.reindex(prize.index).fillna(False)
    ment = ment[[c for c in ment.columns if int(c) >= 0]]
    nm = {int(r[idc]): str(r[namec]) for _, r in
          MK.dropna(subset=[idc]).drop_duplicates(subset=[idc]).iterrows()}
    sent = MK.dropna(subset=[idc]).groupby(idc)["sentiment"].apply(
        lambda x: pd.to_numeric(x, errors="coerce").mean())
    y = prize.values.astype(float)

    rows = []
    for c in sorted(int(x) for x in MK[idc].dropna().unique()):
        if c not in ment.columns: continue
        m = ment[c].values.astype(bool)
        if m.sum() < MIN_VOICE or (~m).sum() < MIN_VOICE: continue
        beta = 100.0 * (y[m].mean() - y[~m].mean())
        bs = np.empty(N_BOOT); bs[:] = np.nan
        for b, ix in enumerate(BOOT):
            mm = m[ix]
            if mm.sum() >= 3 and (~mm).sum() >= 3:
                bs[b] = 100.0 * (y[ix][mm].mean() - y[ix][~mm].mean())
        bs = bs[np.isfinite(bs)]
        if bs.size < N_BOOT * 0.5: continue
        lo, hi = np.percentile(bs, [2.5, 97.5])
        p_two = 2 * min((bs <= 0).mean(), (bs >= 0).mean())   # bootstrap two-sided p
        rows.append(dict(level=lv, cluster=c, name=nm.get(c, f"{pfx}{c}"),
                         beta=beta, lo=lo, hi=hi, n_ment=int(m.sum()),
                         n_teams=int(pteam[m].nunique()),
                         rate_ment=float(y[m].mean()), p_boot=min(1.0, max(p_two, 1/N_BOOT)),
                         sent=float(sent.get(c, np.nan))))
    if not rows:
        print(f"\n  {lv}: no theme clears the support gate - skipped"); continue
    R = pd.DataFrame(rows)
    R["q_fdr"] = _bh(R.p_boot.values); R["sig"] = R.q_fdr < 0.05
    R = R.sort_values("beta", ascending=False).reset_index(drop=True)
    R.to_csv(f"{POOLED_DIR}/figureA_pp_{lv}.csv", index=False)
    psave(f"pooled_figA_pp_{lv}", R)
    ns = int(R.sig.sum())
    print(f"\n  {lv.upper()}: {len(R)} themes tested (>= {MIN_VOICE} on each side) · "
          f"{ns} survive BH-FDR")
    print(f"    range {R.beta.min():+.1f} to {R.beta.max():+.1f} pp · "
          f"every interval is a team-clustered bootstrap")
    for _, r in R[R.sig].iterrows():
        print(f"      * {r['name'][:44]:<44} {r.beta:+.1f} pp  q={r.q_fdr:.3f}")

    # ---------------- figure: solid rows, bar from zero, whisker for the CI ------
    S = R.reindex(R.beta.abs().sort_values(ascending=False).index).head(TOP_N)
    S = S.sort_values("beta").reset_index(drop=True)
    h = max(3.8, 0.48 * len(S) + 2.3)
    fig, ax = plt.subplots(figsize=(11.5, min(h, 10.0)))
    # figure height in inches - title, footer and legend are all positioned
    # from it so their gaps stay constant whatever the row count
    _fh = fig.get_size_inches()[1]
    for i, (_, r) in enumerate(S.iterrows()):
        if i % 2 == 0:
            ax.axhspan(i - .5, i + .5, color="#F4F6F7", zorder=0)
        col = _CMAP(_NORM(r.sent)) if np.isfinite(r.sent) else "#C4CAD0"
        ax.barh(i, r.beta, height=0.66, color=col, edgecolor="white", lw=.7, zorder=3)
        ax.errorbar(r.beta, i, xerr=[[max(0, r.beta - r.lo)], [max(0, r.hi - r.beta)]],
                    fmt="none", ecolor="#3A4650", elinewidth=1.1, capsize=2.5, zorder=4)
        if r.sig:
            side = 1 if r.beta >= 0 else -1
            ax.text(r.hi + side*1.2 if side > 0 else r.lo - 1.2, i, "*",
                    ha="left" if side > 0 else "right", va="center",
                    fontsize=12, fontweight="bold", color=_INK, zorder=5)
    ax.axvline(0, color=_INK, lw=1.2, zorder=2)
    ax.set_yticks(range(len(S)))
    ax.set_yticklabels([f"{pfx}{int(r.cluster)}  " + _w2(r["name"]) +
                        f"  (n={int(r.n_ment)})" for _, r in S.iterrows()], fontsize=7.6)
    ax.set_ylim(-.6, len(S) - .4)
    ax.set_xlabel("difference in the share on a prize team, between participants who "
                  "voice the theme\nand those who do not  [percentage points]",
                  fontsize=9.2)
    for sp in ("top", "right", "left"): ax.spines[sp].set_visible(False)
    ax.xaxis.grid(True, color="#E4E8EB"); ax.set_axisbelow(True)
    ax.tick_params(length=0)
    verdict = (f"{ns} theme(s) survive BH-FDR" if ns else
               "no theme survives BH-FDR - an honest null")
    ttl = f"Theme mention and prize status, participant level — {lv} level"
    # Position title and verdict from the figure's HEIGHT IN INCHES, not from a
    # fixed fraction: a fraction that clears the title on a tall figure collides
    # with it on a short one, which is exactly what was happening.
    fig.suptitle(ttl, fontsize=13, fontweight="bold", y=1 - 0.28/_fh)
    fig.text(0.5, 1 - 0.62/_fh, verdict, ha="center", va="top",
             fontsize=9, color="#5B6670")
    fig.subplots_adjust(right=0.87)
    cax = fig.add_axes([0.895, 0.18, 0.014, 0.64])
    from matplotlib.cm import ScalarMappable
    _sm = ScalarMappable(norm=_NORM, cmap=_CMAP); _sm.set_array([])
    _cb = fig.colorbar(_sm, cax=cax); _cb.outline.set_visible(False)
    _cb.ax.tick_params(labelsize=7, length=0)
    _cb.set_label("theme mean sentiment\n(orange - / grey 0 / teal +)", fontsize=7.4)
    fig.text(0.44, 0.16/_fh,
             f"'Prize' = the participant's team won a special award or was nominated for one "
             f"({n_prize_teams} of {n_teams} teams with a known result; medals are not used, "
             f"73% take Gold) · bar = the difference, whisker = 95% team-cluster bootstrap CI "
             f"· * = BH-FDR q<0.05 · bar colour = the theme's mean sentiment, not an effect",
             ha="center", fontsize=6.8, color="#8A939B")
    plt.tight_layout(rect=[0, 0.90/_fh, .87, 1 - 0.95/_fh])
    try:    save_fig(fig, f"chart_figureA_pp_{lv}")
    except Exception:
        fig.savefig(f"{POOLED_DIR}/chart_figureA_pp_{lv}.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

print("-" * 78)
print("  This is the PARTICIPANT-level view; Stage 40 answers the same question at the")
print("  TEAM level. With 125 teams the minimum detectable difference is about 23")
print("  percentage points, so a null here rules out large effects, not small ones.")
print("  files: chart_figureA_pp_{main,sub} · figureA_pp_{main,sub}.csv")
print("=" * 78)

### Stage 34 · Composite and ordered tests

**Question.** Do composite predictors - thematic breadth, negative tone - carry prize signal that single themes do not, and is there an ordered trend across outcome tiers?

**How to read it.** These tests are deliberately more powerful than the per-theme ones: fewer hypotheses, each using more of the data. A null here is more informative than a null in the per-theme forest.

In [ ]:
# @title
# ==============================================================================
# STAGE 34  -  COMPOSITE AND ORDERED PRIZE TESTS
#             section 6 of 9  -  Themes and competition outcome
# ==============================================================================
# ==============================================================================
# HIGH-POWER PRIZE TESTS  -  composite predictors + ordered trend + precision
# ------------------------------------------------------------------------------
# Companion to Stages 40 / 41 / 41b, which test ~1500 individual theme-mention
# hypotheses and therefore pay a very heavy multiple-testing price. This cell
# does the opposite: it asks FEW, well-powered questions of the same data.
#   A. THREE composite team-level predictors instead of 54 binary mentions:
#        1 team mean sentiment      2 thematic diversity (Shannon H over subs)
#        3 challenge share (share of the team's responses in negative-toned themes)
#      Continuous predictors carry far more information per team than a 0/1
#      mention, so 3 tests here have more power than 54 tests there.
#      Test = Mann-Whitney U (no distributional assumption) + Cliff's delta
#      (effect size) + bootstrap 95% CI, BH-FDR over the three.
#   B. COCHRAN-ARMITAGE TREND across ORDERED medal tiers (None<Bronze<Silver<Gold)
#      at TEAM level - one ordered test per theme at the claim level, strictly
#      than unordered tier-vs-tier comparisons. BH-FDR over the themes tested.
#   C. PRECISION / MINIMUM DETECTABLE EFFECT: with this many teams, what is the
#      smallest difference detectable at 80% power? A null is only interpretable
#      next to this number - it converts "no effect" into a bounded claim.
# Everything is computed from pooled_master. Nothing is imputed or hardcoded;
# unusable groups are reported and skipped.
# Outputs: chart_power_prize.png, composite_prize_tests.csv, trend_tests.csv
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt, textwrap
from scipy.stats import mannwhitneyu, norm

def _w2q(t, w=30):
    return "\n".join(textwrap.wrap(str(t), w)[:2])

def _bhq(pv):
    pv = np.asarray(pv, float); q = np.full(pv.shape, np.nan)
    ok = np.where(np.isfinite(pv))[0]
    if ok.size:
        p = pv[ok]; n = p.size; o = np.argsort(p)
        r = p[o]*n/np.arange(1, n+1)
        r = np.minimum.accumulate(r[::-1])[::-1].clip(max=1.0)
        qq = np.empty(n); qq[o] = r; q[ok] = qq
    return q

def _cliffs_delta(a, b):
    """Nonparametric effect size: P(a>b) - P(a<b). |d| 0.15 small, 0.33 medium, 0.47 large."""
    a = np.asarray(a, float); b = np.asarray(b, float)
    if len(a) == 0 or len(b) == 0: return np.nan
    gt = sum((a[:, None] > b[None, :]).sum(axis=1))
    lt = sum((a[:, None] < b[None, :]).sum(axis=1))
    return float((gt - lt) / (len(a) * len(b)))

def _ca_trend(n_i, r_i, t_i):
    """Cochran-Armitage trend test. n_i = group sizes, r_i = responders,
    t_i = ordered scores. Returns (z, two-sided p)."""
    n_i = np.asarray(n_i, float); r_i = np.asarray(r_i, float); t_i = np.asarray(t_i, float)
    N = n_i.sum(); R = r_i.sum()
    if N == 0 or R == 0 or R == N: return (np.nan, np.nan)
    p = R / N
    T = float((t_i * r_i).sum() - p * (t_i * n_i).sum())
    V = float(p * (1 - p) * ((n_i * t_i**2).sum() - ((n_i * t_i).sum())**2 / N))
    if V <= 0: return (np.nan, np.nan)
    z = T / np.sqrt(V)
    return (float(z), float(2 * norm.sf(abs(z))))

def _mde(n1, n2, p_base, power=0.80, alpha=0.05):
    """Smallest difference in two proportions this design could detect, in
    PERCENTAGE POINTS.

    F2 - this notebook used to carry two different power calculations under this
    one name: a closed-form normal approximation here and an iterative search in
    the prize-precision cell. They disagreed by up to 2.1 percentage points, and
    one returned a proportion while the other returned percentage points. The
    iterative version is the more accurate of the two (it evaluates power at the
    actual p1/p2 rather than at the pooled baseline), so it is now the only one,
    and it returns percentage points in both places.
    """
    from scipy.stats import norm as _norm
    if min(n1, n2) < 2:
        return np.nan
    za = _norm.ppf(1 - alpha / 2)
    for d in np.arange(0.005, 0.80, 0.005):
        p1 = min(max(p_base + d / 2, 1e-6), 1 - 1e-6)
        p2 = min(max(p_base - d / 2, 1e-6), 1 - 1e-6)
        se = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
        if se <= 0:
            continue
        if _norm.cdf(abs(p1 - p2) / se - za) >= power:
            return float(100 * d)
    return np.nan

_PL = globals().get("_orig_pload", pload)
M = _PL("pooled_master")
NB_BOOT = 2000

if M is None:
    print("pooled_master missing - run NB2 Stage 33b first. Skipped.")
elif not {"Team", "medal", "sentiment"}.issubset(M.columns):
    print("pooled_master lacks Team / medal / sentiment - skipped.")
else:
    MK = M[M["medal"].astype(str) != "Unknown"].copy()
    if MK.empty:
        print("no award-matched rows - skipped.")
    else:
        sp = MK["special"].astype(bool) if "special" in MK.columns else False
        nmn = MK["nominated"].astype(bool) if "nominated" in MK.columns else False
        MK["_win"] = sp | nmn

        # The claim level, and a view of it with the unthemed rows removed.
        # Everything below composes team-level predictors out of theme columns,
        # and a column for id -1 would mean "these members said something that
        # could not be placed" - which is not a topic and must not enter a
        # breadth count, a tone composite or an ordered trend test.
        _LV = CLAIM_LEVEL
        _IDC, _NMC, _HASC = _cov_cols(_LV, MK)
        print(_cov_note(_LV, MK))
        MKv = MK[MK[_HASC].astype(bool)] if _HASC in MK.columns else MK[MK[_IDC] >= 0]
        assert (MKv[_IDC] >= 0).all(), "an unthemed row survived the filter"
        team_win = MK.groupby("Team")["_win"].max()
        rng = np.random.default_rng(CFG.seed)

        # ---------- A. three composite team-level predictors ------------------
        g = MK.groupby("Team")
        comp = pd.DataFrame(index=team_win.index)
        comp["sent_mean"] = g["sentiment"].mean()

        if _IDC in MK.columns:
            ct = pd.crosstab(MKv["Team"], MKv[_IDC])
            pr = ct.div(ct.sum(axis=1).replace(0, np.nan), axis=0)
            comp["diversity_H"] = -(pr * np.log(pr.where(pr > 0))).sum(axis=1)
        else:
            comp["diversity_H"] = np.nan

        if _IDC in MK.columns:
            tone = MKv.groupby(_IDC)["sentiment"].mean()
            neg = set(tone[tone < 0].index)                 # negative-toned themes
            MK["_chal"] = MK[_IDC].isin(neg) & MK[_HASC].astype(bool)
            comp["challenge_share"] = MK.groupby("Team")["_chal"].mean()
        else:
            comp["challenge_share"] = np.nan

        comp["prize"] = team_win.reindex(comp.index).fillna(False).astype(bool)
        LABEL = {"sent_mean": "Team mean sentiment",
                 "diversity_H": "Thematic diversity (Shannon H)",
                 "challenge_share": "Challenge share (negative-toned themes)"}
        rows = []
        for k in ("sent_mean", "diversity_H", "challenge_share"):
            v = comp[[k, "prize"]].dropna()
            a = v.loc[v.prize, k].values; b = v.loc[~v.prize, k].values
            if min(len(a), len(b)) < 8:
                print(f"  (skip {k}: only {len(a)}/{len(b)} teams per arm)"); continue
            try:
                _, pv = mannwhitneyu(a, b, alternative="two-sided")
            except Exception:
                pv = np.nan
            d = _cliffs_delta(a, b)
            bs = [_cliffs_delta(rng.choice(a, len(a)), rng.choice(b, len(b)))
                  for _ in range(300)]
            lo, hi = np.percentile(bs, [2.5, 97.5])
            rows.append(dict(predictor=k, label=LABEL[k], n_prize=len(a), n_noprize=len(b),
                             mean_prize=float(a.mean()), mean_noprize=float(b.mean()),
                             cliffs_delta=d, lo=float(lo), hi=float(hi), p=pv))
        CP = pd.DataFrame(rows)
        if len(CP):
            CP["q_fdr"] = _bhq(CP["p"].values)
            CP["sig_fdr"] = (CP["q_fdr"] < 0.05) & np.isfinite(CP["q_fdr"])
            CP.to_csv(f"{POOLED_DIR}/composite_prize_tests.csv", index=False)

        # ---------- B. Cochran-Armitage trend across ordered medal tiers ------
        TR = pd.DataFrame()
        if _IDC in MK.columns:
            ORDER = ["None", "Bronze", "Silver", "Gold"]
            tmedal = MK.drop_duplicates("Team").set_index("Team")["medal"].astype(str)
            tiers = [t for t in ORDER if (tmedal == t).sum() >= 5]
            if len(tiers) >= 3:
                tment = (MKv.groupby(["Team", _IDC]).size()
                            .unstack(fill_value=0) > 0)
                tment = tment[[c for c in tment.columns if int(c) >= 0]]
                names = {int(c): (MKv.loc[MKv[_IDC] == c, _NMC].iloc[0]
                                  if _NMC in MKv.columns else f"{_LV[0].upper()}{c}")
                         for c in tment.columns}
                trows = []
                for c in tment.columns:
                    n_i, r_i, t_i = [], [], []
                    for si, t in enumerate(tiers):
                        ids = tmedal[tmedal == t].index
                        mm = tment[c].reindex(ids).fillna(False)
                        n_i.append(len(ids)); r_i.append(int(mm.sum())); t_i.append(si)
                    if sum(r_i) < 10: continue
                    z, pv = _ca_trend(n_i, r_i, t_i)
                    if not np.isfinite(pv): continue
                    trows.append(dict(theme_id=int(c), name=str(names[int(c)]),
                                      z=z, p=pv, n_voicing=sum(r_i),
                                      rates=" / ".join(f"{100*r/n:.0f}%" for r, n in zip(r_i, n_i))))
                TR = pd.DataFrame(trows)
                if len(TR):
                    TR["q_fdr"] = _bhq(TR["p"].values)
                    TR["sig_fdr"] = (TR["q_fdr"] < 0.05) & np.isfinite(TR["q_fdr"])
                    TR = TR.sort_values("p").reset_index(drop=True)
                    TR.to_csv(f"{POOLED_DIR}/trend_tests.csv", index=False)
            else:
                print(f"  (trend test needs >=3 medal tiers with >=5 teams; found {len(tiers)})")

        # ---------- C. precision / minimum detectable effect ------------------
        n1 = int(comp.prize.sum()); n0 = int((~comp.prize).sum())
        base = float(comp.prize.mean())
        mde = _mde(n1, n0, 0.5)

        # ---------- figure: composites + trend + precision --------------------
        ncol = 2 if len(TR) else 1
        fig, axes = plt.subplots(1, ncol, figsize=(11.0 if ncol == 2 else 7.0, 4.8),
                                 squeeze=False)
        ax = axes[0][0]
        if len(CP):
            yv = np.arange(len(CP))
            for y, (_, r) in zip(yv, CP.iterrows()):
                col = SENT_POS if r["cliffs_delta"] > 0 else SENT_NEG
                ax.barh(y, r["cliffs_delta"], height=.55, color=col, edgecolor="white",
                        zorder=3, alpha=1.0 if r["sig_fdr"] else .45)
                ax.errorbar(r["cliffs_delta"], y,
                            xerr=[[max(0.0, r["cliffs_delta"]-r["lo"])],
                                  [max(0.0, r["hi"]-r["cliffs_delta"])]],
                            fmt="none", ecolor="#3A4650", elinewidth=1, capsize=3, zorder=4)
                ax.text(r["hi"] + .02, y, f"q={r['q_fdr']:.3f}" + ("  *" if r["sig_fdr"] else ""),
                        va="center", fontsize=6.8,
                        color=INK if r["sig_fdr"] else "#8A9098")
            for xt, lab in ((0.15, "small"), (0.33, "medium"), (0.47, "large")):
                for sgn in (1, -1):
                    ax.axvline(sgn*xt, color="#D8DCE0", lw=.8, ls=":", zorder=1)
            ax.axvline(0, color=ZERO, lw=1.2, zorder=2)
            ax.set_yticks(yv)
            ax.set_yticklabels([_w2q(r["label"], 26) for _, r in CP.iterrows()], fontsize=7)
            ax.set_xlim(-1.05, 1.05)
        ax.tick_params(labelsize=6.8, length=0)
        for sp_ in ("top", "right", "left"): ax.spines[sp_].set_visible(False)
        ax.xaxis.grid(True, color=GRID); ax.set_axisbelow(True)
        ax.set_title("A · Composite team predictors vs prize", fontsize=9,
                     fontweight="bold", loc="left", pad=6)
        ax.set_xlabel("Cliff's delta  (prize vs no-prize teams)", fontsize=7.5)

        if len(TR):
            ax2 = axes[0][1]
            S = TR.head(12).iloc[::-1]
            yv = np.arange(len(S))
            for y, (_, r) in zip(yv, S.iterrows()):
                col = SENT_POS if r["z"] > 0 else SENT_NEG
                ax2.barh(y, r["z"], height=.6, color=col, edgecolor="white", zorder=3,
                         alpha=1.0 if r["sig_fdr"] else .45)
                ax2.text(r["z"] + (.06 if r["z"] > 0 else -.06), y,
                         f"q={r['q_fdr']:.2f}" + ("  *" if r["sig_fdr"] else ""),
                         va="center", ha="left" if r["z"] > 0 else "right", fontsize=6.2,
                         color=INK if r["sig_fdr"] else "#8A9098")
            for zc in (-1.96, 1.96):
                ax2.axvline(zc, color="#D8DCE0", lw=.9, ls=":", zorder=1)
            ax2.axvline(0, color=ZERO, lw=1.2, zorder=2)
            # widen symmetrically so the q-labels never run into the theme names
            _zm = float(np.nanmax(np.abs(S["z"].values))) if len(S) else 2.0
            ax2.set_xlim(-_zm * 1.55, _zm * 1.55)
            ax2.set_yticks(yv)
            ax2.set_yticklabels([f"{_LV[0].upper()}{r['theme_id']} " + _w2q(r["name"], 22)
                                 for _, r in S.iterrows()], fontsize=6.2)
            ax2.tick_params(labelsize=6.8, length=0)
            for sp_ in ("top", "right", "left"): ax2.spines[sp_].set_visible(False)
            ax2.xaxis.grid(True, color=GRID); ax2.set_axisbelow(True)
            ax2.set_title(f"B · Ordered medal trend (Cochran-Armitage, {len(TR)} themes)",
                          fontsize=9, fontweight="bold", loc="left", pad=6)
            ax2.set_xlabel("trend z  (dotted = ±1.96)", fontsize=7.5)

        nsig = (int(CP["sig_fdr"].sum()) if len(CP) else 0) + (int(TR["sig_fdr"].sum()) if len(TR) else 0)
        fig_title(fig, "High-power prize tests: composite predictors and ordered trend")
        fig.text(.5, .015,
                 f"A: Mann-Whitney U + Cliff's delta (bootstrap 95% CI), BH-FDR over {len(CP)} composite tests · "
                 f"B: Cochran-Armitage trend over ordered medal tiers, BH-FDR over {len(TR)} themes · "
                 f"{nsig} significant in total\n"
                 f"PRECISION: with {n1} prize and {n0} non-prize teams, the smallest detectable difference "
                 f"in a 50%-baseline rate is ~{mde:.0f} pp at 80% power — effects smaller than this "
                 f"cannot be resolved by this sample, whatever the test.",
                 ha="center", fontsize=6.6, color=FOOT)
        plt.tight_layout(rect=[0, .07, 1, .92])
        save_fig(fig, "chart_power_prize"); plt.close(fig)

        print("=" * 68)
        print(f"  HIGH-POWER PRIZE TESTS   teams: {n1} prize / {n0} no-prize")
        if len(CP):
            for _, r in CP.iterrows():
                print(f"    {r['label']:<40} delta={r['cliffs_delta']:+.3f} "
                      f"[{r['lo']:+.3f},{r['hi']:+.3f}]  q={r['q_fdr']:.3f}"
                      f"{'  SIGNIFICANT' if r['sig_fdr'] else ''}")
        if len(TR):
            print(f"    Cochran-Armitage trend: {len(TR)} themes, "
                  f"{int(TR['sig_fdr'].sum())} significant after FDR")
            for _, r in TR.head(3).iterrows():
                print(f"      {_LV[0].upper()}{r['theme_id']:<3} {str(r['name'])[:34]:<34} z={r['z']:+.2f} "
                      f"q={r['q_fdr']:.3f}  rates {r['rates']}")
        print(f"  PRECISION: minimum detectable difference ~{mde:.0f} pp at 80% power.")
        print( "    Report this next to any null: it bounds what the sample could have found.")
        print(f"  files: composite_prize_tests.csv · trend_tests.csv · chart_power_prize")
        print("=" * 68)


### Stage 35 · Thematic diversity and prize status

**Question.** Do teams that talk about more different things do better?

**How to read it.** Diversity is the Shannon entropy of a team's mention mix; richness is the count of distinct themes. Both are adjusted for team size, because a larger team mechanically covers more themes.

In [ ]:
# @title
# ==============================================================================
# STAGE 35  -  TEAM THEMATIC DIVERSITY AND PRIZE STATUS
#             section 6 of 9  -  Themes and competition outcome
# Diversity = Shannon entropy H of a team's SUB mention mix (primary level; K is
# read from the live hierarchy, never written into this comment);
# richness = number of distinct sub-themes. OLS prize(0/1/2) ~ H + team size,
# team-bootstrap 95% CI, Mann-Whitney robustness. Reads saved artifacts only.
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
_PL = globals().get("_orig_pload", pload)
P = _PL("pooled_hier_participant").copy(); award = _PL("pooled_award"); meta = _PL("pooled_meta").reset_index(drop=True)
N = len(meta)
special = np.asarray(award.get("special", np.zeros(N, bool)), bool)
nominated = np.asarray(award.get("nominated", np.zeros(N, bool)), bool)
prize_row = np.where(special, 2, np.where(nominated, 1, 0)).astype(float)
trow = meta["Team"].astype(str).str.strip().values
team_prize = pd.Series(prize_row, index=trow).groupby(level=0).max()
P["Team"] = P["Team"].astype(str).str.strip()
mcols = [c for c in P.columns if c.startswith("sub_ment_")]      # SUB-43 = primary level
tm = P.groupby("Team")[mcols].sum(); tsz = P.groupby("Team").size().astype(float)
D = pd.DataFrame(index=tm.index)
tot = tm.sum(axis=1).replace(0, np.nan); pmat = tm.div(tot, axis=0)
D["H"] = -(pmat * np.log(pmat.where(pmat > 0))).sum(axis=1)
D["richness"] = (tm > 0).sum(axis=1).astype(float); D["size"] = tsz
D["prize"] = D.index.map(team_prize); D = D.dropna(subset=["prize", "H"])
D["size_z"] = (D["size"] - D["size"].mean()) / (D["size"].std() + 1e-9)
T = len(D); rng = np.random.default_rng(CFG.seed)
X = np.column_stack([np.ones(T), D["H"].values, D["size_z"].values]); y = D["prize"].values
b_all, *_ = np.linalg.lstsq(X, y, rcond=None)
bs = []
for _ in range(2000):
    ix = rng.integers(0, T, T)
    try:
        bb, _r, rk, _s = np.linalg.lstsq(X[ix], y[ix], rcond=None)
        if rk == X.shape[1]: bs.append(bb)
    except Exception: pass
bs = np.array(bs); bH = float(b_all[1])
loH, hiH = (np.percentile(bs[:, 1], [2.5, 97.5]) if len(bs) >= 50 else (np.nan, np.nan))
g2 = D.loc[D["prize"] == 2, "H"].values; g0 = D.loc[D["prize"] == 0, "H"].values
mw = mannwhitneyu(g2, g0, alternative="two-sided") if (len(g2) >= 3 and len(g0) >= 3) else None
psave("pooled_team_diversity", D.reset_index())
try: D.reset_index().to_csv(f"{POOLED_DIR}/team_diversity.csv", index=False)
except Exception: pass
# ---- figure ----
# explicit size: save_fig no longer force-resizes (that was what
# squashed these figures), so each figure must declare its canvas.
fig = plt.figure(figsize=(11.5, 5.5))
axL = fig.add_axes([0.07, 0.13, 0.53, 0.66]); axR = fig.add_axes([0.68, 0.13, 0.28, 0.66])
axL.scatter(D["H"], D["prize"] + rng.normal(0, 0.06, T), s=42, c=D["size"], cmap="cividis", edgecolor="#667", lw=0.4, alpha=0.85)
gg = np.linspace(float(D["H"].min()), float(D["H"].max()), 100)
if len(bs):
    YB = bs[:, 0][None, :] + gg[:, None] * bs[:, 1][None, :]
    axL.fill_between(gg, np.percentile(YB, 2.5, axis=1), np.percentile(YB, 97.5, axis=1), color=SENT_POS, alpha=0.16, lw=0)
axL.plot(gg, b_all[0] + bH * gg, color=SENT_POS, lw=2.4)
axL.set_yticks([0, 1, 2]); axL.set_yticklabels(["No prize", "Nominated", "Special"])
axlabel(axL, xlabel="team thematic diversity  H",
        xhint="Shannon entropy of the team's sub-theme mention mix: higher = the team talks about a wider spread of themes")
_sg = " *" if (loH > 0 or hiH < 0) else ""
axL.set_title(f"Diversity vs prize   [beta={bH:+.3f}, 95% CI {loH:+.3f}..{hiH:+.3f}{_sg} · {T} teams · size-adjusted]", loc="left", fontsize=12, fontweight="bold")
for k, col in [(0, "#AEB3B9"), (1, "#7FC9BE"), (2, SENT_POS)]:
    v = D.loc[D["prize"] == k, "H"].values
    axR.scatter(np.full(len(v), k) + rng.normal(0, 0.05, len(v)), v, s=24, color=col, alpha=0.75, edgecolor="#667", lw=0.3)
    if len(v) >= 2:
        m = v.mean(); se = v.std(ddof=1) / np.sqrt(len(v)); axR.errorbar(k, m, yerr=1.96 * se, fmt="o", color=INK, capsize=5, ms=7, zorder=5)
axR.set_xticks([0, 1, 2]); axR.set_xticklabels(["None", "Nom.", "Special"])
axlabel(axR, xlabel="prize group", xhint="teams grouped by best outcome",
        ylabel="diversity H (nats)")
_mw = f"Mann-Whitney p={mw.pvalue:.3f}" if mw is not None else "mean ± 95% CI"
axR.set_title(f"H by prize group   [{_mw}]", loc="left", fontsize=11)
fig.suptitle("Team thematic diversity and prize status", x=0.5, y=0.965, ha="center", fontsize=TS["title"] - 1, fontweight="bold", color=INK)
fig.text(0.995, 0.02, "Refs: Shannon (1948) entropy · Efron (1979) bootstrap · Mann & Whitney (1947)", ha="right", va="bottom", fontsize=8, color=FOOT)
save_fig(fig, "team_diversity_prize"); plt.close(fig)
print("=" * 64)
print(f"  Stage 43  TEAM DIVERSITY (sub-{int(np.asarray(_PL('pooled_labels_sub')).max())+1}) -> PRIZE  ({T} teams)   beta_H={bH:+.4f}  CI [{loH:+.4f}, {hiH:+.4f}]  {'SIG' if (loH>0 or hiH<0) else 'ns'}")
print("=" * 64)

### Stage 36 · Theme pairs and prize status

**Question.** Do combinations of themes carry information the single themes do not?

**How to read it.** The interaction coefficient is the synergy. A pair must clear a joint-mention floor to be tested; if none does, the printout gives the support numbers, because *no pair could be tested* and *combinations do not matter* are different statements.

In [ ]:
# @title
# ==============================================================================
# STAGE 36  -  THEME PAIRS AND PRIZE STATUS
#             section 6 of 9  -  Themes and competition outcome
# ==============================================================================
# ==============================================================================
# THEME-PAIR SYNERGY AND COMPETITION OUTCOME   (participant level)
# ------------------------------------------------------------------------------
# Do theme COMBINATIONS carry information that the single themes do not? For each
# qualifying pair, OLS  prize (0/1/2) ~ A + B + A*B ; the interaction coefficient
# is the synergy. Intervals come from a team-clustered bootstrap, because
# participants on one team share language and resampling them independently would
# understate the spread. BH-FDR across the tested pairs.
#
# REBUILT. It ran on the mid level and read pooled_labels_mid and mid_ment_*
# columns, none of which exist. Pairs are now formed at the CLAIM level, because
# an interaction coefficient with a p-value is an inferential claim and those are
# made at one level only.
#
# The support gates are the honest constraint here, and they are reported whether
# or not anything survives them: a pair needs MIN_EACH mentioners of each theme
# and MIN_BOTH participants who voiced BOTH. If nothing qualifies, that is a
# statement about co-mention support in this corpus, not evidence that
# combinations do not matter, and the printout says so with the numbers.
#
# Refs: Cameron, Gelbach & Miller (2008) cluster bootstrap; Benjamini & Hochberg
# (1995) FDR; NB2 Stage 26B LLM sentiment.
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import norm as _norm
from statsmodels.stats.multitest import multipletests

MIN_EACH, MIN_BOTH, MAX_PAIRS, NB = 30, 20, 120, 800
LV = CLAIM_LEVEL

R = pload("pooled_theme_synergy")
if R is None:
    rng = np.random.default_rng(CFG.seed)
    P = pload("pooled_hier_participant")
    if P is None:
        raise RuntimeError("pooled_hier_participant missing - run the participant "
                           "matrix cell (it sits right after the preflight).")
    P = P.copy()
    P["Team"] = P["Team"].astype(str).str.strip()

    # team outcome from the canonical table; unknown-award teams are excluded
    _md = MASTER.get("medal", pd.Series("Unknown", index=MASTER.index)).astype(str)
    _sp = MASTER.get("special", pd.Series(False, index=MASTER.index)).astype(bool)
    _nm = MASTER.get("nominated", pd.Series(False, index=MASTER.index)).astype(bool)
    prize_row = np.where(_sp, 2, np.where(_nm, 1, 0)).astype(float)
    _team = MASTER["Team"].astype(str).str.strip().values
    team_prize = (pd.Series(prize_row[(_md != "Unknown").values],
                            index=_team[(_md != "Unknown").values])
                  .groupby(level=0).max())

    P["__p"] = P["Team"].map(team_prize).astype(float)
    P = P.dropna(subset=["__p"]).reset_index(drop=True)
    y = P["__p"].values
    tarr = P["Team"].values
    ut = np.unique(tarr)
    by = {t: np.where(tarr == t)[0] for t in ut}
    BIX = [np.concatenate([by[ut[j]] for j in rng.integers(0, len(ut), len(ut))])
           for _ in range(NB)]

    # theme-level mean sentiment, from themed rows only
    _IDW, _NMW, _HCW = _cov_cols(LV, MASTER)
    print(_cov_note(LV, MASTER))
    _Mv = MASTER[MASTER[_HCW].astype(bool)]
    ts = (_Mv.groupby(_IDW)["sentiment"]
          .apply(lambda x: pd.to_numeric(x, errors="coerce").mean()).to_dict())

    cols = {c: f"{LV}_ment_{c}" for c in range(K[LV]) if f"{LV}_ment_{c}" in P.columns}
    keep = [c for c, cn in cols.items() if P[cn].sum() >= MIN_EACH]
    cand = []
    for i, a in enumerate(keep):
        xa = P[cols[a]].values
        for b in keep[i + 1:]:
            bo = int((xa * P[cols[b]].values).sum())
            if bo >= MIN_BOTH:
                cand.append((bo, a, b))
    _n_all_pairs = len(keep) * (len(keep) - 1) // 2
    cand.sort(reverse=True)
    cand = cand[:MAX_PAIRS]
    ONES = np.ones(len(P))
    rows = []
    for bo, a, b in cand:
        x1 = P[cols[a]].astype(float).values
        x2 = P[cols[b]].astype(float).values
        x12 = x1 * x2
        if np.unique(x12).size < 2:
            continue
        Xf = np.column_stack([ONES, x1, x2, x12])
        bf, _r, rk, _s = np.linalg.lstsq(Xf, y, rcond=None)
        if rk < 4:
            continue
        bb = []
        for ix in BIX:
            try:
                bt, _r2, rk2, _s2 = np.linalg.lstsq(Xf[ix], y[ix], rcond=None)
                if rk2 == 4:
                    bb.append(bt[3])
            except Exception:
                pass
        if len(bb) < NB * 0.5:
            continue
        bb = np.array(bb)
        lo, hi = np.percentile(bb, [2.5, 97.5])
        se = float(bb.std(ddof=1))
        p = float(2 * _norm.sf(abs(bf[3]) / se)) if se > 0 else np.nan
        rows.append(dict(A=int(a), B=int(b),
                         name=f"{theme_name(LV, a)} x {theme_name(LV, b)}",
                         n_both=bo, beta_syn=float(bf[3]), lo=float(lo), hi=float(hi),
                         se=se, p=p,
                         sent=float(np.nanmean([ts.get(a, np.nan), ts.get(b, np.nan)]))))
    R = pd.DataFrame(rows)
    R.attrs["n_themes_supported"] = len(keep)
    R.attrs["n_pairs_possible"] = _n_all_pairs
    R.attrs["n_teams"] = int(P["Team"].nunique())
    if len(R):
        mk = R["p"].notna(); R["q"] = np.nan
        if mk.any():
            R.loc[mk, "q"] = multipletests(R.loc[mk, "p"].values, method="fdr_bh")[1]
        R["sig"] = fdr_flags(R["beta_syn"].values, R["lo"].values, R["hi"].values)
        R = R.sort_values("beta_syn", ascending=False).reset_index(drop=True)
    psave("pooled_theme_synergy", R)

print("=" * 72)
print("  THEME-PAIR SYNERGY AND COMPETITION OUTCOME")
print("=" * 72)
_sup = R.attrs.get("n_themes_supported")
_poss = R.attrs.get("n_pairs_possible")
if R is None or len(R) == 0:
    print(f"  No theme pair met the co-mention support gates at the {LV} level.")
    if _sup is not None:
        print(f"  {_sup} of {K[LV]} themes have at least {MIN_EACH} mentioners, giving "
              f"{_poss} possible pairs,")
        print(f"  and none of them was voiced together by {MIN_BOTH} or more participants.")
    print("  That is a statement about co-mention support in this corpus, not evidence")
    print("  that theme combinations carry no signal. Reported rather than relaxed.")
else:
    nsig = int(R["sig"].sum())
    nq = int((R["q"] < 0.05).sum()) if R["q"].notna().any() else 0
    fig = forest(
        R, beta="beta_syn", lo="lo", hi="hi", name="name", sent="sent", sig="sig",
        name_n=56,
        title=("Participant level: do theme combinations carry information beyond the "
               "single themes?"),
        sub_lines=[
            f"OLS  prize (0/1/2) ~ A + B + A x B  over {LV}-level pairs  |  "
            f"{len(R)} pairs tested · {nsig} interval excludes 0 · {nq} FDR q<0.05",
            "bar = 95% interval (team-clustered bootstrap) · white tick = interaction "
            "coefficient · * = BH-FDR q<0.05 · colour = pair mean sentiment"],
        footer=("Refs: Cameron, Gelbach & Miller (2008) cluster bootstrap · "
                "Benjamini & Hochberg (1995) FDR · NB2 Stage 26B LLM sentiment"),
        xlabel="Interaction coefficient (synergy of voicing BOTH themes)")
    save_fig(fig, f"theme_synergy_forest_{LV}"); plt.close(fig)
    print(f"  {len(R)} pair(s) tested from {_sup} sufficiently-voiced themes")
    print(f"  {nsig} with an interval excluding zero · {nq} surviving BH-FDR at q<0.05")
    if nq == 0:
        print("  No combination survives multiple-comparison control. Stated as a null,")
        print("  not omitted.")
print("=" * 72)

try:
    R.to_csv(f"{POOLED_DIR}/theme_synergy_{LV}.csv", index=False)
except Exception as _e:
    print(f"  (could not write theme_synergy_{LV}.csv: {type(_e).__name__})")


### Stage 37 · Predictive resolution

**Question.** Does the finer theme set predict competition outcome better than the coarse one?

**How to read it.** Out-of-fold AUC with a team-bootstrap interval at each level. With two levels this is a two-point comparison rather than a ladder. The verdict in the title is derived from the numbers at render time.

In [ ]:
# @title
# ==============================================================================
# STAGE 37  -  PREDICTIVE RESOLUTION
#             section 6 of 9  -  Themes and competition outcome
# ==============================================================================
# ==============================================================================
# PREDICTIVE RESOLUTION: DOES A FINER THEME SET CARRY MORE PRIZE SIGNAL?
# ------------------------------------------------------------------------------
# A team is described by its mention-rate profile - for each theme, the share of
# that team's participants who voiced it - and the profile is used to predict the
# team's competition outcome. The question is whether resolution buys anything: a
# finer partition gives the model more columns, and if the signal is real the
# finer level should carry it better than the coarse one. Out-of-fold AUC with a
# team-bootstrap 95% CI, so the comparison is honest about sampling error.
#
# REBUILT. The previous version read pooled_hier_participant, an artifact of the
# deleted three-level stage that Notebook 2 no longer writes, so the computation
# branch could not run at all; and it indexed prof["mid"], which no longer exists.
# Profiles are now built from pooled_master, the one canonical table.
#
# Unthemed responses are excluded from the mention counts but every participant
# stays in the team denominator - a member whose answers could not be clustered
# is still a member of the team, and removing them would inflate every rate.
#
# WHAT THIS FIGURE CAN AND CANNOT SHOW. With two levels it is a two-point
# comparison, not a ladder, and it is reported as such. The verdict in the title
# is derived from the numbers at render time; it is never a fixed string.
# Refs: Hanley & McNeil (1982) AUC; Efron (1979) bootstrap.
# ==============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

_PL = globals().get("_orig_pload", pload)

# WHY THIS READ CHANGED.  This cell used to do:
#     G = _PL("pooled_granularity_auc");  if G is None or not len(G): recompute
# `_PL` is the UNWRAPPED pload, so it stepped around the stale-artifact guard on
# purpose, and the test only asked "is it empty", never "was it computed against
# THIS hierarchy". Both halves failed on the last run: the object on disk had
# been written when the hierarchy still had a mid level, so the figure printed
#     Main (7)  ·  Mid (32)  ·  Sub (43)  ·  0 teams with a known outcome
# on a notebook whose live hierarchy is main-7 / sub-39 and whose other cells
# report 120-125 teams. The "0 teams" was not a separate bug - it was the
# n_teams column of that stale row.
#
# pload_result (cell 12, INFRA-1) returns the object ONLY if its stamp matches
# the live hierarchy fingerprint, wherever it was restored from - including the
# Drive mirror, which the rename-based guard cannot reach on a fresh runtime.
G = pload_result("pooled_granularity_auc")

if G is None or not len(G):
    Mg = MASTER.copy()
    if "Team" not in Mg.columns:
        raise RuntimeError("pooled_master has no Team column - cannot build team profiles.")
    Mg["Team"] = Mg["Team"].astype(str).str.strip()

    # team outcome, from the canonical table
    _sp = Mg.get("special", pd.Series(False, index=Mg.index)).astype(bool)
    _nm = Mg.get("nominated", pd.Series(False, index=Mg.index)).astype(bool)
    prize_row = np.where(_sp, 2, np.where(_nm, 1, 0)).astype(float)
    known = (Mg.get("medal", pd.Series("Unknown", index=Mg.index)).astype(str) != "Unknown")
    team_prize = (pd.Series(prize_row, index=Mg["Team"].values)[known.values]
                  .groupby(level=0).max())

    # team x theme mention-rate profile, per level
    team_size = Mg.groupby("Team")["Participant"].nunique()
    prof = {}
    for lv in LEVELS:
        idc, hasc = f"{lv}_id", f"has_{lv}"
        if idc not in Mg.columns:
            continue
        Mv = Mg[Mg[hasc].astype(bool)] if hasc in Mg.columns else Mg[Mg[idc] >= 0]
        pp = (Mv.groupby(["Participant", idc]).size().unstack(fill_value=0) > 0).astype(int)
        p2t = Mv.drop_duplicates("Participant").set_index("Participant")["Team"]
        pp["_t"] = p2t.reindex(pp.index).values
        cnt = pp.groupby("_t").sum()
        prof[lv] = cnt.div(team_size.reindex(cnt.index), axis=0).fillna(0.0)
    if not prof:
        raise RuntimeError("no level produced a team profile.")

    idx = sorted(set.intersection(*(set(v.index) for v in prof.values()))
                 & set(team_prize.index))
    if len(idx) < 20:
        raise RuntimeError(f"only {len(idx)} teams have both a profile and a known "
                           f"outcome - too few to cross-validate.")
    y_ord = team_prize.reindex(idx).values.astype(float)
    y2 = (y_ord == 2).astype(int)
    y_bin, OUT = ((y2, "special prize vs rest") if y2.sum() >= 8
                  else ((y_ord > 0).astype(int), "any prize vs none"))
    npos, nneg = int(y_bin.sum()), int(len(y_bin) - y_bin.sum())
    if min(npos, nneg) < 2:
        raise RuntimeError(f"outcome is almost constant ({npos} positive of "
                           f"{len(y_bin)}) - nothing to predict.")
    nsp = max(2, min(5, npos, nneg))
    skf = StratifiedKFold(n_splits=nsp, shuffle=True, random_state=CFG.seed)
    rng = np.random.default_rng(CFG.seed)

    res = []
    for lv in LEVELS:
        if lv not in prof:
            continue
        Xk = prof[lv].reindex(idx).fillna(0.0).values
        oof = np.zeros(len(y_bin))
        for tr, te in skf.split(Xk, y_bin):
            clf = LogisticRegression(max_iter=5000, C=1.0)
            clf.fit(Xk[tr], y_bin[tr])
            oof[te] = clf.predict_proba(Xk[te])[:, 1]
        auc = float(roc_auc_score(y_bin, oof))
        aa = []
        for _ in range(2000):
            ix = rng.integers(0, len(y_bin), len(y_bin))
            if y_bin[ix].min() != y_bin[ix].max():
                aa.append(roc_auc_score(y_bin[ix], oof[ix]))
        lo, hi = np.percentile(aa, [2.5, 97.5]) if aa else (np.nan, np.nan)
        sp = spearmanr(oof, y_ord)
        rho = float(getattr(sp, "statistic", getattr(sp, "correlation", np.nan)))
        res.append(dict(level=level_label(lv), key=lv, K=Xk.shape[1], auc=auc,
                        lo=float(lo), hi=float(hi), rho=rho,
                        n_teams=len(idx), outcome=OUT, n_positive=npos))
    G = pd.DataFrame(res)
    psave_result("pooled_granularity_auc", G)

# ------------------------------------------------------------------- figure
fig = plt.figure(figsize=(9.6, 5.6))
ax = fig.add_axes([0.10, 0.16, 0.84, 0.60])
for i, (_, r) in enumerate(G.iterrows()):
    ax.errorbar(i, r["auc"], yerr=[[max(0.0, r["auc"] - r["lo"])], [max(0.0, r["hi"] - r["auc"])]],
                fmt="o", ms=12, color=SENT_POS, ecolor="#333", elinewidth=2,
                capsize=7, zorder=4)
    ax.text(i, r["hi"] + 0.018, f"AUC = {r['auc']:.3f}", ha="center", fontsize=11.5,
            fontweight="bold")
    ax.text(i, r["lo"] - 0.032, f"rho = {r['rho']:+.2f}", ha="center", fontsize=9.5,
            color=MUT)
ax.axhline(0.5, color="#999", lw=1.4, ls="--")
ax.text(len(G) - 0.5, 0.505, "chance (0.5)", fontsize=9, color="#888",
        ha="right", va="bottom")
ax.set_xticks(range(len(G)))
ax.set_xticklabels([f"{r.level}\n({int(r.K)} themes)" for r in G.itertuples()],
                   fontsize=11.5)
ax.set_xlim(-0.6, len(G) - 0.4)
ax.set_ylim(min(0.42, float(G["lo"].min()) - 0.05),
            min(1.0, float(G["hi"].max()) + 0.08))
ax.set_ylabel("Out-of-fold AUC")

# The verdict is derived from the table at render time. A fixed string would state
# a conclusion the figure might contradict if the data changed.
_beats = G[G["lo"] > 0.5]
if len(_beats) == 0:
    _verdict = "thematic composition does not predict competition outcome at either resolution"
    _sub_v = "every confidence interval includes chance (0.5)"
else:
    _best = _beats.sort_values("auc", ascending=False).iloc[0]
    _verdict = (f"{_best['level']} composition predicts outcome above chance "
                f"(AUC = {_best['auc']:.2f})")
    _sub_v = (f"{len(_beats)} of {len(G)} level(s) have a 95% interval entirely above "
              "chance: " + ", ".join(f"{r['level']} AUC {r['auc']:.2f}"
                                     for _, r in _beats.iterrows()))
_o = str(G["outcome"].iloc[0]) if "outcome" in G.columns else "prize vs none"
_nt = int(G["n_teams"].iloc[0]) if "n_teams" in G.columns else 0
fig_title(fig, f"Predictive resolution — {_verdict}")
fig.text(0.5, 0.885,
         f"{_o} ~ team mention-rate profile · L2 logistic regression · stratified "
         f"cross-validation · {_nt} teams · error bar = team-bootstrap 95% interval · {_sub_v}",
         ha="center", fontsize=9.2, color=MUT, wrap=True)
fig.text(0.995, 0.02, "Refs: Hanley & McNeil (1982) AUC · Efron (1979) bootstrap",
         ha="right", va="bottom", fontsize=8, color=FOOT)
save_fig(fig, "granularity_auc_ladder"); plt.close(fig)

print("=" * 70)
print("  PREDICTIVE RESOLUTION")
print("=" * 70)
for _, r in G.iterrows():
    print(f"    {str(r['level']):<12} ({int(r['K']):>3} themes): AUC = {r['auc']:.3f}   "
          f"95% CI [{r['lo']:.3f}, {r['hi']:.3f}]   rho = {r['rho']:+.2f}")
print(f"  outcome: {_o} · {_nt} teams with a known outcome")
print(f"  reading: {_verdict}")
print("  With two levels this is a two-point comparison, not a ladder; it says")
print("  whether the finer partition carries more signal than the coarse one.")
print("=" * 70)
try:
    G.to_csv(f"{POOLED_DIR}/granularity_auc.csv", index=False)
except Exception as _e:
    print(f"  (could not write granularity_auc.csv: {type(_e).__name__})")

### Stage 38 · Secondary check on the same relationship

**How to read it.** The same question at a different aggregation, as a robustness check rather than a new finding.

In [ ]:
# @title
# ==============================================================================
# STAGE 38  -  PRIZE SECONDARY CHECK: THE SAME MACRO PROFILE
#             section 6 of 9  -  Themes and competition outcome
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import statsmodels.api as sm
from statsmodels.stats.multitest import multipletests
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler

# The preflight rebinds _PL() to follow the EXPLORER's detail level. The
# statistical stages must always read the true artifacts, so they use the
# unpatched loader like every other analysis cell in this notebook.
_PL = globals().get("_orig_pload", pload)

part     = _PL("pooled_hier_participant")
award    = _PL("pooled_award") or {}
meta     = _PL("pooled_meta").reset_index(drop=True)
main_llm = _PL("pooled_main_llm") or {}
ml_res   = _PL("pooled_multilevel")
if part is None or ml_res is None:
    raise RuntimeError("Run Stage 52 (multilevel model) first.")

def _mname(g): return (main_llm.get(g) or main_llm.get(str(g)) or {}).get("name") or f"m{g}"

# (font clobbering removed: Stage 0 resolves the report font ONCE. This cell
#  re-picked a font from a fallback list, overriding it mid-notebook, so
#  figures before and after this point rendered in DIFFERENT typefaces.)

print("=" * 64)
print("  STAGE 53  PRIZE SECONDARY CHECK  (team-level logistic, macro themes)")
print("=" * 64)

P = part.reset_index()
P["Team"] = P["Team"].astype(str).str.strip()
main_cols = sorted([c for c in P.columns if c.startswith("main_ment_")], key=lambda c: int(c.split("_")[-1]))
macro_ids = [int(c.split("_")[-1]) for c in main_cols]

team_macro = P.groupby("Team")[main_cols].mean()

meta["Team"] = meta["Team"].astype(str).str.strip()
special = np.asarray(award.get("special", np.zeros(len(meta), bool)), bool)
nominated = np.asarray(award.get("nominated", np.zeros(len(meta), bool)), bool)
prize_row = np.where(special, 2, np.where(nominated, 1, 0)).astype(float)
team_prize = pd.Series(prize_row, index=meta["Team"]).groupby(level=0).max()
# An unmatched team has an UNKNOWN award, not a lost one. fillna(0) would
# score it as 'no prize' and silently deflate every prize rate, so any
# unmatched team is reported and dropped instead. Today this is a no-op:
# all 125 teams match the registry - the guard exists so it cannot become
# a silent error on a future run.
_pr = team_prize.reindex(team_macro.index)
if _pr.isna().any():
    print(f"  !! {int(_pr.isna().sum())} team(s) have no award-registry "
          f"match; they are EXCLUDED from the prize comparison, not "
          f"counted as no-prize: {list(_pr.index[_pr.isna()])[:8]}")
team_won = (_pr.fillna(0) > 0).astype(int)

D = team_macro.copy()
D["won"] = team_won.values
print(f"  teams: {len(D)}  ·  won a prize: {int(D['won'].sum())}  ·  base rate: {D['won'].mean():.0%}")

X = sm.add_constant(D[main_cols])
try:
    logit_res = sm.Logit(D["won"], X).fit(disp=0)
    rowsA = []
    for cname, mid in zip(main_cols, macro_ids):
        b = logit_res.params.get(cname, np.nan); se = logit_res.bse.get(cname, np.nan); p = logit_res.pvalues.get(cname, np.nan)
        rowsA.append(dict(macro=mid, name=_mname(mid), beta=b, se=se, p=p, lo=b-1.96*se, hi=b+1.96*se))
    RA = pd.DataFrame(rowsA)
    # ------------------------------------------------------------------------
    # MULTIPLICITY. This line used to read  RA["sig"] = RA["p"] < 0.05  - seven
    # coefficients tested simultaneously and no correction, in the only
    # inferential cell of this notebook that omitted one. Every sibling cell
    # (the findings engine, Stage 63, Stage 41b, Stage 52, the org panels)
    # applies BH-FDR, and the sentiment side of Panel B below is already
    # FDR-controlled by Stage 52 - so the two halves of that comparison were
    # being held to different standards.
    #
    # WHAT THIS COSTS, STATED PLAINLY. On this data the cell reported two
    # "significant" coefficients at raw p (0.032 and 0.041). With seven tests
    # the loosest BH threshold is 0.05 x 1/7 = 0.007, so neither survives. They
    # were the only positive prize findings in the notebook, and they were an
    # artefact of the missing correction rather than a result. The raw p is
    # kept in the table beside q, so nothing is hidden - only reclassified.
    # ------------------------------------------------------------------------
    RA["q"] = multipletests(RA["p"].fillna(1.0).values, method="fdr_bh")[1]
    RA["sig"] = (RA["q"] < 0.05) & RA["p"].notna()
    RA["sig_raw"] = RA["p"] < 0.05
    RA = RA.sort_values("beta", ascending=False).reset_index(drop=True)
    pseudo_r2 = logit_res.prsquared
    print(f"  McFadden pseudo-R2 = {pseudo_r2:.3f}")
    print(RA[["name", "beta", "p", "q", "sig"]].to_string(index=False))
    _n_raw, _n_fdr = int(RA["sig_raw"].sum()), int(RA["sig"].sum())
    print(f"  {len(RA)} coefficients tested together · {_n_raw} at raw p<0.05 · "
          f"{_n_fdr} surviving BH-FDR q<0.05")
    if _n_raw and not _n_fdr:
        print("  The raw-p hits do NOT survive correction. With 7 simultaneous tests")
        print("  that is the expected number of false positives, and this cell is the")
        print("  only one in the notebook that used to report them as findings.")
except Exception as e:
    print(f"  !! logistic model failed: {e}")
    RA = pd.DataFrame(columns=["macro", "name", "beta", "se", "p", "q", "lo", "hi",
                               "sig", "sig_raw"]); pseudo_r2 = np.nan

Xs = StandardScaler().fit_transform(D[main_cols].values)
y = D["won"].values
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=CFG.seed)
try:
    aucs = cross_val_score(LogisticRegression(max_iter=1000), Xs, y, cv=cv, scoring="roc_auc")
    print(f"  5-fold CV AUC = {aucs.mean():.3f} +/- {aucs.std():.3f}")
except Exception as e:
    aucs = np.array([np.nan]); print(f"  !! CV AUC failed: {e}")

psave("pooled_prize_secondary", dict(logit=RA, pseudo_r2=pseudo_r2, cv_auc=aucs,
                                     multiplicity="BH-FDR over the "
                                     f"{len(RA)} macro coefficients"))

RA_sent = ml_res["macro"][["macro", "name", "beta", "sig"]].rename(columns={"beta": "beta_sentiment", "sig": "sig_sentiment"})
RA_prize = RA[["macro", "beta", "sig"]].rename(columns={"beta": "beta_prize", "sig": "sig_prize"})
comp = RA_sent.merge(RA_prize, on="macro").sort_values("beta_sentiment", ascending=False).reset_index(drop=True)

# ==============================================================================
# FIGURE
# ==============================================================================
fig = plt.figure(figsize=(11.5, 6.4), facecolor="white")
gs = fig.add_gridspec(2, 2, height_ratios=[5.4, 0.34], width_ratios=[1.1, 1], hspace=0.55, wspace=0.55)
ax1 = fig.add_subplot(gs[0, 0]); ax2 = fig.add_subplot(gs[0, 1])
ax_ref = fig.add_subplot(gs[1, :]); ax_ref.axis("off")

yv = np.arange(len(RA))[::-1]
ax1.barh(yv, RA["beta"],
         xerr=[(RA["beta"] - RA["lo"]).clip(lower=0),
               (RA["hi"] - RA["beta"]).clip(lower=0)],
         color=["#1B9E8A" if s else "#AEB3B9" for s in RA["sig"]], height=0.55,
         edgecolor="white", ecolor="#555", capsize=3, zorder=2)
ax1.axvline(0, color="#999", lw=0.9, zorder=1)
for i, (_, r) in zip(yv, RA.iterrows()):
    if r["sig"]:
        ax1.text(r["hi"]+0.08 if r["beta"]>=0 else r["lo"]-0.08, i, "*", va="center",
                  ha="left" if r["beta"]>=0 else "right", fontsize=12, fontweight="bold", zorder=3)
ax1.set_yticks(yv); ax1.set_yticklabels(RA["name"], fontsize=9)
ax1.set_xlabel("effect on prize odds  (log-odds beta, 95% CI)", fontsize=9)
_lbl = ("teal = survives BH-FDR q<0.05"
        + (f"; {int(RA['sig_raw'].sum())} reach raw p<0.05 but not FDR"
           if "sig_raw" in RA.columns and int(RA["sig_raw"].sum()) else ""))
ax1.set_title(f"Panel A  ·  team-level logistic  ·  pseudo-R2={pseudo_r2:.2f}  ·  CV AUC={aucs.mean():.2f}",
              fontsize=11, fontweight="bold", loc="left", pad=10)
for sp in ("top", "right"): ax1.spines[sp].set_visible(False)

code = np.zeros((len(comp), 2))
labtxt = np.empty((len(comp), 2), dtype=object)
for i, (_, r) in enumerate(comp.iterrows()):
    code[i, 0] = (1 if r["beta_sentiment"] > 0 else -1) if r["sig_sentiment"] else 0
    code[i, 1] = (1 if r["beta_prize"] > 0 else -1) if r["sig_prize"] else 0
    labtxt[i, 0] = f"{r['beta_sentiment']:+.2f}" + ("*" if r["sig_sentiment"] else "")
    labtxt[i, 1] = f"{r['beta_prize']:+.2f}" + ("*" if r["sig_prize"] else "")

cmap_tri = ListedColormap(["#E08214", "#D9D9D9", "#1B9E8A"])
norm_tri = BoundaryNorm([-1.5, -0.5, 0.5, 1.5], cmap_tri.N)
ax2.imshow(code, cmap=cmap_tri, norm=norm_tri, aspect="auto")
for i in range(code.shape[0]):
    for j in range(code.shape[1]):
        ax2.text(j, i, labtxt[i, j], ha="center", va="center", fontsize=9, fontweight="bold",
                  color="white" if code[i, j] != 0 else "#333")
ax2.set_xticks([0, 1]); ax2.set_xticklabels(["personal\nsentiment\n(Stage 52)", "team\nprize\n(this stage)"], fontsize=9, fontweight="bold")
ax2.set_yticks(range(len(comp))); ax2.set_yticklabels(comp["name"], fontsize=9)
ax2.set_title("Panel B  ·  triangulation: same theme, two outcomes  (* = significant)", fontsize=11, fontweight="bold", loc="left", pad=10)
for sp in ax2.spines.values(): sp.set_visible(False)
ax2.tick_params(length=0)

ax_ref.text(0.5, 0.7, f"Panel A: logistic regression, won-any-prize ~ {len(main_cols)} macro "
            f"mention rates, one row per team (n={len(D)})  ·  {_lbl}",
            ha="center", va="center", fontsize=8.3, color="#444", transform=ax_ref.transAxes)
ax_ref.text(0.5, 0.18, "Panel B: colour = direction when significant (teal=positive, orange=negative, grey=not significant)  ·  numbers = actual beta in each model's own units (not directly comparable in scale)  ·  "
            "CV AUC via 5-fold stratified cross-validation.",
            ha="center", va="center", fontsize=7.4, style="italic", color="#888", transform=ax_ref.transAxes)

fig.suptitle("Does the same team profile predict prize as predicts personal sentiment?",
              x=0.01, ha="left", fontsize=16, fontweight="bold", color=NAT_TEXT, y=0.99)
fig.subplots_adjust(top=0.88, bottom=0.06, left=0.10, right=0.97)
save_fig(fig, "chart_pooled_prize_secondary"); plt.close(fig)
print("  Saved: chart_pooled_prize_secondary.png")
print("=" * 64)

### Stage 39 · Predictive model of competition outcome

**Question.** Taken together - themes, diversity, sentiment, team size - how much of the outcome can be predicted?

**How to read it.** Nested cross-validation, so the reported AUC is out-of-sample and the penalty is tuned inside each fold. The fold design is derived from the class sizes; if the outcome is too rare to support one, the cell says so instead of reporting a number with no meaning.

In [ ]:
# @title
# ==============================================================================
# STAGE 39  -  PREDICTIVE CAPSTONE MODEL OF COMPETITION OUTCOME
#             section 6 of 9  -  Themes and competition outcome
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as _tdist
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler

# The preflight rebinds _PL() to follow the EXPLORER's detail level. The
# statistical stages must always read the true artifacts, so they use the
# unpatched loader like every other analysis cell in this notebook.
_PL = globals().get("_orig_pload", pload)

part     = _PL("pooled_hier_participant")
org      = _PL("pooled_team_org")
award    = _PL("pooled_award") or {}
meta     = _PL("pooled_meta").reset_index(drop=True)
main_llm = _PL("pooled_main_llm") or {}
sub_llm  = _PL("pooled_sub_llm") or {}
if part is None or org is None:
    raise RuntimeError("Need pooled_hier_participant and pooled_team_org (Stage 46).")

def _mname(g): return (main_llm.get(g) or main_llm.get(str(g)) or {}).get("name") or f"m{g}"

# (font clobbering removed: Stage 0 resolves the report font ONCE. This cell
#  re-picked a font from a fallback list, overriding it mid-notebook, so
#  figures before and after this point rendered in DIFFERENT typefaces.)

print("=" * 64)
print("  STAGE 54  PREDICTIVE CAPSTONE MODEL")
print("=" * 64)

P = part.reset_index()
P["Team"] = P["Team"].astype(str).str.strip()
main_cols = sorted([c for c in P.columns if c.startswith("main_ment_")], key=lambda c: int(c.split("_")[-1]))
sub_cols  = sorted([c for c in P.columns if c.startswith("sub_ment_")],  key=lambda c: int(c.split("_")[-1]))
sub_ids   = [int(c.split("_")[-1]) for c in sub_cols]

team_macro = P.groupby("Team")[main_cols].mean()
team_macro.columns = [f"macro__{_mname(int(c.split('_')[-1]))}" for c in main_cols]
team_size = P.groupby("Team").size().rename("team_size")

# Thematic diversity is measured at the CLAIM level. It used to be measured on
# the mid level, which no longer exists; the claim level is the finest partition
# the analysis stands behind, so it is the one that carries the entropy.
_div_cols = sub_cols
_div_sum = P.groupby("Team")[_div_cols].sum()
_div_share = _div_sum.div(_div_sum.sum(axis=1).replace(0, np.nan), axis=0)
team_diversity = (-(_div_share * np.log(_div_share.replace(0, np.nan)))
                  .sum(axis=1)).fillna(0).rename("diversity_H")

_SENT_MAP = {"positive": 1.0, "pos": 1.0, "negative": -1.0, "neg": -1.0, "mixed": 0.0, "neutral": 0.0}
def _sent_score(c):
    rec = sub_llm.get(c) or sub_llm.get(str(c)) or {}
    return _SENT_MAP.get(str(rec.get("sentiment", "")).strip().lower(), 0.0)
sent_scores = {c: _sent_score(c) for c in sub_ids}
M = P[sub_cols].values.astype(float); S = np.array([sent_scores[c] for c in sub_ids])
n_ment = M.sum(axis=1)
with np.errstate(invalid="ignore", divide="ignore"):
    P["valence"] = np.where(n_ment > 0, (M * S).sum(axis=1) / n_ment, np.nan)
team_valence = P.groupby("Team")["valence"].mean().rename("valence_mean")

def _clean_cat(s, min_n=8):
    s = s.astype(str)
    vc = s.value_counts()
    keep = set(vc[vc >= min_n].index) - {"nan", "None"}
    return s.where(s.isin(keep), "Other")

org_feat = pd.DataFrame(index=org.index)
org_feat["Section"] = _clean_cat(org["Section"])
org_feat["Region"]  = _clean_cat(org["Region"])
org_feat["Track"]   = _clean_cat(org["Track"], min_n=10)

D = team_macro.join([team_size, team_diversity, team_valence], how="inner").join(org_feat, how="left")
D = pd.get_dummies(D, columns=["Section", "Region", "Track"], drop_first=True)

meta["Team"] = meta["Team"].astype(str).str.strip()
special = np.asarray(award.get("special", np.zeros(len(meta), bool)), bool)
nominated = np.asarray(award.get("nominated", np.zeros(len(meta), bool)), bool)
prize_row = np.where(special, 2, np.where(nominated, 1, 0)).astype(float)
team_prize = pd.Series(prize_row, index=meta["Team"]).groupby(level=0).max()
D["won"] = (team_prize.reindex(D.index).fillna(0) > 0).astype(int)
D = D.dropna()
print(f"  teams in model: {len(D)}  ·  features: {D.shape[1]-1}  ·  base prize rate: {D['won'].mean():.0%}")

feat_cols = [c for c in D.columns if c != "won"]
Xs = StandardScaler().fit_transform(D[feat_cols].values.astype(float))
y  = D["won"].values

# Nested cross-validation needs enough of the MINORITY class to survive being split
# twice: the outer fold takes a share of it, and the inner fold splits that share
# again to tune the penalty. With a rare outcome the inner fold can end up with a
# single class, sklearn raises "All the N fits failed", and the whole notebook
# stops on a figure. The fold counts are therefore derived from the data, and if
# even the smallest workable design is impossible the cell says so and stops
# rather than pretending it fitted something.
_minor = int(min(np.sum(y == 0), np.sum(y == 1)))
if _minor < 8:
    raise RuntimeError(
        f"the outcome has only {_minor} team(s) in its smaller class out of {len(y)}. "
        f"A nested cross-validated model cannot be fitted on that, and forcing it "
        f"would produce an AUC with no meaning. Report the descriptive prize figures "
        f"instead and state the class imbalance.")
N_SPLITS = max(3, min(5, _minor // 3))
N_REPEATS = 10
_INNER = max(2, min(5, (_minor - _minor // N_SPLITS) // 2))
print(f"  nested CV design from the data: outer {N_SPLITS}-fold x {N_REPEATS} repeats, "
      f"inner {_INNER}-fold  (smaller class = {_minor} of {len(y)} teams)")
outer = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=CFG.seed)
model = LogisticRegressionCV(penalty="elasticnet", solver="saga", l1_ratios=[0.1, 0.5, 0.9],
                             Cs=10, cv=_INNER, scoring="roc_auc", max_iter=5000,
                             random_state=CFG.seed)

cvres = cross_validate(model, Xs, y, cv=outer, scoring="roc_auc", return_estimator=True)
aucs = cvres["test_score"]
print(f"  nested CV AUC = {aucs.mean():.3f} +/- {aucs.std():.3f}  across {len(aucs)} folds  (baseline 0.5 = no skill)")

# ---- is this actually better than chance? Nadeau-Bengio corrected test for repeated k-fold CV ----
# (repeated-CV folds share training data, so a naive t-test overstates confidence; this correction fixes that)
n_tot = len(aucs)
test_train_ratio = (1 / N_SPLITS) / (1 - 1 / N_SPLITS)
corrected_var = np.var(aucs, ddof=1) * (1 / n_tot + test_train_ratio)
t_stat = (aucs.mean() - 0.5) / np.sqrt(corrected_var)
p_vs_chance = 2 * (1 - _tdist.cdf(abs(t_stat), df=n_tot - 1))
beats_chance = (aucs.mean() > 0.5) and (p_vs_chance < 0.05)
print(f"  AUC vs. chance (0.5): corrected t={t_stat:.2f}, p={p_vs_chance:.3f}  ->  "
      f"{'beats chance' if beats_chance else 'NOT distinguishable from chance'}")

# ---- fold-level coefficients: mean +/- SD across the 50 already-fitted nested-CV models ----
fold_coefs = np.array([est.coef_[0] for est in cvres["estimator"]])
coef_mean = fold_coefs.mean(axis=0)
coef_sd   = fold_coefs.std(axis=0)
sel_freq  = (np.abs(fold_coefs) > 1e-6).mean(axis=0)

importance = pd.DataFrame({"feature": feat_cols, "coef_mean": coef_mean, "coef_sd": coef_sd,
                            "selection_freq": sel_freq}).set_index("feature")
importance = importance.reindex(importance["coef_mean"].abs().sort_values(ascending=False).index)
print("  top features (mean +/- SD across 50 nested-CV folds):")
print(importance.head(15).to_string())

psave("pooled_capstone", dict(cv_auc=aucs, p_vs_chance=p_vs_chance, beats_chance=beats_chance,
                               importance=importance, n_teams=len(D), feat_cols=feat_cols))

# ==============================================================================
# FIGURE
# ==============================================================================
n_top = 15
top_imp = importance.head(n_top).iloc[::-1]
fig = plt.figure(figsize=(14, max(7.8, 0.36*n_top + 4.2)), facecolor="white")
gs = fig.add_gridspec(2, 1, height_ratios=[max(n_top*0.36, 4), 0.5], hspace=0.5)
ax1 = fig.add_subplot(gs[0]); ax_ref = fig.add_subplot(gs[1]); ax_ref.axis("off")

colors = ["#1B9E8A" if v > 0 else "#E08214" for v in top_imp["coef_mean"]]
alphas = 0.35 + 0.65 * top_imp["selection_freq"].values
ax1.barh(range(len(top_imp)), top_imp["coef_mean"], xerr=top_imp["coef_sd"],
         color=colors, height=0.6, edgecolor="white", ecolor="#555", capsize=3,
         error_kw=dict(elinewidth=1.1))
for i, a in enumerate(alphas):
    ax1.patches[i].set_alpha(a)
ax1.axvline(0, color="#999", lw=0.9)
ax1.set_yticks(range(len(top_imp))); ax1.set_yticklabels([c.replace("macro__","").replace("_"," ") for c in top_imp.index], fontsize=9)
ax1.set_xlabel("elastic-net coefficient  (mean +/- SD across 50 nested-CV folds; opacity = selection frequency)", fontsize=9.5)

verdict = ("does NOT beat chance" if not beats_chance else "beats chance")
ax1.set_title(f"Team prize prediction  ·  nested CV AUC = {aucs.mean():.2f} \u00b1 {aucs.std():.2f}  ({verdict}, p={p_vs_chance:.2f} vs. 0.50)  ·  n={len(D)} teams",
              fontsize=12.5, fontweight="bold", loc="left", pad=12,
              color=("#9B1B30" if not beats_chance else "#1B6E63"))
for sp in ("top", "right"): ax1.spines[sp].set_visible(False)

warn = ("Honest read: this model's cross-validated AUC is not reliably better than a coin flip, so the bars above "
        "describe which features this particular feature set leans on, not a validated predictor of winning.") if not beats_chance else \
       "This model's cross-validated AUC is reliably better than chance; bars show which features drive that."
ax_ref.text(0.5, 0.68, warn, ha="center", va="center", fontsize=8.6, color="#444", wrap=True, transform=ax_ref.transAxes)
ax_ref.text(0.5, 0.18, "error bars + selection frequency come directly from the 50 nested-CV fold refits (no extra bootstrap)  ·  "
            "AUC-vs-chance test corrected for repeated-CV fold non-independence (Nadeau & Bengio, 2003)  ·  "
            "Zou & Hastie (2005) elastic net; Cawley & Talbot (2010) on nested CV and selection bias.",
            ha="center", va="center", fontsize=7.4, style="italic", color="#888", transform=ax_ref.transAxes)

fig.subplots_adjust(top=0.86, bottom=0.08, left=0.30, right=0.97)
save_fig(fig, "chart_pooled_capstone"); plt.close(fig)
print("  Saved: chart_pooled_capstone.png")
print("=" * 64)

---

## Section 7 - Themes and expressed sentiment

The theme a response belongs to, as a predictor of the sentiment measured in that response. This is the best-powered analysis in the project: the unit is the response, not the team.

### Stage 40 · Does the theme a response belongs to predict its sentiment?

**Question.** Are some themes expressed more positively or negatively than others, once the question and the response length are accounted for?

**Reads.** `pooled_master`. **Writes.** `pooled_sentiment_model`.

**How to read it.** A multilevel model on the measured sentiment score: theme and question as fixed effects, participant nested in team as random effects, and log response length as a mandatory covariate - length predicts both measurability and tone, and omitting it turns length into theme effects. Contrasts are deviations from the grand mean, so a bar, its interval and its star all come from the same coefficient.

**Confound to carry into the write-up.** The themes were induced from the same text the sentiment was scored on, so theme and sentiment share a source. This is a confound, not circularity: the lexicon never saw the theme labels.

In [ ]:
# @title
# ==============================================================================
# STAGE 40  -  DOES A RESPONSE'S THEME PREDICT ITS SENTIMENT?
#             section 7 of 9  -  Themes and sentiment
# ==============================================================================
# WHY THIS REPLACES THE OLD CELL
#   The previous version modelled a variable called "valence", built like this:
#       valence(participant) = mean over the sub-themes they mentioned of the
#                              LLM's own sentiment label for that theme (+1/0/-1)
#   and then regressed it on theme-mention dummies. The outcome was therefore a
#   deterministic function of the predictors: regressing the sub-theme dummies on
#   it returns R2 = 0.85, and the main-theme dummies R2 = 0.46. A coefficient from
#   that model says "the LLM called this theme negative", not "participants who
#   raised this theme wrote more negatively". It cannot support a finding.
#
#   This version uses the sentiment that was MEASURED from the participant's own
#   words - the sentiment score already computed in NB2 Stage 26B and carried
#   in pooled_master. The scorer never saw the theme labels, so the relationship is no
#   longer true by construction.
#
# THE MODEL
#   unit        one survey RESPONSE (a participant answers up to 5 questions)
#   outcome     sentiment score, -1..+1, read from the raw text by an LLM
#   fixed       the theme this response belongs to, plus which QUESTION it answers
#               (questions differ in tone, so the theme effect is adjusted for it)
#   random      participant, nested in team - responses from one person are not
#               independent, and neither are people from one team
#   inference   each theme's adjusted mean against the grand mean, BH-FDR across
#               the themes at that level
#
# HONEST LIMITATION, printed with the result: the themes were clustered from the
# same text the sentiment scorer read, so theme membership and sentiment are related through
# the language itself. This is a real confound and is reported - but it is a
# substantive one, unlike the previous version, which was a tautology.
#
# Outputs: chart_sentiment_by_theme_{main,sub}, sentiment_model_{main,sub}.csv,
#          pooled_sentiment_model.
# Refs: NB2 Stage 26B (LLM categorical sentiment); Benjamini-Hochberg (1995);
#       Snijders & Bosker (2012) multilevel ICC.
# ==============================================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

_PL = globals().get("_orig_pload", pload)
M = _PL("pooled_master")
if M is None:
    raise RuntimeError("Need pooled_master - run NB2 Stage 33b first.")
M = M.copy()

MIN_N = 20          # a theme needs this many scored responses to be estimated
# _LVS, not LEVELS: the preflight owns that name and every later cell reads it.
_LVS = [(lv,) + _cov_cols(lv, MASTER)[:2] for lv in LEVELS]

print("=" * 74)
print("  STAGE 52  SENTIMENT ~ THEME  (measured sentiment, response level, multilevel)")
print("=" * 74)

D0 = M[np.isfinite(pd.to_numeric(M["sentiment"], errors="coerce"))].copy()
D0["sentiment"] = pd.to_numeric(D0["sentiment"], errors="coerce")
n_drop = len(M) - len(D0)
print(f"  responses: {len(M)} total, {len(D0)} with a sentiment score, {n_drop} without")
if "sent_status" in M.columns:
    _why = M.loc[~M.index.isin(D0.index), "sent_status"].value_counts().to_dict()
    print(f"  the {n_drop} without a score, by reason: {_why}")
    print("  ('withheld' = the model returned no verdict, or the span it quoted as")
    print("   measure; 'excluded-upstream' = NB2's language detection rejected the text.")
    print("   These are NOT the same thing and are not all non-English - see Stage 14b.)")
else:
    print(f"  ({n_drop} carry no usable score. Run Stage 14b first to see why.)")
print(f"  participants {D0.Participant.nunique()} · teams {D0.Team.nunique()} · "
      f"questions {D0.question.nunique()}")

OUT, FIGS = {}, {}
for lv, idc, namec in _LVS:
    # NOISE. -1 is not NaN, so dropna kept it and the model gained a "theme"
    # level made of every response that belongs to no theme - a fixed effect
    # with no meaning, estimated against several hundred rows, that shifts the
    # deviation contrasts for every real theme.
    _hasc = _cov_cols(lv, D0)[2]
    print(_cov_note(lv, D0))
    D = (D0[D0[_hasc].astype(bool)] if _hasc in D0.columns
         else D0[D0[idc] >= 0]).dropna(subset=[idc]).copy()
    D["theme"] = D[idc].astype(int).astype(str)
    assert (D[idc].astype(int) >= 0).all(), f"a -1 {lv} id reached the model"
    D["q"] = D["question"].astype(str)
    keep = [t for t, n in D.theme.value_counts().items() if n >= MIN_N]
    D = D[D.theme.isin(keep)]
    if D.theme.nunique() < 3:
        print(f"\n  {lv}: fewer than 3 themes clear the {MIN_N}-response gate - skipped")
        continue
    nm = {str(int(r[idc])): str(r[namec]) for _, r in
          D.drop_duplicates(subset=[idc]).iterrows()}

    print(f"\n  --- {lv.upper()} level: {D.theme.nunique()} themes, {len(D)} responses "
          f"(themes with < {MIN_N} responses excluded) ---")

    # Sum (deviation) contrasts, NOT the default treatment contrasts. With treatment
    # coding every theme is tested against whichever theme happens to sort first, so
    # the p-values answer a question nobody asked and - worse - they would not match
    # the bars, which show each theme's distance from the OVERALL mean. Sum coding
    # makes each coefficient exactly that distance, so the figure and the test agree.
    # log(words) is a REQUIRED covariate on substantive grounds: longer answers say
    # more and can express stronger affect, and response length varies
    # enormously across themes here (2.6 to 32.1 words per response, Kruskal-Wallis
    # p = 4e-111). Without this term, part of what looks like "this theme is less
    # positive" is really "this theme's answers are shorter". On the uncorrected data
    # the term alone took the theme-level count from 16 significant themes to 9.
    D["lw"] = np.log(D["n_words"].clip(lower=1)) if "n_words" in D.columns else \
              np.log(D["text"].astype(str).str.split().str.len().clip(lower=1))
    fml = "sentiment ~ C(theme, Sum) + C(q, Sum) + lw"
    # A ladder, tried in order, stopping at the first rung that CONVERGES. Accepting a
    # non-converged fit silently is the thing to avoid: statsmodels still returns
    # coefficients, and they can sit on the boundary of the parameter space. Whichever
    # rung supplies the numbers is recorded and printed with the result.
    #   re_formula="1" is REQUIRED on the two-level rungs: with a vc_formula and no
    #   re_formula, statsmodels drops the grouping factor's own random intercept and
    #   the team level silently disappears from the variance decomposition.
    _LADDER = [
        ("participant within team",
         dict(groups="Team", re_formula="1",
              vc_formula={"Participant": "0 + C(Participant)"}), dict(reml=True)),
        ("participant within team (powell)",
         dict(groups="Team", re_formula="1",
              vc_formula={"Participant": "0 + C(Participant)"}),
         dict(reml=True, method="powell")),
        ("participant only",
         dict(groups="Participant"), dict(reml=True)),
        ("participant only (powell)",
         dict(groups="Participant"), dict(reml=True, method="powell")),
    ]
    md, nest, converged, tried = None, None, False, []
    for _lbl, _kw, _fkw in _LADDER:
        try:
            _kw = dict(_kw); _grp = D[_kw.pop("groups")]
            # The ladder EXPECTS some rungs to fail - that is the point of having
            # rungs. statsmodels emitted a ConvergenceWarning for every attempt,
            # which buried the actual result under a wall of red text. The attempts
            # are silenced here; the OUTCOME is reported by the ladder below, which
            # is the honest channel and states plainly when nothing converged.
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                _m = smf.mixedlm(fml, data=D, groups=_grp, **_kw).fit(**_fkw)
        except Exception as _e:
            tried.append(f"{_lbl}: {type(_e).__name__}"); continue
        _ok = bool(getattr(_m, "converged", True))
        tried.append(f"{_lbl}: {'converged' if _ok else 'did not converge'}")
        if md is None:                       # keep the first fit as a last resort
            md, nest, converged = _m, _lbl, _ok
        if _ok:
            md, nest, converged = _m, _lbl, True
            break
    if md is None:
        print(f"  !! no specification could be fitted: {'; '.join(tried)}")
        continue
    print(f"  model: {nest}" + ("" if converged else "  (DID NOT CONVERGE)"))
    if not converged:
        print(f"    every specification tried: {'; '.join(tried)}")
        print("    the estimates below are reported because they are stable across the")
        print("    rungs, but the intervals are approximate and this must be stated in")
        print("    the paper - do not present them as exact.")

    # variance decomposition
    # variance components: statsmodels reports the grouping factor in cov_re (a
    # DataFrame or an array depending on version) and any vc_formula terms in vcomp
    def _v(x):
        try:
            a = np.asarray(x.values if hasattr(x, "values") else x, dtype=float)
            return float(a.reshape(-1)[0]) if a.size else np.nan
        except Exception:
            return np.nan
    # cov_re and vcomp are already in the response's own units - verified against a
    # simulation with a known 18/32/50 split, which they recover as 14/30/55.
    # Multiplying either by `scale` is wrong and understates both random levels.
    v_group = _v(getattr(md, "cov_re", None))
    try:
        v_part = float(np.sum(np.asarray(md.vcomp, dtype=float)))
    except Exception:
        v_part = np.nan
    v_res = float(md.scale)
    lbl_group = "between teams" if "within team" in nest else "between participants"
    tot = np.nansum([v_group, v_part, v_res])
    print(f"  variance in how positively a response is written (nesting: {nest}):")
    if np.isfinite(v_group) and tot > 0:
        print(f"    {lbl_group:<22} {100*v_group/tot:5.1f}%")
    if np.isfinite(v_part) and tot > 0 and "within team" in nest:
        print(f"    {'between participants':<22} {100*v_part/tot:5.1f}%")
    print(f"    {'within a participant':<22} {100*v_res/tot:5.1f}%   <- most of it, as expected")

    # Under Sum coding patsy names the terms by the level's POSITION and omits the
    # last level, whose deviation is minus the sum of the others. Recover it so every
    # theme appears, and mark it as not directly tested rather than pretending.
    levs = sorted(D.theme.unique(), key=lambda x: int(x))
    rows = []
    for pos, t in enumerate(levs):
        key = f"C(theme, Sum)[S.{t}]"
        if key not in md.params.index:
            key = next((k for k in md.params.index
                        if k.startswith("C(theme, Sum)[") and k.rstrip("]").endswith(f".{t}")), None)
        sub = D[D.theme == t]
        if key is not None:
            b, se = float(md.params[key]), float(md.bse[key]); p = float(md.pvalues[key])
        else:                                   # the omitted level
            oth = [k for k in md.params.index if k.startswith("C(theme, Sum)[")]
            b = -float(md.params[oth].sum()); se = np.nan; p = np.nan
        rows.append(dict(level=lv, id=int(t), name=nm.get(t, f"{lv}{t}"),
                         n_resp=len(sub), n_part=sub.Participant.nunique(),
                         mean_sent=float(sub.sentiment.mean()),
                         # the RAW mean's own spread, so the descriptive panel
                         # below carries an interval of its own rather than
                         # borrowing the model's
                         sd_sent=(float(sub.sentiment.std(ddof=1))
                                  if sub.sentiment.notna().sum() > 1 else np.nan),
                         n_sent=int(sub.sentiment.notna().sum()),
                         dev_from_mean=b, se=se, p=p,
                         lo=b - 1.96*se if np.isfinite(se) else np.nan,
                         hi=b + 1.96*se if np.isfinite(se) else np.nan,
                         tested=key is not None))
    R = pd.DataFrame(rows)
    ok = R.p.notna()
    R["q_fdr"] = np.nan
    if ok.any():
        R.loc[ok, "q_fdr"] = multipletests(R.loc[ok, "p"].values, method="fdr_bh")[1]
    R["sig_fdr"] = R.q_fdr < 0.05
    R = R.sort_values("dev_from_mean").reset_index(drop=True)
    R.to_csv(f"{POOLED_DIR}/sentiment_model_{lv}.csv", index=False)
    OUT[lv] = R
    ns = int(R.sig_fdr.sum())
    _lwb = md.params.get("lw", np.nan); _lwp = md.pvalues.get("lw", np.nan)
    _grand = float(D.sentiment.mean())
    if np.isfinite(_lwb):
        print(f"  length covariate log(words): beta={_lwb:+.3f}, p={_lwp:.1e}")
        print( "    Longer answers score more positively, and the model adjusts that out.")
        print( "    WHY THAT IS WORTH STATING, not just doing. Stage 14b already measured")
        print( "    that scoring COVERAGE is not length-dependent (3 points across length")
        print( "    bands), so length is not a measurement artefact here - it was kept as")
        print( "    a covariate on substantive grounds. But if a theme makes people write")
        print( "    at length, and length carries tone, then length is a MEDIATOR of that")
        print( "    theme's effect rather than a confounder of it, and adjusting for a")
        print( "    mediator removes part of the effect being measured. Neither reading is")
        print( "    obviously right on this data, so BOTH numbers are reported below and")
        print( "    in the figure: the raw mean a theme's responses actually have, and")
        print( "    the deviation after adjustment.")
    print(f"\n  overall mean sentiment of themed responses at this level: {_grand:+.3f}")
    print( "    Note these are two different quantities. The 'difference from the mean'")
    print( "    below is a coefficient against the MODEL's own baseline (Sum coding,")
    print( "    adjusted for question and length); the number above is the plain mean of")
    print( "    the responses. They are shown side by side and never added together.")
    print(f"  themes whose responses are written more/less positively than the overall")
    print(f"  mean, ADJUSTED for question and length, after BH-FDR: "
          f"{ns} of {int(R.tested.sum())} tested")
    for _, r in R[R.sig_fdr].sort_values("dev_from_mean").iterrows():
        print(f"    {r['name'][:42]:<42} {r.dev_from_mean:+.3f}  q={r.q_fdr:.3f}  n={r.n_resp}")

    # ---- RAW vs ADJUSTED, printed because the two really can disagree --------
    # THE READING ERROR THIS EXISTS TO PREVENT. The figure used to plot
    # dev_from_mean - a difference from the model's baseline - under the title
    # "Measured sentiment by theme", coloured with this project's SENTIMENT ramp,
    # in which orange means negative sentiment in every other figure. So a theme
    # whose responses are genuinely positive, but less positive than average, was
    # drawn as an orange bar and read as a negative theme. The numbers were
    # right; the encoding said something the numbers did not.
    R["raw_dev"] = R["mean_sent"] - _grand
    _pos_below = R[(R.mean_sent > 0) & (R.raw_dev < 0)]
    if len(_pos_below):
        print(f"\n  {len(_pos_below)} theme(s) are POSITIVE on the raw scale yet sit BELOW")
        print( "  the overall mean - the ones the old single-panel figure drew as")
        print( "  negative bars:")
        for _, r in _pos_below.sort_values("mean_sent").head(10).iterrows():
            print(f"    {str(r['name'])[:38]:<40} raw {r.mean_sent:+.3f}   "
                  f"vs mean {r.raw_dev:+.3f}   adjusted {r.dev_from_mean:+.3f}")
    _both = R[R.raw_dev.notna() & R.dev_from_mean.notna()]
    _flip = _both[np.sign(_both.raw_dev) != np.sign(_both.dev_from_mean)]
    if len(_flip):
        print(f"\n  {len(_flip)} theme(s) where ADJUSTMENT CHANGES THE DIRECTION - the raw")
        print( "  mean and the question/length-adjusted deviation disagree about which")
        print( "  side of average the theme is on. These are exactly the themes the")
        print( "  mediator question above bites on, so read both before claiming either:")
        for _, r in _flip.sort_values("raw_dev").iterrows():
            print(f"    {str(r['name'])[:38]:<40} raw {r.raw_dev:+.3f}   "
                  f"adjusted {r.dev_from_mean:+.3f}"
                  + ("   (survives FDR)" if bool(r.sig_fdr) else ""))
    else:
        print("\n  No theme changes direction between the raw and the adjusted view, so")
        print("  the question/length adjustment does not reverse any effect here.")

    # =========================================================================
    # THE FIGURE - two panels, because one panel was answering a different
    # question than the one it appeared to answer
    # -------------------------------------------------------------------------
    #   A  the RAW mean sentiment of each theme's responses, on the sentiment
    #      ramp - where the ramp finally means what it means in every other
    #      figure - with a reference line at the overall mean of themed
    #      responses. This panel answers "is this theme positive or negative?"
    #   B  the model's ADJUSTED deviation with its 95% interval and FDR star.
    #      Deliberately NOT on the sentiment ramp: a difference from average is
    #      not a sentiment, so it gets a neutral encoding - solid where it
    #      survives FDR, faded where it does not. This panel answers "does this
    #      theme differ from average once question and length are held constant?"
    #
    # The two are different quantities and are now allowed to look different.
    # Rows are sorted by the RAW mean so panel A reads top to bottom; panel B
    # keeps that row order so one theme can be followed across both.
    # =========================================================================
    Rf = R.sort_values("mean_sent").reset_index(drop=True)
    h = max(3.4, 0.38 * len(Rf) + 2.2)
    fig, (axA, axB) = plt.subplots(
        1, 2, figsize=(13.6, min(h, 10.0)), sharey=True,
        gridspec_kw=dict(width_ratios=[1.25, 1.0], wspace=0.06))
    y = np.arange(len(Rf))
    _NORM = sent_norm(0.6)
    _HL = globals().get("HIGHLIGHT", "#9B1B30")

    # ---- A: the raw mean, on the sentiment ramp ------------------------------
    for i, (_, r) in enumerate(Rf.iterrows()):
        col = SENT_CMAP(_NORM(r.mean_sent)) if np.isfinite(r.mean_sent) else "#C4CAD0"
        axA.barh(i, r.mean_sent, height=.62, color=col, edgecolor="white", zorder=3)
        _sd, _n = r.get("sd_sent", np.nan), int(r.get("n_sent", 0) or 0)
        if np.isfinite(_sd) and _n > 1:
            axA.errorbar(r.mean_sent, i, xerr=1.96 * _sd / np.sqrt(_n), fmt="none",
                         ecolor="#5B6670", elinewidth=.8, capsize=2, zorder=4)
    axA.axvline(0, color="#14202B", lw=1, zorder=2)
    axA.axvline(_grand, color=_HL, ls="--", lw=1.3, zorder=5)
    axA.annotate(f"overall mean {_grand:+.2f}", xy=(_grand, len(Rf) - 0.4),
                 xytext=(4, 0), textcoords="offset points", fontsize=7.4,
                 color=_HL, va="top", ha="left")
    axA.set_xlabel("measured sentiment of the theme's responses\n"
                   "(raw mean, -1 to +1; whisker = 95% CI of that mean)", fontsize=8.8)
    axA.set_title("A  -  how positive is this theme?", fontsize=10,
                  fontweight="bold", loc="left", pad=8)

    # ---- B: the adjusted deviation, NOT on the sentiment ramp ----------------
    for i, (_, r) in enumerate(Rf.iterrows()):
        axB.barh(i, r.dev_from_mean, height=.62,
                 color="#3A4650" if r.sig_fdr else "#C9CDD2",
                 edgecolor="white", zorder=3)
        if np.isfinite(r.se):
            axB.errorbar(r.dev_from_mean, i, xerr=1.96 * r.se, fmt="none",
                         ecolor="#5B6670", elinewidth=.9, capsize=2, zorder=4)
    axB.axvline(0, color="#14202B", lw=1)
    axB.set_xlabel("difference from the overall mean, adjusted for\n"
                   "question and response length\n"
                   "(whisker = 95% CI; solid = BH-FDR q<0.05)", fontsize=8.8)
    axB.set_title("B  -  does it differ from average?", fontsize=10,
                  fontweight="bold", loc="left", pad=8)

    import textwrap
    def _lab(r):                       # wrap onto a second line, never cut a word
        head = f"{lv[0]}{int(r.id)}  {r['name']}"
        w = textwrap.wrap(head, 38)[:2]
        w[-1] = w[-1] + f"  (n={int(r.n_resp)})" + ("  *" if r.sig_fdr else "")
        return "\n".join(w)
    axA.set_yticks(y)
    axA.set_yticklabels([_lab(r) for _, r in Rf.iterrows()], fontsize=7.2)
    for _a in (axA, axB):
        for sp in ("top", "right", "left"):
            _a.spines[sp].set_visible(False)
        _a.xaxis.grid(True, color="#E4E8EB")
        _a.set_axisbelow(True)
    _t = (f"How positively each theme is written, and whether that differs from "
          f"average - {lv} level")
    try:    fig_title(fig, _t)
    except Exception: fig.suptitle(_t, fontsize=13)
    fig.text(0.5, 0.004,
             "Panel A is a raw descriptive mean drawn on the sentiment ramp; panel B is "
             "a mixed-model coefficient (participant nested in team) and is deliberately "
             "NOT on that ramp - a difference from average is not a sentiment. A theme "
             "can be positive in A and below average in B at the same time.",
             ha="center", fontsize=7.4, style="italic",
             color=globals().get("FOOT", "#98A2AD"), wrap=True)
    plt.tight_layout(rect=[0, .035, 1, .94])
    # the colourbar is added AFTER tight_layout, on explicit axes - the same
    # pattern Stage 0b's forest() uses. Attaching it to an axis beforehand makes
    # tight_layout warn and mis-size the panels.
    _cax = fig.add_axes([0.965, 0.30, 0.010, 0.34])
    _sm = plt.cm.ScalarMappable(norm=_NORM, cmap=SENT_CMAP); _sm.set_array([])
    _cb = fig.colorbar(_sm, cax=_cax)
    _cb.set_label("measured sentiment (panel A)", fontsize=7.2)
    _cb.ax.tick_params(labelsize=6.4, length=0)
    _cb.outline.set_visible(False)
    try:    save_fig(fig, f"chart_sentiment_by_theme_{lv}")
    except Exception: fig.savefig(f"{POOLED_DIR}/chart_sentiment_by_theme_{lv}.png",
                                  dpi=300, bbox_inches="tight")
    plt.close(fig)

psave("pooled_sentiment_model", OUT)

# Stage 53 consumes `pooled_multilevel["macro"]` with columns macro/name/beta/sig.
# The old Stage 52 wrote it from its circular valence model; write the same shape
# here from the CORRECTED main-level results so Stage 53 compares against a real
# measurement instead of the model's own labels. Same numbers as the table above.
if "main" in OUT:
    _c = OUT["main"].rename(columns={"id": "macro", "dev_from_mean": "beta",
                                     "sig_fdr": "sig", "q_fdr": "q"})
    _keep = [c for c in ("macro", "name", "beta", "se", "p", "lo", "hi", "q", "sig")
             if c in _c.columns]
    psave("pooled_multilevel", dict(macro=_c[_keep].copy(), source="corrected Stage 52",
                                    note="beta = deviation from the overall mean of the "
                                         "MEASURED sentiment score, adjusted for question and "
                                         "response length"))
    if not globals().get("_ML_SHIM_SAID"):
        print("  wrote pooled_multilevel (main level) for Stage 53 to consume")
        globals()["_ML_SHIM_SAID"] = True


# Stage 53 consumes `pooled_multilevel["macro"]` with columns macro/name/beta/sig.
# The old Stage 52 wrote it from its circular valence model; write the same shape
# here from the CORRECTED main-level results so Stage 53 compares against a real
# measurement instead of the model's own labels. Same numbers as the table above.
if "main" in OUT:
    _c = OUT["main"].rename(columns={"id": "macro", "dev_from_mean": "beta",
                                     "sig_fdr": "sig", "q_fdr": "q"})
    _keep = [c for c in ("macro", "name", "beta", "se", "p", "lo", "hi", "q", "sig")
             if c in _c.columns]
    psave("pooled_multilevel", dict(macro=_c[_keep].copy(), source="corrected Stage 52",
                                    note="beta = deviation from the overall mean of the "
                                         "MEASURED sentiment score, adjusted for question and "
                                         "response length"))
    if not globals().get("_ML_SHIM_SAID"):
        print("  wrote pooled_multilevel (main level) for Stage 53 to consume")
        globals()["_ML_SHIM_SAID"] = True

print("-" * 74)
print("  LIMITATION, to carry into the paper: the themes were clustered from the same")
print("  text the sentiment scorer read, so theme membership and sentiment share a source. This is a")
print("  genuine confound and the coefficients should be read as 'responses grouped in")
print("  this theme are written more/less positively', not as a causal effect of the")
print("  theme on the participant. It is NOT, however, circular: the sentiment scorer is independent")
print("  instrument applied to the raw text and never saw the theme labels.")
print("  files: sentiment_model_{main,sub}.csv · chart_sentiment_by_theme_{main,sub}")
print("=" * 74)

---

## Section 8 - Organisational context

Team-level structure from the official registry - section, region, track, size - and how it relates to what teams talk about.

### Stage 41 · Team profiles

**Question.** One row per team: size, outcome, registry metadata and thematic profile.

**Writes.** `team_profiles.csv` - the table behind every team-level analysis in this notebook.

In [ ]:
# @title
# ==============================================================================
# STAGE 41  -  TEAM PROFILE TABLE
#             section 8 of 9  -  Teams
# One row per team: size, country/region/track, medal, special/nominated, its
# dominant main theme and share, mean sentiment. The manual-coder companion.
# ==============================================================================
import numpy as np, pandas as pd
_PL = globals().get("_orig_pload", pload)
M = _PL("pooled_master")
if M is None:
    print("pooled_master missing.")
else:
    _gT = M.groupby("Team")
    prof = pd.DataFrame(dict(
        n_participants=_gT["Participant"].nunique(), n_responses=_gT.size(),
        Country=_gT["Country"].first(), Region=_gT["Region"].first(), Track=_gT["Track"].first(),
        medal=_gT["medal"].first(), special=_gT["special"].max(), nominated=_gT["nominated"].max(),
        mean_sentiment=_gT["sentiment"].mean().round(3)))
    top = M.groupby(["Team", "main_name"]).size().rename("n").reset_index()
    top = top.sort_values(["Team", "n"], ascending=[True, False]).drop_duplicates("Team")
    prof["top_main_theme"] = top.set_index("Team")["main_name"]
    prof["top_theme_share"] = (top.set_index("Team")["n"] / prof["n_responses"]).round(2)
    prof = prof.sort_values("n_responses", ascending=False)
    psave("team_profiles", prof)
    prof.to_csv(f"{POOLED_DIR}/team_profiles.csv")
    print(f"  team_profiles.csv written: {len(prof)} teams x {prof.shape[1]} cols")
    print(prof.head(8).to_string())


### Stage 42 · Section, region and track

**Question.** Does organisational context relate to what teams talk about?

**How to read it.** Registry categories, so these are real organisational facts rather than anything derived from the text. Small categories are reported with their counts and are not tested.

In [ ]:
# @title
# ==============================================================================
# STAGE 42  -  TEAM ORGANIZATION: SECTION, REGION AND TRACK
#             section 8 of 9  -  Teams
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

# Read artifacts through the UNPATCHED loader: the preflight rebinds pload()
# to follow the explorer's detail level, which must not reach the analyses.
# Defined here, before first use, so this cell also runs on its own.
_PL = globals().get("_orig_pload", pload)

org     = _PL("pooled_team_org")
part    = _PL("pooled_hier_participant")
award   = _PL("pooled_award") or {}
meta    = _PL("pooled_meta").reset_index(drop=True)
if org is None or part is None:
    raise RuntimeError("Run Stage 46 (team org join) first.")
_subllm = _PL("pooled_sub_llm") or {}; _subname = _PL("pooled_sub_name") or {}
def _sname(g): return (_subllm.get(g) or _subllm.get(str(g)) or {}).get("name") or _subname.get(g) or _subname.get(str(g)) or f"s{g}"

# (font clobbering removed: Stage 0 resolves the report font ONCE. This cell
#  re-picked a font from a fallback list, overriding it mid-notebook, so
#  figures before and after this point rendered in DIFFERENT typefaces.)

def _wrap_word(t, w=16):
    """Wrap on WORD boundaries only. textwrap's default splits long single
    words, which is how "Biomanufacturing" became "Biomanufacturi ng"."""
    import textwrap as _tw
    return "\n".join(_tw.wrap(str(t), w, break_long_words=False,
                              break_on_hyphens=False))

def _wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return p, max(0, centre - half), min(1, centre + half)

print("=" * 64)
print("  STAGE 50  TEAM ORGANIZATION: TRACK + COMBINED COMPARISON")
print("=" * 64)

meta["Team"] = meta["Team"].astype(str).str.strip()
special = np.asarray(award.get("special", np.zeros(len(meta), bool)), bool)
nominated = np.asarray(award.get("nominated", np.zeros(len(meta), bool)), bool)
prize_row = np.where(special, 2, np.where(nominated, 1, 0)).astype(float)
team_prize = pd.Series(prize_row, index=meta["Team"]).groupby(level=0).max()
teams = org.index.tolist()
# An unmatched team has an UNKNOWN award, not a lost one. fillna(0) would
# score it as 'no prize' and silently deflate every prize rate, so any
# unmatched team is reported and dropped instead. Today this is a no-op:
# all 125 teams match the registry - the guard exists so it cannot become
# a silent error on a future run.
_pr = team_prize.reindex(teams)
if _pr.isna().any():
    print(f"  !! {int(_pr.isna().sum())} team(s) have no award-registry "
          f"match; they are EXCLUDED from the prize comparison, not "
          f"counted as no-prize: {list(_pr.index[_pr.isna()])[:8]}")
team_won = (_pr.fillna(0) > 0).astype(int)

P = part.reset_index()
P["Team"] = P["Team"].astype(str).str.strip()
cols = [c for c in P.columns if c.startswith("sub_ment_")]
P["_section"] = P["Team"].map(org["Section"].astype(str).to_dict())
P["_region"]  = P["Team"].map(org["Region"].astype(str).to_dict())

# ---- Track: prize rate + effect-size test (same recipe as Section/Region) ----
track = org["Track"].astype(str)
counts_tr = track.value_counts()
print("  track counts:", dict(counts_tr))
MIN_TEAMS_TRK = 10   # raised from 5: keeps only well-powered tracks, avoids a cramped/noisy panel
keep_trk = [t for t in counts_tr.index if counts_tr[t] >= MIN_TEAMS_TRK and t not in ("nan", "None")]
print(f"  tracks with >= {MIN_TEAMS_TRK} teams (used below): {keep_trk}")
P["_track"] = P["Team"].map(track.to_dict())

# =============================================================================
# ONE ROW PER TEAM - the unit these three factors actually vary over
# -----------------------------------------------------------------------------
# THE DEFECT THIS REPLACES. Every sub-theme test below used to run over
# PARTICIPANT rows grouped by their team's Section / Region / Track. All three
# are TEAM attributes: every member of a team shares them, so a participant-level
# Kruskal-Wallis treats 63 undergrad teams as 164 independent observations. That
# is pseudo-replication, and it inflates significance in exactly one direction.
#
# It was not hypothetical here. This cell reported "Region: 4 sub-themes
# significant at FDR", while Stage 63 - which permutes the group label across
# TEAMS - reported 0 of 108 across all three factors. Two cells in one notebook,
# the same question, opposite answers. Stage 63's own header already explained
# why this one was wrong: "a participant-level test would count teammates as
# independent and report significance that is not there."
#
# The variable names said so too. MIN_TEAMS_TRK / MIN_TEAMS_GRP were compared
# against len(group) - which was a count of PARTICIPANTS, not teams. The gate
# never measured what it was named after.
#
# THE FIX. Aggregate first: one row per team, each column that team's mention
# RATE for a sub-theme (the same construction Stage 53 already uses for the macro
# level). Every test below then has one observation per team, the group-size
# gates finally count teams, and the answer becomes comparable with Stage 63's.
#
# Expect the four Region "hits" to disappear. They are an artefact of counting
# teammates twice, not a finding being suppressed - and the prize-rate panels,
# which were already team-level, are unaffected.
# =============================================================================
PT = P.groupby("Team")[cols].mean()
for _c, _src in (("_section", "Section"), ("_region", "Region"), ("_track", "Track")):
    PT[_c] = (org[_src].astype(str).reindex(PT.index).values
              if _src in org.columns else "")
print(f"  unit of analysis for the sub-theme tests: {len(PT)} TEAMS "
      f"(was {len(P)} participants - Section/Region/Track are team attributes, "
      f"so teammates are not independent observations)")

statsA_trk = []
for t in keep_trk:
    m = (track == t).values
    k, n = int(team_won[m].sum()), int(m.sum())
    p, lo, hi = _wilson(k, n)
    statsA_trk.append(dict(label=t, k=k, n=n, p=p, lo=lo, hi=hi))
statsA_trk = sorted(statsA_trk, key=lambda s: -s["p"])
print("  Track - prize rate:")
for s in statsA_trk:
    print(f"    {s['label']:<18} {s['k']}/{s['n']} = {s['p']:.0%}  (95% CI {s['lo']:.0%}-{s['hi']:.0%})")

if len(keep_trk) < 2:
    print("  Not enough tracks with sufficient teams to compare.")
    R_trk = pd.DataFrame(columns=["cluster", "name", "H", "p", "eps2", "q", "sig"])
else:
    Ptr = PT[PT["_track"].isin(keep_trk)]        # one row per TEAM
    rows = []
    for cname in cols:
        c = int(cname.split("_")[-1])
        groups = [Ptr.loc[Ptr._track == t, cname].astype(float).values for t in keep_trk]
        groups = [g for g in groups if len(g) >= MIN_TEAMS_TRK]   # now really teams
        if len(groups) < 2 or all(np.all(g == g[0]) for g in groups):
            continue
        try:
            H, p_kw = kruskal(*groups)
        except Exception:
            continue
        n_tot = sum(len(g) for g in groups); k_grp = len(groups)
        eps2 = max(0.0, (H - k_grp + 1) / (n_tot - k_grp))
        rows.append(dict(cluster=c, name=_sname(c), H=H, p=p_kw, eps2=eps2))
    R_trk = pd.DataFrame(rows)
    if len(R_trk):
        R_trk["q"] = multipletests(R_trk["p"].values, method="fdr_bh")[1]
        R_trk["sig"] = R_trk["q"] < 0.05
        R_trk = R_trk.sort_values("eps2", ascending=False).reset_index(drop=True)
    print(f"  Track: {len(R_trk)} sub-themes tested on {len(Ptr)} teams, "
          f"{int(R_trk['sig'].sum()) if len(R_trk) else 0} significant at FDR q<0.05")

# ---- Section and Region, computed HERE with the same recipe as Track ---------
# These two panels used to come from a stage that is not part of this notebook, so
# they could only ever arrive from an earlier run - and in practice they arrived
# from the retired 43-sub-theme hierarchy, which is why they were being dropped.
# Recomputing them here puts every panel on the same data as the rest of the figure
# and removes an artifact nothing in this notebook could rebuild.
MIN_TEAMS_GRP = 8

def _org_panel(groupcol, label):
    """Prize rate per group (Wilson CI) + Kruskal-Wallis effect size per sub-theme."""
    if label not in org.columns:
        print(f"  {label}: not in the team registry - panel skipped"); return None
    team_g = org[label].astype(str)
    cnt = team_g.value_counts()
    keep = [x for x in cnt.index if cnt[x] >= MIN_TEAMS_GRP
            and x not in ("nan", "None", "", "Unknown")]
    if len(keep) < 2:
        print(f"  {label}: fewer than two groups with >= {MIN_TEAMS_GRP} teams - panel skipped")
        return None
    stats = []
    for x in keep:
        m = (team_g == x).values
        kk, nn = int(team_won[m].sum()), int(m.sum())
        p_, lo_, hi_ = _wilson(kk, nn)
        stats.append(dict(label=x, k=kk, n=nn, p=p_, lo=lo_, hi=hi_))
    stats = sorted(stats, key=lambda z: -z["p"])
    print(f"  {label} - prize rate:")
    for z in stats:
        print(f"    {z['label']:<18} {z['k']}/{z['n']} = {z['p']:.0%}  "
              f"(95% CI {z['lo']:.0%}-{z['hi']:.0%})")
    # one row per TEAM - see the note above PT. The old version grouped
    # PARTICIPANT rows here, which is what produced the four Region "hits" that
    # Stage 63's team-level permutation could not reproduce.
    g = PT[groupcol].astype(str)
    Pg = PT[g.isin(keep)]
    gg = g[g.isin(keep)]
    rr = []
    for cname in cols:
        c_ = int(cname.split("_")[-1])
        grp = [Pg.loc[gg == x, cname].astype(float).values for x in keep]
        grp = [z for z in grp if len(z) >= MIN_TEAMS_GRP]   # now really teams
        if len(grp) < 2 or all(np.all(z == z[0]) for z in grp):
            continue
        try:    H_, p_ = kruskal(*grp)
        except Exception: continue
        n_tot = sum(len(z) for z in grp); k_grp = len(grp)
        rr.append(dict(cluster=c_, name=_sname(c_), H=H_, p=p_,
                       eps2=max(0.0, (H_ - k_grp + 1) / (n_tot - k_grp))))
    if not rr:
        print(f"  {label}: no sub-theme testable - panel skipped"); return None
    R_ = pd.DataFrame(rr)
    R_["q"] = multipletests(R_["p"].values, method="fdr_bh")[1]
    R_["sig"] = R_["q"] < 0.05
    R_ = R_.sort_values("eps2", ascending=False).reset_index(drop=True)
    print(f"  {label}: {len(R_)} sub-themes tested on {len(PT)} teams, "
          f"{int(R_['sig'].sum())} significant at FDR")
    return dict(prize=stats, theme_test=R_, groups=keep)

sec_res = _org_panel("_section", "Section")
reg_res = _org_panel("_region", "Region")


psave("pooled_org_track", dict(theme_test=R_trk, tracks=keep_trk, prize=statsA_trk))

# ---- Team-clustered bootstrap CI for the displayed top-N effect sizes -----
def _boot_eps2_ci(PTf, groupcol, keep_list, cname, n_boot=400, seed=CFG.seed, ci=95):
    """Bootstrap the effect size by resampling TEAMS - the exchangeable unit.

    The previous version already resampled teams, but then concatenated all the
    PARTICIPANT rows of the drawn teams and ran the test on those, so the interval
    was built around a point estimate computed at the wrong unit. PTf now holds
    one row per team, so a resample is simply a draw of rows and the interval and
    the estimate describe the same quantity."""
    pool = {g: PTf.loc[PTf[groupcol] == g, cname].astype(float).values
            for g in keep_list}
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        groups_b = []
        for g in keep_list:
            v = pool[g]
            if len(v) == 0: continue
            vals = rng.choice(v, size=len(v), replace=True)
            if len(vals) >= 2: groups_b.append(vals)
        if len(groups_b) < 2 or all(np.all(g == g[0]) for g in groups_b):
            continue
        try:
            H, _ = kruskal(*groups_b)
        except Exception:
            continue
        n_tot = sum(len(g) for g in groups_b); k_grp = len(groups_b)
        boots.append(max(0.0, (H - k_grp + 1) / (n_tot - k_grp)))
    if len(boots) < 20:
        return (np.nan, np.nan)
    return (float(np.percentile(boots, (100 - ci) / 2)), float(np.percentile(boots, 100 - (100 - ci) / 2)))

print("  bootstrapping 95% CI for displayed effect sizes "
      "(resampling TEAMS, 400 resamples each)...")
n_top = 12
group_spec = {**({"Section": ("_section", sec_res["groups"])} if sec_res else {}),
              **({"Region": ("_region", reg_res["groups"])} if reg_res else {}), "Track": ("_track", keep_trk)}
theme_by_factor = {**({"Section": sec_res["theme_test"]} if sec_res else {}),
                   **({"Region": reg_res["theme_test"]} if reg_res else {}), "Track": R_trk}
prize_by_factor = {**({"Section": sec_res.get("prize", [])} if sec_res else {}),
                   **({"Region": reg_res.get("prize", [])} if reg_res else {}), "Track": statsA_trk}
factors = [f for f in theme_by_factor if theme_by_factor[f] is not None and len(theme_by_factor[f])]

Rtop_by_factor = {}
for f in factors:
    groupcol, keep_list = group_spec[f]
    Rt = theme_by_factor[f].head(n_top).copy()
    los, his = [], []
    for _, r in Rt.iterrows():
        lo, hi = _boot_eps2_ci(PT, groupcol, keep_list, f"sub_ment_{int(r.cluster)}")
        los.append(lo); his.append(hi)
    Rt["eps2_lo"] = los; Rt["eps2_hi"] = his
    Rtop_by_factor[f] = Rt
print("  bootstrap CI done.")

n_f = len(factors)
global_max_hi = max(np.nanmax(np.where(np.isnan(Rtop_by_factor[f]["eps2_hi"]),
                    Rtop_by_factor[f]["eps2"], Rtop_by_factor[f]["eps2_hi"])) for f in factors)
xmax_eps = global_max_hi * 1.15
import textwrap

def _w2(t, w=18):
    # F1 - the body used to be `"\n".join(textwrap.wrap(str(t), w)[:2])`,
    # which wrapped and then threw away every line past the second WITHOUT a
    # marker: "Financial Limitations and Costs" rendered as "Financial /
    # Limitations and" and read as a complete label. Measured on the 39 real
    # sub-theme names, that silently cut 15 of them here.
    # fit_label keeps max_lines=2 so this figure's layout is unchanged, widens
    # up to 60% before conceding, and only then truncates - always visibly.
    return fit_label(canon_theme_name(t), width=w, max_lines=2)

# COMPACT best-fit canvas (was ~20.6 x 11.9 in - stretched fonts, screen overflow)
fig = plt.figure(figsize=(11.5, 7.6), facecolor="white")
gs = fig.add_gridspec(3, n_f, height_ratios=[1.5, 3.6, 0.4], hspace=0.55, wspace=0.5)

for j, fname in enumerate(factors):
    axp = fig.add_subplot(gs[0, j])
    stats = prize_by_factor[fname]
    xs = np.arange(len(stats))
    # ONE neutral colour. The old categorical palette gave every group its own
    # hue, which carries no information - the group is named on the x axis -
    # and it clashed with row B, where colour means 'survives FDR'. Colour is
    # reserved for significance so it means the same thing across the figure.
    for x, s in zip(xs, stats):
        axp.bar(x, s["p"],
                yerr=[[max(0.0, float(s["p"]) - float(s["lo"]))],
                      [max(0.0, float(s["hi"]) - float(s["p"]))]],
                color="#8FA5B0", width=0.6, capsize=3, edgecolor="white",
                error_kw=dict(elinewidth=0.9))
        # above the whisker, on two lines: at five categories the bars are narrower
        # than the text, so an inside label gets clipped ('49%  n=39' -> '49%  n=3')
        axp.text(x, min(s["hi"] + 0.025, 0.94),
                 f"{s['p']:.0%}\nn={s['n']}",
                 va="bottom", ha="center", fontsize=6.2, fontweight="bold",
                 linespacing=1.05, color=INK)
    axp.set_xticks(xs)
    axp.set_xticklabels([_wrap_word(s["label"], 16) for s in stats], fontsize=6.3,
                        rotation=25, ha="right")
    # A prize RATE is a proportion: it cannot exceed 1.0, and an axis running to
    # 1.3 invites the reader to think the scale continues past certainty.
    axp.set_ylim(0, 1.0); axp.tick_params(axis="y", labelsize=6.5)
    if j == 0:
        axp.set_ylabel("prize rate", fontsize=8)
    axp.set_title(f"A{j+1} · {fname} · prize rate (95% Wilson CI)",
                  fontsize=8.5, fontweight="bold", loc="left", pad=6)
    for sp in ("top", "right"):
        axp.spines[sp].set_visible(False)

for j, fname in enumerate(factors):
    ax = fig.add_subplot(gs[1, j])
    Rt = Rtop_by_factor[fname]
    yv = np.arange(len(Rt))[::-1]
    eps2 = Rt["eps2"].values
    # A failed bootstrap has NO interval. Substituting lo=0 and hi=the point
    # estimate draws a whisker that looks measured but is invented. Those rows
    # now get a zero-length whisker (i.e. none) and are reported instead.
    _lo_raw = Rt["eps2_lo"].values.astype(float)
    _hi_raw = Rt["eps2_hi"].values.astype(float)
    _ok_ci = np.isfinite(_lo_raw) & np.isfinite(_hi_raw)
    lo = np.where(_ok_ci, _lo_raw, eps2)
    hi = np.where(_ok_ci, _hi_raw, eps2)
    if (~_ok_ci).any():
        print(f"    {int((~_ok_ci).sum())} of {len(_ok_ci)} rows have no bootstrap "
              f"interval - shown without a whisker rather than with a fabricated one")
    xerr = np.vstack([np.clip(eps2 - lo, 0, None), np.clip(hi - eps2, 0, None)])
    ax.barh(yv, eps2, xerr=xerr, color=[SENT_POS if s else SENT_NEU for s in Rt["sig"]],
            height=0.6, edgecolor="white", ecolor="#555", capsize=2,
            error_kw=dict(elinewidth=0.8), zorder=2)
    for i, (_, r) in zip(yv, Rt.iterrows()):
        if r["sig"]:
            sx = (r["eps2_hi"] if not np.isnan(r["eps2_hi"]) else r["eps2"]) + xmax_eps * 0.02
            ax.text(sx, i, "*", va="center", fontsize=9, fontweight="bold",
                    color="#1A1A1A", zorder=3)
    ax.set_xlim(0, xmax_eps)
    # the shared scale is kept so the three factors stay comparable, but the
    # reader must be told what the axis IS and how small the range really is
    ax.set_xlabel("Kruskal-Wallis epsilon-squared: the share of the variance in\n"
                  "whether a participant voices the theme that this factor explains",
                  fontsize=6.4)
    ax.text(0.99, 0.012, f"largest here: {float(np.nanmax(eps2)):.3f}",
            transform=ax.transAxes, ha="right", va="bottom",
            fontsize=6.0, color="#8A939B")
    ax.set_yticks(yv)
    ax.set_yticklabels([f"s{r.cluster} " + _w2(r["name"], 22) for _, r in Rt.iterrows()],
                       fontsize=6.2)
    ax.tick_params(axis="x", labelsize=6.5)
    ax.set_title(f"B{j+1} · {fname} · top sub-themes (95% bootstrap CI)",
                 fontsize=8.5, fontweight="bold", loc="left", pad=6)
    for sp in ("left", "right", "top"):
        ax.spines[sp].set_visible(False)
    ax.tick_params(length=0)

ax_ref = fig.add_subplot(gs[2, :]); ax_ref.axis("off")
ax_ref.text(0.5, 0.7, "A: bars = prize rate, whiskers = 95% Wilson CI · "
            "B: bars = Kruskal-Wallis epsilon-squared, whiskers = 95% team-clustered "
            "bootstrap CI (400x), shared x-scale · teal + * = BH-FDR q<0.05",
            ha="center", va="center", fontsize=6.8, color="#444",
            transform=ax_ref.transAxes)
ax_ref.text(0.5, 0.12, "Kruskal & Wallis (1952) · Efron & Tibshirani (1993) · "
            "Cameron et al. (2008) · Benjamini & Hochberg (1995)",
            ha="center", va="center", fontsize=6.2, style="italic", color="#888",
            transform=ax_ref.transAxes)

fig.suptitle("Team organization factors: prize rate and effect-size overview",
             x=0.5, ha="center", fontsize=12.5, fontweight="bold", color=INK, y=0.985)
fig.subplots_adjust(top=0.90, bottom=0.04, left=0.10, right=0.985)
save_fig(fig, "chart_pooled_org_combined")
# extra ultra-crisp copy for print/submission (400 dpi)
fig.savefig(f"{POOLED_DIR}/chart_pooled_org_combined_400dpi.png",
            dpi=400, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("  Saved: chart_pooled_org_combined.png (+ 400dpi copy)")
print("=" * 64)

### Stage 43 · Team archetypes

**Question.** Do teams fall into recognisable thematic profiles?

**How to read it.** Clustering of team-level theme profiles. This is exploratory: the archetypes are a description of this sample, and no claim is made that they generalise.

In [ ]:
# @title
# ==============================================================================
# STAGE 43  -  TEAM ARCHETYPE CLUSTERING
#             section 8 of 9  -  Teams
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# The preflight rebinds _PL() to follow the EXPLORER's detail level. The
# statistical stages must always read the true artifacts, so they use the
# unpatched loader like every other analysis cell in this notebook.
_PL = globals().get("_orig_pload", pload)

part  = _PL("pooled_hier_participant")
award = _PL("pooled_award") or {}
meta  = _PL("pooled_meta").reset_index(drop=True)
_subllm = _PL("pooled_sub_llm") or {}; _subname = _PL("pooled_sub_name") or {}
def _sname(g): return (_subllm.get(g) or _subllm.get(str(g)) or {}).get("name") or _subname.get(g) or _subname.get(str(g)) or f"s{g}"

# (font clobbering removed: Stage 0 resolves the report font ONCE. This cell
#  re-picked a font from a fallback list, overriding it mid-notebook, so
#  figures before and after this point rendered in DIFFERENT typefaces.)

def _wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan, np.nan)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = (z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))) / denom
    return p, max(0, centre - half), min(1, centre + half)

print("=" * 64)
print("  STAGE 51  TEAM ARCHETYPE CLUSTERING")
print("=" * 64)

P = part.reset_index()
P["Team"] = P["Team"].astype(str).str.strip()
sub_cols = sorted([c for c in P.columns if c.startswith("sub_ment_")], key=lambda c: int(c.split("_")[-1]))
team_profile = P.groupby("Team")[sub_cols].mean()
print(f"  team profile matrix: {team_profile.shape[0]} teams x {team_profile.shape[1]} sub-themes")

X = StandardScaler().fit_transform(team_profile.values)

# data-driven K via silhouette (K=2..6)
best_k, best_s, best_labels = None, -1, None
sil_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=CFG.seed)
    labels = km.fit_predict(X)
    s = silhouette_score(X, labels)
    sil_scores[k] = s
    if s > best_s:
        best_k, best_s, best_labels = k, s, labels
print("  silhouette by K:", {k: round(v, 3) for k, v in sil_scores.items()})
print(f"  chosen K = {best_k}  (silhouette = {best_s:.3f})")

team_profile["archetype"] = best_labels
archetype_ids = sorted(team_profile["archetype"].unique())

global_mean = team_profile[sub_cols].mean()
global_std  = team_profile[sub_cols].std().replace(0, np.nan)
archetype_top = {}
for a in archetype_ids:
    grp_mean = team_profile.loc[team_profile.archetype == a, sub_cols].mean()
    z = (grp_mean - global_mean) / global_std
    top3 = z.sort_values(ascending=False).head(3)
    archetype_top[a] = [(int(c.split("_")[-1]), _sname(int(c.split("_")[-1])), float(z[c])) for c in top3.index]
    print(f"  archetype {a} (n={int((team_profile.archetype==a).sum())}): " +
          ", ".join(f"{n} (z={zz:+.2f})" for _, n, zz in archetype_top[a]))

meta["Team"] = meta["Team"].astype(str).str.strip()
special = np.asarray(award.get("special", np.zeros(len(meta), bool)), bool)
nominated = np.asarray(award.get("nominated", np.zeros(len(meta), bool)), bool)
prize_row = np.where(special, 2, np.where(nominated, 1, 0)).astype(float)
team_prize = pd.Series(prize_row, index=meta["Team"]).groupby(level=0).max()
# An unmatched team has an UNKNOWN award, not a lost one. fillna(0) would
# score it as 'no prize' and silently deflate every prize rate, so any
# unmatched team is reported and dropped instead. Today this is a no-op:
# all 125 teams match the registry - the guard exists so it cannot become
# a silent error on a future run.
_pr = team_prize.reindex(team_profile.index)
if _pr.isna().any():
    print(f"  !! {int(_pr.isna().sum())} team(s) have no award-registry "
          f"match; they are EXCLUDED from the prize comparison, not "
          f"counted as no-prize: {list(_pr.index[_pr.isna()])[:8]}")
team_won = (_pr.fillna(0) > 0).astype(int)

statsA = []
for a in archetype_ids:
    m = (team_profile.archetype == a).values
    k, n = int(team_won[m].sum()), int(m.sum())
    p, lo, hi = _wilson(k, n)
    label = f"A{a}: " + "/".join(n_ for _, n_, _ in archetype_top[a][:2])
    statsA.append(dict(label=label, k=k, n=n, p=p, lo=lo, hi=hi))
print("  prize rate by archetype:")
for s in statsA:
    print(f"    {s['label']:<45} {s['k']}/{s['n']} = {s['p']:.0%}  (95% CI {s['lo']:.0%}-{s['hi']:.0%})")

psave("pooled_team_archetype", dict(team_profile=team_profile, archetype_top=archetype_top,
                                     silhouette=sil_scores, best_k=best_k, prize=statsA))

pca = PCA(n_components=2, random_state=CFG.seed)
xy = pca.fit_transform(X)
evr = pca.explained_variance_ratio_

fig = plt.figure(figsize=(11.5, 5.8), facecolor="white")
gs = fig.add_gridspec(2, 2, height_ratios=[5.2, 0.35], width_ratios=[1.15, 1], hspace=0.55, wspace=0.32)
ax1 = fig.add_subplot(gs[0, 0]); ax2 = fig.add_subplot(gs[0, 1]); ax_ref = fig.add_subplot(gs[1, :]); ax_ref.axis("off")

cmap_a = plt.get_cmap("viridis")
colors_a = {a: cmap_a(i / max(1, len(archetype_ids) - 1)) for i, a in enumerate(archetype_ids)}
for a in archetype_ids:
    m = team_profile.archetype.values == a
    lbl = f"A{a}  ({m.sum()} teams)"
    ax1.scatter(xy[m, 0], xy[m, 1], s=42, color=colors_a[a], edgecolor="white", linewidth=0.6, label=lbl, zorder=3)
ax1.axhline(0, color="#DDD", lw=0.8, zorder=1); ax1.axvline(0, color="#DDD", lw=0.8, zorder=1)
ax1.set_xlabel(f"PC1  ({evr[0]:.0%} variance)", fontsize=9.5)
ax1.set_ylabel(f"PC2  ({evr[1]:.0%} variance)", fontsize=9.5)
ax1.set_title("Panel A  ·  teams in theme-profile space, coloured by archetype", fontsize=12, fontweight="bold", loc="left", pad=10)
ax1.legend(frameon=False, fontsize=8.5, loc="best")
for sp in ("top", "right"): ax1.spines[sp].set_visible(False)

xs = np.arange(len(statsA))
for x, s, a in zip(xs, statsA, archetype_ids):
    # A Wilson interval can round to a hair outside the point estimate, which
    # makes one of these lengths -1e-17 and matplotlib refuses the whole figure.
    # Clamping at zero changes no drawn pixel and keeps the bar from vanishing.
    _lo_len = max(0.0, float(s["p"]) - float(s["lo"]))
    _hi_len = max(0.0, float(s["hi"]) - float(s["p"]))
    ax2.bar(x, s["p"], yerr=[[_lo_len], [_hi_len]], color=colors_a[a], width=0.55,
            capsize=5, edgecolor="white")
    ax2.text(x, s["hi"] + 0.05, f"{s['p']:.0%}\n(n={s['n']})", ha="center", va="bottom", fontsize=8.5, fontweight="bold", linespacing=1.15)
ax2.set_xticks(xs); ax2.set_xticklabels([f"A{a}" for a in archetype_ids], fontsize=10)
ax2.set_ylim(0, 1.35); ax2.set_ylabel("prize rate", fontsize=9.5)
ax2.set_title("Panel B  ·  prize rate by archetype  (95% Wilson CI)", fontsize=12, fontweight="bold", loc="left", pad=10)
for sp in ("top", "right"): ax2.spines[sp].set_visible(False)

ax_ref.text(0.5, 0.5, f"K={best_k} chosen by max silhouette score ({best_s:.3f}) over K=2..6  ·  features = team-mean sub-theme mention rate, standardised  ·  "
            "KMeans (Lloyd 1982); PCA shown for display only (clustering done in full feature space).",
            ha="center", va="center", fontsize=8, style="italic", color="#888", transform=ax_ref.transAxes)

fig.suptitle("Team archetypes: discourse-profile clusters and their prize outcomes", x=0.01, ha="left",
              fontsize=16, fontweight="bold", color=NAT_TEXT, y=0.985)
fig.subplots_adjust(top=0.87, bottom=0.06, left=0.06, right=0.98)
save_fig(fig, "chart_pooled_team_archetype"); plt.close(fig)
print("  Saved: chart_pooled_team_archetype.png")
print("=" * 64)

---

## Section 9 - Methods, limitations and export

The methods paragraph and the limitations list are written from the artifacts themselves, so they cannot drift from what the notebook actually did. Then the figure inventory and the export bundle.

### Stage 44 · Methods and limitations

**How to read it.** Written from the artifacts at run time, so the numbers in the paragraph are the numbers in the figures. The limitations list is part of the output, not an afterthought.

In [ ]:
# @title
# ==============================================================================
# STAGE 44  -  METHODS, LIMITATIONS AND FIGURE MAP
#             section 9 of 9  -  Methods, inventory and export
# ==============================================================================
# WHY IT IS GENERATED RATHER THAN TYPED
#   Every stale number in this project came from a figure or a paragraph that had
#   been written by hand and then not updated when the analysis moved underneath
#   it: "Sub (43 themes)" on a 54-theme run, "all five questions" on a four-
#   question pool, a naming verdict from a threshold the code no longer used. A
#   methods section typed once has exactly the same failure mode, and it is the
#   part a reviewer reads most closely.
#
#   So this cell writes it from the artifacts. Every count, K, rate and p-value
#   below is read at run time. If an artifact is missing the sentence says so
#   instead of quietly leaving a number out.
#
# Outputs: methods_and_limitations.md, figure_map.csv, and the same text printed
#          here so it can be pasted straight into the write-up.
# ==============================================================================
import os, glob
import numpy as np, pandas as pd

_PL = globals().get("_orig_pload", pload)
def _get(n):
    try:
        return _PL(n)
    except Exception:
        return None

M    = globals().get("MASTER") if globals().get("MASTER") is not None else _get("pooled_master")
KSEL = _get("pooled_k_selection") or {}
AUD  = _get("pooled_naming_audit")
LVR  = _get("pooled_label_verification")
EV   = _get("pooled_theme_evidence")
CBA  = _get("pooled_codebook_alignment")
LEX  = _get("pooled_oneword_lexicon")
BRG  = _get("pooled_bridge")
if M is None:
    raise RuntimeError("pooled_master missing - nothing can be described without it.")

# ---------------------------------------------------------------------------
# The sentiment sentence is DERIVED, because it was wrong for a whole run.
# The list said "Sentiment is lexicon-based and English-only" while cell 19, in
# the same notebook, printed "instrument: LLM categorical verdict", 98% coverage
# and non-Latin responses scored at 100%. A typed sentence cannot follow the
# pipeline; a generated one cannot drift from it.
MV = _get("pooled_measurement_validity") or {}
DET = _get("pooled_sentiment_detail")


def _sentiment_limitation():
    inst = str(MV.get("instrument", "")).lower()
    n_tot = int(MV.get("n_total", len(M)))
    n_sc = int(MV.get("n_scored", 0))
    if "llm" in inst:
        _rng = float(MV.get("coverage_range_pts", float("nan")))
        _len = (f" Coverage varies by only {_rng:.0f} percentage points across "
                f"response-length bands, so measurability is not length-dependent."
                if _rng == _rng else "")
        return (f"Sentiment is read per response by an instruction-following model "
                f"(NB2 Stage 26B), not matched against a lexicon: it is scored from "
                f"the meaning of the sentence and is multilingual. "
                f"{n_sc} of {n_tot} responses carry a score; a verdict is withheld "
                f"when the model returns nothing parsable or when the span it quoted "
                f"as its evidence is not present in the response, and those are "
                f"treated as unmeasured rather than neutral.{_len}")
    if inst:
        return (f"Sentiment was produced by an instrument recorded as '{inst}'; "
                f"{n_sc} of {n_tot} responses carry a score, and the remainder are "
                f"treated as unmeasured rather than neutral.")
    return ("The sentiment instrument could not be identified from the artifacts "
            "(pooled_measurement_validity is missing) - run the measurement-validity "
            "cell before quoting anything about sentiment.")


L, miss = [], []
def w(s=""): L.append(s)
def need(obj, what):
    if obj is None:
        miss.append(what)
        return False
    return True

K = {}
for lv, key in (("main", "pooled_labels_main7"),
                ("sub", "pooled_labels_sub")):
    a = _get(key)
    if a is not None:
        K[lv] = int(np.asarray(a).max()) + 1

QL = globals().get("QL_POOL") or sorted(M["question"].dropna().unique())
QALL = list(getattr(CFG, "short_labels", QL)) if "CFG" in globals() else list(QL)

# ------------------------------------------------------------------ corpus ----
w("## Corpus and unit of analysis")
w()
w(f"The thematic analysis is fitted on **{len(M)} responses** from "
  f"**{M['Participant'].nunique()} participants** in **{M['Team'].nunique()} teams**, "
  f"covering **{len(QL)} of the {len(QALL)} survey questions** "
  f"({', '.join(QL)}). The unit of analysis is the individual response, not the "
  f"participant: a participant may contribute one response per question, so "
  f"participant- and team-level results are reported with their own denominators "
  f"and never inferred from response counts.")
w()
# Nothing is held out: all five questions are pooled. What DOES need stating is
# the share of the corpus the clustering placed in no theme.
if K.get("sub") and N_ALL:
    _nz = NOISE.get(CLAIM_LEVEL, 0)
    w(f"Density-based clustering does not assign every response. **{_nz} of "
      f"{N_ALL} responses ({_nz / N_ALL:.0%}) were left in no theme** at the "
      f"{CLAIM_LEVEL} level and **{NOISE.get('main', 0)}** at the main level. "
      f"They were not discarded and not forced into a theme: they are reported "
      f"as their own category in descriptive figures, are excluded from every "
      f"significance test, and remain in the denominator of every prevalence, "
      f"so the reported rates sum to less than 100% and the remainder is exactly "
      f"this unplaced share.")
w()
# --------------------------------------------------------------- hierarchy ----
w("## Theme structure")
w()
if K:
    # DERIVED, not typed. The previous sentence said "One Ward tree ... three-level
    # hierarchy", which was wrong twice: SUB comes from HDBSCAN (Ward is used only
    # to build MAIN from the sub centroids), and there are two levels, not three.
    _HH = _get("pooled_hdbscan_hierarchy") or {}
    _lv = list(globals().get("LEVELS", ["main", "sub"]))
    _nlv = {1: "single-level", 2: "two-level", 3: "three-level"}.get(len(_lv), f"{len(_lv)}-level")
    _mm = str(_HH.get("main_method", ""))
    _crit = str(_HH.get("main_criterion", ""))
    _ssrc = str(_HH.get("sub_source", ""))
    if _mm.startswith("ward"):
        _how = (f"Sub-themes are the analytical level and come from HDBSCAN on the "
                f"UMAP embedding, with the configuration chosen at a local maximum "
                f"of the DBCV density-validity index"
                + (f" ({_ssrc})" if _ssrc else "") + ". "
                f"Main themes are then built by size-weighted Ward agglomeration of "
                f"the sub-theme centroids"
                + (f", cut at the {_crit}" if _crit else "") + ". Because Ward merges "
                f"whole sub-themes, every sub-theme lies inside exactly one main "
                f"theme by construction, and a response the clustering left unplaced "
                f"stays unplaced at both levels.")
    else:
        _how = (f"The hierarchy was produced by the procedure recorded in "
                f"pooled_hdbscan_hierarchy"
                + (f" (main: {_mm})" if _mm else "") + ".")
    w(f"The analysis uses a **{_nlv}** hierarchy of **{K.get('main','?')} main and "
      f"{K.get('sub','?')} sub-themes**. {_how}")
    _st = (_HH.get("stability") or {})
    _si = (_HH.get("silhouette") or {})
    if _st:
        w()
        w(f"- bootstrap stability (adjusted Rand over 80% subsamples): "
          + ", ".join(f"**{lv}** {_st[lv]:.3f}" for lv in _lv if lv in _st)
          + (". Cluster-wise Jaccard, which is what decides whether an individual "
             "theme may carry a claim, is reported per theme in Notebook 2 Stage 25."
             ))
    if _si:
        w(f"- silhouette on themed responses: "
          + ", ".join(f"**{lv}** {_si[lv]:+.3f}" for lv in _lv if lv in _si) + ".")
if KSEL:
    _j = dict(zip(KSEL.get("K", []), KSEL.get("jaccard", [])))
    _s = dict(zip(KSEL.get("K", []), KSEL.get("sil", [])))
    _m = dict(zip(KSEL.get("K", []), KSEL.get("min_size", [])))
    for lv in LEVELS:
        k = K.get(lv)
        if k in _j:
            w(f"- **{lv} (K={k})**: bootstrap Jaccard {_j[k]:.3f}, silhouette "
              f"{_s.get(k, float('nan')):+.3f}, smallest cluster {int(_m.get(k, 0))} "
              f"responses, median {len(M)//k}.")
    w()
    w(f"K was chosen inside a stated researcher window at each level. At the sub "
      f"level the pick is **not** made on bootstrap Jaccard: across that window the "
      f"minimum cluster size is constant and the silhouette spread is negligible, "
      f"while Jaccard rises with K because resampling reproduces very small isolated "
      f"clusters almost every time. Selecting on it would therefore have returned "
      f"the most fragmented partition rather than the most stable one. Among the K "
      f"clearing the stability and silhouette floors, the one leaving the most "
      f"evidence per theme was taken, since a theme's mentioner count cannot exceed "
      f"its response count.")
w()

# ------------------------------------------------------------------ naming ----
w("## Theme naming and its verification")
w()
if need(AUD, "pooled_naming_audit") and isinstance(AUD, pd.DataFrame):
    for lv in LEVELS:
        s = AUD[AUD.level == lv]
        if len(s):
            w(f"- **{lv}**: {int((s.status == 'accepted').sum())} of {len(s)} labels "
              f"accepted by the naming engine.")
    _sg = pd.to_numeric(AUD.get("summary_grounding"), errors="coerce").dropna() \
        if "summary_grounding" in AUD.columns else pd.Series(dtype=float)
    if len(_sg):
        _held = int(AUD.get("flags", pd.Series(dtype=str)).fillna("")
                    .str.contains("summary withheld").sum())
        w(f"- Theme **summaries** are held to the same evidence standard as the "
          f"labels (median grounding {_sg.median():.2f}); **{_held}** were withheld "
          f"and replaced by the cluster's own terms rather than published.")
    w()
    w("A label is accepted only if the model declined to abstain, cited responses "
      "that exist, used wording grounded in those responses, survived a judge shown "
      "members it had not been shown, and sat closer to its own cluster than to "
      "any other when embedded. Labels that fail are withheld, never invented.")
    w()
_V = LVR.get("table") if isinstance(LVR, dict) else LVR
if isinstance(_V, pd.DataFrame) and len(_V):
    w(f"Independently, an optimal (Hungarian) assignment between the labels and the "
      f"cluster centroids keeps **{int(_V.verified.sum())} of {len(_V)}** labels on "
      f"their own theme.")
    for lv in LEVELS:
        s = _V[_V.level == lv]
        if len(s):
            Kx = int(s.of_K.iloc[0])
            w(f"  - {lv}: label→theme rank-1 for "
              f"{int((s.rank_label_to_theme == 1).sum())}/{len(s)} "
              f"(chance {100.0/Kx:.1f}%).")
    w()

# ------------------------------------------------------------------ tiers -----
if need(EV, "pooled_theme_evidence") and isinstance(EV, pd.DataFrame):
    w("## What may be claimed, and what may only be described")
    w()
    for lv in LEVELS:
        s = EV[EV.level == lv]
        if len(s):
            w(f"- **{lv}**: {int((s.tier=='CLAIMABLE').sum())} themes carry enough "
              f"evidence for an inferential claim, "
              f"{int((s.tier=='DESCRIPTIVE').sum())} are described but not tested, "
              f"{int((s.tier=='ARTIFACT').sum())} had no label accepted.")
    _ib = int(EV.instrument_bound.sum()) if "instrument_bound" in EV.columns else 0
    if _ib:
        w(f"- **{_ib} theme(s)** draw most of their responses from a single question "
          f"and may describe the instrument rather than a cross-cutting experience; "
          f"they are named as such.")
    w()
    w("Support gates are **reported, not applied as a veto**: every theme appears in "
      "every figure, drawn faded when it was not tested, with the reason recorded in "
      "`claims_coverage.csv`. A theme that was never tested must not be read as a "
      "null result.")
    w()

# ------------------------------------------------------------- convergence ----
if need(CBA, "pooled_codebook_alignment") and isinstance(CBA.get("summary"), pd.DataFrame):
    S = CBA["summary"]
    w("## Convergence with an independent manual coding")
    w()
    for lv in S.level.unique():
        s = S[S.level == lv]
        w(f"- Against the **{len(s)}** hand-coded themes, at the {lv} level: "
          f"{int((s.verdict=='ONE-TO-ONE').sum())} recovered as a single cluster, "
          f"{int((s.verdict=='SPLIT').sum())} split across several, "
          f"{int((s.verdict=='ABSENT').sum())} not recovered.")
    w()
    w("**This is a conceptual alignment, not inter-rater agreement.** Each hand theme "
      "was embedded from its own label, definition and example quotes and compared to "
      "the clusters' centroids; no response was coded twice, so no kappa is reported. "
      "The two analyses also cover different rows: the manual coding used the whole "
      "instrument, this pipeline the narrative answers above the length threshold.")
    w()

# ----------------------------------------------------------------- lexicon ----
# PROVENANCE GATE.  LEX / BRG are artifacts of the one-word lexicon and one-word
# bridge cells, both of which were removed from this notebook. Nothing rebuilds
# them, so anything still on disk under those names belongs to an earlier
# pipeline - and on the last run cell 88 read exactly that and published
# "282 answers, 219 distinct forms, grouped into 8 lexical groups" as if the
# analysis had been performed. INFRA-1 sets orphans aside, so these are normally
# None; the explicit check below means that even if a copy reappears from a
# backup, the section states its provenance instead of quoting it.
_LEX_OK = LEX is not None and any(
    "oneword" in str(k).lower() or "lexicon" in str(k).lower()
    for k in (globals().get("NB3_PRODUCES") or []))
if _LEX_OK:
    w("## The one-word item")
    w()
    w(f"The one-word answers are analysed as a **lexicon**, not as a theme level: "
      f"{LEX.get('n_answers','?')} answers, {len(LEX['types'])} distinct forms, "
      f"grouped into {LEX.get('k','?')} lexical groups named from their own most "
      f"frequent words. A single word carries no clause to interpret and its "
      f"embedding is dominated by length, so forcing it into the theme hierarchy "
      f"would manufacture structure. No result from this section is reported as a "
      f"theme.")
    w()
if False and need(BRG, "pooled_bridge"):   # producer removed - see the note above
    _p = BRG.get("p_value", 1.0)
    w(f"Whether the two agree was tested **within person**: for the "
      f"{BRG.get('n_pairs','?')} participants who answered both, the one-word answer "
      f"landed on a theme that participant had themselves voiced "
      f"{BRG.get('observed', float('nan')):.1%} of the time against a permutation "
      f"null of {BRG.get('null_mean', float('nan')):.1%} (p = {_p:.4f}), "
      + ("**a real association**." if _p < .05 else
         "**which is not evidence of an association**; it is reported as a null.")
      + " Absolute similarities are not interpretable here and are not quoted.")
    w()

# ------------------------------------------------------------- limitations ----
w("## Limitations")
w()
if (LEX is not None) or (BRG is not None):
    w("- An artifact from the removed one-word lexicon / bridge stages was found on "
      "disk. It is **not** described above: no cell in this notebook computes it, so "
      "its numbers belong to an earlier pipeline. It has been set aside as `.stale`.")
    w()
lim = [
    f"The design is cross-sectional and self-selected; nothing here supports a causal "
    f"reading.",
    f"Themes were induced from the same text that the sentiment score was computed on, "
    f"so theme and sentiment share a source. This is a confound, not circularity: the "
    f"sentiment instrument never saw the theme labels.",
    _sentiment_limitation(),
    f"Theme membership is exhaustive only for the responses the clustering could "
    f"place; the rest are described as unplaced and enter no test.",
    f"Multiple testing is corrected within each family separately (level x unit x "
    f"factor); comparisons across families are descriptive.",
]
if K.get("sub"):
    # N1. This used to print len(M)//K['sub'] and call it the median: the whole
    # corpus - including the 403 responses no theme contains - divided by K. On
    # this run that produced "38" against a true distribution of 10 / 20 / 115.
    # A single number cannot describe that spread, so the range is quoted too.
    _lab_sub = np.asarray(_get("pooled_labels_sub"))
    _sz = np.bincount(_lab_sub[_lab_sub >= 0], minlength=K["sub"]) if _lab_sub is not None else None
    if _sz is not None and len(_sz):
        lim.append(f"At the sub level theme sizes run from {int(_sz.min())} to "
                   f"{int(_sz.max())} responses (median {int(np.median(_sz))}), so the "
                   f"smaller sub-themes are under-powered for group comparisons. They "
                   f"are shown and described, not tested.")
    else:
        lim.append("Sub-theme sizes could not be read from the label array, so no "
                   "statement is made about their spread.")
# The unplaced share is the limitation that replaces it, and it is real.
if NOISE.get(CLAIM_LEVEL, 0):
    _nz = NOISE[CLAIM_LEVEL]
    lim.append(f"{_nz} of {N_ALL} responses ({_nz / N_ALL:.0%}) were placed in "
               f"no theme by the clustering. They are described but carry no "
               f"claim, so every thematic finding here speaks for the "
               f"{N_ALL - _nz} responses that could be placed, not for the "
               f"whole corpus.")
for x in lim:
    w(f"- {x}")
w()

if miss:
    w("## Not described here")
    w()
    w("The following artifacts were not present when this section was generated, so "
      "the corresponding paragraphs are absent rather than approximated: "
      + ", ".join(f"`{m}`" for m in miss) + ".")
    w()

TXT = "\n".join(L)
open(f"{POOLED_DIR}/methods_and_limitations.md", "w", encoding="utf-8").write(TXT)

# -------------------------------------------------------------- figure map ----
rows = []
for p in sorted(glob.glob(f"{POOLED_DIR}/*.png")):
    b = os.path.basename(p)
    rows.append(dict(figure=b, kb=round(os.path.getsize(p) / 1024, 1)))
FM = pd.DataFrame(rows)
FM.to_csv(f"{POOLED_DIR}/figure_map.csv", index=False)

print("=" * 78)
print("  STAGE 70   METHODS, LIMITATIONS AND FIGURE MAP")
print("=" * 78)
print(TXT)
print("-" * 78)
print(f"  -> methods_and_limitations.md ({len(TXT.split())} words)"
      f"  ·  figure_map.csv ({len(FM)} figures)")
if miss:
    print(f"  missing artifacts, reported rather than glossed over: {', '.join(miss)}")
print("  Every number above was read from an artifact at run time. Re-run this cell")
print("  after any change and the prose follows the analysis instead of drifting.")
print("=" * 78)

### Stage 45 · Figure inventory

**How to read it.** Every figure and table this notebook produced, with the artifact behind it.

In [ ]:
# @title
# ==============================================================================
# STAGE 45  -  FIGURE INVENTORY
#             section 9 of 9  -  Methods, inventory and export
# ==============================================================================
# ==============================================================================
# NOTEBOOK 3  -  FIGURE INVENTORY  (visual-only; produces no new artifacts)
# Lists the figure files in the shared store so you can confirm what was written.
# Re-running any explorer cell above just overwrites its own figure from cache.
# ==============================================================================
import os, glob

# The preflight rebinds pload() to follow the EXPLORER's detail level. The
# statistical stages must always read the true artifacts, so they use the
# unpatched loader like every other analysis cell in this notebook.
_PL = globals().get("_orig_pload", pload)
figs = []
for ext in ("*.png", "*.svg", "*.html"):
    figs += glob.glob(os.path.join(POOLED_DIR, ext))
figs = sorted(set(figs))
print("=" * 64)
print(f"  NOTEBOOK 3 COMPLETE  -  {len(figs)} figure files in")
print(f"  {POOLED_DIR}")
if MIRROR_POOLED_DIR:
    print(f"  (mirrored to {MIRROR_POOLED_DIR}; files over the size cap stay local)")
print("=" * 64)
for f in figs[-30:]:
    print("   ", os.path.basename(f))

# ---- reproducibility manifest ------------------------------------------------
# Records the seed, cache version, embedding model and config fingerprint behind
# this run, so a reviewer can tell which configuration produced these figures.
try:
    _n = len(_PL("pooled_meta")) if _PL("pooled_meta") is not None else 0
    write_manifest("nb3_manifest.json",
                   fingerprints={k: CFG.fingerprint(k, _n) for k in
                                 ("embed", "umap", "ward", "subtheme")},
                   n_rows=_n)
    print("  wrote nb3_manifest.json (seed, cache_version, embed model, fingerprints)")
except Exception as _e:
    print(f"  (manifest not written: {type(_e).__name__}: {str(_e)[:70]})")


### Stage 46 · Export

**How to read it.** Bundles the figures, tables and artifacts for the submission. Run last.

In [ ]:
# @title
# ==============================================================================
# STAGE 46  -  EXPORT AND CLEANUP
#             section 9 of 9  -  Methods, inventory and export
# INPUT:  All charts, checkpoints, and data files in CHECKPOINT_DIR
# OUTPUT: iGEM_Pipeline_Outputs.zip  (in Drive AND offered for local download)
# ==============================================================================
# Bundles every output the pipeline produced into a single zip,
# organised into 4 folders:
#   charts/        all chart_*.png  (static figures)
#   interactive/   all chart_*.html  (Plotly)
#   checkpoints/   all *.pkl  (data - embeddings, UMAP, clusters, sentiment)
#   data/          all *.csv  (exported tables, etc.)
# Plus a MANIFEST.txt at the root listing every file and its size.
#
# Encoder model weights (parametric_umap_encoders/) are NOT bundled
# because they are large and only needed in Drive for projecting
# new survey waves - they stay on Drive.
#
# BUG FIX vs. previous version: the old cell used glob.glob('chart_*.png')
# which searched the working directory (/content/) and missed every
# PNG saved to CHECKPOINT_DIR.  Now all globs use CHECKPOINT_DIR.

import glob
import os
import shutil
import zipfile

try:
    from google.colab import files as _colab_files
    _IS_COLAB = True
except ImportError:
    _IS_COLAB = False

# ── What to archive ───────────────────────────────────────
POOLED_DIR = globals().get('POOLED_DIR', f'{CHECKPOINT_DIR}/pooled_combined')
PATTERNS = {
    'charts':      [f'{CHECKPOINT_DIR}/chart_*.png', f'{CHECKPOINT_DIR}/chart_*.svg'],
    'interactive': [f'{CHECKPOINT_DIR}/chart_*.html'],
    'checkpoints': [f'{CHECKPOINT_DIR}/*.pkl'],
    'data':        [f'{CHECKPOINT_DIR}/*.csv', f'{CHECKPOINT_DIR}/*.json'],
    'pooled':      [f'{POOLED_DIR}/*.png', f'{POOLED_DIR}/*.svg', f'{POOLED_DIR}/*.html', f'{POOLED_DIR}/*.json', f'{POOLED_DIR}/*.csv', f'{POOLED_DIR}/*.pkl'],
}

zip_filename   = 'iGEM_Pipeline_Outputs.zip'
zip_path_drive = f'{CHECKPOINT_DIR}/{zip_filename}'
zip_path_local = zip_filename     # CWD copy for Colab download

# Remove any stale zip from a previous run BEFORE globbing
for old_zip in (zip_path_drive, zip_path_local):
    if os.path.exists(old_zip):
        os.remove(old_zip)

# ── Collect files (organised by category) ─────────────────
to_archive   = {}        # arcname -> real path
totals       = {}
manifest     = []

for category, globs in PATTERNS.items():
    files = []
    for g in globs:
        files.extend(sorted(glob.glob(g)))
    files = sorted(set(files))
    # Belt-and-braces: never include the zip itself if it slipped through
    files = [f for f in files if os.path.basename(f) != zip_filename]
    totals[category] = len(files)
    for f in files:
        arcname = f'{category}/{os.path.basename(f)}'
        to_archive[arcname] = f

if not to_archive:
    raise RuntimeError(
        f'No files found in {CHECKPOINT_DIR}. '
        'Run the pipeline stages first to produce charts and checkpoints.')

# ── Build manifest ────────────────────────────────────────
manifest.append('iGEM Pipeline Outputs - Manifest')
manifest.append('=' * 50)
manifest.append('')
manifest.append(f'Total files archived: {len(to_archive)}')
for cat, n in totals.items():
    manifest.append(f'  {cat:<14} {n}')
manifest.append('')
manifest.append('Files (by category):')
manifest.append('')

for category in PATTERNS.keys():
    cat_arcs = sorted(a for a in to_archive if a.startswith(category + '/'))
    if not cat_arcs:
        continue
    manifest.append(f'[{category}]')
    for arcname in cat_arcs:
        size_kb = os.path.getsize(to_archive[arcname]) / 1024
        manifest.append(f'  {arcname:<60} {size_kb:>10.1f} KB')
    manifest.append('')

# Note encoders (kept on Drive, not bundled)
encoder_dir = f'{CHECKPOINT_DIR}/parametric_umap_encoders'
if os.path.isdir(encoder_dir):
    enc_count = sum(len(fs) for _, _, fs in os.walk(encoder_dir))
    manifest.append('[encoder weights - NOT bundled, kept on Drive]')
    manifest.append(f'  parametric_umap_encoders/ ({enc_count} files)')
    manifest.append('  -> Use these on Drive to project new survey waves '
                    'without retraining.')

manifest_str = '\n'.join(manifest)

# ── Write zip (Drive copy first) ──────────────────────────
print('=' * 60)
print('  PACKAGING PIPELINE OUTPUTS')
print('=' * 60)

with zipfile.ZipFile(zip_path_drive, 'w', zipfile.ZIP_DEFLATED, allowZip64=True) as zf:
    for arcname, fpath in to_archive.items():
        zf.write(fpath, arcname)
    zf.writestr('MANIFEST.txt', manifest_str)

drive_size_mb = os.path.getsize(zip_path_drive) / (1024 * 1024)

# ── Print summary ─────────────────────────────────────────
print(manifest_str)
print()
print('=' * 60)
print(f'  OK Zip saved to Drive: {zip_path_drive}')
print(f'    Size: {drive_size_mb:.2f} MB · {len(to_archive)} files + MANIFEST.txt')
print('=' * 60)

# ── Local download (Colab only) ───────────────────────────
if _IS_COLAB:
    # Copy to CWD so files.download() works
    shutil.copy2(zip_path_drive, zip_path_local)
    try:
        _colab_files.download(zip_path_local)
        print(f'  OK Browser download started: {zip_filename}')
    except Exception as e:
        print(f'  WARNING Browser download failed ({e}).')
        print(f'    Grab the zip from Drive: {zip_path_drive}')
else:
    print(f'  Non-Colab environment - zip available at: {os.path.abspath(zip_path_drive)}')